AIGP - 165

# Graph Attention Networks


orginal code and inspiration - [Graph Attention Networks: Self-Attention for GNNs
Graph Neural Network Course: Chapter 2](https://mlabonne.github.io/blog/posts/2022-03-09-Graph_Attention_Network.html)

In [ ]:
# We assume that PyTorch is already installed
import torch
torchversion = torch.__version__

# Install PyTorch Scatter, PyTorch Sparse, and PyTorch Geometric
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{torchversion}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-{torchversion}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

# Numpy for matrices
import numpy as np
np.random.seed(0)

# Visualization
import networkx as nx
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Dataset

In [ ]:
from torch_geometric.datasets import Planetoid

# Import dataset from PyTorch Geometric
dataset = Planetoid(root=".", name="CiteSeer")

data = dataset[0]

# Print information about the dataset
print(f'Dataset: {dataset}')
print('-------------------')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of nodes: {data.x.shape[0]}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

# Print information about the graph
print(f'\nGraph:')
print('------')
print(f'Edges are directed: {data.is_directed()}')
print(f'Graph has isolated nodes: {data.has_isolated_nodes()}')
print(f'Graph has loops: {data.has_self_loops()}')

In [ ]:
from torch_geometric.utils import remove_isolated_nodes

isolated = (remove_isolated_nodes(data['edge_index'])[2] == False).sum(dim=0).item()
print(f'Number of isolated nodes = {isolated}')

# Plot dataset

In [ ]:
from torch_geometric.utils import to_networkx

G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(18,18))
plt.axis('off')
nx.draw_networkx(G,
                pos=nx.spring_layout(G, seed=0),
                with_labels=False,
                node_size=50,
                node_color=data.y,
                width=2,
                edge_color="grey"
                )
plt.show()

# Plot node degrees

In [ ]:
from torch_geometric.utils import degree
from collections import Counter

# Get list of degrees for each node
degrees = degree(data.edge_index[0]).numpy()

# Count the number of nodes for each degree
numbers = Counter(degrees)

# Bar plot
fig, ax = plt.subplots(figsize=(18, 7))
ax.set_xlabel('Node degree')
ax.set_ylabel('Number of nodes')
plt.bar(numbers.keys(),
        numbers.values(),
        color='#0A047A')

# Implement GAT vs. GCN

In [ ]:
import torch.nn.functional as F
from torch.nn import Linear, Dropout
from torch_geometric.nn import GCNConv, GATv2Conv


class GCN(torch.nn.Module):
  """Graph Convolutional Network"""
  def __init__(self, dim_in, dim_h, dim_out):
    super().__init__()
    self.gcn1 = GCNConv(dim_in, dim_h)
    self.gcn2 = GCNConv(dim_h, dim_out)
    self.optimizer = torch.optim.Adam(self.parameters(),
                                      lr=0.01,
                                      weight_decay=5e-4)

  def forward(self, x, edge_index):
    h = F.dropout(x, p=0.5, training=self.training)
    h = self.gcn1(h, edge_index)
    h = torch.relu(h)
    h = F.dropout(h, p=0.5, training=self.training)
    h = self.gcn2(h, edge_index)
    return h, F.log_softmax(h, dim=1)


class GAT(torch.nn.Module):
  """Graph Attention Network"""
  def __init__(self, dim_in, dim_h, dim_out, heads=8):
    super().__init__()
    self.gat1 = GATv2Conv(dim_in, dim_h, heads=heads)
    self.gat2 = GATv2Conv(dim_h*heads, dim_out, heads=1)
    self.optimizer = torch.optim.Adam(self.parameters(),
                                      lr=0.005,
                                      weight_decay=5e-4)

  def forward(self, x, edge_index):
    h = F.dropout(x, p=0.6, training=self.training)
    h = self.gat1(x, edge_index)
    h = F.elu(h)
    h = F.dropout(h, p=0.6, training=self.training)
    h = self.gat2(h, edge_index)
    return h, F.log_softmax(h, dim=1)

def accuracy(pred_y, y):
    """Calculate accuracy."""
    return ((pred_y == y).sum() / len(y)).item()

def train(model, data):
    """Train a GNN model and return the trained model."""
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = model.optimizer
    epochs = 200

    model.train()
    for epoch in range(epochs+1):
        # Training
        optimizer.zero_grad()
        _, out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        acc = accuracy(out[data.train_mask].argmax(dim=1), data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        # Validation
        val_loss = criterion(out[data.val_mask], data.y[data.val_mask])
        val_acc = accuracy(out[data.val_mask].argmax(dim=1), data.y[data.val_mask])

        # Print metrics every 10 epochs
        if(epoch % 10 == 0):
            print(f'Epoch {epoch:>3} | Train Loss: {loss:.3f} | Train Acc: '
                  f'{acc*100:>6.2f}% | Val Loss: {val_loss:.2f} | '
                  f'Val Acc: {val_acc*100:.2f}%')

    return model

def test(model, data):
    """Evaluate the model on test set and print the accuracy score."""
    model.eval()
    _, out = model(data.x, data.edge_index)
    acc = accuracy(out.argmax(dim=1)[data.test_mask], data.y[data.test_mask])
    return acc

# Train GCN

In [ ]:
%%time

# Create GCN model
gcn = GCN(dataset.num_features, 16, dataset.num_classes)
print(gcn)

# Train
train(gcn, data)

# Test
acc = test(gcn, data)
print(f'\nGCN test accuracy: {acc*100:.2f}%\n')

# Train GAT

In [ ]:
%%time

# Create GAT model
gat = GAT(dataset.num_features, 8, dataset.num_classes)
print(gat)

# Train
train(gat, data)

# Test
acc = test(gat, data)
print(f'\nGAT test accuracy: {acc*100:.2f}%\n')

# t-SNE plots

In [ ]:
# Initialize new untrained model
untrained_gat = GAT(dataset.num_features, 8, dataset.num_classes)

# Get embeddings
h, _ = untrained_gat(data.x, data.edge_index)

# Train TSNE
tsne = TSNE(n_components=2, learning_rate='auto',
         init='pca').fit_transform(h.detach())

# Plot TSNE
plt.figure(figsize=(10, 10))
plt.axis('off')
plt.scatter(tsne[:, 0], tsne[:, 1], s=50, c=data.y)
plt.show()

In [ ]:
# Get embeddings
h, _ = gat(data.x, data.edge_index)

# Train TSNE
tsne = TSNE(n_components=2, learning_rate='auto',
         init='pca').fit_transform(h.detach())

# Plot TSNE
plt.figure(figsize=(10, 10))
plt.axis('off')
plt.scatter(tsne[:, 0], tsne[:, 1], s=50, c=data.y)
plt.show()

# Plot accuracy for each node degree

In [ ]:
from torch_geometric.utils import degree

# Get model's classifications
_, out = gat(data.x, data.edge_index)

# Calculate the degree of each node
degrees = degree(data.edge_index[0]).numpy()

# Store accuracy scores and sample sizes
accuracies = []
sizes = []

# Accuracy for degrees between 0 and 5
for i in range(0, 6):
  mask = np.where(degrees == i)[0]
  accuracies.append(accuracy(out.argmax(dim=1)[mask], data.y[mask]))
  sizes.append(len(mask))

# Accuracy for degrees > 5
mask = np.where(degrees > 5)[0]
accuracies.append(accuracy(out.argmax(dim=1)[mask], data.y[mask]))
sizes.append(len(mask))

# Bar plot
fig, ax = plt.subplots(figsize=(18, 9))
ax.set_xlabel('Node degree')
ax.set_ylabel('Accuracy score')
plt.bar(['0','1','2','3','4','5','>5'],
        accuracies,
        color='#0A047A')
for i in range(0, 7):
    plt.text(i, accuracies[i], f'{accuracies[i]*100:.2f}%',
             ha='center', color='#0A047A')
for i in range(0, 7):
    plt.text(i, accuracies[i]//2, sizes[i],
             ha='center', color='white')

Graph Attention Networks (GAT) – Code Explained (Point Form)



0) Setup & Installs
	•	torchversion = torch.__version__ → grabs your PyTorch version to fetch matching PyG wheels.
	•	Install extras:
	•	torch-scatter, torch-sparse → required tensor ops for graphs.
	•	pytorch_geometric (PyG) → GNN layers, datasets, utils.
	•	Imports: numpy, networkx, sklearn.manifold.TSNE, matplotlib.pyplot for data/plots and t-SNE.

1) Load Dataset (Planetoid → CiteSeer)
	•	dataset = Planetoid(root=".", name="CiteSeer") → auto-downloads and caches graph.
	•	data = dataset[0] → a single citation graph (nodes=papers, edges=citations).
	•	data.x → node features (bag-of-words).
data.y → node labels (paper category).
data.edge_index → COO edge list (shape [2, num_edges]).
data.train_mask / val_mask / test_mask → boolean splits for semi-supervised training.
	•	Prints: graphs count, nodes, feature dim, class count, directed/self loops/isolated nodes.

2) Explore Graph Structure
	•	to_networkx + nx.draw_networkx → quick 2D visualization of the undirected version.
	•	Node degree distribution:
	•	degree(data.edge_index[0]) → degree per node.
	•	Counter + bar chart → shows many low-degree nodes (1–2 neighbors common).

3) Models Implemented

A) GCN (Graph Convolutional Network)
	•	Layers: GCNConv(dim_in → dim_h) → ReLU → GCNConv(dim_h → dim_out).
	•	Dropout 0.5 before each conv.
	•	Optimizer: Adam(lr=0.01, weight_decay=5e-4).
	•	Output:
	•	h (logits) and log_softmax(h, dim=1) for NLL training.

B) GAT (Graph Attention Network, v2)
	•	Layers:
	•	GATv2Conv(dim_in → dim_h, heads=8) → concatenates 8 head outputs (dim becomes dim_h * heads).
	•	GATv2Conv(dim_h*heads → dim_out, heads=1) → final logits.
	•	Activations & regularization: Dropout 0.6 → ELU → Dropout 0.6.
	•	Optimizer: Adam(lr=0.005, weight_decay=5e-4).
	•	Output: logits + log-softmax.

What GATv2 does (conceptually)
	•	Learns attention coefficients αᵢⱼ between neighbors (including self) via a small neural scoring function.
	•	Applies softmax over neighbors to normalize importance.
	•	Uses multi-head attention: parallel attention mechanisms whose outputs are concatenated (hidden) or averaged (final layer).

4) Training Utilities
	•	accuracy(pred_y, y) → simple accuracy helper.
	•	train(model, data):
	•	Loss: CrossEntropyLoss on data.train_mask.
	•	Backprop each epoch; also compute validation loss/acc on data.val_mask.
	•	Prints metrics every 10 epochs.
	•	test(model, data):
	•	Eval mode → forward once.
	•	Accuracy on data.test_mask.

5) Run Experiments
	•	GCN run: instantiate → train (200 epochs) → test → print test accuracy.
	•	GAT run: instantiate → train (200 epochs) → test → print test accuracy.
	•	Expectation: GAT often slightly better accuracy; costs more time per epoch.

6) Embedding Visualization (t-SNE)
	•	Build an untrained GAT → forward → get logits h as embeddings → t-SNE to 2D → scatter colored by true labels → looks random.
	•	Build a trained GAT → same pipeline → clusters align better with labels → shows learned separability.

7) Accuracy vs Node Degree
	•	Forward trained GAT → predictions.
	•	Compute degree again; slice nodes by degree buckets (0,1,2,3,4,5,>5).
	•	For each bucket: accuracy and sample size.
	•	Bar plot: low-degree nodes have lower accuracy (less information to aggregate).



Small Gotchas / Notes
	•	Stability: GAT uses higher dropout (0.6) to regularize attention heads.
	•	Learning rates: GAT often needs a slightly smaller LR than GCN (as shown).
	•	Runtime: Attention is heavier than simple graph conv; expect longer training.

Common Tweaks to Improve Results
	•	Early stopping with patience (monitor val_loss or val_acc).
	•	L2 reg / weight decay sweeps and dropout tuning.
	•	Increase hidden dim or heads (watch memory).
	•	Add feature normalization (e.g., row-norm x) or edge self-loops if not present.
	•	Try GATConv vs GATv2Conv; v2 usually better but benchmark both.
	•	Set random seeds for reproducibility (PyTorch, NumPy, Python).

Quick mental model
	•	GCN: averages neighbor features with degree-based normalization → then linear map.
	•	GAT: learns how much each neighbor matters (attention) → weighted sum → better when some neighbors are more informative than others.

# Modifications to the above code -

	•	Feature engineering: add log-degree (and optional PageRank) as node features
	•	Structural regularization: DropEdge (random edge dropout)
	•	Stronger blocks: GATv2 + LayerNorm + Residual (when dims match)
	•	Better training: early stopping + ReduceLROnPlateau scheduler + reproducible seeds
	•	Interpretability: extract attention weights (top attentive neighbors)
	•	Cheap boost: Label Propagation refinement on logits
	•	Diagnostics: accuracy by node degree buckets

⚙️ I. What’s Limiting Accuracy

	1.Sparse connectivity → many nodes have 0–2 neighbors → attention has little to aggregate.
	2.Static training masks (Planetoid split) → only ~120 labelled nodes train the entire network.
	3.Shallow message-passing horizon → 3 layers see ≤ 3-hop neighbors; longer dependencies lost.
	4.High-dimensional sparse bag-of-words features → difficult to learn robust embeddings.
	5.Over-smoothing / over-fitting beyond 600–800 epochs.

In [ ]:
# ============================================ #
# Graph Attention Networks +++  (Perf + Visuals)
# ============================================ #
# Includes:
#  - Annealed DropEdge regularization
#  - Residual MLP refinement head
#  - Label-consistency auxiliary loss
#  - Log-degree feature augmentation + scaling
#  - 3-layer GATv2 backbone w/ LayerNorm
#  - Long training (no early stop) + cosine LR decay + grad clipping
#  - Visuals: learning curves, t-SNE (animated), interactive 3D graph,
#             attention heatmap, accuracy-by-degree, confusion matrix
# ============================================ #

# Uncomment if running in Colab
# !pip install -q torch-geometric scikit-learn matplotlib plotly networkx

import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import degree, add_self_loops, to_undirected
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn.models.label_prop import LabelPropagation
import numpy as np, random, matplotlib.pyplot as plt, time, networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
import plotly.graph_objs as go

# ----------------------- #
# 0) Reproducibility
# ----------------------- #
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ----------------------- #
# 1) Dataset + features
# ----------------------- #
dataset = Planetoid(root="./data", name="CiteSeer")
data = dataset[0].to(device)
# Stabilize structure for attention
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# Scale + augment features with log-degree
scaler = StandardScaler()
x_scaled = scaler.fit_transform(data.x.cpu().numpy())
deg = degree(data.edge_index[0], num_nodes=data.num_nodes).cpu().numpy()
log_deg = np.log1p(deg).reshape(-1, 1)
x_aug = np.concatenate([x_scaled, log_deg], axis=1)
data.x = torch.tensor(x_aug, dtype=torch.float32, device=device)
in_dim = data.x.size(1)
print(f"Nodes:{data.num_nodes}, Edges:{data.edge_index.size(1)}, Features:{in_dim}, Classes:{dataset.num_classes}")

# ----------------------- #
# 2) Annealed DropEdge
# ----------------------- #
def dropedge(edge_index, epoch, total_epochs, base_p=0.2):
    """Linearly decreases edge-drop prob from base_p -> 0 over training."""
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0: return edge_index
    E = edge_index.size(1)
    keep = (torch.rand(E, device=edge_index.device) > p)
    # Keep self-loops
    self_mask = (edge_index[0] == edge_index[1])
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)
    return edge_index[:, keep]

# ----------------------- #
# 3) GATv2 block
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(self, in_d, out_d, heads=8, dropout=0.6, residual=True, concat=True):
        super().__init__()
        self.conv = GATv2Conv(in_d, out_d, heads=heads, dropout=dropout, concat=concat)
        self.norm = nn.LayerNorm(out_d*heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.use_res = residual and (in_d == (out_d*heads if concat else out_d))
    def forward(self, x, ei):
        h = self.conv(x, ei)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        return h + x if self.use_res else h

# ----------------------- #
# 4) GAT+++ model
# ----------------------- #
class BetterGAT(nn.Module):
    def __init__(self, dim_in, dim_h, dim_out, heads=(8,8,4), dropout=0.6):
        super().__init__()
        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0], dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h*heads[0], dim_h, heads=heads[1], dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h*heads[1], dim_h, heads=heads[2], dropout=dropout, residual=False)
        self.out = GATv2Conv(dim_h*heads[2], dim_out, heads=1, dropout=dropout, concat=False)
        self.res_fc = nn.Linear(dim_in, dim_out)  # residual MLP head for isolated/low-degree nodes
        self.opt = torch.optim.Adam(self.parameters(), lr=0.003, weight_decay=5e-4)
        self.sch = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=200)
    def forward(self, x, ei, training=False):
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, ei)
        h = self.g2(h, ei)
        h = self.g3(h, ei)
        h = F.dropout(h, p=0.6, training=training)
        logits = self.out(h, ei)
        logits = logits + 0.1 * self.res_fc(x)  # residual refinement
        return h, F.log_softmax(logits, dim=1)

# ----------------------- #
# 5) Train (no early stop) + consistency loss
# ----------------------- #
def accuracy(pred, y): return (pred.eq(y).sum() / y.numel()).item()

def train(model, data, epochs=3000, clip=1.0, consistency_w=0.2, base_dropedge=0.2):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    train_curve, val_curve = [], []
    print(f"\n🧠 Training for {epochs} epochs (annealed DropEdge + consistency) ...\n")
    for ep in range(1, epochs+1):
        model.train(); model.opt.zero_grad()
        # Annealed DropEdge
        ei_aug = dropedge(data.edge_index, ep, epochs, base_p=base_dropedge)
        _, out = model(data.x, ei_aug, training=True)
        ce_loss = ce(out[data.train_mask], data.y[data.train_mask])

        # Label-consistency: match logits distribution to LP-smoothed probs
        with torch.no_grad():
            lp_probs = lp_layer(out.exp(), data.edge_index)
        consistency = F.mse_loss(out.exp(), lp_probs)

        loss = ce_loss + consistency_w * consistency
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        model.opt.step(); model.sch.step()

        # Metrics
        model.eval()
        with torch.no_grad():
            _, outv = model(data.x, data.edge_index, training=False)
            val_loss = ce(outv[data.val_mask], data.y[data.val_mask]).item()
            val_acc  = accuracy(outv[data.val_mask].argmax(1), data.y[data.val_mask])
            tr_acc   = accuracy(out[data.train_mask].argmax(1), data.y[data.train_mask])
        train_curve.append(tr_acc); val_curve.append(val_acc)

        if ep % 50 == 0 or ep == 1:
            lr = model.opt.param_groups[0]['lr']
            print(f"Epoch {ep:04d} | TrainLoss:{loss:.3f} | ValLoss:{val_loss:.3f} | ValAcc:{val_acc*100:5.2f}% | LR:{lr:.5f}")
    print("✅ Training complete.\n")
    return model, train_curve, val_curve

@torch.no_grad()
def test(model, data, use_lp=False):
    model.eval()
    if use_lp:
        _, logp = model(data.x, data.edge_index)
        probs   = logp.exp()
        lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
        refined = lp_layer(probs, data.edge_index)
        pred    = refined.argmax(1)
    else:
        _, out = model(data.x, data.edge_index)
        pred    = out.argmax(1)
    return accuracy(pred[data.test_mask], data.y[data.test_mask]), pred

# ----------------------- #
# 6) Run
# ----------------------- #
model = BetterGAT(in_dim, 8, dataset.num_classes).to(device)
print(model)
model, train_curve, val_curve = train(model, data, epochs=3000, clip=1.0, consistency_w=0.2, base_dropedge=0.2)
raw_acc,  preds_raw  = test(model, data, use_lp=False)
lp_acc,   preds_lp   = test(model, data, use_lp=True)
print(f"📈 Test Accuracy (Raw): {raw_acc*100:.2f}% | (Label Prop): {lp_acc*100:.2f}%")

# =======================================================================
# 7) Visualizations — spaced out & labelled
# =======================================================================
def spacer(title):
    print("\n" + "─"*90)
    print("🎨", title)
    print("─"*90)
    time.sleep(0.5)

# 7.1 Learning curves
spacer("Learning Curves (Train vs Val Accuracy)")
plt.figure(figsize=(9,4))
plt.plot(train_curve, label="Train Acc", color="#1f77b4")
plt.plot(val_curve,   label="Val Acc",   color="#ff7f0e")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Learning Curves (GAT+++)"); plt.legend()
plt.grid(alpha=0.4); plt.tight_layout(); plt.show()

# 7.2 t-SNE animation (Untrained vs Trained)
@torch.no_grad()
def tsne_animation(model, data):
    spacer("t-SNE Animation (Untrained → Trained Embeddings)")
    # Untrained baseline (fresh init with same dims)
    untrained = BetterGAT(in_dim, 8, dataset.num_classes).to(device)
    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)
    tsne0 = TSNE(n_components=2, init='pca', learning_rate='auto').fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(n_components=2, init='pca', learning_rate='auto').fit_transform(emb1.cpu().numpy())
    frames = [
        go.Frame(data=[go.Scatter(x=tsne0[:,0], y=tsne0[:,1], mode='markers',
                                  marker=dict(color=data.y.cpu(), colorscale='Viridis', size=6, opacity=0.85))],
                 name="Untrained"),
        go.Frame(data=[go.Scatter(x=tsne1[:,0], y=tsne1[:,1], mode='markers',
                                  marker=dict(color=data.y.cpu(), colorscale='Viridis', size=6, opacity=0.85))],
                 name="Trained")
    ]
    layout = go.Layout(
        title="t-SNE Embeddings Evolution",
        xaxis=dict(title="t-SNE-1"), yaxis=dict(title="t-SNE-2"),
        updatemenus=[dict(type="buttons", showactive=False,
                          buttons=[dict(label="Play", method="animate", args=[None])])],
        height=600
    )
    fig = go.Figure(frames=frames, layout=layout)
    fig.add_trace(frames[0].data[0])
    fig.show()
tsne_animation(model, data)

# 7.3 Interactive 3D graph (colored by PREDICTIONS)
def interactive_graph(data, pred, title="Interactive Graph (Predictions)"):
    spacer(title)
    G = nx.Graph()
    edges = data.edge_index.cpu().t().numpy()
    G.add_edges_from(map(tuple, edges))
    pos = nx.spring_layout(G, seed=42, dim=3)
    Xn, Yn, Zn = zip(*[pos[k] for k in G.nodes()])
    Xe, Ye, Ze = [], [], []
    for e in G.edges():
        Xe += [pos[e[0]][0], pos[e[1]][0], None]
        Ye += [pos[e[0]][1], pos[e[1]][1], None]
        Ze += [pos[e[0]][2], pos[e[1]][2], None]
    edge_trace = go.Scatter3d(x=Xe,y=Ye,z=Ze, mode="lines", line=dict(color="rgba(120,120,120,0.4)", width=1), hoverinfo="none")
    node_trace = go.Scatter3d(
        x=Xn,y=Yn,z=Zn, mode="markers",
        marker=dict(size=5, color=pred.cpu(), colorscale="Rainbow", opacity=0.9),
        text=[f"Node {i} | True {int(data.y[i])} | Pred {int(pred[i])}" for i in range(data.num_nodes)],
        hoverinfo="text"
    )
    layout = go.Layout(title=title, height=680, margin=dict(l=0,r=0,b=0,t=60), showlegend=False)
    fig = go.Figure(data=[edge_trace, node_trace], layout=layout)
    fig.update_layout(scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)))
    fig.show()

interactive_graph(data, preds_raw, title="Interactive Graph (Raw Predictions)")
interactive_graph(data, preds_lp,  title="Interactive Graph (Label-Prop Predictions)")

# 7.4 Accuracy-by-degree bar chart
@torch.no_grad()
def accuracy_by_degree(model, data):
    spacer("Accuracy by Node Degree")
    _, out = model(data.x, data.edge_index, training=False)
    pred = out.argmax(1); degs = degree(data.edge_index[0], num_nodes=data.num_nodes)
    bins = [0,1,2,3,4,5]
    accs, counts = [], []
    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)
    mask = (degs > 5)
    counts.append(int(mask.sum()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)
    plt.figure(figsize=(10,5))
    plt.bar([str(b) for b in bins] + [">5"], accs, color="#0A047A")
    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)
    plt.title("Accuracy by Node Degree"); plt.xlabel("Degree"); plt.ylabel("Accuracy")
    plt.ylim(0, 1.05); plt.grid(axis="y", linestyle="--", alpha=0.5); plt.tight_layout(); plt.show()

accuracy_by_degree(model, data)

# 7.5 Attention heatmap for a specific node (from test mask)
@torch.no_grad()
def attention_heatmap(model, data, node_id):
    spacer(f"Attention Heatmap for Node {node_id}")
    _, att = model.g1.conv(data.x, data.edge_index, return_attention_weights=True)
    ei, alpha = att
    alpha = alpha.mean(1)  # mean over heads
    mask = (ei[0] == node_id)
    nbrs = ei[1, mask].cpu().numpy()
    weights = alpha[mask].cpu().numpy()
    if len(nbrs) == 0:
        print("No neighbors for this node.")
        return
    idx = np.argsort(weights)[::-1]
    nbrs, weights = nbrs[idx], weights[idx]
    plt.figure(figsize=(8, max(1.6, 0.25*len(nbrs))))
    plt.barh(range(len(nbrs)), weights, color="skyblue")
    plt.yticks(range(len(nbrs)), [str(n) for n in nbrs]); plt.gca().invert_yaxis()
    plt.xlabel("Attention Weight αᵢⱼ"); plt.title(f"Node {node_id} → Neighbor Attention")
    plt.tight_layout(); plt.show()

test_nodes = torch.where(data.test_mask)[0].tolist()
if test_nodes:
    attention_heatmap(model, data, test_nodes[0])

# 7.6 Confusion matrix on test mask (raw predictions)
@torch.no_grad()
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)"):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()
    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))
    plt.figure(figsize=(6,5))
    plt.imshow(cm, cmap="Blues"); plt.title(title); plt.colorbar()
    plt.xlabel("Predicted"); plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks); plt.yticks(ticks, ticks)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    plt.tight_layout(); plt.show()

plot_confusion(data, preds_raw, "Confusion Matrix (Raw Predictions, Test Mask)")
plot_confusion(data, preds_lp,  "Confusion Matrix (Label-Prop Predictions, Test Mask)")

print("\n✅ All visuals rendered.")

In [ ]:
# ============================================ #
# Graph Attention Networks +++  (Perf + Visuals)
# ============================================ #
# Includes:
#  - Annealed DropEdge regularization
#  - Residual MLP refinement head
#  - Label-consistency auxiliary loss
#  - Log-degree feature augmentation + scaling
#  - 3-layer GATv2 backbone w/ LayerNorm
#  - Long training (no early stop) + cosine LR decay + grad clipping
#  - Visuals: learning curves, t-SNE (animated), interactive 3D graph,
#             attention heatmap, accuracy-by-degree, confusion matrix
# ============================================ #

# Uncomment if running in Colab
# !pip install -q torch-geometric scikit-learn matplotlib plotly networkx

import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import degree, add_self_loops, to_undirected
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn.models.label_prop import LabelPropagation
import numpy as np, random, matplotlib.pyplot as plt, time, networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
import plotly.graph_objs as go

# ----------------------- #
# 0) Reproducibility
# ----------------------- #
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ----------------------- #
# 1) Dataset + features
# ----------------------- #
dataset = Planetoid(root="./data", name="CiteSeer")
data = dataset[0].to(device)
# Stabilize structure for attention
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# Scale + augment features with log-degree
scaler = StandardScaler()
x_scaled = scaler.fit_transform(data.x.cpu().numpy())
deg = degree(data.edge_index[0], num_nodes=data.num_nodes).cpu().numpy()
log_deg = np.log1p(deg).reshape(-1, 1)
x_aug = np.concatenate([x_scaled, log_deg], axis=1)
data.x = torch.tensor(x_aug, dtype=torch.float32, device=device)
in_dim = data.x.size(1)
print(f"Nodes:{data.num_nodes}, Edges:{data.edge_index.size(1)}, Features:{in_dim}, Classes:{dataset.num_classes}")

# ----------------------- #
# 2) Annealed DropEdge
# ----------------------- #
def dropedge(edge_index, epoch, total_epochs, base_p=0.2):
    """Linearly decreases edge-drop prob from base_p -> 0 over training."""
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0: return edge_index
    E = edge_index.size(1)
    keep = (torch.rand(E, device=edge_index.device) > p)
    # Keep self-loops
    self_mask = (edge_index[0] == edge_index[1])
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)
    return edge_index[:, keep]

# ----------------------- #
# 3) GATv2 block
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(self, in_d, out_d, heads=8, dropout=0.6, residual=True, concat=True):
        super().__init__()
        self.conv = GATv2Conv(in_d, out_d, heads=heads, dropout=dropout, concat=concat)
        self.norm = nn.LayerNorm(out_d*heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.use_res = residual and (in_d == (out_d*heads if concat else out_d))
    def forward(self, x, ei):
        h = self.conv(x, ei)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        return h + x if self.use_res else h

# ----------------------- #
# 4) GAT+++ model
# ----------------------- #
class BetterGAT(nn.Module):
    def __init__(self, dim_in, dim_h, dim_out, heads=(8,8,4), dropout=0.6):
        super().__init__()
        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0], dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h*heads[0], dim_h, heads=heads[1], dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h*heads[1], dim_h, heads=heads[2], dropout=dropout, residual=False)
        self.out = GATv2Conv(dim_h*heads[2], dim_out, heads=1, dropout=dropout, concat=False)
        self.res_fc = nn.Linear(dim_in, dim_out)  # residual MLP head for isolated/low-degree nodes
        self.opt = torch.optim.Adam(self.parameters(), lr=0.003, weight_decay=5e-4)
        self.sch = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=200)
    def forward(self, x, ei, training=False):
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, ei)
        h = self.g2(h, ei)
        h = self.g3(h, ei)
        h = F.dropout(h, p=0.6, training=training)
        logits = self.out(h, ei)
        logits = logits + 0.1 * self.res_fc(x)  # residual refinement
        return h, F.log_softmax(logits, dim=1)

# ----------------------- #
# 5) Train (no early stop) + consistency loss
# ----------------------- #
def accuracy(pred, y): return (pred.eq(y).sum() / y.numel()).item()

def train(model, data, epochs=10000, clip=1.0, consistency_w=0.2, base_dropedge=0.2):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    train_curve, val_curve = [], []
    print(f"\n🧠 Training for {epochs} epochs (annealed DropEdge + consistency) ...\n")
    for ep in range(1, epochs+1):
        model.train(); model.opt.zero_grad()
        # Annealed DropEdge
        ei_aug = dropedge(data.edge_index, ep, epochs, base_p=base_dropedge)
        _, out = model(data.x, ei_aug, training=True)
        ce_loss = ce(out[data.train_mask], data.y[data.train_mask])

        # Label-consistency: match logits distribution to LP-smoothed probs
        with torch.no_grad():
            lp_probs = lp_layer(out.exp(), data.edge_index)
        consistency = F.mse_loss(out.exp(), lp_probs)

        loss = ce_loss + consistency_w * consistency
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        model.opt.step(); model.sch.step()

        # Metrics
        model.eval()
        with torch.no_grad():
            _, outv = model(data.x, data.edge_index, training=False)
            val_loss = ce(outv[data.val_mask], data.y[data.val_mask]).item()
            val_acc  = accuracy(outv[data.val_mask].argmax(1), data.y[data.val_mask])
            tr_acc   = accuracy(out[data.train_mask].argmax(1), data.y[data.train_mask])
        train_curve.append(tr_acc); val_curve.append(val_acc)

        if ep % 50 == 0 or ep == 1:
            lr = model.opt.param_groups[0]['lr']
            print(f"Epoch {ep:04d} | TrainLoss:{loss:.3f} | ValLoss:{val_loss:.3f} | ValAcc:{val_acc*100:5.2f}% | LR:{lr:.5f}")
    print("✅ Training complete.\n")
    return model, train_curve, val_curve

@torch.no_grad()
def test(model, data, use_lp=False):
    model.eval()
    if use_lp:
        _, logp = model(data.x, data.edge_index)
        probs   = logp.exp()
        lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
        refined = lp_layer(probs, data.edge_index)
        pred    = refined.argmax(1)
    else:
        _, out = model(data.x, data.edge_index)
        pred    = out.argmax(1)
    return accuracy(pred[data.test_mask], data.y[data.test_mask]), pred

# ----------------------- #
# 6) Run
# ----------------------- #
model = BetterGAT(in_dim, 8, dataset.num_classes).to(device)
print(model)
model, train_curve, val_curve = train(model, data, epochs=10000, clip=1.0, consistency_w=0.2, base_dropedge=0.2)
raw_acc,  preds_raw  = test(model, data, use_lp=False)
lp_acc,   preds_lp   = test(model, data, use_lp=True)
print(f"📈 Test Accuracy (Raw): {raw_acc*100:.2f}% | (Label Prop): {lp_acc*100:.2f}%")

# =======================================================================
# 7) Visualizations — spaced out & labelled
# =======================================================================
def spacer(title):
    print("\n" + "─"*90)
    print("🎨", title)
    print("─"*90)
    time.sleep(0.5)

# 7.1 Learning curves
spacer("Learning Curves (Train vs Val Accuracy)")
plt.figure(figsize=(9,4))
plt.plot(train_curve, label="Train Acc", color="#1f77b4")
plt.plot(val_curve,   label="Val Acc",   color="#ff7f0e")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Learning Curves (GAT+++)"); plt.legend()
plt.grid(alpha=0.4); plt.tight_layout(); plt.show()

# 7.2 t-SNE animation (Untrained vs Trained)
@torch.no_grad()
def tsne_animation(model, data):
    spacer("t-SNE Animation (Untrained → Trained Embeddings)")
    # Untrained baseline (fresh init with same dims)
    untrained = BetterGAT(in_dim, 8, dataset.num_classes).to(device)
    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)
    tsne0 = TSNE(n_components=2, init='pca', learning_rate='auto').fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(n_components=2, init='pca', learning_rate='auto').fit_transform(emb1.cpu().numpy())
    frames = [
        go.Frame(data=[go.Scatter(x=tsne0[:,0], y=tsne0[:,1], mode='markers',
                                  marker=dict(color=data.y.cpu(), colorscale='Viridis', size=6, opacity=0.85))],
                 name="Untrained"),
        go.Frame(data=[go.Scatter(x=tsne1[:,0], y=tsne1[:,1], mode='markers',
                                  marker=dict(color=data.y.cpu(), colorscale='Viridis', size=6, opacity=0.85))],
                 name="Trained")
    ]
    layout = go.Layout(
        title="t-SNE Embeddings Evolution",
        xaxis=dict(title="t-SNE-1"), yaxis=dict(title="t-SNE-2"),
        updatemenus=[dict(type="buttons", showactive=False,
                          buttons=[dict(label="Play", method="animate", args=[None])])],
        height=600
    )
    fig = go.Figure(frames=frames, layout=layout)
    fig.add_trace(frames[0].data[0])
    fig.show()
tsne_animation(model, data)

# 7.3 Interactive 3D graph (colored by PREDICTIONS)
def interactive_graph(data, pred, title="Interactive Graph (Predictions)"):
    spacer(title)
    G = nx.Graph()
    edges = data.edge_index.cpu().t().numpy()
    G.add_edges_from(map(tuple, edges))
    pos = nx.spring_layout(G, seed=42, dim=3)
    Xn, Yn, Zn = zip(*[pos[k] for k in G.nodes()])
    Xe, Ye, Ze = [], [], []
    for e in G.edges():
        Xe += [pos[e[0]][0], pos[e[1]][0], None]
        Ye += [pos[e[0]][1], pos[e[1]][1], None]
        Ze += [pos[e[0]][2], pos[e[1]][2], None]
    edge_trace = go.Scatter3d(x=Xe,y=Ye,z=Ze, mode="lines", line=dict(color="rgba(120,120,120,0.4)", width=1), hoverinfo="none")
    node_trace = go.Scatter3d(
        x=Xn,y=Yn,z=Zn, mode="markers",
        marker=dict(size=5, color=pred.cpu(), colorscale="Rainbow", opacity=0.9),
        text=[f"Node {i} | True {int(data.y[i])} | Pred {int(pred[i])}" for i in range(data.num_nodes)],
        hoverinfo="text"
    )
    layout = go.Layout(title=title, height=680, margin=dict(l=0,r=0,b=0,t=60), showlegend=False)
    fig = go.Figure(data=[edge_trace, node_trace], layout=layout)
    fig.update_layout(scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)))
    fig.show()

interactive_graph(data, preds_raw, title="Interactive Graph (Raw Predictions)")
interactive_graph(data, preds_lp,  title="Interactive Graph (Label-Prop Predictions)")

# 7.4 Accuracy-by-degree bar chart
@torch.no_grad()
def accuracy_by_degree(model, data):
    spacer("Accuracy by Node Degree")
    _, out = model(data.x, data.edge_index, training=False)
    pred = out.argmax(1); degs = degree(data.edge_index[0], num_nodes=data.num_nodes)
    bins = [0,1,2,3,4,5]
    accs, counts = [], []
    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)
    mask = (degs > 5)
    counts.append(int(mask.sum()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)
    plt.figure(figsize=(10,5))
    plt.bar([str(b) for b in bins] + [">5"], accs, color="#0A047A")
    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)
    plt.title("Accuracy by Node Degree"); plt.xlabel("Degree"); plt.ylabel("Accuracy")
    plt.ylim(0, 1.05); plt.grid(axis="y", linestyle="--", alpha=0.5); plt.tight_layout(); plt.show()

accuracy_by_degree(model, data)

# 7.5 Attention heatmap for a specific node (from test mask)
@torch.no_grad()
def attention_heatmap(model, data, node_id):
    spacer(f"Attention Heatmap for Node {node_id}")
    _, att = model.g1.conv(data.x, data.edge_index, return_attention_weights=True)
    ei, alpha = att
    alpha = alpha.mean(1)  # mean over heads
    mask = (ei[0] == node_id)
    nbrs = ei[1, mask].cpu().numpy()
    weights = alpha[mask].cpu().numpy()
    if len(nbrs) == 0:
        print("No neighbors for this node.")
        return
    idx = np.argsort(weights)[::-1]
    nbrs, weights = nbrs[idx], weights[idx]
    plt.figure(figsize=(8, max(1.6, 0.25*len(nbrs))))
    plt.barh(range(len(nbrs)), weights, color="skyblue")
    plt.yticks(range(len(nbrs)), [str(n) for n in nbrs]); plt.gca().invert_yaxis()
    plt.xlabel("Attention Weight αᵢⱼ"); plt.title(f"Node {node_id} → Neighbor Attention")
    plt.tight_layout(); plt.show()

test_nodes = torch.where(data.test_mask)[0].tolist()
if test_nodes:
    attention_heatmap(model, data, test_nodes[0])

# 7.6 Confusion matrix on test mask (raw predictions)
@torch.no_grad()
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)"):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()
    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))
    plt.figure(figsize=(6,5))
    plt.imshow(cm, cmap="Blues"); plt.title(title); plt.colorbar()
    plt.xlabel("Predicted"); plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks); plt.yticks(ticks, ticks)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    plt.tight_layout(); plt.show()

plot_confusion(data, preds_raw, "Confusion Matrix (Raw Predictions, Test Mask)")
plot_confusion(data, preds_lp,  "Confusion Matrix (Label-Prop Predictions, Test Mask)")

print("\n✅ All visuals rendered.")

In [ ]:
import torch
torchversion = torch.__version__

# Install PyTorch Scatter, PyTorch Sparse, and PyTorch Geometric
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{torchversion}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-{torchversion}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

# Numpy for matrices
import numpy as np
np.random.seed(0)

# Visualization
import networkx as nx
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

In [ ]:
# ============================================ #
# Graph Attention Networks +++
# (Cosine-Aligned Aux Loss + DropEdge + Visuals)
# ============================================ #
# Includes:
#  - Annealed DropEdge regularization
#  - Residual MLP refinement head
#  - Cosine-aligned label-consistency auxiliary loss
#    (a "second compass" when neighbourhoods get thin)
#  - Log-degree feature augmentation + scaling
#  - 3-layer GATv2 backbone w/ LayerNorm
#  - Long training + cosine LR decay + grad clipping
#  - Visuals: learning curves, t-SNE, accuracy-by-degree,
#             attention heatmap, confusion matrix
# ============================================ #

!pip install -q torch-geometric scikit-learn matplotlib networkx

import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
)
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ----------------------- #
# 0) Reproducibility
# ----------------------- #
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

# 🔒 Force everything to CPU to avoid device mismatches
device = torch.device("cpu")
print("Device:", device)

# ----------------------- #
# 1) Dataset + basic stats
# ----------------------- #
dataset = Planetoid(root="./data", name="CiteSeer")

data = dataset[0]  # single graph
# keep on CPU; we'll just ensure everything is on same device = CPU

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Stabilize structure for attention
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# ----------------------- #
# 2) Quick structure visuals
# ----------------------- #
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y,
    width=0.5,
    edge_color="grey",
)
plt.title("CiteSeer Graph (Node-colored by Label)")
plt.show()

degrees_arr = degree(data.edge_index[0]).numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.show()

# ----------------------- #
# 3) Feature scaling + log-degree augmentation
# ----------------------- #
scaler = StandardScaler()
x_scaled = scaler.fit_transform(data.x.numpy())

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

x_aug = np.concatenate([x_scaled, log_deg], axis=1)
data.x = torch.tensor(x_aug, dtype=torch.float32)

in_dim = data.x.size(1)
print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

# ----------------------- #
# 4) Annealed DropEdge
# ----------------------- #
def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


# ----------------------- #
# 5) GATv2 Block
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


# ----------------------- #
# 6) GAT+++ Model
# ----------------------- #
class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(8, 8, 4),
        dropout: float = 0.6,
    ):
        super().__init__()

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # residual head for low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        self.opt = torch.optim.Adam(self.parameters(), lr=0.003, weight_decay=5e-4)
        self.sch = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=200)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ----------------------- #
# 7) Training / Eval (with cosine-aligned aux loss)
# ----------------------- #
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return (pred.eq(y).sum() / y.numel()).item()


def train(
    model: BetterGAT,
    data,
    epochs: int = 3000,
    clip: float = 1.0,
    consistency_w: float = 0.2,
    base_dropedge: float = 0.2,
):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    train_curve, val_curve = [], []
    print(f"\n🧠 Training for {epochs} epochs (DropEdge + cosine LP consistency)...\n")
    print(
        "Note: cosine-aligned auxiliary loss acts as a second compass\n"
        "      for the attention heads when neighbourhoods get thin —\n"
        "      dual-signal routing to expose real structure vs pattern-matching.\n"
    )

    for ep in range(1, epochs + 1):
        model.train()
        model.opt.zero_grad()

        # Annealed DropEdge
        edge_index_aug = dropedge(data.edge_index, ep, epochs, base_p=base_dropedge)

        # Forward
        _, log_probs = model(data.x, edge_index_aug, training=True)

        # CE loss on labeled nodes
        ce_loss = ce(log_probs[data.train_mask], data.y[data.train_mask])

        # Label propagation on same (CPU) device
        with torch.no_grad():
            probs = log_probs.exp()
            lp_probs = lp_layer(probs, data.edge_index)

        # Cosine-aligned auxiliary loss ("second compass")
        cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
        consistency = 1.0 - cos_sim.mean()

        loss = ce_loss + consistency_w * consistency
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        model.opt.step()
        model.sch.step()

        model.eval()
        with torch.no_grad():
            _, log_probs_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(
                log_probs_val[data.val_mask], data.y[data.val_mask]
            ).item()
            tr_acc = accuracy(
                log_probs[data.train_mask].argmax(1),
                data.y[data.train_mask],
            )
            val_acc = accuracy(
                log_probs_val[data.val_mask].argmax(1),
                data.y[data.val_mask],
            )

        train_curve.append(tr_acc)
        val_curve.append(val_acc)

        if ep % 50 == 0 or ep == 1:
            lr = model.opt.param_groups[0]["lr"]
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss: {loss:.3f} | "
                f"ValLoss: {val_loss:.3f} | "
                f"TrainAcc: {tr_acc*100:5.2f}% | "
                f"ValAcc: {val_acc*100:5.2f}% | "
                f"LR: {lr:.5f}"
            )

    print("\n✅ Training complete.")
    print(
        "\nNow this setup is ready to be pushed into harsher regimes —\n"
        "extreme sparsity, label starvation, corrupted edges — to map\n"
        "out its true failure geometry. That’s usually where the good\n"
        "stories hide.\n"
    )
    return model, train_curve, val_curve


@torch.no_grad()
def test(model: BetterGAT, data, use_lp: bool = False):
    model.eval()
    _, log_probs = model(data.x, data.edge_index, training=False)

    if use_lp:
        probs = log_probs.exp()
        lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
        lp_probs = lp_layer(probs, data.edge_index)
        pred = lp_probs.argmax(1)
    else:
        pred = log_probs.argmax(1)

    return accuracy(pred[data.test_mask], data.y[data.test_mask]), pred


# ----------------------- #
# 8) Run training
# ----------------------- #
model = BetterGAT(in_dim, 8, dataset.num_classes)
print(model)

model, train_curve, val_curve = train(
    model,
    data,
    epochs=3000,
    clip=1.0,
    consistency_w=0.2,
    base_dropedge=0.2,
)

raw_acc, preds_raw = test(model, data, use_lp=False)
lp_acc, preds_lp = test(model, data, use_lp=True)

print(f"📈 Test Accuracy (Raw):       {raw_acc*100:.2f}%")
print(f"📈 Test Accuracy (LabelProp): {lp_acc*100:.2f}%")

# ----------------------- #
# 9) Visual helpers
# ----------------------- #
def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.5)


spacer("Learning Curves (Train vs Val Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(train_curve, label="Train Acc", color="#1f77b4")
plt.plot(val_curve, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Learning Curves (BetterGAT + Cosine Consistency)")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()


@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained)")

    untrained = BetterGAT(in_dim, 8, dataset.num_classes)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained GAT Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.numpy(), s=8)
    plt.axis("off")
    plt.title("Trained BetterGAT Embeddings")

    plt.tight_layout()
    plt.show()


tsne_static(model, data)


@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree")
    _, log_probs = model(data.x, data.edge_index, training=False)
    pred = log_probs.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


accuracy_by_degree(model, data)


@torch.no_grad()
def attention_heatmap(model: BetterGAT, data, node_id: int):
    spacer(f"Attention Heatmap for Node {node_id}")
    _, att = model.g1.conv(
        data.x,
        data.edge_index,
        return_attention_weights=True,
    )
    edge_idx, alpha = att
    alpha = alpha.mean(1)

    mask = edge_idx[0] == node_id
    nbrs = edge_idx[1, mask].numpy()
    weights = alpha[mask].numpy()

    if len(nbrs) == 0:
        print("No neighbors for this node.")
        return

    idx = np.argsort(weights)[::-1]
    nbrs, weights = nbrs[idx], weights[idx]

    plt.figure(figsize=(8, max(1.6, 0.25 * len(nbrs))))
    plt.barh(range(len(nbrs)), weights, color="skyblue")
    plt.yticks(range(len(nbrs)), [str(n) for n in nbrs])
    plt.gca().invert_yaxis()
    plt.xlabel("Attention Weight αᵢⱼ")
    plt.title(f"Node {node_id} → Neighbor Attention")
    plt.tight_layout()
    plt.show()


test_nodes = torch.where(data.test_mask)[0].tolist()
if test_nodes:
    attention_heatmap(model, data, int(test_nodes[0]))


@torch.no_grad()
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)"):
    spacer(title)
    y_true = data.y[data.test_mask].numpy()
    y_pred = pred[data.test_mask].numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    plt.show()


plot_confusion(data, preds_raw, "Confusion Matrix (Raw Predictions, Test Mask)")
plot_confusion(data, preds_lp, "Confusion Matrix (Label-Prop Predictions, Test Mask)")

print("\n✅ All visuals rendered.")

In [ ]:
# ============================================ #
# Graph Attention Networks +++
# (Cosine-Aligned Aux Loss + DropEdge + Visuals)
# ============================================ #
# Includes:
#  - Annealed DropEdge regularization
#  - Residual MLP refinement head
#  - Cosine-aligned label-consistency auxiliary loss
#    (a "second compass" when neighbourhoods get thin)
#  - Log-degree feature augmentation + scaling
#  - 3-layer GATv2 backbone w/ LayerNorm
#  - Long training + cosine LR decay + grad clipping
#  - Visuals:
#       * Learning curves (acc)
#       * Loss curves
#       * t-SNE (untrained vs trained)
#       * Accuracy-by-degree
#       * Attention heatmap
#       * Confusion matrices
#       * Per-class accuracy (raw vs LP)
#       * Cosine alignment histogram
#       * Ego-graph around a misclassified node
# ============================================ #

!pip install -q torch-geometric scikit-learn matplotlib networkx

import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
)
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ----------------------- #
# 0) Reproducibility
# ----------------------- #
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

# 🔒 Force everything to CPU to avoid device mismatches
device = torch.device("cpu")
print("Device:", device)

# ----------------------- #
# 1) Dataset + basic stats
# ----------------------- #
dataset = Planetoid(root="./data", name="CiteSeer")

data = dataset[0]  # single graph on CPU

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Stabilize structure for attention
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# ----------------------- #
# 2) Quick structure visuals
# ----------------------- #
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y,
    width=0.5,
    edge_color="grey",
)
plt.title("CiteSeer Graph (Node-colored by Label)")
plt.show()

degrees_arr = degree(data.edge_index[0]).numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.show()

# ----------------------- #
# 3) Feature scaling + log-degree augmentation
# ----------------------- #
scaler = StandardScaler()
x_scaled = scaler.fit_transform(data.x.numpy())

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

x_aug = np.concatenate([x_scaled, log_deg], axis=1)
data.x = torch.tensor(x_aug, dtype=torch.float32)

in_dim = data.x.size(1)
print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

# ----------------------- #
# 4) Annealed DropEdge
# ----------------------- #
def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


# ----------------------- #
# 5) GATv2 Block
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


# ----------------------- #
# 6) GAT+++ Model
# ----------------------- #
class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(8, 8, 4),
        dropout: float = 0.6,
    ):
        super().__init__()

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # residual head for low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        self.opt = torch.optim.Adam(self.parameters(), lr=0.003, weight_decay=5e-4)
        self.sch = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=200)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ----------------------- #
# 7) Training / Eval (with cosine-aligned aux loss)
# ----------------------- #
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return (pred.eq(y).sum() / y.numel()).item()


def train(
    model: BetterGAT,
    data,
    epochs: int = 3000,
    clip: float = 1.0,
    consistency_w: float = 0.2,
    base_dropedge: float = 0.2,
):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    print(f"\n🧠 Training for {epochs} epochs (DropEdge + cosine LP consistency)...\n")
    print(
        "Note: cosine-aligned auxiliary loss acts as a second compass\n"
        "      for the attention heads when neighbourhoods get thin —\n"
        "      dual-signal routing to expose real structure vs pattern-matching.\n"
    )

    for ep in range(1, epochs + 1):
        model.train()
        model.opt.zero_grad()

        # Annealed DropEdge
        edge_index_aug = dropedge(data.edge_index, ep, epochs, base_p=base_dropedge)

        # Forward
        _, log_probs = model(data.x, edge_index_aug, training=True)

        # CE loss on labeled nodes
        ce_loss = ce(log_probs[data.train_mask], data.y[data.train_mask])

        # Label propagation on same device
        with torch.no_grad():
            probs = log_probs.exp()
            lp_probs = lp_layer(probs, data.edge_index)

        # Cosine-aligned auxiliary loss ("second compass")
        cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
        consistency = 1.0 - cos_sim.mean()

        loss = ce_loss + consistency_w * consistency
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        model.opt.step()
        model.sch.step()

        # Eval on val
        model.eval()
        with torch.no_grad():
            _, log_probs_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(
                log_probs_val[data.val_mask], data.y[data.val_mask]
            ).item()
            tr_acc = accuracy(
                log_probs[data.train_mask].argmax(1),
                data.y[data.train_mask],
            )
            val_acc = accuracy(
                log_probs_val[data.val_mask].argmax(1),
                data.y[data.val_mask],
            )

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_loss)

        if ep % 50 == 0 or ep == 1:
            lr = model.opt.param_groups[0]["lr"]
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss: {loss:.3f} | "
                f"ValLoss: {val_loss:.3f} | "
                f"TrainAcc: {tr_acc*100:5.2f}% | "
                f"ValAcc: {val_acc*100:5.2f}% | "
                f"LR: {lr:.5f}"
            )

    print("\n✅ Training complete.")
    print(
        "\nNow this setup is ready to be pushed into harsher regimes —\n"
        "extreme sparsity, label starvation, corrupted edges — to map\n"
        "out its true failure geometry. That’s usually where the good\n"
        "stories hide.\n"
    )
    return model, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve


@torch.no_grad()
def test(model: BetterGAT, data, use_lp: bool = False):
    model.eval()
    _, log_probs = model(data.x, data.edge_index, training=False)

    if use_lp:
        probs = log_probs.exp()
        lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
        lp_probs = lp_layer(probs, data.edge_index)
        pred = lp_probs.argmax(1)
    else:
        pred = log_probs.argmax(1)

    return accuracy(pred[data.test_mask], data.y[data.test_mask]), pred


# ----------------------- #
# 8) Run training
# ----------------------- #
model = BetterGAT(in_dim, 8, dataset.num_classes)
print(model)

model, train_acc, val_acc, train_loss, val_loss = train(
    model,
    data,
    epochs=3000,
    clip=1.0,
    consistency_w=0.2,
    base_dropedge=0.2,
)

raw_acc, preds_raw = test(model, data, use_lp=False)
lp_acc, preds_lp = test(model, data, use_lp=True)

print(f"📈 Test Accuracy (Raw):       {raw_acc*100:.2f}%")
print(f"📈 Test Accuracy (LabelProp): {lp_acc*100:.2f}%")

# ----------------------- #
# 9) Visual helpers
# ----------------------- #
def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.5)


# 9.1 Accuracy curves
spacer("Learning Curves (Train vs Val Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(train_acc, label="Train Acc", color="#1f77b4")
plt.plot(val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Learning Curves (BetterGAT + Cosine Consistency)")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 9.2 Loss curves
spacer("Loss Curves (Train vs Val)")
plt.figure(figsize=(9, 4))
plt.plot(train_loss, label="Train Loss", color="#1f77b4")
plt.plot(val_loss, label="Val Loss", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves (BetterGAT + Cosine Consistency)")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()


# 9.3 t-SNE (untrained vs trained)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained)")

    untrained = BetterGAT(in_dim, 8, dataset.num_classes)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained GAT Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.numpy(), s=8)
    plt.axis("off")
    plt.title("Trained BetterGAT Embeddings")

    plt.tight_layout()
    plt.show()


tsne_static(model, data)


# 9.4 Accuracy-by-degree
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree")
    _, log_probs = model(data.x, data.edge_index, training=False)
    pred = log_probs.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


accuracy_by_degree(model, data)


# 9.5 Attention heatmap
@torch.no_grad()
def attention_heatmap(model: BetterGAT, data, node_id: int):
    spacer(f"Attention Heatmap for Node {node_id}")
    _, att = model.g1.conv(
        data.x,
        data.edge_index,
        return_attention_weights=True,
    )
    edge_idx, alpha = att
    alpha = alpha.mean(1)

    mask = edge_idx[0] == node_id
    nbrs = edge_idx[1, mask].numpy()
    weights = alpha[mask].numpy()

    if len(nbrs) == 0:
        print("No neighbors for this node.")
        return

    idx = np.argsort(weights)[::-1]
    nbrs, weights = nbrs[idx], weights[idx]

    plt.figure(figsize=(8, max(1.6, 0.25 * len(nbrs))))
    plt.barh(range(len(nbrs)), weights, color="skyblue")
    plt.yticks(range(len(nbrs)), [str(n) for n in nbrs])
    plt.gca().invert_yaxis()
    plt.xlabel("Attention Weight αᵢⱼ")
    plt.title(f"Node {node_id} → Neighbor Attention")
    plt.tight_layout()
    plt.show()


test_nodes = torch.where(data.test_mask)[0].tolist()
if test_nodes:
    attention_heatmap(model, data, int(test_nodes[0]))


# 9.6 Confusion matrices (and capture them for per-class accuracy)
@torch.no_grad()
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)"):
    spacer(title)
    y_true = data.y[data.test_mask].numpy()
    y_pred = pred[data.test_mask].numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    plt.show()

    return cm


cm_raw = plot_confusion(data, preds_raw, "Confusion Matrix (Raw Predictions, Test Mask)")
cm_lp = plot_confusion(data, preds_lp, "Confusion Matrix (Label-Prop Predictions, Test Mask)")


# 9.7 Per-class accuracy (Raw vs LabelProp)
def per_class_accuracy_from_cm(cm):
    n_classes = cm.shape[0]
    accs = []
    for c in range(n_classes):
        total = cm[c].sum()
        accs.append(cm[c, c] / total if total > 0 else 0.0)
    return accs


spacer("Per-Class Accuracy (Raw vs LabelProp)")
raw_per_class = per_class_accuracy_from_cm(cm_raw)
lp_per_class = per_class_accuracy_from_cm(cm_lp)

classes = np.arange(dataset.num_classes)
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(classes - width / 2, raw_per_class, width=width, label="Raw", color="#1f77b4")
plt.bar(classes + width / 2, lp_per_class, width=width, label="LabelProp", color="#ff7f0e")
plt.xticks(classes, classes)
plt.ylim(0, 1.05)
plt.xlabel("Class")
plt.ylabel("Per-Class Accuracy")
plt.title("Per-Class Accuracy (Raw vs LabelProp)")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


# 9.8 Cosine alignment histogram (probs vs LP probs)
@torch.no_grad()
def cosine_alignment_hist(model: BetterGAT, data):
    spacer("Cosine Alignment: Probs vs Label-Prop Probs")
    _, log_probs = model(data.x, data.edge_index, training=False)
    probs = log_probs.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    cos_sim = F.cosine_similarity(probs, lp_probs, dim=1).numpy()

    plt.figure(figsize=(8, 4))
    plt.hist(cos_sim, bins=30, color="#1f77b4", alpha=0.85)
    plt.xlabel("Cosine Similarity")
    plt.ylabel("Number of Nodes")
    plt.title("Cosine Alignment between GAT Probs and Label-Prop Probs")
    plt.grid(alpha=0.4)
    plt.tight_layout()
    plt.show()


cosine_alignment_hist(model, data)


# 9.9 Ego-graph around a misclassified test node
@torch.no_grad()
def misclassified_ego_graph(G, data, pred):
    spacer("Ego-Graph Around a Misclassified Test Node")
    mask = data.test_mask & (pred != data.y)
    idx = torch.where(mask)[0]
    if idx.numel() == 0:
        print("No misclassified test nodes, skipping ego-graph.")
        return

    node_id = int(idx[0])
    print(f"Showing ego-graph for misclassified node {node_id}")

    # Ego-graph radius 2
    H = nx.ego_graph(G, node_id, radius=2)
    pos = nx.spring_layout(H, seed=42)

    colors = []
    for n in H.nodes():
        if not bool(data.test_mask[n]):
            colors.append("lightgrey")
        else:
            colors.append(
                "tab:red" if pred[n] != data.y[n] else "tab:green"
            )

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        H,
        pos=pos,
        node_color=colors,
        node_size=80,
        edge_color="lightgrey",
        with_labels=False,
    )
    plt.title("Ego-Graph (Red = misclassified test nodes, Green = correct)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


misclassified_ego_graph(G, data, preds_raw)

print("\n✅ All visuals rendered.")

In [ ]:
# ============================================ #
# Graph Attention Networks +++
# (Cosine-Aligned Aux Loss + DropEdge + Heavy Visuals)
# ============================================ #
# Includes:
#  - Annealed DropEdge regularization
#  - Residual MLP refinement head
#  - Cosine-aligned label-consistency auxiliary loss
#  - Log-degree feature augmentation + scaling
#  - 3-layer GATv2 backbone w/ LayerNorm
#  - Long training + cosine LR decay + grad clipping
#  - Visuals:
#       * Graph + degree distribution
#       * Learning curves (acc) + Loss curves
#       * t-SNE (untrained vs trained, static)
#       * Accuracy-by-degree
#       * Attention heatmap
#       * Confusion matrices
#       * Per-class accuracy (raw vs LP)
#       * Cosine alignment histogram
#       * Ego-graph around a misclassified node
#       * Confidence histogram (test)
#       * Margin distribution (test)
#       * Class frequency vs per-class accuracy
#       * 🔥 Interactive 3D graph (Plotly)
#       * 🔥 Interactive 2D t-SNE (Plotly)
# ============================================ #

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

import plotly.graph_objs as go

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
)
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ----------------------- #
# 0) Reproducibility
# ----------------------- #
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

# 🔒 Force everything to CPU to avoid device mismatches
device = torch.device("cpu")
print("Device:", device)

# ----------------------- #
# 1) Dataset + basic stats
# ----------------------- #
dataset = Planetoid(root="./data", name="CiteSeer")
data = dataset[0]  # single graph on CPU

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Stabilize structure for attention
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# ----------------------- #
# 2) Quick structure visuals
# ----------------------- #
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y,
    width=0.5,
    edge_color="grey",
)
plt.title("CiteSeer Graph (Node-colored by Label)")
plt.show()

degrees_arr = degree(data.edge_index[0]).numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.show()

# ----------------------- #
# 3) Feature scaling + log-degree augmentation
# ----------------------- #
scaler = StandardScaler()
x_scaled = scaler.fit_transform(data.x.numpy())

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

x_aug = np.concatenate([x_scaled, log_deg], axis=1)
data.x = torch.tensor(x_aug, dtype=torch.float32)

in_dim = data.x.size(1)
print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

# ----------------------- #
# 4) Annealed DropEdge
# ----------------------- #
def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


# ----------------------- #
# 5) GATv2 Block
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


# ----------------------- #
# 6) GAT+++ Model
# ----------------------- #
class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(8, 8, 4),
        dropout: float = 0.6,
    ):
        super().__init__()

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # residual head for low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        self.opt = torch.optim.Adam(self.parameters(), lr=0.003, weight_decay=5e-4)
        self.sch = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=200)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ----------------------- #
# 7) Training / Eval (with cosine-aligned aux loss)
# ----------------------- #
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return (pred.eq(y).sum() / y.numel()).item()


def train(
    model: BetterGAT,
    data,
    epochs: int = 3000,
    clip: float = 1.0,
    consistency_w: float = 0.2,
    base_dropedge: float = 0.2,
):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    print(f"\n🧠 Training for {epochs} epochs (DropEdge + cosine LP consistency)...\n")
    print(
        "Note: cosine-aligned auxiliary loss acts as a second compass\n"
        "      for the attention heads when neighbourhoods get thin —\n"
        "      dual-signal routing to expose real structure vs pattern-matching.\n"
    )

    for ep in range(1, epochs + 1):
        model.train()
        model.opt.zero_grad()

        # Annealed DropEdge
        edge_index_aug = dropedge(data.edge_index, ep, epochs, base_p=base_dropedge)

        # Forward
        _, log_probs = model(data.x, edge_index_aug, training=True)

        # CE loss on labeled nodes
        ce_loss = ce(log_probs[data.train_mask], data.y[data.train_mask])

        # Label propagation on same device
        with torch.no_grad():
            probs = log_probs.exp()
            lp_probs = lp_layer(probs, data.edge_index)

        # Cosine-aligned auxiliary loss ("second compass")
        cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
        consistency = 1.0 - cos_sim.mean()

        loss = ce_loss + consistency_w * consistency
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        model.opt.step()
        model.sch.step()

        # Eval on val
        model.eval()
        with torch.no_grad():
            _, log_probs_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(
                log_probs_val[data.val_mask], data.y[data.val_mask]
            ).item()
            tr_acc = accuracy(
                log_probs[data.train_mask].argmax(1),
                data.y[data.train_mask],
            )
            val_acc = accuracy(
                log_probs_val[data.val_mask].argmax(1),
                data.y[data.val_mask],
            )

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_loss)

        if ep % 50 == 0 or ep == 1:
            lr = model.opt.param_groups[0]["lr"]
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss: {loss:.3f} | "
                f"ValLoss: {val_loss:.3f} | "
                f"TrainAcc: {tr_acc*100:5.2f}% | "
                f"ValAcc: {val_acc*100:5.2f}% | "
                f"LR: {lr:.5f}"
            )

    print("\n✅ Training complete.")
    print(
        "\nNow this setup is ready to be pushed into harsher regimes —\n"
        "extreme sparsity, label starvation, corrupted edges — to map\n"
        "out its true failure geometry. That’s usually where the good\n"
        "stories hide.\n"
    )
    return model, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve


@torch.no_grad()
def test(model: BetterGAT, data, use_lp: bool = False):
    model.eval()
    _, log_probs = model(data.x, data.edge_index, training=False)

    if use_lp:
        probs = log_probs.exp()
        lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
        lp_probs = lp_layer(probs, data.edge_index)
        pred = lp_probs.argmax(1)
    else:
        pred = log_probs.argmax(1)

    return accuracy(pred[data.test_mask], data.y[data.test_mask]), pred


# ----------------------- #
# 8) Run training
# ----------------------- #
model = BetterGAT(in_dim, 8, dataset.num_classes)
print(model)

model, train_acc, val_acc, train_loss, val_loss = train(
    model,
    data,
    epochs=3000,
    clip=1.0,
    consistency_w=0.2,
    base_dropedge=0.2,
)

raw_acc, preds_raw = test(model, data, use_lp=False)
lp_acc, preds_lp = test(model, data, use_lp=True)

print(f"📈 Test Accuracy (Raw):       {raw_acc*100:.2f}%")
print(f"📈 Test Accuracy (LabelProp): {lp_acc*100:.2f}%")

# ----------------------- #
# 9) Visual helpers
# ----------------------- #
def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.5)


# 9.1 Accuracy curves
spacer("Learning Curves (Train vs Val Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(train_acc, label="Train Acc", color="#1f77b4")
plt.plot(val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Learning Curves (BetterGAT + Cosine Consistency)")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 9.2 Loss curves
spacer("Loss Curves (Train vs Val)")
plt.figure(figsize=(9, 4))
plt.plot(train_loss, label="Train Loss", color="#1f77b4")
plt.plot(val_loss, label="Val Loss", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves (BetterGAT + Cosine Consistency)")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()


# 9.3 t-SNE (untrained vs trained)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained)")

    untrained = BetterGAT(in_dim, 8, dataset.num_classes)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained GAT Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.numpy(), s=8)
    plt.axis("off")
    plt.title("Trained BetterGAT Embeddings")

    plt.tight_layout()
    plt.show()


tsne_static(model, data)


# 9.4 Accuracy-by-degree
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree")
    _, log_probs = model(data.x, data.edge_index, training=False)
    pred = log_probs.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


accuracy_by_degree(model, data)


# 9.5 Attention heatmap
@torch.no_grad()
def attention_heatmap(model: BetterGAT, data, node_id: int):
    spacer(f"Attention Heatmap for Node {node_id}")
    _, att = model.g1.conv(
        data.x,
        data.edge_index,
        return_attention_weights=True,
    )
    edge_idx, alpha = att
    alpha = alpha.mean(1)

    mask = edge_idx[0] == node_id
    nbrs = edge_idx[1, mask].numpy()
    weights = alpha[mask].numpy()

    if len(nbrs) == 0:
        print("No neighbors for this node.")
        return

    idx = np.argsort(weights)[::-1]
    nbrs, weights = nbrs[idx], weights[idx]

    plt.figure(figsize=(8, max(1.6, 0.25 * len(nbrs))))
    plt.barh(range(len(nbrs)), weights, color="skyblue")
    plt.yticks(range(len(nbrs)), [str(n) for n in nbrs])
    plt.gca().invert_yaxis()
    plt.xlabel("Attention Weight αᵢⱼ")
    plt.title(f"Node {node_id} → Neighbor Attention")
    plt.tight_layout()
    plt.show()


test_nodes = torch.where(data.test_mask)[0].tolist()
if test_nodes:
    attention_heatmap(model, data, int(test_nodes[0]))


# 9.6 Confusion matrices (and capture them for per-class accuracy)
@torch.no_grad()
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)"):
    spacer(title)
    y_true = data.y[data.test_mask].numpy()
    y_pred = pred[data.test_mask].numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    plt.show()

    return cm


cm_raw = plot_confusion(data, preds_raw, "Confusion Matrix (Raw Predictions, Test Mask)")
cm_lp = plot_confusion(data, preds_lp, "Confusion Matrix (Label-Prop Predictions, Test Mask)")


# 9.7 Per-class accuracy (Raw vs LabelProp)
def per_class_accuracy_from_cm(cm):
    n_classes = cm.shape[0]
    accs = []
    for c in range(n_classes):
        total = cm[c].sum()
        accs.append(cm[c, c] / total if total > 0 else 0.0)
    return accs


spacer("Per-Class Accuracy (Raw vs LabelProp)")
raw_per_class = per_class_accuracy_from_cm(cm_raw)
lp_per_class = per_class_accuracy_from_cm(cm_lp)

classes = np.arange(dataset.num_classes)
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(classes - width / 2, raw_per_class, width=width, label="Raw", color="#1f77b4")
plt.bar(classes + width / 2, lp_per_class, width=width, label="LabelProp", color="#ff7f0e")
plt.xticks(classes, classes)
plt.ylim(0, 1.05)
plt.xlabel("Class")
plt.ylabel("Per-Class Accuracy")
plt.title("Per-Class Accuracy (Raw vs LabelProp)")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


# 9.8 Cosine alignment histogram (probs vs LP probs)
@torch.no_grad()
def cosine_alignment_hist(model: BetterGAT, data):
    spacer("Cosine Alignment: Probs vs Label-Prop Probs")
    _, log_probs = model(data.x, data.edge_index, training=False)
    probs = log_probs.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    cos_sim = F.cosine_similarity(probs, lp_probs, dim=1).numpy()

    plt.figure(figsize=(8, 4))
    plt.hist(cos_sim, bins=30, color="#1f77b4", alpha=0.85)
    plt.xlabel("Cosine Similarity")
    plt.ylabel("Number of Nodes")
    plt.title("Cosine Alignment between GAT Probs and Label-Prop Probs")
    plt.grid(alpha=0.4)
    plt.tight_layout()
    plt.show()


cosine_alignment_hist(model, data)


# 9.9 Ego-graph around a misclassified test node
@torch.no_grad()
def misclassified_ego_graph(G, data, pred):
    spacer("Ego-Graph Around a Misclassified Test Node")
    mask = data.test_mask & (pred != data.y)
    idx = torch.where(mask)[0]
    if idx.numel() == 0:
        print("No misclassified test nodes, skipping ego-graph.")
        return

    node_id = int(idx[0])
    print(f"Showing ego-graph for misclassified node {node_id}")

    # Ego-graph radius 2
    H = nx.ego_graph(G, node_id, radius=2)
    pos = nx.spring_layout(H, seed=42)

    colors = []
    for n in H.nodes():
        if not bool(data.test_mask[n]):
            colors.append("lightgrey")
        else:
            colors.append(
                "tab:red" if pred[n] != data.y[n] else "tab:green"
            )

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        H,
        pos=pos,
        node_color=colors,
        node_size=80,
        edge_color="lightgrey",
        with_labels=False,
    )
    plt.title("Ego-Graph (Red = misclassified test nodes, Green = correct)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


misclassified_ego_graph(G, data, preds_raw)


# 9.10 Confidence + margin distributions on test nodes
@torch.no_grad()
def confidence_and_margin_plots(model: BetterGAT, data):
    spacer("Confidence & Margin Distributions (Test Mask)")
    _, log_probs = model(data.x, data.edge_index, training=False)
    probs = log_probs.exp()
    test_mask = data.test_mask

    test_probs = probs[test_mask]
    top_vals, _ = test_probs.max(dim=1)
    # top-2 margins
    sorted_vals, _ = torch.sort(test_probs, dim=1, descending=True)
    margins = (sorted_vals[:, 0] - sorted_vals[:, 1]).numpy()

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.hist(top_vals.numpy(), bins=20, color="#1f77b4", alpha=0.9)
    plt.xlabel("Top-1 confidence")
    plt.ylabel("Count")
    plt.title("Top-1 Confidence (Test Nodes)")
    plt.grid(alpha=0.4)

    plt.subplot(1, 2, 2)
    plt.hist(margins, bins=20, color="#ff7f0e", alpha=0.9)
    plt.xlabel("Top-1 - Top-2 margin")
    plt.ylabel("Count")
    plt.title("Decision Margin (Test Nodes)")
    plt.grid(alpha=0.4)

    plt.tight_layout()
    plt.show()


confidence_and_margin_plots(model, data)


# 9.11 Class frequency vs per-class accuracy
@torch.no_grad()
def class_freq_vs_accuracy(data, cm, title_suffix="Raw"):
    spacer(f"Class Frequency vs Per-Class Accuracy ({title_suffix})")
    y_true = data.y[data.test_mask].numpy()
    class_counts = np.bincount(y_true, minlength=dataset.num_classes)
    per_class_acc = per_class_accuracy_from_cm(cm)

    fig, ax1 = plt.subplots(figsize=(8, 4))
    classes = np.arange(dataset.num_classes)

    ax1.bar(classes, class_counts, alpha=0.6, label="Frequency", color="#1f77b4")
    ax1.set_xlabel("Class")
    ax1.set_ylabel("Frequency", color="#1f77b4")

    ax2 = ax1.twinx()
    ax2.plot(classes, per_class_acc, "o-", color="#ff7f0e", label="Accuracy")
    ax2.set_ylabel("Accuracy", color="#ff7f0e")
    ax2.set_ylim(0, 1.05)

    plt.title(f"Class Frequency vs Per-Class Accuracy ({title_suffix})")
    fig.tight_layout()
    plt.show()


class_freq_vs_accuracy(data, cm_raw, "Raw")
class_freq_vs_accuracy(data, cm_lp, "LabelProp")


# 9.12 🔥 Interactive 3D graph (Plotly)
@torch.no_grad()
def interactive_3d_graph(G, data, pred, title="Interactive 3D Graph (Predictions)"):
    spacer(title)
    # 3D spring layout
    pos_3d = nx.spring_layout(G, seed=42, dim=3)

    Xn, Yn, Zn = [], [], []
    node_text = []
    for i in range(data.num_nodes):
        x, y, z = pos_3d[i]
        Xn.append(x)
        Yn.append(y)
        Zn.append(z)
        node_text.append(f"Node {i} | True {int(data.y[i])} | Pred {int(pred[i])}")

    Xe, Ye, Ze = [], [], []
    for u, v in G.edges():
        x0, y0, z0 = pos_3d[u]
        x1, y1, z1 = pos_3d[v]
        Xe += [x0, x1, None]
        Ye += [y0, y1, None]
        Ze += [z0, z1, None]

    edge_trace = go.Scatter3d(
        x=Xe,
        y=Ye,
        z=Ze,
        mode="lines",
        line=dict(color="rgba(150,150,150,0.35)", width=1),
        hoverinfo="none",
    )
    node_trace = go.Scatter3d(
        x=Xn,
        y=Yn,
        z=Zn,
        mode="markers",
        marker=dict(
            size=4,
            color=pred.numpy(),
            colorscale="Rainbow",
            opacity=0.9,
        ),
        text=node_text,
        hoverinfo="text",
    )

    layout = go.Layout(
        title=title,
        height=720,
        margin=dict(l=0, r=0, b=0, t=50),
        showlegend=False,
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
        ),
    )

    fig = go.Figure(data=[edge_trace, node_trace], layout=layout)
    fig.show()


interactive_3d_graph(G, data, preds_raw, title="Interactive 3D Graph (Raw Predictions)")


# 9.13 🔥 Interactive 2D t-SNE (Plotly)
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, title="Interactive t-SNE (Trained Embeddings)"):
    spacer(title)
    emb, _ = model(data.x, data.edge_index, training=False)
    tsne_emb = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb.numpy())

    x_tsne = tsne_emb[:, 0]
    y_tsne = tsne_emb[:, 1]

    node_text = [
        f"Node {i} | True {int(data.y[i])}"
        for i in range(data.num_nodes)
    ]

    trace = go.Scatter(
        x=x_tsne,
        y=y_tsne,
        mode="markers",
        marker=dict(
            size=6,
            color=data.y.numpy(),
            colorscale="Viridis",
            showscale=True,
            opacity=0.9,
        ),
        text=node_text,
        hoverinfo="text",
    )

    layout = go.Layout(
        title=title,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        height=700,
        margin=dict(l=0, r=0, b=0, t=50),
    )

    fig = go.Figure(data=[trace], layout=layout)
    fig.show()


interactive_tsne(model, data, title="Interactive t-SNE (BetterGAT Embeddings)")

print("\n✅ All visuals rendered (static + interactive).")

In [ ]:
# ============================================ #
# Graph Attention Networks +++ (CiteSeer Lab)
# ============================================ #
# Features:
#   • Log-degree + PageRank feature augmentation
#   • Annealed DropEdge + Random Feature Masking
#   • 3-layer GATv2 backbone with LayerNorm & residuals
#   • Optional DropPath (stochastic depth) in GAT blocks
#   • Cosine-aligned LabelPropagation auxiliary loss
#   • Cosine LR schedule + grad clipping
#   • Diagnostics:
#       - Graph + degree distribution
#       - Train/Val accuracy + loss curves
#       - t-SNE (untrained vs trained)
#       - Accuracy by node degree
#       - Attention heatmap (GAT head)
#       - Confusion matrices + per-class accuracy
#       - Confidence & margin distributions (test)
#       - Cosine alignment histogram (probs vs LP)
#       - Ego-graph around misclassified node
#       - Interactive 3D graph & 2D t-SNE (Plotly)
# ============================================ #

# If in Colab, uncomment:
!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

import plotly.graph_objs as go

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
)
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn.models.label_prop import LabelPropagation

# ----------------------- #
# 0) Reproducibility
# ----------------------- #
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

device = torch.device("cpu")   # keep on CPU for stability/portability
print("Device:", device)

# ----------------------- #
# 1) Dataset + basic stats
# ----------------------- #
dataset = Planetoid(root="./data", name="CiteSeer")
data = dataset[0]  # single homogenous graph

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Stabilize structure for attention (undirected + self-loops)
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# Build networkx graph once (for PageRank + ego-graphs)
G = to_networkx(data, to_undirected=True)

# ----------------------- #
# 2) Structure visuals
# ----------------------- #
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y,
    width=0.5,
    edge_color="grey",
)
plt.title("CiteSeer Graph (Node-colored by Label)")
plt.show()

degrees_arr = degree(data.edge_index[0]).numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.show()

# ----------------------- #
# 3) Feature scaling + structural augmentation
# ----------------------- #
# Base features -> StandardScaler
scaler = StandardScaler()
x_scaled = scaler.fit_transform(data.x.numpy())

# Log-degree
deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

# PageRank (normalized)
pr_dict = nx.pagerank(G, alpha=0.85)
pr_vals = np.array([pr_dict[i] for i in range(data.num_nodes)]).reshape(-1, 1)
pr_vals = (pr_vals - pr_vals.mean()) / (pr_vals.std() + 1e-8)

# Concatenate: [scaled_x, log_deg, pagerank]
x_aug = np.concatenate([x_scaled, log_deg, pr_vals], axis=1)
data.x = torch.tensor(x_aug, dtype=torch.float32)

in_dim = data.x.size(1)
print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

# ----------------------- #
# 4) Regularization helpers
# ----------------------- #
def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Annealed DropEdge:
      p_start = base_p, linearly decays to 0 by final epoch.
      Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]

def random_feature_mask(x: torch.Tensor,
                        p: float = 0.1,
                        training: bool = True) -> torch.Tensor:
    """
    Simple feature dropout: randomly zero-out feature dims
    for all nodes (same mask across nodes).
    """
    if (not training) or p <= 0:
        return x
    mask = (torch.rand(x.size(1)) > p).float()
    return x * mask

class DropPath(nn.Module):
    """
    Stochastic depth: drop entire residual branch with prob p.
    """
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.dim() - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor

# ----------------------- #
# 5) GATv2 Block with LayerNorm + DropPath
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
        droppath_prob: float = 0.0,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)
        self.droppath = DropPath(droppath_prob) if self.use_res and droppath_prob > 0 else nn.Identity()

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = x + self.droppath(h)
        return h

# ----------------------- #
# 6) BetterGAT model
# ----------------------- #
class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(8, 8, 4),
        dropout: float = 0.6,
    ):
        super().__init__()

        # depth-dependent droppath (stochastic depth)
        dp1, dp2, dp3 = 0.0, 0.05, 0.1

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False, droppath_prob=dp1)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False, droppath_prob=dp2)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False, droppath_prob=dp3)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # residual head for low-degree / structurally weak nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        self.opt = torch.optim.Adam(self.parameters(), lr=0.003, weight_decay=5e-4)
        self.sch = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=200)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        # Random feature masking as extra regularization
        x = random_feature_mask(x, p=0.1, training=training)

        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)

# ----------------------- #
# 7) Training / Eval with cosine LP aux loss
# ----------------------- #
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return (pred.eq(y).sum() / y.numel()).item()

def train(
    model: BetterGAT,
    data,
    epochs: int = 2000,
    clip: float = 1.0,
    consistency_w: float = 0.2,
    base_dropedge: float = 0.2,
):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    print(f"\n🧠 Training for {epochs} epochs (DropEdge + cosine LP consistency)...\n")
    print(
        "Cosine-aligned aux loss: forces GAT probabilities to stay aligned\n"
        "with Label Propagation on the same graph – a second geometric\n"
        "signal when neighbourhoods become too thin.\n"
    )

    for ep in range(1, epochs + 1):
        model.train()
        model.opt.zero_grad()

        # Annealed DropEdge
        edge_index_aug = dropedge(data.edge_index, ep, epochs, base_p=base_dropedge)

        # Forward
        _, log_probs = model(data.x, edge_index_aug, training=True)

        # CE loss on labeled nodes
        ce_loss = ce(log_probs[data.train_mask], data.y[data.train_mask])

        # Label propagation on same graph
        with torch.no_grad():
            probs = log_probs.exp()
            lp_probs = lp_layer(probs, data.edge_index)

        # Cosine-aligned auxiliary loss ("second compass")
        cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
        consistency = 1.0 - cos_sim.mean()

        loss = ce_loss + consistency_w * consistency
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        model.opt.step()
        model.sch.step()

        # Eval on val
        model.eval()
        with torch.no_grad():
            _, log_probs_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(
                log_probs_val[data.val_mask], data.y[data.val_mask]
            ).item()
            tr_acc = accuracy(
                log_probs[data.train_mask].argmax(1),
                data.y[data.train_mask],
            )
            val_acc = accuracy(
                log_probs_val[data.val_mask].argmax(1),
                data.y[data.val_mask],
            )

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_loss)

        if ep % 50 == 0 or ep == 1:
            lr = model.opt.param_groups[0]["lr"]
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss: {loss:.3f} | "
                f"ValLoss: {val_loss:.3f} | "
                f"TrainAcc: {tr_acc*100:5.2f}% | "
                f"ValAcc: {val_acc*100:5.2f}% | "
                f"LR: {lr:.5f}"
            )

    print("\n✅ Training complete.\n")
    return model, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve

@torch.no_grad()
def test(model: BetterGAT, data, use_lp: bool = False):
    model.eval()
    _, log_probs = model(data.x, data.edge_index, training=False)

    if use_lp:
        probs = log_probs.exp()
        lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
        lp_probs = lp_layer(probs, data.edge_index)
        pred = lp_probs.argmax(1)
    else:
        pred = log_probs.argmax(1)

    return accuracy(pred[data.test_mask], data.y[data.test_mask]), pred

# ----------------------- #
# 8) Run training + test
# ----------------------- #
model = BetterGAT(in_dim, 8, dataset.num_classes)
print(model)

model, train_acc, val_acc, train_loss, val_loss = train(
    model,
    data,
    epochs=2000,          # can bump to 3000 if patient
    clip=1.0,
    consistency_w=0.2,
    base_dropedge=0.2,
)

raw_acc, preds_raw = test(model, data, use_lp=False)
lp_acc, preds_lp = test(model, data, use_lp=True)

print(f"📈 Test Accuracy (Raw):       {raw_acc*100:.2f}%")
print(f"📈 Test Accuracy (LabelProp): {lp_acc*100:.2f}%")

# ----------------------- #
# 9) Visual helpers
# ----------------------- #
def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.4)

# 9.1 Accuracy curves
spacer("Learning Curves (Train vs Val Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(train_acc, label="Train Acc", color="#1f77b4")
plt.plot(val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Learning Curves (BetterGAT + Cosine Consistency)")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 9.2 Loss curves
spacer("Loss Curves (Train vs Val)")
plt.figure(figsize=(9, 4))
plt.plot(train_loss, label="Train Loss", color="#1f77b4")
plt.plot(val_loss, label="Val Loss", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves (BetterGAT + Cosine Consistency)")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 9.3 t-SNE (untrained vs trained)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained)")

    untrained = BetterGAT(in_dim, 8, dataset.num_classes)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained GAT Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.numpy(), s=8)
    plt.axis("off")
    plt.title("Trained BetterGAT Embeddings")

    plt.tight_layout()
    plt.show()

tsne_static(model, data)

# 9.4 Accuracy-by-degree
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree")
    _, log_probs = model(data.x, data.edge_index, training=False)
    pred = log_probs.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

accuracy_by_degree(model, data)

# 9.5 Attention heatmap
@torch.no_grad()
def attention_heatmap(model: BetterGAT, data, node_id: int):
    spacer(f"Attention Heatmap for Node {node_id}")
    _, att = model.g1.conv(
        data.x,
        data.edge_index,
        return_attention_weights=True,
    )
    edge_idx, alpha = att
    alpha = alpha.mean(1)

    mask = edge_idx[0] == node_id
    nbrs = edge_idx[1, mask].numpy()
    weights = alpha[mask].numpy()

    if len(nbrs) == 0:
        print("No neighbors for this node.")
        return

    idx = np.argsort(weights)[::-1]
    nbrs, weights = nbrs[idx], weights[idx]

    plt.figure(figsize=(8, max(1.6, 0.25 * len(nbrs))))
    plt.barh(range(len(nbrs)), weights, color="skyblue")
    plt.yticks(range(len(nbrs)), [str(n) for n in nbrs])
    plt.gca().invert_yaxis()
    plt.xlabel("Attention Weight αᵢⱼ")
    plt.title(f"Node {node_id} → Neighbor Attention")
    plt.tight_layout()
    plt.show()

test_nodes = torch.where(data.test_mask)[0].tolist()
if test_nodes:
    attention_heatmap(model, data, int(test_nodes[0]))

# 9.6 Confusion matrices (for per-class metrics)
@torch.no_grad()
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)"):
    spacer(title)
    y_true = data.y[data.test_mask].numpy()
    y_pred = pred[data.test_mask].numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    plt.show()

    return cm

cm_raw = plot_confusion(data, preds_raw, "Confusion Matrix (Raw Predictions, Test Mask)")
cm_lp = plot_confusion(data, preds_lp, "Confusion Matrix (Label-Prop Predictions, Test Mask)")

def per_class_accuracy_from_cm(cm):
    n_classes = cm.shape[0]
    accs = []
    for c in range(n_classes):
        total = cm[c].sum()
        accs.append(cm[c, c] / total if total > 0 else 0.0)
    return accs

# 9.7 Per-class accuracy (Raw vs LabelProp)
spacer("Per-Class Accuracy (Raw vs LabelProp)")
raw_per_class = per_class_accuracy_from_cm(cm_raw)
lp_per_class = per_class_accuracy_from_cm(cm_lp)

classes = np.arange(dataset.num_classes)
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(classes - width / 2, raw_per_class, width=width, label="Raw", color="#1f77b4")
plt.bar(classes + width / 2, lp_per_class, width=width, label="LabelProp", color="#ff7f0e")
plt.xticks(classes, classes)
plt.ylim(0, 1.05)
plt.xlabel("Class")
plt.ylabel("Per-Class Accuracy")
plt.title("Per-Class Accuracy (Raw vs LabelProp)")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# 9.8 Cosine alignment histogram (probs vs LP probs)
@torch.no_grad()
def cosine_alignment_hist(model: BetterGAT, data):
    spacer("Cosine Alignment: Probs vs Label-Prop Probs")
    _, log_probs = model(data.x, data.edge_index, training=False)
    probs = log_probs.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    cos_sim = F.cosine_similarity(probs, lp_probs, dim=1).numpy()

    plt.figure(figsize=(8, 4))
    plt.hist(cos_sim, bins=30, color="#1f77b4", alpha=0.85)
    plt.xlabel("Cosine Similarity")
    plt.ylabel("Number of Nodes")
    plt.title("Cosine Alignment between GAT Probs and Label-Prop Probs")
    plt.grid(alpha=0.4)
    plt.tight_layout()
    plt.show()

cosine_alignment_hist(model, data)

# 9.9 Ego-graph around a misclassified test node
@torch.no_grad()
def misclassified_ego_graph(G, data, pred):
    spacer("Ego-Graph Around a Misclassified Test Node")
    mask = data.test_mask & (pred != data.y)
    idx = torch.where(mask)[0]
    if idx.numel() == 0:
        print("No misclassified test nodes, skipping ego-graph.")
        return

    node_id = int(idx[0])
    print(f"Showing ego-graph for misclassified node {node_id}")

    H = nx.ego_graph(G, node_id, radius=2)
    pos = nx.spring_layout(H, seed=42)

    colors = []
    for n in H.nodes():
        if not bool(data.test_mask[n]):
            colors.append("lightgrey")
        else:
            colors.append("tab:red" if pred[n] != data.y[n] else "tab:green")

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        H,
        pos=pos,
        node_color=colors,
        node_size=80,
        edge_color="lightgrey",
        with_labels=False,
    )
    plt.title("Ego-Graph (Red = misclassified test nodes, Green = correct)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

misclassified_ego_graph(G, data, preds_raw)

# 9.10 Confidence + margin distributions on test nodes
@torch.no_grad()
def confidence_and_margin_plots(model: BetterGAT, data):
    spacer("Confidence & Margin Distributions (Test Mask)")
    _, log_probs = model(data.x, data.edge_index, training=False)
    probs = log_probs.exp()
    test_mask = data.test_mask

    test_probs = probs[test_mask]
    top_vals, _ = test_probs.max(dim=1)
    sorted_vals, _ = torch.sort(test_probs, dim=1, descending=True)
    margins = (sorted_vals[:, 0] - sorted_vals[:, 1]).numpy()

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.hist(top_vals.numpy(), bins=20, color="#1f77b4", alpha=0.9)
    plt.xlabel("Top-1 confidence")
    plt.ylabel("Count")
    plt.title("Top-1 Confidence (Test Nodes)")
    plt.grid(alpha=0.4)

    plt.subplot(1, 2, 2)
    plt.hist(margins, bins=20, color="#ff7f0e", alpha=0.9)
    plt.xlabel("Top-1 - Top-2 margin")
    plt.ylabel("Count")
    plt.title("Decision Margin (Test Nodes)")
    plt.grid(alpha=0.4)

    plt.tight_layout()
    plt.show()

confidence_and_margin_plots(model, data)

# 9.11 Class frequency vs per-class accuracy
@torch.no_grad()
def class_freq_vs_accuracy(data, cm, title_suffix="Raw"):
    spacer(f"Class Frequency vs Per-Class Accuracy ({title_suffix})")
    y_true = data.y[data.test_mask].numpy()
    class_counts = np.bincount(y_true, minlength=dataset.num_classes)
    per_class_acc = per_class_accuracy_from_cm(cm)

    fig, ax1 = plt.subplots(figsize=(8, 4))
    classes = np.arange(dataset.num_classes)

    ax1.bar(classes, class_counts, alpha=0.6, label="Frequency", color="#1f77b4")
    ax1.set_xlabel("Class")
    ax1.set_ylabel("Frequency", color="#1f77b4")

    ax2 = ax1.twinx()
    ax2.plot(classes, per_class_acc, "o-", color="#ff7f0e", label="Accuracy")
    ax2.set_ylabel("Accuracy", color="#ff7f0e")
    ax2.set_ylim(0, 1.05)

    plt.title(f"Class Frequency vs Per-Class Accuracy ({title_suffix})")
    fig.tight_layout()
    plt.show()

class_freq_vs_accuracy(data, cm_raw, "Raw")
class_freq_vs_accuracy(data, cm_lp, "LabelProp")

# 9.12 Interactive 3D graph (Plotly)
@torch.no_grad()
def interactive_3d_graph(G, data, pred, title="Interactive 3D Graph (Predictions)"):
    spacer(title)
    pos_3d = nx.spring_layout(G, seed=42, dim=3)

    Xn, Yn, Zn, node_text = [], [], [], []
    for i in range(data.num_nodes):
        x, y, z = pos_3d[i]
        Xn.append(x); Yn.append(y); Zn.append(z)
        node_text.append(f"Node {i} | True {int(data.y[i])} | Pred {int(pred[i])}")

    Xe, Ye, Ze = [], [], []
    for u, v in G.edges():
        x0, y0, z0 = pos_3d[u]
        x1, y1, z1 = pos_3d[v]
        Xe += [x0, x1, None]
        Ye += [y0, y1, None]
        Ze += [z0, z1, None]

    edge_trace = go.Scatter3d(
        x=Xe,
        y=Ye,
        z=Ze,
        mode="lines",
        line=dict(color="rgba(150,150,150,0.35)", width=1),
        hoverinfo="none",
    )
    node_trace = go.Scatter3d(
        x=Xn,
        y=Yn,
        z=Zn,
        mode="markers",
        marker=dict(size=4, color=pred.numpy(), colorscale="Rainbow", opacity=0.9),
        text=node_text,
        hoverinfo="text",
    )

    layout = go.Layout(
        title=title,
        height=720,
        margin=dict(l=0, r=0, b=0, t=50),
        showlegend=False,
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
        ),
    )

    fig = go.Figure(data=[edge_trace, node_trace], layout=layout)
    fig.show()

interactive_3d_graph(G, data, preds_raw, title="Interactive 3D Graph (Raw Predictions)")

# 9.13 Interactive 2D t-SNE (Plotly)
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, title="Interactive t-SNE (Trained Embeddings)"):
    spacer(title)
    emb, _ = model(data.x, data.edge_index, training=False)
    tsne_emb = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb.numpy())

    x_tsne = tsne_emb[:, 0]
    y_tsne = tsne_emb[:, 1]

    node_text = [f"Node {i} | True {int(data.y[i])}" for i in range(data.num_nodes)]

    trace = go.Scatter(
        x=x_tsne,
        y=y_tsne,
        mode="markers",
        marker=dict(
            size=6,
            color=data.y.numpy(),
            colorscale="Viridis",
            showscale=True,
            opacity=0.9,
        ),
        text=node_text,
        hoverinfo="text",
    )

    layout = go.Layout(
        title=title,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        height=700,
        margin=dict(l=0, r=0, b=0, t=50),
    )

    fig = go.Figure(data=[trace], layout=layout)
    fig.show()

interactive_tsne(model, data, title="Interactive t-SNE (BetterGAT Embeddings)")

print("\n✅ All visuals rendered (static + interactive).")

In [ ]:
# ============================================================= #
# Ultra-GAT Lab (CiteSeer/Cora/PubMed) – One-File Playground
# ============================================================= #
# Features:
#   * Config-driven: dataset, epochs, hidden dim, loss weights
#   * Log-degree + PageRank node features (toggle)
#   * Annealed DropEdge + feature masking + DropPath
#   * 3-layer BetterGAT backbone (teacher)
#   * Cosine-aligned Label Prop aux loss (toggle)
#   * Supervised contrastive loss on embeddings (toggle)
#   * Early stopping (best val) + checkpointing
#   * Student GCN with distillation from GAT teacher
#   * MC-Dropout uncertainty on test (entropy hist)
#   * Calibration: ECE + reliability diagram
#   * Diagnostics: t-SNE, confusion matrices, per-class acc, etc.
# ============================================================= #

# In Colab, uncomment:
# !pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

import plotly.graph_objs as go

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation

# ----------------------- #
# 0) CONFIG
# ----------------------- #
CONFIG = {
    "dataset_name": "CiteSeer",         # "Cora", "CiteSeer", or "PubMed"
    "teacher_hidden_dim": 8,
    "teacher_heads": (8, 8, 4),
    "epochs_teacher": 2000,
    "epochs_student": 1000,
    "dropedge_base_p": 0.2,
    "feature_mask_p": 0.1,
    "lp_aux_weight": 0.2,               # cosine-aligned LP loss weight
    "contrastive_weight": 0.1,          # supervised contrastive loss
    "temperature_contrastive": 0.5,
    "early_stopping_patience": 200,
    "use_struct_features": True,        # log-degree + PageRank
    "use_lp_aux": True,
    "use_contrastive": True,
    "mc_dropout_samples": 30,           # MC-dropout passes
    "distill_alpha": 0.7,               # KD: weight on soft teacher
    "distill_temperature": 3.0,
    "seed": 42,
}

# ----------------------- #
# 1) Reproducibility
# ----------------------- #
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(CONFIG["seed"])
device = torch.device("cpu")  # keep CPU-safe
print("Device:", device)

# ----------------------- #
# 2) Dataset + structural features
# ----------------------- #
dataset = Planetoid(root="./data", name=CONFIG["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Normalize edge structure
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)
G = to_networkx(data, to_undirected=True)

# Quick structure visuals
plt.figure(figsize=(7, 7))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=15,
    node_color=data.y,
    width=0.4,
    edge_color="grey",
)
plt.title(f"{CONFIG['dataset_name']} Graph (Nodes colored by label)")
plt.show()

degrees_arr = degree(data.edge_index[0]).numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(7, 3))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.show()

# ---- Feature scaling + structural aug ----
scaler = StandardScaler()
x_scaled = scaler.fit_transform(data.x.numpy())

if CONFIG["use_struct_features"]:
    deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
    log_deg = np.log1p(deg_np).reshape(-1, 1)
    pr_dict = nx.pagerank(G, alpha=0.85)
    pr_vals = np.array([pr_dict[i] for i in range(data.num_nodes)]).reshape(-1, 1)
    pr_vals = (pr_vals - pr_vals.mean()) / (pr_vals.std() + 1e-8)
    x_aug = np.concatenate([x_scaled, log_deg, pr_vals], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)
print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

# ----------------------- #
# 3) Regularisation helpers
# ----------------------- #
def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)
    return edge_index[:, keep]

def random_feature_mask(x: torch.Tensor,
                        p: float = 0.1,
                        training: bool = True) -> torch.Tensor:
    if (not training) or p <= 0:
        return x
    mask = (torch.rand(x.size(1)) > p).float()
    return x * mask

class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.dim() - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x / keep_prob * random_tensor

# ----------------------- #
# 4) GATv2 Blocks + Teacher Model
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
        droppath_prob: float = 0.0,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)
        self.droppath = DropPath(droppath_prob) if self.use_res and droppath_prob > 0 else nn.Identity()

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = x + self.droppath(h)
        return h

class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(8, 8, 4),
        dropout: float = 0.6,
        feature_mask_p: float = 0.1,
    ):
        super().__init__()
        dp1, dp2, dp3 = 0.0, 0.05, 0.1
        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False, droppath_prob=dp1)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False, droppath_prob=dp2)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False, droppath_prob=dp3)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )
        self.res_fc = nn.Linear(dim_in, dim_out)

        self.opt = torch.optim.Adam(self.parameters(), lr=0.003, weight_decay=5e-4)
        self.sch = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=200)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)
        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)
        return h, F.log_softmax(logits, dim=1)

# ----------------------- #
# 5) Student GCN (for distillation)
# ----------------------- #
class StudentGCN(nn.Module):
    def __init__(self, dim_in, dim_h, dim_out, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(dim_in, dim_h)
        self.conv2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x, edge_index, training=False):
        x = F.dropout(x, p=self.dropout, training=training)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# ----------------------- #
# 6) Losses: accuracy, contrastive, distillation
# ----------------------- #
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return (pred.eq(y).sum() / y.numel()).item()

def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask_nodes: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    """
    Simple supervised contrastive loss on embeddings.
    emb: [N, D], labels: [N], mask_nodes: bool mask for nodes to include.
    """
    idx = mask_nodes.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, dtype=emb.dtype)

    z = emb[idx]  # [M, D]
    y = labels[idx]
    z = F.normalize(z, dim=1)
    sim = torch.matmul(z, z.T) / temperature  # [M,M]

    # mask out self
    self_mask = torch.eye(sim.size(0), dtype=torch.bool)
    sim = sim.masked_fill(self_mask, -1e9)

    labels_matrix = y.unsqueeze(0) == y.unsqueeze(1)  # [M,M]
    positives = labels_matrix & (~self_mask)

    log_probs = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    loss = -log_probs[positives].mean() if positives.any() else torch.tensor(0.0)
    return loss

def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    true_y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float = 0.7,
    T: float = 3.0,
):
    """
    Student loss: alpha * KD + (1-alpha) * CE on hard labels.
    """
    ce = nn.NLLLoss()
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0)

    s = student_logp[idx]        # log-probs
    t = teacher_logp[idx]

    # CE with true labels
    ce_loss = ce(s, true_y[idx])

    # KL divergence between softened distributions
    s_T = F.log_softmax(s / T, dim=1)
    t_T = F.softmax(t / T, dim=1).detach()
    kl = F.kl_div(s_T, t_T, reduction="batchmean") * (T * T)

    return alpha * kl + (1 - alpha) * ce_loss

# ----------------------- #
# 7) Train TEACHER (BetterGAT) w/ aux losses + early stopping
# ----------------------- #
def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    best_state = None
    best_val_acc = -1.0
    best_epoch = -1
    patience = cfg["early_stopping_patience"]
    bad_count = 0

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]

    print(f"\n🧠 Training TEACHER for up to {E} epochs (early stopping, DropEdge, aux losses)...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)
        emb, logp = model(data.x, edge_index_aug, training=True)

        ce_loss = ce(logp[data.train_mask], data.y[data.train_mask])
        loss = ce_loss

        # Cosine LP auxiliary
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0)

        # Supervised contrastive on embeddings
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()
        model.sch.step()

        # Eval
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(logp_val[data.val_mask], data.y[data.val_mask]).item()
            tr_acc = accuracy(logp[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_val[data.val_mask].argmax(1), data.y[data.val_mask])

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_loss)

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad_count = 0
        else:
            bad_count += 1

        if ep % 50 == 0 or ep == 1:
            lr = model.opt.param_groups[0]["lr"]
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | Contr:{contr:.3f} | LR:{lr:.5f}"
            )

        if bad_count >= patience:
            print(f"\n⏹ Early stopping at epoch {ep}, best val at epoch {best_epoch} ({best_val_acc*100:.2f}%).")
            break

    print("\n✅ Teacher training complete. Restoring best checkpoint.\n")
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve

@torch.no_grad()
def eval_model(model, data, use_lp=False):
    model.eval()
    emb, logp = model(data.x, data.edge_index, training=False)
    if use_lp:
        probs = logp.exp()
        lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
        lp_probs = lp_layer(probs, data.edge_index)
        pred = lp_probs.argmax(1)
        return emb, logp, accuracy(pred[data.test_mask], data.y[data.test_mask]), pred
    else:
        pred = logp.argmax(1)
        return emb, logp, accuracy(pred[data.test_mask], data.y[data.test_mask]), pred

# ----------------------- #
# 8) Train TEACHER
# ----------------------- #
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=CONFIG["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=CONFIG["teacher_heads"],
    dropout=0.6,
    feature_mask_p=CONFIG["feature_mask_p"],
)
print(teacher)

teacher, tr_acc_t, val_acc_t, tr_loss_t, val_loss_t = train_teacher(teacher, data, CONFIG)

emb_raw, logp_raw, raw_acc, preds_raw = eval_model(teacher, data, use_lp=False)
emb_lp, logp_lp, lp_acc, preds_lp = eval_model(teacher, data, use_lp=True)

print(f"📈 Teacher Test Accuracy (Raw):       {raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {lp_acc*100:.2f}%")

# ----------------------- #
# 9) Train STUDENT with distillation
# ----------------------- #
def train_student(student: StudentGCN, teacher_logp: torch.Tensor, data, cfg):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    best_state = None
    best_val_acc = -1.0
    bad = 0
    patience = cfg["early_stopping_patience"] // 2

    train_acc_curve, val_acc_curve = [], []

    print(f"\n🎓 Training STUDENT GCN for up to {E} epochs (KD from teacher)...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        # KD loss
        kd = distillation_loss(logp_s, teacher_logp, data.y, data.train_mask,
                               alpha=alpha, T=T)
        kd.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()

        # Eval
        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(logp_s[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_sv[data.val_mask].argmax(1), data.y[data.val_mask])
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in student.state_dict().items()}
            bad = 0
        else:
            bad += 1

        if ep % 50 == 0 or ep == 1:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )
        if bad >= patience:
            print(f"\n⏹ Student early stopping at epoch {ep}.")
            break

    if best_state is not None:
        student.load_state_dict(best_state)
    return student, train_acc_curve, val_acc_curve

student = StudentGCN(in_dim, dim_h=16, dim_out=dataset.num_classes)
student, tr_acc_s, val_acc_s = train_student(student, logp_raw.detach(), data, CONFIG)

@torch.no_grad()
def eval_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    return logp, accuracy(pred[data.test_mask], data.y[data.test_mask]), pred

logp_s, stud_acc, preds_s = eval_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {stud_acc*100:.2f}%")

# ----------------------- #
# 10) Basic plots: teacher learning curves
# ----------------------- #
def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.4)

spacer("Teacher Learning Curves (Train vs Val Accuracy)")
plt.figure(figsize=(8, 3))
plt.plot(tr_acc_t, label="Train Acc")
plt.plot(val_acc_t, label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher: BetterGAT")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

spacer("Teacher Loss Curves (Train vs Val)")
plt.figure(figsize=(8, 3))
plt.plot(tr_loss_t, label="Train Loss")
plt.plot(val_loss_t, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher: BetterGAT")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# ----------------------- #
# 11) t-SNE (untrained vs trained teacher)
# ----------------------- #
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")
    untrained = BetterGAT(in_dim, CONFIG["teacher_hidden_dim"],
                          dataset.num_classes, heads=CONFIG["teacher_heads"])
    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(n_components=2, init="pca", learning_rate="auto").fit_transform(emb0.numpy())
    tsne1 = TSNE(n_components=2, init="pca", learning_rate="auto").fit_transform(emb1.numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")
    plt.tight_layout()
    plt.show()

tsne_static(teacher, data)

# ----------------------- #
# 12) Confusion matrices + per-class acc (teacher raw/LP)
# ----------------------- #
@torch.no_grad()
def plot_confusion(data, pred, title):
    spacer(title)
    y_true = data.y[data.test_mask].numpy()
    y_pred = pred[data.test_mask].numpy()
    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))
    plt.figure(figsize=(6, 4))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    plt.tight_layout()
    plt.show()
    return cm

cm_raw = plot_confusion(data, preds_raw,  "Teacher Confusion (Raw, Test)")
cm_lp  = plot_confusion(data, preds_lp,   "Teacher Confusion (LabelProp, Test)")
cm_stu = plot_confusion(data, preds_s,    "Student Confusion (KD, Test)")

def per_class_accuracy_from_cm(cm):
    n_classes = cm.shape[0]
    accs = []
    for c in range(n_classes):
        total = cm[c].sum()
        accs.append(cm[c, c] / total if total > 0 else 0.0)
    return accs

spacer("Per-Class Accuracy (Teacher Raw vs LP vs Student)")
raw_per = per_class_accuracy_from_cm(cm_raw)
lp_per  = per_class_accuracy_from_cm(cm_lp)
stu_per = per_class_accuracy_from_cm(cm_stu)

classes = np.arange(dataset.num_classes)
w = 0.25
plt.figure(figsize=(9, 4))
plt.bar(classes - w,     raw_per, width=w, label="Teacher Raw")
plt.bar(classes,         lp_per,  width=w, label="Teacher LP")
plt.bar(classes + w,     stu_per, width=w, label="Student KD")
plt.xticks(classes, classes)
plt.ylim(0, 1.05)
plt.xlabel("Class")
plt.ylabel("Accuracy")
plt.title("Per-Class Accuracy")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# ----------------------- #
# 13) Accuracy by degree (teacher raw)
# ----------------------- #
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 3))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")
    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)
    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

accuracy_by_degree(teacher, data)

# ----------------------- #
# 14) MC-Dropout uncertainty + calibration (Teacher)
# ----------------------- #
@torch.no_grad()
def mc_dropout_predictions(model: BetterGAT, data, mc_samples: int):
    model.train()   # keep dropout on
    T = mc_samples
    probs_list = []
    for _ in range(T):
        _, logp = model(data.x, data.edge_index, training=True)
        probs_list.append(logp.exp().unsqueeze(0))
    probs_mc = torch.cat(probs_list, dim=0)          # [T,N,C]
    probs_mean = probs_mc.mean(dim=0)                # [N,C]
    return probs_mc, probs_mean

def entropy(p: torch.Tensor, dim=-1):
    return -(p * (p.clamp_min(1e-12).log())).sum(dim=dim)

spacer("MC-Dropout Uncertainty (Teacher)")
probs_mc, probs_mean = mc_dropout_predictions(teacher, data, CONFIG["mc_dropout_samples"])
test_mask = data.test_mask
test_probs = probs_mean[test_mask]
test_entropy = entropy(test_probs, dim=1).numpy()

plt.figure(figsize=(7, 3))
plt.hist(test_entropy, bins=20, color="#1f77b4", alpha=0.9)
plt.xlabel("Predictive entropy")
plt.ylabel("Count")
plt.title("MC-Dropout Predictive Entropy (Test Nodes)")
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# Calibration: ECE + reliability diagram
@torch.no_grad()
def expected_calibration_error(probs, labels, n_bins=15):
    """
    probs: [N,C], labels: [N]
    """
    conf, predicted = probs.max(dim=1)
    labels = labels
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    bin_centers = (bins[:-1] + bins[1:]) / 2.0
    accs, confs, counts = [], [], []

    for i in range(n_bins):
        in_bin = (conf >= bins[i]) & (conf < bins[i+1])
        count = in_bin.sum().item()
        if count > 0:
            acc = (predicted[in_bin] == labels[in_bin]).float().mean().item()
            avg_conf = conf[in_bin].mean().item()
            ece += (count / len(conf)) * abs(acc - avg_conf)
            accs.append(acc); confs.append(avg_conf); counts.append(count)
        else:
            accs.append(0.0); confs.append(0.0); counts.append(0)
    return ece, bin_centers, accs, confs, counts

spacer("Calibration (Teacher Raw Probs)")
test_probs_raw = logp_raw.exp()[test_mask]
test_labels = data.y[test_mask]
ece_val, centers, accs_bin, confs_bin, _ = expected_calibration_error(test_probs_raw, test_labels, n_bins=10)
print(f"Expected Calibration Error (ECE): {ece_val:.4f}")

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "--", color="grey")
plt.scatter(confs_bin, accs_bin)
plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.title("Reliability Diagram (Teacher Raw)")
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# ----------------------- #
# 15) Attention heatmap for a test node
# ----------------------- #
@torch.no_grad()
def attention_heatmap(model: BetterGAT, data, node_id: int):
    spacer(f"Attention Heatmap for Node {node_id}")
    _, att = model.g1.conv(
        data.x,
        data.edge_index,
        return_attention_weights=True,
    )
    edge_idx, alpha = att
    alpha = alpha.mean(1)

    mask = edge_idx[0] == node_id
    nbrs = edge_idx[1, mask].numpy()
    weights = alpha[mask].numpy()

    if len(nbrs) == 0:
        print("No neighbors for this node.")
        return

    idx = np.argsort(weights)[::-1]
    nbrs, weights = nbrs[idx], weights[idx]

    plt.figure(figsize=(8, max(1.6, 0.25 * len(nbrs))))
    plt.barh(range(len(nbrs)), weights, color="skyblue")
    plt.yticks(range(len(nbrs)), [str(n) for n in nbrs])
    plt.gca().invert_yaxis()
    plt.xlabel("Attention Weight αᵢⱼ")
    plt.title(f"Node {node_id} → Neighbor Attention (Layer 1)")
    plt.tight_layout()
    plt.show()

test_nodes = torch.where(data.test_mask)[0].tolist()
if test_nodes:
    attention_heatmap(teacher, data, int(test_nodes[0]))

# ----------------------- #
# 16) Ego-graph misclassified node (teacher raw)
# ----------------------- #
@torch.no_grad()
def misclassified_ego_graph(G, data, pred):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")
    mask = data.test_mask & (pred != data.y)
    idx = torch.where(mask)[0]
    if idx.numel() == 0:
        print("No misclassified test nodes, skipping ego-graph.")
        return
    node_id = int(idx[0])
    print(f"Showing ego-graph for misclassified node {node_id}")
    H = nx.ego_graph(G, node_id, radius=2)
    pos = nx.spring_layout(H, seed=42)
    colors = []
    for n in H.nodes():
        if not bool(data.test_mask[n]):
            colors.append("lightgrey")
        else:
            colors.append("tab:red" if pred[n] != data.y[n] else "tab:green")
    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        H,
        pos=pos,
        node_color=colors,
        node_size=80,
        edge_color="lightgrey",
        with_labels=False,
    )
    plt.title("Ego-Graph (Red = misclassified test nodes, Green = correct)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

misclassified_ego_graph(G, data, preds_raw)

# ----------------------- #
# 17) Interactive 3D graph + 2D t-SNE (teacher)
# ----------------------- #
@torch.no_grad()
def interactive_3d_graph(G, data, pred, title="Interactive 3D Graph (Predictions)"):
    spacer(title)
    pos_3d = nx.spring_layout(G, seed=42, dim=3)
    Xn, Yn, Zn, node_text = [], [], [], []
    for i in range(data.num_nodes):
        x, y, z = pos_3d[i]
        Xn.append(x); Yn.append(y); Zn.append(z)
        node_text.append(f"Node {i} | True {int(data.y[i])} | Pred {int(pred[i])}")
    Xe, Ye, Ze = [], [], []
    for u, v in G.edges():
        x0, y0, z0 = pos_3d[u]; x1, y1, z1 = pos_3d[v]
        Xe += [x0, x1, None]
        Ye += [y0, y1, None]
        Ze += [z0, z1, None]
    edge_trace = go.Scatter3d(
        x=Xe, y=Ye, z=Ze, mode="lines",
        line=dict(color="rgba(150,150,150,0.35)", width=1),
        hoverinfo="none",
    )
    node_trace = go.Scatter3d(
        x=Xn, y=Yn, z=Zn, mode="markers",
        marker=dict(size=4, color=pred.numpy(), colorscale="Rainbow", opacity=0.9),
        text=node_text, hoverinfo="text",
    )
    layout = go.Layout(
        title=title, height=720,
        margin=dict(l=0, r=0, b=0, t=50),
        showlegend=False,
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
        ),
    )
    fig = go.Figure(data=[edge_trace, node_trace], layout=layout)
    fig.show()

interactive_3d_graph(G, data, preds_raw, title="Interactive 3D Graph (Teacher Raw Predictions)")

@torch.no_grad()
def interactive_tsne(emb: torch.Tensor, data, title="Interactive t-SNE (Embeddings)"):
    spacer(title)
    tsne_emb = TSNE(n_components=2, init="pca", learning_rate="auto").fit_transform(emb.numpy())
    x_tsne, y_tsne = tsne_emb[:, 0], tsne_emb[:, 1]
    node_text = [f"Node {i} | True {int(data.y[i])}" for i in range(data.num_nodes)]
    trace = go.Scatter(
        x=x_tsne, y=y_tsne,
        mode="markers",
        marker=dict(
            size=6,
            color=data.y.numpy(),
            colorscale="Viridis",
            showscale=True,
            opacity=0.9,
        ),
        text=node_text,
        hoverinfo="text",
    )
    layout = go.Layout(
        title=title,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        height=700,
        margin=dict(l=0, r=0, b=0, t=50),
    )
    fig = go.Figure(data=[trace], layout=layout)
    fig.show()

interactive_tsne(emb_raw, data, title="Interactive t-SNE (Teacher Embeddings)")

print("\n✅ Ultra-GAT Lab finished: teacher, student, uncertainty, calibration & rich diagnostics ready.\n")

In [ ]:
# ============================================================
# Graph Attention Networks ++ with Teacher–Student KD
# CiteSeer   |   GATv2 Teacher  +  GCN Student
# - No early stopping
# - Constant LR (no scheduler collapse)
# - DropEdge + log-degree features + LP aux loss
# - Knowledge Distillation from Teacher -> Student
# - Core diagnostics & plots
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ----------------------- #
# 0) Config + Repro
# ----------------------- #
CONFIG = {
    "dataset_name": "CiteSeer",
    "teacher_hidden_dim": 8,
    "teacher_heads": (8, 8, 4),
    "epochs_teacher": 1500,          # full run, no early stopping
    "epochs_student": 800,           # student KD epochs
    "dropedge_base_p": 0.15,
    "feature_mask_p": 0.05,
    "lp_aux_weight": 0.1,            # label-prop aux weight
    "contrastive_weight": 0.0,       # OFF for this version
    "temperature_contrastive": 0.5,
    "use_struct_features": True,
    "use_lp_aux": True,
    "use_contrastive": False,        # OFF
    "distill_alpha": 0.7,            # KD: weight for soft-teacher loss
    "distill_temperature": 3.0,
    "seed": 42,
}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(CONFIG["seed"])

# You can change to "cuda" if you want GPU & have it:
device = torch.device("cpu")
print("Device:", device)


# ----------------------- #
# 1) Dataset + basic stats
# ----------------------- #
dataset = Planetoid(root="./data", name=CONFIG["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Undirect + add self-loops (better for attention)
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# ----------------------- #
# 2) Quick structure visuals
# ----------------------- #
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y,
    width=0.5,
    edge_color="grey",
)
plt.title(f"{CONFIG['dataset_name']} Graph (Node-colored by Label)")
plt.show()

degrees_arr = degree(data.edge_index[0]).numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.show()

# ----------------------- #
# 3) Feature scaling + structural aug
# ----------------------- #
x_np = data.x.numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

if CONFIG["use_struct_features"]:
    x_aug = np.concatenate([x_scaled, log_deg], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)

print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

# Move to device
data = data.to(device)


# ----------------------- #
# 4) Utilities
# ----------------------- #
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.2)


# ----------------------- #
# 5) GATv2 Block + BetterGAT
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(8, 8, 4),
        dropout: float = 0.6,
        feature_mask_p: float = 0.05,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # residual head for low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        # constant LR, no scheduler
        self.opt = torch.optim.Adam(self.parameters(), lr=0.005, weight_decay=5e-4)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ----------------------- #
# 6) (Optional) Contrastive loss stub
#     (kept here but OFF in config)
# ----------------------- #
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    """
    Simple supervised contrastive loss on training nodes.
    Here for completeness; CONFIG["use_contrastive"] = False.
    """
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    # Remove self-similarity
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9

    labels_eq = y.unsqueeze(0) == y.unsqueeze(1)
    labels_eq = labels_eq.float()

    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ----------------------- #
# 7) Teacher training (NO early stopping)
# ----------------------- #
def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]

    best_val_acc = -1.0
    best_epoch = -1

    print(f"\n🧠 Training TEACHER for {E} epochs (NO early stopping, constant LR)...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        # DropEdge schedule
        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)

        emb, logp = model(data.x, edge_index_aug, training=True)

        ce_loss = ce(logp[data.train_mask], data.y[data.train_mask])
        loss = ce_loss

        # Label-prop aux
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)

        # (Optional) contrastive
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()

        # Eval on full graph
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(logp_val[data.val_mask], data.y[data.val_mask]).item()
            tr_acc = accuracy(logp[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_val[data.val_mask].argmax(1), data.y[data.val_mask])

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | Contr:{contr:.3f}"
            )

    print(f"\n✅ Teacher finished all {E} epochs.")
    print(f"   Best Val Accuracy seen: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    # Return last logp for distillation
    model.eval()
    with torch.no_grad():
        _, final_logp = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)

    # Raw
    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    # LabelProp refinement
    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp


# ----------------------- #
# 8) Student GCN + KD
# ----------------------- #
class StudentGCN(nn.Module):
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.gcn1(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.gcn2(h, edge_index)
        return F.log_softmax(h, dim=1)


def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    """
    Standard KD loss: alpha * KL + (1-alpha) * CE
    """
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_student(
    student: StudentGCN,
    teacher_logp: torch.Tensor,
    data,
    cfg,
):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    train_acc_curve, val_acc_curve = [], []

    print(f"\n🎓 Training STUDENT GCN for {E} epochs (KD, no early stopping)...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(logp_s[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_sv[data.val_mask].argmax(1), data.y[data.val_mask])
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

    print(f"\n✅ Student finished all {E} epochs.\n")
    return student, train_acc_curve, val_acc_curve


@torch.no_grad()
def test_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred


# ----------------------- #
# 9) Run training
# ----------------------- #
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=CONFIG["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=CONFIG["teacher_heads"],
    dropout=0.6,
    feature_mask_p=CONFIG["feature_mask_p"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss = train_teacher(
    teacher, data, CONFIG
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, _ = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

student = StudentGCN(in_dim, 16, dataset.num_classes).to(device)
student, s_tr_acc, s_val_acc = train_student(student, teacher_logp, data, CONFIG)
s_acc, s_pred = test_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {s_acc*100:.2f}%")


# ----------------------- #
# 10) Plots & Diagnostics
# ----------------------- #
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)"):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    plt.show()
    return cm


# 10.1 Teacher learning curves
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(t_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss", color="#1f77b4")
plt.plot(t_val_loss, label="Val Loss", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 10.2 Student learning curves
spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(s_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student GCN (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 10.3 t-SNE (untrained vs trained teacher)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
    ).to(device)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plt.tight_layout()
    plt.show()


tsne_static(teacher, data)

# 10.4 Accuracy by degree (teacher raw)
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


accuracy_by_degree(teacher, data)

# 10.5 Confusion matrices
cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)")
cm_student = plot_confusion(data, s_pred, "Student Confusion (KD, Test)")

print("\n✅ Full GAT++ Teacher + GCN Student KD pipeline finished.")

In [ ]:
# ============================================================
# Graph Attention Networks ++ with Teacher–Student KD
# CiteSeer   |   GATv2 Teacher  +  GCN Student
# - No early stopping
# - Constant LR (no scheduler collapse)
# - DropEdge + log-degree features + LP aux loss
# - Knowledge Distillation from Teacher -> Student
# - Rich diagnostics & plots (static + interactive)
# - Auto-uses GPU (e.g. T4) on Colab / Kaggle if available
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
    k_hop_subgraph,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ----------------------- #
# 0) Config + Repro
# ----------------------- #
CONFIG = {
    "dataset_name": "CiteSeer",
    "teacher_hidden_dim": 8,
    "teacher_heads": (8, 8, 4),
    "epochs_teacher": 3000,          # full run, no early stopping
    "epochs_student": 1600,           # student KD epochs
    "dropedge_base_p": 0.15,
    "feature_mask_p": 0.05,
    "lp_aux_weight": 0.1,            # label-prop aux weight
    "contrastive_weight": 0.0,       # OFF for this version
    "temperature_contrastive": 0.5,
    "use_struct_features": True,
    "use_lp_aux": True,
    "use_contrastive": False,        # OFF
    "distill_alpha": 0.7,            # KD: weight for soft-teacher loss
    "distill_temperature": 3.0,
    "seed": 42,
}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(CONFIG["seed"])

# Auto GPU (T4 on Colab/Kaggle) if available
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Device: CPU")


# ----------------------- #
# 1) Dataset + basic stats
# ----------------------- #
dataset = Planetoid(root="./data", name=CONFIG["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Undirect + add self-loops (better for attention)
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# ----------------------- #
# 2) Quick structure visuals
# ----------------------- #
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y,
    width=0.5,
    edge_color="grey",
)
plt.title(f"{CONFIG['dataset_name']} Graph (Node-colored by Label)")
plt.show()

degrees_arr = degree(data.edge_index[0]).numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.show()

# ----------------------- #
# 3) Feature scaling + structural aug
# ----------------------- #
x_np = data.x.numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

if CONFIG["use_struct_features"]:
    x_aug = np.concatenate([x_scaled, log_deg], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)

print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

# Move to device
data = data.to(device)


# ----------------------- #
# 4) Utilities
# ----------------------- #
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.1)


# ----------------------- #
# 5) GATv2 Block + BetterGAT
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(8, 8, 4),
        dropout: float = 0.6,
        feature_mask_p: float = 0.05,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # residual head for low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        # constant LR, no scheduler
        self.opt = torch.optim.Adam(self.parameters(), lr=0.005, weight_decay=5e-4)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ----------------------- #
# 6) (Optional) Contrastive loss stub
#     (kept here but OFF in config)
# ----------------------- #
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    """
    Simple supervised contrastive loss on training nodes.
    Here for completeness; CONFIG["use_contrastive"] = False.
    """
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    # Remove self-similarity
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9

    labels_eq = y.unsqueeze(0) == y.unsqueeze(1)
    labels_eq = labels_eq.float()

    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ----------------------- #
# 7) Teacher training (NO early stopping)
# ----------------------- #
def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]

    best_val_acc = -1.0
    best_epoch = -1

    print(f"\n🧠 Training TEACHER for {E} epochs (NO early stopping, constant LR)...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        # DropEdge schedule
        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)

        emb, logp = model(data.x, edge_index_aug, training=True)

        ce_loss = ce(logp[data.train_mask], data.y[data.train_mask])
        loss = ce_loss

        # Label-prop aux
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)

        # (Optional) contrastive
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()

        # Eval on full graph
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(logp_val[data.val_mask], data.y[data.val_mask]).item()
            tr_acc = accuracy(logp[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_val[data.val_mask].argmax(1), data.y[data.val_mask])

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | Contr:{contr:.3f}"
            )

    print(f"\n✅ Teacher finished all {E} epochs.")
    print(f"   Best Val Accuracy seen: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    # Return last logp for distillation
    model.eval()
    with torch.no_grad():
        _, final_logp = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)

    # Raw
    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    # LabelProp refinement
    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp


# ----------------------- #
# 8) Student GCN + KD
# ----------------------- #
class StudentGCN(nn.Module):
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.gcn1(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.gcn2(h, edge_index)
        return F.log_softmax(h, dim=1)


def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    """
    Standard KD loss: alpha * KL + (1-alpha) * CE
    """
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_student(
    student: StudentGCN,
    teacher_logp: torch.Tensor,
    data,
    cfg,
):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    train_acc_curve, val_acc_curve = [], []

    print(f"\n🎓 Training STUDENT GCN for {E} epochs (KD, no early stopping)...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(logp_s[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_sv[data.val_mask].argmax(1), data.y[data.val_mask])
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

    print(f"\n✅ Student finished all {E} epochs.\n")
    return student, train_acc_curve, val_acc_curve


@torch.no_grad()
def test_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred


# ----------------------- #
# 9) Run training
# ----------------------- #
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=CONFIG["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=CONFIG["teacher_heads"],
    dropout=0.6,
    feature_mask_p=CONFIG["feature_mask_p"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss = train_teacher(
    teacher, data, CONFIG
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, _ = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

student = StudentGCN(in_dim, 16, dataset.num_classes).to(device)
student, s_tr_acc, s_val_acc = train_student(student, teacher_logp, data, CONFIG)
s_acc, s_pred = test_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {s_acc*100:.2f}%")


# ----------------------- #
# 10) Plots & Diagnostics
# ----------------------- #
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)"):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    plt.show()
    return cm


# 10.1 Teacher learning curves
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(t_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss", color="#1f77b4")
plt.plot(t_val_loss, label="Val Loss", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 10.2 Student learning curves
spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(s_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student GCN (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 10.3 t-SNE (untrained vs trained teacher)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
    ).to(device)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plt.tight_layout()
    plt.show()


tsne_static(teacher, data)

# 10.4 Accuracy by degree (teacher raw)
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


accuracy_by_degree(teacher, data)

# 10.5 Confusion matrices
cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)")
cm_student = plot_confusion(data, s_pred, "Student Confusion (KD, Test)")


# 10.6 Per-Class Accuracy (Teacher Raw vs LP vs Student)
@torch.no_grad()
def per_class_accuracy(data, raw_pred, lp_pred, stu_pred, title="Per-Class Accuracy"):
    spacer("Per-Class Accuracy (Teacher Raw vs LabelProp vs Student)")
    y = data.y.cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()
    classes = np.arange(dataset.num_classes)

    raw = raw_pred.cpu().numpy()
    lp = lp_pred.cpu().numpy()
    stu = stu_pred.cpu().numpy()

    acc_raw, acc_lp, acc_stu = [], [], []

    for c in classes:
        mask = (y == c) & test_mask
        if mask.sum() == 0:
            acc_raw.append(0.0)
            acc_lp.append(0.0)
            acc_stu.append(0.0)
        else:
            acc_raw.append((raw[mask] == c).mean())
            acc_lp.append((lp[mask] == c).mean())
            acc_stu.append((stu[mask] == c).mean())

    x = np.arange(len(classes))
    width = 0.25

    plt.figure(figsize=(9, 4))
    plt.bar(x - width, acc_raw, width, label="Teacher Raw")
    plt.bar(x, acc_lp, width, label="Teacher LP")
    plt.bar(x + width, acc_stu, width, label="Student")

    plt.xticks(x, classes)
    plt.ylim(0, 1.05)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


per_class_accuracy(data, t_raw_pred, t_lp_pred, s_pred)


# 10.7 MC-Dropout Uncertainty (Teacher)
@torch.no_grad()
def mc_dropout_uncertainty(model: BetterGAT, data, num_samples: int = 30):
    spacer("MC-Dropout Uncertainty (Teacher)")
    model.train()  # enable dropout

    probs_list = []
    for _ in range(num_samples):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))

    probs_mc = torch.cat(probs_list, dim=0)  # [S, N, C]
    mean_probs = probs_mc.mean(dim=0)        # [N, C]

    # predictive entropy
    entropy = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)  # [N]

    test_mask = data.test_mask
    _, logp_det = model(data.x, data.edge_index, training=False)
    pred_det = logp_det.argmax(1)

    correct_mask = (pred_det == data.y) & test_mask
    wrong_mask = (pred_det != data.y) & test_mask

    ent_correct = entropy[correct_mask].cpu().numpy()
    ent_wrong = entropy[wrong_mask].cpu().numpy()

    print(f"Avg entropy (correct test): {ent_correct.mean():.4f}")
    print(f"Avg entropy (wrong   test): {ent_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(ent_correct, bins=20, alpha=0.7, label="Correct", density=True)
    plt.hist(ent_wrong, bins=20, alpha=0.7, label="Wrong", density=True)
    plt.xlabel("Predictive Entropy")
    plt.ylabel("Density")
    plt.title("MC-Dropout Uncertainty on Test Nodes")
    plt.legend()
    plt.tight_layout()
    plt.show()

    model.eval()
    return entropy, pred_det


entropy_mc, pred_det = mc_dropout_uncertainty(teacher, data)


# 10.8 Calibration (Teacher Raw Probs + ECE)
@torch.no_grad()
def calibration_plot(model: BetterGAT, data, n_bins: int = 10):
    spacer("Calibration (Teacher Raw Probs)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    probs = logp.exp()

    test_mask = data.test_mask
    y_true = data.y[test_mask]
    probs_test = probs[test_mask]

    conf, preds = probs_test.max(dim=1)
    conf = conf.cpu().numpy()
    preds = preds.cpu().numpy()
    y_true = y_true.cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    accs, avg_confs, counts = [], [], []
    total = len(conf)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            accs.append(0.0)
            avg_confs.append(0.0)
            counts.append(0)
            continue
        counts.append(int(mask.sum()))
        avg_conf = conf[mask].mean()
        accuracy_b = (preds[mask] == y_true[mask]).mean()
        accs.append(accuracy_b)
        avg_confs.append(avg_conf)
        ece += (mask.sum() / total) * abs(accuracy_b - avg_conf)

    print(f"Expected Calibration Error (ECE): {ece:.4f}")

    centers = 0.5 * (bins[:-1] + bins[1:])
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    plt.bar(centers, accs, width=1.0 / n_bins, alpha=0.7, edgecolor="k", label="Accuracy")
    plt.plot(centers, avg_confs, "o-", label="Avg Confidence")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Teacher)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return ece


ece_val = calibration_plot(teacher, data)


# 10.9 Ego-Graph Around a Misclassified Test Node (Teacher Raw)
@torch.no_grad()
def ego_graph_misclassified(model: BetterGAT, data, center_k: int = 2):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)

    test_mask = data.test_mask
    wrong_nodes = torch.where((pred != data.y) & test_mask)[0]
    if wrong_nodes.numel() == 0:
        print("No misclassified test nodes – nice!")
        return

    center = int(wrong_nodes[0].item())
    print(f"Visualising ego-graph for misclassified test node {center}")

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        center, num_hops=center_k, edge_index=data.edge_index, relabel_nodes=True
    )

    G_sub = nx.Graph()
    G_sub.add_edges_from(edge_index_sub.cpu().t().numpy())
    pos = nx.spring_layout(G_sub, seed=0)

    true_labels = data.y[subset].cpu().numpy()
    pred_labels = pred[subset].cpu().numpy()
    correct_flags = (true_labels == pred_labels)

    colors = ["#1f77b4" if c else "#d62728" for c in correct_flags]

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        G_sub,
        pos=pos,
        node_color=colors,
        node_size=200,
        with_labels=False,
        edge_color="gray",
    )
    plt.title("Ego-Graph Around Misclassified Node (Blue=Correct, Red=Wrong)")
    plt.axis("off")
    plt.show()


ego_graph_misclassified(teacher, data)


# 10.10 Interactive t-SNE (2D + 3D) with Plotly
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, n_components: int = 2):
    if n_components not in (2, 3):
        raise ValueError("n_components must be 2 or 3")

    spacer(f"Interactive t-SNE ({n_components}D) – Teacher Embeddings")

    model.eval()
    emb, _ = model(data.x, data.edge_index, training=False)
    emb_np = emb.cpu().numpy()
    labels_np = data.y.cpu().numpy()

    tsne = TSNE(
        n_components=n_components, init="pca", learning_rate="auto"
    ).fit_transform(emb_np)

    if n_components == 2:
        fig = px.scatter(
            x=tsne[:, 0],
            y=tsne[:, 1],
            color=labels_np.astype(str),
            title="Interactive t-SNE (2D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Label"},
        )
    else:
        fig = px.scatter_3d(
            x=tsne[:, 0],
            y=tsne[:, 1],
            z=tsne[:, 2],
            color=labels_np.astype(str),
            title="Interactive t-SNE (3D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3", "color": "Label"},
        )

    fig.show()


interactive_tsne(teacher, data, n_components=2)
interactive_tsne(teacher, data, n_components=3)

print("\n✅ Ultra-GAT++ Teacher + GCN Student KD lab finished: "
      "teacher, student, uncertainty, calibration & rich diagnostics ready.")

In [ ]:
# ============================================================
# Ultra-GAT Lab: Graph Attention Networks ++ with Teacher–Student KD
# Dataset: CiteSeer   |   GATv2 Teacher  +  GCN Student
#
# Novel bits (your "project" contributions):
# - DropEdge schedule + feature masking + log-degree structural features
# - Label-Propagation consistency auxiliary loss (graph-regularised teacher)
# - Knowledge Distillation: GATv2 -> compact GCN student
# - Rich diagnostics:
#     * Learning curves, confusion matrices, per-class accuracy
#     * Accuracy vs node degree, MC-Dropout uncertainty, calibration (ECE)
#     * Ego-graph visualisation around misclassified node
#     * Static & interactive t-SNE (2D & 3D) of embeddings
# - Auto GPU support (e.g. T4 on Colab / Kaggle)
# - Experiment summary + artifact export + reusable experiment runner
#   => ready for reports, blog posts, and portfolio demos.
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import os
import json
import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
    k_hop_subgraph,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ----------------------- #
# 0) Config + Repro
# ----------------------- #
CONFIG = {
    "dataset_name": "CiteSeer",
    "teacher_hidden_dim": 8,
    "teacher_heads": (8, 8, 4),
    "epochs_teacher": 3000,          # full run, no early stopping
    "epochs_student": 1600,          # student KD epochs
    "dropedge_base_p": 0.15,
    "feature_mask_p": 0.05,
    "lp_aux_weight": 0.1,            # label-prop aux weight
    "contrastive_weight": 0.0,       # OFF for this version
    "temperature_contrastive": 0.5,
    "use_struct_features": True,
    "use_lp_aux": True,
    "use_contrastive": False,        # OFF
    "distill_alpha": 0.7,            # KD: weight for soft-teacher loss
    "distill_temperature": 3.0,
    "seed": 42,
}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(CONFIG["seed"])

# Auto GPU (T4 on Colab/Kaggle) if available
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Device: CPU")


# ----------------------- #
# 1) Dataset + basic stats
# ----------------------- #
dataset = Planetoid(root="./data", name=CONFIG["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Undirect + add self-loops (better for attention)
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# ----------------------- #
# 2) Quick structure visuals
# ----------------------- #
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y,
    width=0.5,
    edge_color="grey",
)
plt.title(f"{CONFIG['dataset_name']} Graph (Node-colored by Label)")
plt.show()

degrees_arr = degree(data.edge_index[0]).numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.show()

# ----------------------- #
# 3) Feature scaling + structural aug
# ----------------------- #
x_np = data.x.numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

if CONFIG["use_struct_features"]:
    x_aug = np.concatenate([x_scaled, log_deg], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)

print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

# Move to device
data = data.to(device)


# ----------------------- #
# 4) Utilities
# ----------------------- #
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.1)


# ----------------------- #
# 5) GATv2 Block + BetterGAT
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(8, 8, 4),
        dropout: float = 0.6,
        feature_mask_p: float = 0.05,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # residual head for low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        # constant LR, no scheduler
        self.opt = torch.optim.Adam(self.parameters(), lr=0.005, weight_decay=5e-4)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ----------------------- #
# 6) (Optional) Contrastive loss stub
#     (kept here but OFF in config)
# ----------------------- #
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    """
    Simple supervised contrastive loss on training nodes.
    Here for completeness; CONFIG["use_contrastive"] = False.
    """
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    # Remove self-similarity
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9

    labels_eq = y.unsqueeze(0) == y.unsqueeze(1)
    labels_eq = labels_eq.float()

    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ----------------------- #
# 7) Teacher training (NO early stopping)
# ----------------------- #
def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]

    best_val_acc = -1.0
    best_epoch = -1

    print(f"\n🧠 Training TEACHER for {E} epochs (NO early stopping, constant LR)...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        # DropEdge schedule
        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)

        emb, logp = model(data.x, edge_index_aug, training=True)

        ce_loss = ce(logp[data.train_mask], data.y[data.train_mask])
        loss = ce_loss

        # Label-prop aux
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)

        # (Optional) contrastive
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()

        # Eval on full graph
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(logp_val[data.val_mask], data.y[data.val_mask]).item()
            tr_acc = accuracy(logp[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_val[data.val_mask].argmax(1), data.y[data.val_mask])

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | Contr:{contr:.3f}"
            )

    print(f"\n✅ Teacher finished all {E} epochs.")
    print(f"   Best Val Accuracy seen: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    # Return last logp for distillation
    model.eval()
    with torch.no_grad():
        _, final_logp = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)

    # Raw
    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    # LabelProp refinement
    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp


# ----------------------- #
# 8) Student GCN + KD
# ----------------------- #
class StudentGCN(nn.Module):
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.gcn1(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.gcn2(h, edge_index)
        return F.log_softmax(h, dim=1)


def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    """
    Standard KD loss: alpha * KL + (1-alpha) * CE
    """
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_student(
    student: StudentGCN,
    teacher_logp: torch.Tensor,
    data,
    cfg,
):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    train_acc_curve, val_acc_curve = [], []

    print(f"\n🎓 Training STUDENT GCN for {E} epochs (KD, no early stopping)...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(logp_s[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_sv[data.val_mask].argmax(1), data.y[data.val_mask])
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

    print(f"\n✅ Student finished all {E} epochs.\n")
    return student, train_acc_curve, val_acc_curve


@torch.no_grad()
def test_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred, logp


# ----------------------- #
# 9) Run training
# ----------------------- #
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=CONFIG["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=CONFIG["teacher_heads"],
    dropout=0.6,
    feature_mask_p=CONFIG["feature_mask_p"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss = train_teacher(
    teacher, data, CONFIG
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, teacher_logp_eval = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

student = StudentGCN(in_dim, 16, dataset.num_classes).to(device)
student, s_tr_acc, s_val_acc = train_student(student, teacher_logp, data, CONFIG)
s_acc, s_pred, s_logp_eval = test_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {s_acc*100:.2f}%")


# ----------------------- #
# 10) Plots & Diagnostics
# ----------------------- #
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)"):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    plt.show()
    return cm


# 10.1 Teacher learning curves
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(t_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss", color="#1f77b4")
plt.plot(t_val_loss, label="Val Loss", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 10.2 Student learning curves
spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(s_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student GCN (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 10.3 t-SNE (untrained vs trained teacher)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
    ).to(device)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plt.tight_layout()
    plt.show()


tsne_static(teacher, data)

# 10.4 Accuracy by degree (teacher raw)
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


accuracy_by_degree(teacher, data)

# 10.5 Confusion matrices
cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)")
cm_student = plot_confusion(data, s_pred, "Student Confusion (KD, Test)")


# 10.6 Per-Class Accuracy (Teacher Raw vs LP vs Student)
@torch.no_grad()
def per_class_accuracy(data, raw_pred, lp_pred, stu_pred, title="Per-Class Accuracy"):
    spacer("Per-Class Accuracy (Teacher Raw vs LabelProp vs Student)")
    y = data.y.cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()
    classes = np.arange(dataset.num_classes)

    raw = raw_pred.cpu().numpy()
    lp = lp_pred.cpu().numpy()
    stu = stu_pred.cpu().numpy()

    acc_raw, acc_lp, acc_stu = [], [], []

    for c in classes:
        mask = (y == c) & test_mask
        if mask.sum() == 0:
            acc_raw.append(0.0)
            acc_lp.append(0.0)
            acc_stu.append(0.0)
        else:
            acc_raw.append((raw[mask] == c).mean())
            acc_lp.append((lp[mask] == c).mean())
            acc_stu.append((stu[mask] == c).mean())

    x = np.arange(len(classes))
    width = 0.25

    plt.figure(figsize=(9, 4))
    plt.bar(x - width, acc_raw, width, label="Teacher Raw")
    plt.bar(x, acc_lp, width, label="Teacher LP")
    plt.bar(x + width, acc_stu, width, label="Student")

    plt.xticks(x, classes)
    plt.ylim(0, 1.05)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


per_class_accuracy(data, t_raw_pred, t_lp_pred, s_pred)


# 10.7 MC-Dropout Uncertainty (Teacher)
@torch.no_grad()
def mc_dropout_uncertainty(model: BetterGAT, data, num_samples: int = 30):
    spacer("MC-Dropout Uncertainty (Teacher)")
    model.train()  # enable dropout

    probs_list = []
    for _ in range(num_samples):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))

    probs_mc = torch.cat(probs_list, dim=0)  # [S, N, C]
    mean_probs = probs_mc.mean(dim=0)        # [N, C]

    # predictive entropy
    entropy = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)  # [N]

    test_mask = data.test_mask
    _, logp_det = model(data.x, data.edge_index, training=False)
    pred_det = logp_det.argmax(1)

    correct_mask = (pred_det == data.y) & test_mask
    wrong_mask = (pred_det != data.y) & test_mask

    ent_correct = entropy[correct_mask].cpu().numpy()
    ent_wrong = entropy[wrong_mask].cpu().numpy()

    print(f"Avg entropy (correct test): {ent_correct.mean():.4f}")
    print(f"Avg entropy (wrong   test): {ent_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(ent_correct, bins=20, alpha=0.7, label="Correct", density=True)
    plt.hist(ent_wrong, bins=20, alpha=0.7, label="Wrong", density=True)
    plt.xlabel("Predictive Entropy")
    plt.ylabel("Density")
    plt.title("MC-Dropout Uncertainty on Test Nodes")
    plt.legend()
    plt.tight_layout()
    plt.show()

    model.eval()
    return entropy, pred_det


entropy_mc, pred_det = mc_dropout_uncertainty(teacher, data)


# 10.8 Calibration (Teacher Raw Probs + ECE)
@torch.no_grad()
def calibration_plot(model: BetterGAT, data, n_bins: int = 10):
    spacer("Calibration (Teacher Raw Probs)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    probs = logp.exp()

    test_mask = data.test_mask
    y_true = data.y[test_mask]
    probs_test = probs[test_mask]

    conf, preds = probs_test.max(dim=1)
    conf = conf.cpu().numpy()
    preds = preds.cpu().numpy()
    y_true = y_true.cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    accs, avg_confs, counts = [], [], []
    total = len(conf)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            accs.append(0.0)
            avg_confs.append(0.0)
            counts.append(0)
            continue
        counts.append(int(mask.sum()))
        avg_conf = conf[mask].mean()
        accuracy_b = (preds[mask] == y_true[mask]).mean()
        accs.append(accuracy_b)
        avg_confs.append(avg_conf)
        ece += (mask.sum() / total) * abs(accuracy_b - avg_conf)

    print(f"Expected Calibration Error (ECE): {ece:.4f}")

    centers = 0.5 * (bins[:-1] + bins[1:])
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    plt.bar(centers, accs, width=1.0 / n_bins, alpha=0.7, edgecolor="k", label="Accuracy")
    plt.plot(centers, avg_confs, "o-", label="Avg Confidence")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Teacher)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return ece


ece_val = calibration_plot(teacher, data)


# 10.9 Ego-Graph Around a Misclassified Test Node (Teacher Raw)
@torch.no_grad()
def ego_graph_misclassified(model: BetterGAT, data, center_k: int = 2):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)

    test_mask = data.test_mask
    wrong_nodes = torch.where((pred != data.y) & test_mask)[0]
    if wrong_nodes.numel() == 0:
        print("No misclassified test nodes – nice!")
        return

    center = int(wrong_nodes[0].item())
    print(f"Visualising ego-graph for misclassified test node {center}")

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        center, num_hops=center_k, edge_index=data.edge_index, relabel_nodes=True
    )

    G_sub = nx.Graph()
    G_sub.add_edges_from(edge_index_sub.cpu().t().numpy())
    pos = nx.spring_layout(G_sub, seed=0)

    true_labels = data.y[subset].cpu().numpy()
    pred_labels = pred[subset].cpu().numpy()
    correct_flags = (true_labels == pred_labels)

    colors = ["#1f77b4" if c else "#d62728" for c in correct_flags]

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        G_sub,
        pos=pos,
        node_color=colors,
        node_size=200,
        with_labels=False,
        edge_color="gray",
    )
    plt.title("Ego-Graph Around Misclassified Node (Blue=Correct, Red=Wrong)")
    plt.axis("off")
    plt.show()


ego_graph_misclassified(teacher, data)


# 10.10 Interactive t-SNE (2D + 3D) with Plotly
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, n_components: int = 2):
    if n_components not in (2, 3):
        raise ValueError("n_components must be 2 or 3")

    spacer(f"Interactive t-SNE ({n_components}D) – Teacher Embeddings")

    model.eval()
    emb, _ = model(data.x, data.edge_index, training=False)
    emb_np = emb.cpu().numpy()
    labels_np = data.y.cpu().numpy()

    tsne = TSNE(
        n_components=n_components, init="pca", learning_rate="auto"
    ).fit_transform(emb_np)

    if n_components == 2:
        fig = px.scatter(
            x=tsne[:, 0],
            y=tsne[:, 1],
            color=labels_np.astype(str),
            title="Interactive t-SNE (2D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Label"},
        )
    else:
        fig = px.scatter_3d(
            x=tsne[:, 0],
            y=tsne[:, 1],
            z=tsne[:, 2],
            color=labels_np.astype(str),
            title="Interactive t-SNE (3D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3", "color": "Label"},
        )

    fig.show()


interactive_tsne(teacher, data, n_components=2)
interactive_tsne(teacher, data, n_components=3)


# ----------------------- #
# 11) Experiment Summary & Artifact Export (your "project" layer)
# ----------------------- #
@torch.no_grad()
def export_artifacts(
    teacher: BetterGAT,
    student: StudentGCN,
    data,
    cfg,
    t_raw_acc,
    t_lp_acc,
    s_acc,
    ece_val,
    run_name: str = "ultra_gat_citeseer",
):
    spacer("Exporting Artifacts & Experiment Summary")
    os.makedirs("artifacts", exist_ok=True)

    # 11.1 embeddings & logits
    teacher.eval()
    student.eval()
    t_emb, t_logp = teacher(data.x, data.edge_index, training=False)
    s_logp = student(data.x, data.edge_index, training=False)

    t_emb_np = t_emb.cpu().numpy()
    t_logit_np = t_logp.cpu().numpy()
    s_logit_np = s_logp.cpu().numpy()
    y_np = data.y.cpu().numpy()

    np.savez_compressed(
        os.path.join("artifacts", f"{run_name}_embeddings_logits.npz"),
        teacher_embeddings=t_emb_np,
        teacher_logits=t_logit_np,
        student_logits=s_logit_np,
        labels=y_np,
    )

    # 11.2 model weights
    torch.save(teacher.state_dict(), os.path.join("artifacts", f"{run_name}_teacher.pt"))
    torch.save(student.state_dict(), os.path.join("artifacts", f"{run_name}_student.pt"))

    # 11.3 config + metrics summary
    summary = {
        "run_name": run_name,
        "dataset": cfg["dataset_name"],
        "teacher_hidden_dim": cfg["teacher_hidden_dim"],
        "teacher_heads": cfg["teacher_heads"],
        "epochs_teacher": cfg["epochs_teacher"],
        "epochs_student": cfg["epochs_student"],
        "dropedge_base_p": cfg["dropedge_base_p"],
        "feature_mask_p": cfg["feature_mask_p"],
        "use_struct_features": cfg["use_struct_features"],
        "use_lp_aux": cfg["use_lp_aux"],
        "distill_alpha": cfg["distill_alpha"],
        "distill_temperature": cfg["distill_temperature"],
        "seed": cfg["seed"],
        "metrics": {
            "teacher_raw_acc": float(t_raw_acc),
            "teacher_lp_acc": float(t_lp_acc),
            "student_acc": float(s_acc),
            "teacher_ece": float(ece_val),
        },
    }

    with open(os.path.join("artifacts", f"{run_name}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("✅ Saved:")
    print("  - artifacts/"
          f"{run_name}_embeddings_logits.npz")
    print("  - artifacts/"
          f"{run_name}_teacher.pt")
    print("  - artifacts/"
          f"{run_name}_student.pt")
    print("  - artifacts/"
          f"{run_name}_summary.json")

    return summary


summary = export_artifacts(
    teacher,
    student,
    data,
    CONFIG,
    t_raw_acc,
    t_lp_acc,
    s_acc,
    ece_val,
    run_name="ultra_gat_citeseer",
)

print("\n📊 Experiment Summary (for reports / README):")
print(json.dumps(summary, indent=2))


# ----------------------- #
# 12) Optional: Lightweight Experiment Runner / Ablation Template
#      (not called by default — you can use this for your paper / blog)
# ----------------------- #
def clone_config(base_cfg, overrides=None):
    cfg = dict(base_cfg)
    if overrides:
        cfg.update(overrides)
    return cfg


def run_teacher_student_once(cfg_overrides=None, run_name="ablation_run"):
    """
    Mini wrapper to re-run the whole teacher+student pipeline
    with slight config changes (for ablations).
    NOTE: This will retrain models; reduce epochs in cfg_overrides
    when running multiple times.
    """
    cfg = clone_config(CONFIG, cfg_overrides)
    set_seed(cfg["seed"])

    # reload dataset fresh
    dataset_local = Planetoid(root="./data", name=cfg["dataset_name"])
    data_local = dataset_local[0]
    data_local.edge_index = to_undirected(data_local.edge_index, num_nodes=data_local.num_nodes)
    data_local.edge_index, _ = add_self_loops(data_local.edge_index, num_nodes=data_local.num_nodes)

    # features
    x_np = data_local.x.numpy()
    scaler_local = StandardScaler()
    x_scaled = scaler_local.fit_transform(x_np)

    deg_np = degree(data_local.edge_index[0], num_nodes=data_local.num_nodes).numpy()
    log_deg = np.log1p(deg_np).reshape(-1, 1)

    if cfg["use_struct_features"]:
        x_aug = np.concatenate([x_scaled, log_deg], axis=1)
    else:
        x_aug = x_scaled

    data_local.x = torch.tensor(x_aug, dtype=torch.float32)
    in_dim_local = data_local.x.size(1)
    data_local = data_local.to(device)

    teacher_local = BetterGAT(
        dim_in=in_dim_local,
        dim_h=cfg["teacher_hidden_dim"],
        dim_out=dataset_local.num_classes,
        heads=cfg["teacher_heads"],
        dropout=0.6,
        feature_mask_p=cfg["feature_mask_p"],
    ).to(device)

    teacher_local, t_logp_local, t_tr_acc_local, t_val_acc_local, _, _ = train_teacher(
        teacher_local, data_local, cfg
    )
    t_raw_acc_local, _, t_lp_acc_local, _, _ = test_teacher(teacher_local, data_local)

    student_local = StudentGCN(in_dim_local, 16, dataset_local.num_classes).to(device)
    student_local, s_tr_acc_local, s_val_acc_local = train_student(
        student_local, t_logp_local, data_local, cfg
    )
    s_acc_local, _, _ = test_student(student_local, data_local)

    sum_local = export_artifacts(
        teacher_local,
        student_local,
        data_local,
        cfg,
        t_raw_acc_local,
        t_lp_acc_local,
        s_acc_local,
        ece_val=0.0,  # skip calibration for quicker ablations
        run_name=run_name,
    )

    return sum_local, (t_tr_acc_local, t_val_acc_local, s_tr_acc_local, s_val_acc_local)


# Example usage for ablations (you can run these cells manually later):
#
# 1) Turn OFF DropEdge + LP-aux and reduce epochs for a quick comparison:
# ablation_cfg = {
#     "epochs_teacher": 800,
#     "epochs_student": 400,
#     "dropedge_base_p": 0.0,
#     "use_lp_aux": False,
# }
# summary_ablation, curves_ablation = run_teacher_student_once(
#     cfg_overrides=ablation_cfg,
#     run_name="ultra_gat_citeseer_no_dropedge_no_lp"
# )
# print(json.dumps(summary_ablation, indent=2))
#
# 2) Turn OFF structural features (log-degree) only:
# ablation_cfg2 = {
#     "epochs_teacher": 800,
#     "epochs_student": 400,
#     "use_struct_features": False,
# }
# summary_ablation2, curves_ablation2 = run_teacher_student_once(
#     cfg_overrides=ablation_cfg2,
#     run_name="ultra_gat_citeseer_no_struct"
# )
# print(json.dumps(summary_ablation2, indent=2))


print("\n✅ Ultra-GAT++ Teacher + GCN Student KD lab finished: "
      "teacher, student, uncertainty, calibration, artifacts & ablation hooks ready.")

In [ ]:
# ============================================================
# Ultra-GAT++ with Teacher–Student KD on CiteSeer
#
# Teacher: GATv2 (BetterGAT)
# Student: Compact GCN (KD distilled)
#
# Features:
# - Auto GPU (T4 etc. on Colab/Kaggle) if available
# - DropEdge + feature masking + log-degree structural feature
# - Label Propagation auxiliary consistency loss
# - OPTIONAL: LP-based pseudo-label KL on high-confidence unlabeled nodes
# - Early stopping for Teacher & Student (best-epoch restore)
# - Tuned KD (alpha, temperature)
# - Rich diagnostics: curves, confusions, t-SNE, MC-dropout, calibration, ego-graph, interactive t-SNE
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
    k_hop_subgraph,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ----------------------- #
# 0) Config + Repro
# ----------------------- #
CONFIG = {
    "dataset_name": "CiteSeer",

    # Teacher architecture
    "teacher_hidden_dim": 16,
    "teacher_heads": (4, 4, 2),     # 16*2=32 last dim, but richer internals

    # Training epochs (max) + early stopping
    "epochs_teacher": 3000,
    "use_early_stopping_teacher": True,
    "teacher_patience": 300,

    "epochs_student": 1600,
    "use_early_stopping_student": True,
    "student_patience": 300,

    # Regularisation
    "dropedge_base_p": 0.25,       # stronger DropEdge
    "feature_mask_p": 0.10,        # more aggressive feature masking
    "lp_aux_weight": 0.1,          # label-prop aux weight

    # LP pseudo-label KL (semi-supervised flavour)
    "use_lp_pseudo": True,
    "lp_pseudo_conf_thr": 0.9,
    "lp_pseudo_weight": 0.1,

    # Contrastive stub (kept OFF)
    "contrastive_weight": 0.0,
    "temperature_contrastive": 0.5,
    "use_struct_features": True,
    "use_lp_aux": True,
    "use_contrastive": False,

    # KD hyperparams
    "distill_alpha": 0.6,          # mix between teacher-soft & hard labels
    "distill_temperature": 2.0,

    "seed": 42,
}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])

# Auto GPU (T4 on Colab/Kaggle) if available
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Device: CPU")


# ----------------------- #
# 1) Dataset + basic stats
# ----------------------- #
dataset = Planetoid(root="./data", name=CONFIG["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Undirect + add self-loops (better for attention)
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)


# ----------------------- #
# 2) Quick structure visuals
# ----------------------- #
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y,
    width=0.5,
    edge_color="grey",
)
plt.title(f"{CONFIG['dataset_name']} Graph (Node-colored by Label)")
plt.show()

degrees_arr = degree(data.edge_index[0]).numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.show()


# ----------------------- #
# 3) Feature scaling + structural aug
# ----------------------- #
x_np = data.x.numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

if CONFIG["use_struct_features"]:
    x_aug = np.concatenate([x_scaled, log_deg], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)

print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

# Move to device
data = data.to(device)


# ----------------------- #
# 4) Utilities
# ----------------------- #
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.1)


# ----------------------- #
# 5) GATv2 Block + BetterGAT
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(4, 4, 2),
        dropout: float = 0.6,
        feature_mask_p: float = 0.10,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=True)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=True)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # residual head for low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        # constant LR, stronger weight decay
        self.opt = torch.optim.Adam(
            self.parameters(),
            lr=0.004,
            weight_decay=1e-3
        )

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ----------------------- #
# 6) Optional Contrastive Loss (kept OFF)
# ----------------------- #
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    """
    Simple supervised contrastive loss on training nodes.
    Here for completeness; CONFIG["use_contrastive"] = False.
    """
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9  # remove self-similarity

    labels_eq = y.unsqueeze(0) == y.unsqueeze(1)
    labels_eq = labels_eq.float()

    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ----------------------- #
# 7) Teacher training (with Early Stopping)
# ----------------------- #
def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]

    best_val_acc = -1.0
    best_epoch = -1
    best_state = None
    epochs_no_improve = 0

    print(f"\n🧠 Training TEACHER for up to {E} epochs "
          f"(early stopping={cfg['use_early_stopping_teacher']}, constant LR)...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        # DropEdge schedule
        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)

        emb, logp = model(data.x, edge_index_aug, training=True)

        ce_loss = ce(logp[data.train_mask], data.y[data.train_mask])
        loss = ce_loss

        # Label-prop aux
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)
            lp_probs = logp.exp()  # placeholder

        # LP-based pseudo-label KL on high-confidence unlabeled nodes
        if cfg.get("use_lp_pseudo", False):
            with torch.no_grad():
                conf, _ = lp_probs.max(dim=1)
                pseudo_mask = (~data.train_mask) & (conf > cfg["lp_pseudo_conf_thr"])
            if pseudo_mask.any():
                kl_pseudo = F.kl_div(
                    F.log_softmax(logp[pseudo_mask], dim=1),
                    lp_probs[pseudo_mask],
                    reduction="batchmean",
                )
                loss = loss + cfg["lp_pseudo_weight"] * kl_pseudo
            else:
                kl_pseudo = torch.tensor(0.0, device=data.x.device)
        else:
            kl_pseudo = torch.tensor(0.0, device=data.x.device)

        # (Optional) contrastive
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()

        # Eval on full graph
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(logp_val[data.val_mask], data.y[data.val_mask]).item()
            tr_acc = accuracy(logp[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_val[data.val_mask].argmax(1), data.y[data.val_mask])

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_loss)

        # Early stopping book-keeping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | KL_p:{kl_pseudo:.3f} | Contr:{contr:.3f}"
            )

        if cfg["use_early_stopping_teacher"] and epochs_no_improve >= cfg["teacher_patience"]:
            print(f"\n⏹ Early stopping at epoch {ep}, best epoch {best_epoch} "
                  f"with ValAcc={best_val_acc*100:.2f}%")
            break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(
            {k: v.to(device) for k, v in best_state.items()}
        )

    print(f"\n✅ Teacher finished; best ValAcc={best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    model.eval()
    with torch.no_grad():
        _, final_logp = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)

    # Raw
    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    # LabelProp refinement
    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp


# ----------------------- #
# 8) Student GCN + KD (with Early Stopping)
# ----------------------- #
class StudentGCN(nn.Module):
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.gcn1(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.gcn2(h, edge_index)
        return F.log_softmax(h, dim=1)


def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    """
    Standard KD loss: alpha * KL + (1-alpha) * CE
    """
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_student(
    student: StudentGCN,
    teacher_logp: torch.Tensor,
    data,
    cfg,
):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    train_acc_curve, val_acc_curve = [], []

    best_val_acc = -1.0
    best_epoch = -1
    best_state = None
    epochs_no_improve = 0

    print(f"\n🎓 Training STUDENT GCN for up to {E} epochs "
          f"(KD, early stopping={cfg['use_early_stopping_student']})...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(logp_s[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_sv[data.val_mask].argmax(1), data.y[data.val_mask])
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = {k: v.cpu().clone() for k, v in student.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

        if cfg["use_early_stopping_student"] and epochs_no_improve >= cfg["student_patience"]:
            print(f"\n⏹ Student early stopping at epoch {ep}, best epoch {best_epoch} "
                  f"with ValAcc={best_val_acc*100:.2f}%")
            break

    if best_state is not None:
        student.load_state_dict(
            {k: v.to(device) for k, v in best_state.items()}
        )

    print(f"\n✅ Student finished; best ValAcc={best_val_acc*100:.2f}% at epoch {best_epoch}\n")
    return student, train_acc_curve, val_acc_curve


@torch.no_grad()
def test_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred


# ----------------------- #
# 9) Run training
# ----------------------- #
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=CONFIG["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=CONFIG["teacher_heads"],
    dropout=0.6,
    feature_mask_p=CONFIG["feature_mask_p"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss = train_teacher(
    teacher, data, CONFIG
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, _ = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

student = StudentGCN(in_dim, 16, dataset.num_classes).to(device)
student, s_tr_acc, s_val_acc = train_student(student, teacher_logp, data, CONFIG)
s_acc, s_pred = test_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {s_acc*100:.2f}%")


# ----------------------- #
# 10) Plots & Diagnostics
# ----------------------- #
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)"):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    plt.show()
    return cm


# 10.1 Teacher learning curves
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(t_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss", color="#1f77b4")
plt.plot(t_val_loss, label="Val Loss", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# 10.2 Student learning curves
spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(s_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student GCN (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()


# 10.3 t-SNE (untrained vs trained teacher)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
    ).to(device)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plt.tight_layout()
    plt.show()


tsne_static(teacher, data)


# 10.4 Accuracy by degree (teacher raw)
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


accuracy_by_degree(teacher, data)


# 10.5 Confusion matrices
cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)")
cm_student = plot_confusion(data, s_pred, "Student Confusion (KD, Test)")


# 10.6 Per-Class Accuracy (Teacher Raw vs LP vs Student)
@torch.no_grad()
def per_class_accuracy(data, raw_pred, lp_pred, stu_pred, title="Per-Class Accuracy"):
    spacer("Per-Class Accuracy (Teacher Raw vs LabelProp vs Student)")
    y = data.y.cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()
    classes = np.arange(dataset.num_classes)

    raw = raw_pred.cpu().numpy()
    lp = lp_pred.cpu().numpy()
    stu = stu_pred.cpu().numpy()

    acc_raw, acc_lp, acc_stu = [], [], []

    for c in classes:
        mask = (y == c) & test_mask
        if mask.sum() == 0:
            acc_raw.append(0.0)
            acc_lp.append(0.0)
            acc_stu.append(0.0)
        else:
            acc_raw.append((raw[mask] == c).mean())
            acc_lp.append((lp[mask] == c).mean())
            acc_stu.append((stu[mask] == c).mean())

    x = np.arange(len(classes))
    width = 0.25

    plt.figure(figsize=(9, 4))
    plt.bar(x - width, acc_raw, width, label="Teacher Raw")
    plt.bar(x,        acc_lp,  width, label="Teacher LP")
    plt.bar(x + width, acc_stu, width, label="Student")

    plt.xticks(x, classes)
    plt.ylim(0, 1.05)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


per_class_accuracy(data, t_raw_pred, t_lp_pred, s_pred)


# 10.7 MC-Dropout Uncertainty (Teacher)
@torch.no_grad()
def mc_dropout_uncertainty(model: BetterGAT, data, num_samples: int = 30):
    spacer("MC-Dropout Uncertainty (Teacher)")
    model.train()  # enable dropout

    probs_list = []
    for _ in range(num_samples):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))

    probs_mc = torch.cat(probs_list, dim=0)  # [S, N, C]
    mean_probs = probs_mc.mean(dim=0)        # [N, C]

    # predictive entropy
    entropy = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)  # [N]

    test_mask = data.test_mask
    _, logp_det = model(data.x, data.edge_index, training=False)
    pred_det = logp_det.argmax(1)

    correct_mask = (pred_det == data.y) & test_mask
    wrong_mask = (pred_det != data.y) & test_mask

    ent_correct = entropy[correct_mask].cpu().numpy()
    ent_wrong = entropy[wrong_mask].cpu().numpy()

    print(f"Avg entropy (correct test): {ent_correct.mean():.4f}")
    print(f"Avg entropy (wrong   test): {ent_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(ent_correct, bins=20, alpha=0.7, label="Correct", density=True)
    plt.hist(ent_wrong,   bins=20, alpha=0.7, label="Wrong",   density=True)
    plt.xlabel("Predictive Entropy")
    plt.ylabel("Density")
    plt.title("MC-Dropout Uncertainty on Test Nodes")
    plt.legend()
    plt.tight_layout()
    plt.show()

    model.eval()
    return entropy, pred_det


entropy_mc, pred_det = mc_dropout_uncertainty(teacher, data)


# 10.8 Calibration (Teacher Raw Probs + ECE)
@torch.no_grad()
def calibration_plot(model: BetterGAT, data, n_bins: int = 10):
    spacer("Calibration (Teacher Raw Probs)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    probs = logp.exp()

    test_mask = data.test_mask
    y_true = data.y[test_mask]
    probs_test = probs[test_mask]

    conf, preds = probs_test.max(dim=1)
    conf = conf.cpu().numpy()
    preds = preds.cpu().numpy()
    y_true = y_true.cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    accs, avg_confs, counts = [], [], []
    total = len(conf)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            accs.append(0.0)
            avg_confs.append(0.0)
            counts.append(0)
            continue
        counts.append(int(mask.sum()))
        avg_conf = conf[mask].mean()
        accuracy_b = (preds[mask] == y_true[mask]).mean()
        accs.append(accuracy_b)
        avg_confs.append(avg_conf)
        ece += (mask.sum() / total) * abs(accuracy_b - avg_conf)

    print(f"Expected Calibration Error (ECE): {ece:.4f}")

    centers = 0.5 * (bins[:-1] + bins[1:])
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    plt.bar(centers, accs, width=1.0 / n_bins, alpha=0.7, edgecolor="k", label="Accuracy")
    plt.plot(centers, avg_confs, "o-", label="Avg Confidence")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Teacher)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return ece


ece_val = calibration_plot(teacher, data)


# 10.9 Ego-Graph Around a Misclassified Test Node (Teacher Raw)
@torch.no_grad()
def ego_graph_misclassified(model: BetterGAT, data, center_k: int = 2):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)

    test_mask = data.test_mask
    wrong_nodes = torch.where((pred != data.y) & test_mask)[0]
    if wrong_nodes.numel() == 0:
        print("No misclassified test nodes – nice!")
        return

    center = int(wrong_nodes[0].item())
    print(f"Visualising ego-graph for misclassified test node {center}")

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        center, num_hops=center_k, edge_index=data.edge_index, relabel_nodes=True
    )

    G_sub = nx.Graph()
    G_sub.add_edges_from(edge_index_sub.cpu().t().numpy())
    pos = nx.spring_layout(G_sub, seed=0)

    true_labels = data.y[subset].cpu().numpy()
    pred_labels = pred[subset].cpu().numpy()
    correct_flags = (true_labels == pred_labels)

    colors = ["#1f77b4" if c else "#d62728" for c in correct_flags]

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        G_sub,
        pos=pos,
        node_color=colors,
        node_size=200,
        with_labels=False,
        edge_color="gray",
    )
    plt.title("Ego-Graph Around Misclassified Node (Blue=Correct, Red=Wrong)")
    plt.axis("off")
    plt.show()


ego_graph_misclassified(teacher, data)


# 10.10 Interactive t-SNE (2D + 3D) with Plotly
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, n_components: int = 2):
    if n_components not in (2, 3):
        raise ValueError("n_components must be 2 or 3")

    spacer(f"Interactive t-SNE ({n_components}D) – Teacher Embeddings")

    model.eval()
    emb, _ = model(data.x, data.edge_index, training=False)
    emb_np = emb.cpu().numpy()
    labels_np = data.y.cpu().numpy()

    tsne = TSNE(
        n_components=n_components, init="pca", learning_rate="auto"
    ).fit_transform(emb_np)

    if n_components == 2:
        fig = px.scatter(
            x=tsne[:, 0],
            y=tsne[:, 1],
            color=labels_np.astype(str),
            title="Interactive t-SNE (2D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Label"},
        )
    else:
        fig = px.scatter_3d(
            x=tsne[:, 0],
            y=tsne[:, 1],
            z=tsne[:, 2],
            color=labels_np.astype(str),
            title="Interactive t-SNE (3D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3", "color": "Label"},
        )

    fig.show()


interactive_tsne(teacher, data, n_components=2)
interactive_tsne(teacher, data, n_components=3)

print("\n✅ Ultra-GAT++ Teacher + GCN Student KD lab finished: "
      "teacher, student, uncertainty, calibration, LP-pseudo & rich diagnostics ready.")

In [ ]:
# ============================================================
# Ultra-GAT++ with Teacher–Student KD on CiteSeer
#
# Teacher: GATv2 (BetterGAT)
# Student: Compact GCN (KD distilled)
#
# Features:
# - Auto GPU (T4 etc. on Colab/Kaggle) if available
# - DropEdge + feature masking + log-degree structural feature
# - Label Propagation auxiliary consistency loss
# - LP-based pseudo-label KL on high-confidence unlabeled nodes
# - Early stopping for Teacher & Student (best-epoch restore)
# - Tuned KD (alpha, temperature)
# - Simple ensemble (Teacher + LP + Student)
# - Rich diagnostics: curves, confusions, t-SNE, MC-dropout, calibration, ego-graph, interactive t-SNE
# - Markdown report generator (UltraGAT_report.md)
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import os
import time
import random
import json
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
    k_hop_subgraph,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ----------------------- #
# 0) Config + Repro
# ----------------------- #
CONFIG = {
    "experiment_tag": "UltraGAT_CiteSeer_v2",
    "dataset_name": "CiteSeer",

    # Teacher architecture
    "teacher_hidden_dim": 16,
    "teacher_heads": (4, 4, 2),     # 16*2=32 last dim, richer internals

    # Training epochs (max) + early stopping
    "epochs_teacher": 3000,
    "use_early_stopping_teacher": True,
    "teacher_patience": 300,

    "epochs_student": 1600,
    "use_early_stopping_student": True,
    "student_patience": 300,

    # Regularisation
    "dropedge_base_p": 0.25,       # stronger DropEdge
    "feature_mask_p": 0.10,        # more aggressive feature masking
    "lp_aux_weight": 0.1,          # label-prop aux weight

    # LP pseudo-label KL (semi-supervised flavour)
    "use_lp_pseudo": True,
    "lp_pseudo_conf_thr": 0.9,
    "lp_pseudo_weight": 0.1,

    # Contrastive stub (kept OFF)
    "contrastive_weight": 0.0,
    "temperature_contrastive": 0.5,
    "use_struct_features": True,
    "use_lp_aux": True,
    "use_contrastive": False,

    # KD hyperparams
    "distill_alpha": 0.6,          # mix between teacher-soft & hard labels
    "distill_temperature": 2.0,

    "seed": 42,
}

PLOT_DIR = "plots"
os.makedirs(PLOT_DIR, exist_ok=True)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])

# Auto GPU (T4 on Colab/Kaggle) if available
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Device: CPU")


def save_fig(name: str, dpi: int = 150):
    """Save current matplotlib figure into PLOT_DIR and also show it."""
    path = os.path.join(PLOT_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.show()
    print(f"[+] Saved figure to {path}")


# ----------------------- #
# 1) Dataset + basic stats
# ----------------------- #
dataset = Planetoid(root="./data", name=CONFIG["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Undirect + add self-loops (better for attention)
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)


# ----------------------- #
# 2) Quick structure visuals
# ----------------------- #
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y,
    width=0.5,
    edge_color="grey",
)
plt.title(f"{CONFIG['dataset_name']} Graph (Node-colored by Label)")
save_fig("graph_layout.png")

degrees_arr = degree(data.edge_index[0]).numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
plt.title("Node Degree Distribution")
plt.tight_layout()
save_fig("degree_distribution.png")


# ----------------------- #
# 3) Feature scaling + structural aug
# ----------------------- #
x_np = data.x.numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

if CONFIG["use_struct_features"]:
    x_aug = np.concatenate([x_scaled, log_deg], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)

print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

# Move to device
data = data.to(device)


# ----------------------- #
# 4) Utilities
# ----------------------- #
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.1)


# ----------------------- #
# 5) GATv2 Block + BetterGAT
# ----------------------- #
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(4, 4, 2),
        dropout: float = 0.6,
        feature_mask_p: float = 0.10,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=True)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=True)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # residual head for low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        # constant LR, stronger weight decay
        self.opt = torch.optim.Adam(
            self.parameters(),
            lr=0.004,
            weight_decay=1e-3
        )

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ----------------------- #
# 6) Optional Contrastive Loss (kept OFF)
# ----------------------- #
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    """
    Simple supervised contrastive loss on training nodes.
    Here for completeness; CONFIG["use_contrastive"] = False.
    """
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9  # remove self-similarity

    labels_eq = y.unsqueeze(0) == y.unsqueeze(1)
    labels_eq = labels_eq.float()

    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ----------------------- #
# 7) Teacher training (with Early Stopping)
# ----------------------- #
def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]

    best_val_acc = -1.0
    best_epoch = -1
    best_state = None
    epochs_no_improve = 0

    print(f"\n🧠 Training TEACHER for up to {E} epochs "
          f"(early stopping={cfg['use_early_stopping_teacher']}, constant LR)...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        # DropEdge schedule
        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)

        emb, logp = model(data.x, edge_index_aug, training=True)

        ce_loss = ce(logp[data.train_mask], data.y[data.train_mask])
        loss = ce_loss

        # Label-prop aux
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)
            lp_probs = logp.exp()  # placeholder

        # LP-based pseudo-label KL on high-confidence unlabeled nodes
        if cfg.get("use_lp_pseudo", False):
            with torch.no_grad():
                conf, _ = lp_probs.max(dim=1)
                pseudo_mask = (~data.train_mask) & (conf > cfg["lp_pseudo_conf_thr"])
            if pseudo_mask.any():
                kl_pseudo = F.kl_div(
                    F.log_softmax(logp[pseudo_mask], dim=1),
                    lp_probs[pseudo_mask],
                    reduction="batchmean",
                )
                loss = loss + cfg["lp_pseudo_weight"] * kl_pseudo
            else:
                kl_pseudo = torch.tensor(0.0, device=data.x.device)
        else:
            kl_pseudo = torch.tensor(0.0, device=data.x.device)

        # (Optional) contrastive
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()

        # Eval on full graph
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(logp_val[data.val_mask], data.y[data.val_mask]).item()
            tr_acc = accuracy(logp[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_val[data.val_mask].argmax(1), data.y[data.val_mask])

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_loss)

        # Early stopping book-keeping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | KL_p:{kl_pseudo:.3f} | Contr:{contr:.3f}"
            )

        if cfg["use_early_stopping_teacher"] and epochs_no_improve >= cfg["teacher_patience"]:
            print(f"\n⏹ Early stopping at epoch {ep}, best epoch {best_epoch} "
                  f"with ValAcc={best_val_acc*100:.2f}%")
            break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(
            {k: v.to(device) for k, v in best_state.items()}
        )

    print(f"\n✅ Teacher finished; best ValAcc={best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    model.eval()
    with torch.no_grad():
        _, final_logp = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)

    # Raw
    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    # LabelProp refinement
    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp


# ----------------------- #
# 8) Student GCN + KD (with Early Stopping)
# ----------------------- #
class StudentGCN(nn.Module):
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.gcn1(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.gcn2(h, edge_index)
        return F.log_softmax(h, dim=1)


def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    """
    Standard KD loss: alpha * KL + (1-alpha) * CE
    """
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_student(
    student: StudentGCN,
    teacher_logp: torch.Tensor,
    data,
    cfg,
):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    train_acc_curve, val_acc_curve = [], []

    best_val_acc = -1.0
    best_epoch = -1
    best_state = None
    epochs_no_improve = 0

    print(f"\n🎓 Training STUDENT GCN for up to {E} epochs "
          f"(KD, early stopping={cfg['use_early_stopping_student']})...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(logp_s[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_sv[data.val_mask].argmax(1), data.y[data.val_mask])
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = {k: v.cpu().clone() for k, v in student.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

        if cfg["use_early_stopping_student"] and epochs_no_improve >= cfg["student_patience"]:
            print(f"\n⏹ Student early stopping at epoch {ep}, best epoch {best_epoch} "
                  f"with ValAcc={best_val_acc*100:.2f}%")
            break

    if best_state is not None:
        student.load_state_dict(
            {k: v.to(device) for k, v in best_state.items()}
        )

    print(f"\n✅ Student finished; best ValAcc={best_val_acc*100:.2f}% at epoch {best_epoch}\n")
    return student, train_acc_curve, val_acc_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred, logp


# ----------------------- #
# 9) Run training
# ----------------------- #
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=CONFIG["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=CONFIG["teacher_heads"],
    dropout=0.6,
    feature_mask_p=CONFIG["feature_mask_p"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss, t_best_epoch, t_best_val = train_teacher(
    teacher, data, CONFIG
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, t_logp_eval = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

student = StudentGCN(in_dim, 16, dataset.num_classes).to(device)
student, s_tr_acc, s_val_acc, s_best_epoch, s_best_val = train_student(student, teacher_logp, data, CONFIG)
s_acc, s_pred, s_logp_eval = test_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {s_acc*100:.2f}%")


# ----------------------- #
# 9.5 Simple Ensemble (Teacher + LP + Student)
# ----------------------- #
@torch.no_grad()
def ensemble_performance(teacher_logp: torch.Tensor,
                         data,
                         student_logp: torch.Tensor):

    probs_teacher = teacher_logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs_teacher, data.edge_index)
    probs_student = student_logp.exp()

    # Simple average ensemble in probability space
    probs_ens = (probs_teacher + lp_probs + probs_student) / 3.0
    pred_ens = probs_ens.argmax(1)

    ens_acc = accuracy(pred_ens[data.test_mask], data.y[data.test_mask])
    return ens_acc, pred_ens


ens_acc, ens_pred = ensemble_performance(t_logp_eval, data, s_logp_eval)
print(f"🤝 Ensemble Test Accuracy (Teacher + LP + Student): {ens_acc*100:.2f}%")


# ----------------------- #
# 10) Plots & Diagnostics
# ----------------------- #
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)", fname=None):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    if fname is None:
        fname = title.lower().replace(" ", "_") + ".png"
    save_fig(fname)
    return cm


# 10.1 Teacher learning curves
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(t_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
save_fig("teacher_accuracy_curves.png")

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss", color="#1f77b4")
plt.plot(t_val_loss, label="Val Loss", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
save_fig("teacher_loss_curves.png")

# 10.2 Student learning curves
spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(s_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student GCN (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
save_fig("student_accuracy_curves.png")


# 10.3 t-SNE (untrained vs trained teacher)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
    ).to(device)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plt.tight_layout()
    save_fig("tsne_untrained_vs_trained.png")


tsne_static(teacher, data)


# 10.4 Accuracy by degree (teacher raw)
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    save_fig("teacher_accuracy_by_degree.png")


accuracy_by_degree(teacher, data)


# 10.5 Confusion matrices
cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)", "teacher_confusion_raw.png")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)", "teacher_confusion_lp.png")
cm_student = plot_confusion(data, s_pred, "Student Confusion (KD, Test)", "student_confusion_kd.png")
cm_ensemble = plot_confusion(data, ens_pred, "Ensemble Confusion (Test)", "ensemble_confusion.png")


# 10.6 Per-Class Accuracy (Teacher Raw vs LP vs Student vs Ensemble)
@torch.no_grad()
def per_class_accuracy(data, raw_pred, lp_pred, stu_pred, ens_pred, title="Per-Class Accuracy"):
    spacer("Per-Class Accuracy (Teacher Raw vs LabelProp vs Student vs Ensemble)")
    y = data.y.cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()
    classes = np.arange(dataset.num_classes)

    raw = raw_pred.cpu().numpy()
    lp = lp_pred.cpu().numpy()
    stu = stu_pred.cpu().numpy()
    ens = ens_pred.cpu().numpy()

    acc_raw, acc_lp, acc_stu, acc_ens = [], [], [], []

    for c in classes:
        mask = (y == c) & test_mask
        if mask.sum() == 0:
            acc_raw.append(0.0)
            acc_lp.append(0.0)
            acc_stu.append(0.0)
            acc_ens.append(0.0)
        else:
            acc_raw.append((raw[mask] == c).mean())
            acc_lp.append((lp[mask] == c).mean())
            acc_stu.append((stu[mask] == c).mean())
            acc_ens.append((ens[mask] == c).mean())

    x = np.arange(len(classes))
    width = 0.2

    plt.figure(figsize=(10, 4))
    plt.bar(x - 1.5*width, acc_raw, width, label="Teacher Raw")
    plt.bar(x - 0.5*width, acc_lp,  width, label="Teacher LP")
    plt.bar(x + 0.5*width, acc_stu, width, label="Student")
    plt.bar(x + 1.5*width, acc_ens, width, label="Ensemble")

    plt.xticks(x, classes)
    plt.ylim(0, 1.05)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    save_fig("per_class_accuracy.png")

    return {
        "teacher_raw": acc_raw,
        "teacher_lp": acc_lp,
        "student": acc_stu,
        "ensemble": acc_ens,
    }


per_class_stats = per_class_accuracy(data, t_raw_pred, t_lp_pred, s_pred, ens_pred)


# 10.7 MC-Dropout Uncertainty (Teacher)
@torch.no_grad()
def mc_dropout_uncertainty(model: BetterGAT, data, num_samples: int = 30):
    spacer("MC-Dropout Uncertainty (Teacher)")
    model.train()  # enable dropout

    probs_list = []
    for _ in range(num_samples):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))

    probs_mc = torch.cat(probs_list, dim=0)  # [S, N, C]
    mean_probs = probs_mc.mean(dim=0)        # [N, C]

    # predictive entropy
    entropy = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)  # [N]

    test_mask = data.test_mask
    _, logp_det = model(data.x, data.edge_index, training=False)
    pred_det = logp_det.argmax(1)

    correct_mask = (pred_det == data.y) & test_mask
    wrong_mask = (pred_det != data.y) & test_mask

    ent_correct = entropy[correct_mask].cpu().numpy()
    ent_wrong = entropy[wrong_mask].cpu().numpy()

    print(f"Avg entropy (correct test): {ent_correct.mean():.4f}")
    print(f"Avg entropy (wrong   test): {ent_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(ent_correct, bins=20, alpha=0.7, label="Correct", density=True)
    plt.hist(ent_wrong,   bins=20, alpha=0.7, label="Wrong",   density=True)
    plt.xlabel("Predictive Entropy")
    plt.ylabel("Density")
    plt.title("MC-Dropout Uncertainty on Test Nodes")
    plt.legend()
    plt.tight_layout()
    save_fig("mc_dropout_uncertainty.png")

    model.eval()
    return entropy, pred_det, float(ent_correct.mean()), float(ent_wrong.mean())


entropy_mc, pred_det, avg_ent_correct, avg_ent_wrong = mc_dropout_uncertainty(teacher, data)


# 10.8 Calibration (Teacher Raw Probs + ECE)
@torch.no_grad()
def calibration_plot(model: BetterGAT, data, n_bins: int = 10):
    spacer("Calibration (Teacher Raw Probs)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    probs = logp.exp()

    test_mask = data.test_mask
    y_true = data.y[test_mask]
    probs_test = probs[test_mask]

    conf, preds = probs_test.max(dim=1)
    conf = conf.cpu().numpy()
    preds = preds.cpu().numpy()
    y_true = y_true.cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    accs, avg_confs, counts = [], [], []
    total = len(conf)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            accs.append(0.0)
            avg_confs.append(0.0)
            counts.append(0)
            continue
        counts.append(int(mask.sum()))
        avg_conf = conf[mask].mean()
        accuracy_b = (preds[mask] == y_true[mask]).mean()
        accs.append(accuracy_b)
        avg_confs.append(avg_conf)
        ece += (mask.sum() / total) * abs(accuracy_b - avg_conf)

    print(f"Expected Calibration Error (ECE): {ece:.4f}")

    centers = 0.5 * (bins[:-1] + bins[1:])
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    plt.bar(centers, accs, width=1.0 / n_bins, alpha=0.7, edgecolor="k", label="Accuracy")
    plt.plot(centers, avg_confs, "o-", label="Avg Confidence")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Teacher)")
    plt.legend()
    plt.tight_layout()
    save_fig("calibration_reliability_diagram.png")

    return float(ece)


ece_val = calibration_plot(teacher, data)


# 10.9 Ego-Graph Around a Misclassified Test Node (Teacher Raw)
@torch.no_grad()
def ego_graph_misclassified(model: BetterGAT, data, center_k: int = 2):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)

    test_mask = data.test_mask
    wrong_nodes = torch.where((pred != data.y) & test_mask)[0]
    if wrong_nodes.numel() == 0:
        print("No misclassified test nodes – nice!")
        return

    center = int(wrong_nodes[0].item())
    print(f"Visualising ego-graph for misclassified test node {center}")

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        center, num_hops=center_k, edge_index=data.edge_index, relabel_nodes=True
    )

    G_sub = nx.Graph()
    G_sub.add_edges_from(edge_index_sub.cpu().t().numpy())
    pos = nx.spring_layout(G_sub, seed=0)

    true_labels = data.y[subset].cpu().numpy()
    pred_labels = pred[subset].cpu().numpy()
    correct_flags = (true_labels == pred_labels)

    colors = ["#1f77b4" if c else "#d62728" for c in correct_flags]

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        G_sub,
        pos=pos,
        node_color=colors,
        node_size=200,
        with_labels=False,
        edge_color="gray",
    )
    plt.title("Ego-Graph Around Misclassified Node (Blue=Correct, Red=Wrong)")
    plt.axis("off")
    save_fig("ego_graph_misclassified.png")


ego_graph_misclassified(teacher, data)


# 10.10 Interactive t-SNE (2D + 3D) with Plotly
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, n_components: int = 2):
    if n_components not in (2, 3):
        raise ValueError("n_components must be 2 or 3")

    spacer(f"Interactive t-SNE ({n_components}D) – Teacher Embeddings")

    model.eval()
    emb, _ = model(data.x, data.edge_index, training=False)
    emb_np = emb.cpu().numpy()
    labels_np = data.y.cpu().numpy()

    tsne = TSNE(
        n_components=n_components, init="pca", learning_rate="auto"
    ).fit_transform(emb_np)

    if n_components == 2:
        fig = px.scatter(
            x=tsne[:, 0],
            y=tsne[:, 1],
            color=labels_np.astype(str),
            title="Interactive t-SNE (2D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Label"},
        )
    else:
        fig = px.scatter_3d(
            x=tsne[:, 0],
            y=tsne[:, 1],
            z=tsne[:, 2],
            color=labels_np.astype(str),
            title="Interactive t-SNE (3D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3", "color": "Label"},
        )

    fig.show()


interactive_tsne(teacher, data, n_components=2)
interactive_tsne(teacher, data, n_components=3)


# ----------------------- #
# 11) Markdown Report Generator
# ----------------------- #
def generate_markdown_report(
    config,
    dataset,
    t_raw_acc,
    t_lp_acc,
    s_acc,
    ens_acc,
    ece,
    avg_ent_correct,
    avg_ent_wrong,
    t_best_epoch,
    t_best_val,
    s_best_epoch,
    s_best_val,
    per_class_stats,
    output_path="UltraGAT_report.md",
):
    """Write a markdown report summarising the experiment."""
    lines = []

    lines.append(f"# Ultra-GAT++ Teacher–Student KD Report\n")
    lines.append(f"**Experiment tag:** `{config['experiment_tag']}`\n")
    lines.append(f"**Dataset:** `{dataset}`\n")
    lines.append("---\n")

    # High-level results
    lines.append("## 1. Overall Results\n")
    lines.append("| Model | Test Accuracy |\n| --- | --- |\n")
    lines.append(f"| Teacher (Raw logits) | {t_raw_acc*100:.2f}% |\n")
    lines.append(f"| Teacher + Label Propagation | {t_lp_acc*100:.2f}% |\n")
    lines.append(f"| Student GCN (KD) | {s_acc*100:.2f}% |\n")
    lines.append(f"| Ensemble (Teacher + LP + Student) | {ens_acc*100:.2f}% |\n\n")

    lines.append(f"- Teacher best validation accuracy: **{t_best_val*100:.2f}%** at epoch **{t_best_epoch}**\n")
    lines.append(f"- Student best validation accuracy: **{s_best_val*100:.2f}%** at epoch **{s_best_epoch}**\n\n")

    # Calibration & Uncertainty
    lines.append("## 2. Calibration & Uncertainty\n")
    lines.append(f"- Expected Calibration Error (ECE): **{ece:.4f}**\n")
    lines.append(f"- Average MC-Dropout entropy (correct test nodes): **{avg_ent_correct:.4f}**\n")
    lines.append(f"- Average MC-Dropout entropy (wrong test nodes): **{avg_ent_wrong:.4f}**\n\n")
    lines.append("Lower entropy for correct vs. wrong predictions indicates the teacher is more confident when it is right – a desirable behaviour for downstream active learning or human-in-the-loop settings.\n\n")

    # Per-class
    lines.append("## 3. Per-Class Accuracies (Test)\n")
    lines.append("| Class | Teacher Raw | Teacher LP | Student | Ensemble |\n| --- | --- | --- | --- | --- |\n")
    num_classes = len(per_class_stats["teacher_raw"])
    for c in range(num_classes):
        lines.append(
            f"| {c} | "
            f"{per_class_stats['teacher_raw'][c]*100:.1f}% | "
            f"{per_class_stats['teacher_lp'][c]*100:.1f}% | "
            f"{per_class_stats['student'][c]*100:.1f}% | "
            f"{per_class_stats['ensemble'][c]*100:.1f}% |\n"
        )
    lines.append("\n")

    # Config summary
    lines.append("## 4. Configuration Summary\n")
    lines.append("```json\n")
    lines.append(json.dumps(config, indent=2))
    lines.append("\n```\n")

    # Files/plots hint
    lines.append("## 5. Generated Plots & Artifacts\n")
    lines.append("- All static plots are saved under the `plots/` directory:\n")
    lines.append("  - `graph_layout.png`, `degree_distribution.png`\n")
    lines.append("  - `teacher_accuracy_curves.png`, `teacher_loss_curves.png`, `student_accuracy_curves.png`\n")
    lines.append("  - Confusion matrices: `teacher_confusion_raw.png`, `teacher_confusion_lp.png`, `student_confusion_kd.png`, `ensemble_confusion.png`\n")
    lines.append("  - `per_class_accuracy.png`, `teacher_accuracy_by_degree.png`\n")
    lines.append("  - `mc_dropout_uncertainty.png`, `calibration_reliability_diagram.png`, `ego_graph_misclassified.png`, `tsne_untrained_vs_trained.png`\n")
    lines.append("- Interactive Plotly t-SNE figures are rendered inline in the notebook.\n\n")

    lines.append("> This notebook implements Ultra-GAT++: a stacked GATv2 teacher with DropEdge, feature masking, structural augmentation, label-propagation auxiliary consistency, LP-based pseudo-label KD flavour, and a compact GCN student distilled from the teacher. A simple probability-level ensemble of Teacher, LP-smoothed Teacher, and Student yields further gains.\n")

    with open(output_path, "w") as f:
        f.writelines(lines)

    print(f"\n[+] Markdown report written to: {output_path}")


generate_markdown_report(
    config=CONFIG,
    dataset=CONFIG["dataset_name"],
    t_raw_acc=t_raw_acc,
    t_lp_acc=t_lp_acc,
    s_acc=s_acc,
    ens_acc=ens_acc,
    ece=ece_val,
    avg_ent_correct=avg_ent_correct,
    avg_ent_wrong=avg_ent_wrong,
    t_best_epoch=t_best_epoch,
    t_best_val=t_best_val,
    s_best_epoch=s_best_epoch,
    s_best_val=s_best_val,
    per_class_stats=per_class_stats,
)

print("\n✅ Ultra-GAT++ Teacher + GCN Student KD lab finished: "
      "teacher, student, ensemble, uncertainty, calibration, LP-pseudo & markdown report ready.")

In [ ]:
# ============================================================
# Ultra-GAT++ Lab with Teacher–Student KD + Hyperparam Sweep
#
# Main experiment:
#   - BetterGAT (GATv2 Teacher)
#   - GCN Student (KD distilled)
#   - DropEdge + Feature Masking + Log-degree
#   - LP auxiliary consistency term
#   - LP-based pseudo-label KL
#   - Early stopping for Teacher & Student
#   - Ensemble: Teacher + LP-smoothed Teacher + Student
#   - Diagnostics: curves, confusions, t-SNE, MC-Dropout, calibration, ego-graph
#   - Markdown report: UltraGAT_report.md
#
# Hyperparam Sweep:
#   - Multiple CONFIG variants (teacher depth/heads, KD alpha/T, DropEdge)
#   - Shorter runs for sweep mode
#   - Automatic best-config selection
#   - Summary appended to report
#
# Auto GPU (T4 etc. on Colab/Kaggle) if available.
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import os
import time
import random
import json
from copy import deepcopy
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
    k_hop_subgraph,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ============================================================
# 0) Base Config, Seed, Device, Plot Helpers
# ============================================================
BASE_CONFIG = {
    "experiment_tag": "UltraGAT_CiteSeer_v3",
    "dataset_name": "CiteSeer",

    # Teacher architecture
    "teacher_hidden_dim": 16,
    "teacher_heads": (4, 4, 2),

    # Epoch budgets (max)
    "epochs_teacher": 3000,
    "use_early_stopping_teacher": True,
    "teacher_patience": 300,

    "epochs_student": 1600,
    "use_early_stopping_student": True,
    "student_patience": 300,

    # Regularisation
    "dropedge_base_p": 0.25,
    "feature_mask_p": 0.10,
    "lp_aux_weight": 0.1,

    # LP pseudo-label KL (semi-supervised)
    "use_lp_pseudo": True,
    "lp_pseudo_conf_thr": 0.9,
    "lp_pseudo_weight": 0.1,

    # Contrastive stub (OFF)
    "contrastive_weight": 0.0,
    "temperature_contrastive": 0.5,
    "use_struct_features": True,
    "use_lp_aux": True,
    "use_contrastive": False,

    # KD hyperparams
    "distill_alpha": 0.6,
    "distill_temperature": 2.0,

    # Sweep mode override (see later)
    "sweep_short_mode": False,  # when True: override epochs to smaller values

    "seed": 42,
}

PLOT_DIR = "plots"
os.makedirs(PLOT_DIR, exist_ok=True)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(BASE_CONFIG["seed"])

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Device: CPU")


def save_fig(name: str, dpi: int = 150):
    path = os.path.join(PLOT_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.show()
    print(f"[+] Saved figure to {path}")


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.05)


# ============================================================
# 1) Data Prep: load + structural features
# ============================================================
def prepare_dataset(config):
    dataset = Planetoid(root="./data", name=config["dataset_name"])
    data = dataset[0]

    print(f"Dataset: {dataset}")
    print("-------------------")
    print(f"Number of graphs: {len(dataset)}")
    print(f"Number of nodes: {data.num_nodes}")
    print(f"Number of features: {dataset.num_features}")
    print(f"Number of classes: {dataset.num_classes}")

    print("\nGraph:")
    print("------")
    print(f"Edges are directed: {data.is_directed()}")
    print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
    print(f"Graph has self-loops: {data.has_self_loops()}")

    isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
    print(f"Number of isolated nodes = {isolated}")

    # Undirect + add self-loops
    data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
    data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

    # Graph layout + degree dist only for main (not for sweep-short)
    if not config.get("sweep_short_mode", False):
        G = to_networkx(data, to_undirected=True)
        plt.figure(figsize=(8, 8))
        plt.axis("off")
        nx.draw_networkx(
            G,
            pos=nx.spring_layout(G, seed=0),
            with_labels=False,
            node_size=20,
            node_color=data.y,
            width=0.5,
            edge_color="grey",
        )
        plt.title(f"{config['dataset_name']} Graph (Node-colored by Label)")
        save_fig("graph_layout.png")

        degrees_arr = degree(data.edge_index[0]).numpy()
        deg_counts = Counter(degrees_arr)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.set_xlabel("Node degree")
        ax.set_ylabel("Number of nodes")
        plt.bar(deg_counts.keys(), deg_counts.values(), color="#0A047A")
        plt.title("Node Degree Distribution")
        plt.tight_layout()
        save_fig("degree_distribution.png")

    # Feature scaling + log-degree augmentation
    x_np = data.x.numpy()
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x_np)

    deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
    log_deg = np.log1p(deg_np).reshape(-1, 1)

    if config["use_struct_features"]:
        x_aug = np.concatenate([x_scaled, log_deg], axis=1)
    else:
        x_aug = x_scaled

    data.x = torch.tensor(x_aug, dtype=torch.float32)
    in_dim = data.x.size(1)

    print(
        f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
        f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
    )

    data = data.to(device)
    return dataset, data, in_dim


# ============================================================
# 2) Utilities
# ============================================================
def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    if y.numel() == 0:
        return 0.0
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index
    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p
    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)
    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


# ============================================================
# 3) Models: BetterGAT Teacher + StudentGCN
# ============================================================
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(4, 4, 2),
        dropout: float = 0.6,
        feature_mask_p: float = 0.10,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=True)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=True)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        self.res_fc = nn.Linear(dim_in, dim_out)

        self.opt = torch.optim.Adam(
            self.parameters(),
            lr=0.004,
            weight_decay=1e-3
        )

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)
        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


class StudentGCN(nn.Module):
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.gcn1(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.gcn2(h, edge_index)
        return F.log_softmax(h, dim=1)


# Contrastive stub (kept OFF)
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9

    labels_eq = y.unsqueeze(0) == y.unsqueeze(1)
    labels_eq = labels_eq.float()

    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ============================================================
# 4) Training: Teacher & Student + KD
# ============================================================
def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_teacher(model: BetterGAT, data, cfg):
    ce = nn.CrossEntropyLoss()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    # override epochs if sweep_short_mode
    E = cfg["epochs_teacher"]
    if cfg.get("sweep_short_mode", False):
        E = min(E, 1200)

    base_drop = cfg["dropedge_base_p"]

    best_val_acc = -1.0
    best_epoch = -1
    best_state = None
    epochs_no_improve = 0

    print(f"\n🧠 Training TEACHER for up to {E} epochs "
          f"(early stopping={cfg['use_early_stopping_teacher']})...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)

        emb, logp = model(data.x, edge_index_aug, training=True)

        ce_loss = ce(logp[data.train_mask], data.y[data.train_mask])
        loss = ce_loss

        # LP aux
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)
            lp_probs = logp.exp()

        # LP pseudo-label KL
        if cfg.get("use_lp_pseudo", False):
            with torch.no_grad():
                conf, _ = lp_probs.max(dim=1)
                pseudo_mask = (~data.train_mask) & (conf > cfg["lp_pseudo_conf_thr"])
            if pseudo_mask.any():
                kl_pseudo = F.kl_div(
                    F.log_softmax(logp[pseudo_mask], dim=1),
                    lp_probs[pseudo_mask],
                    reduction="batchmean",
                )
                loss = loss + cfg["lp_pseudo_weight"] * kl_pseudo
            else:
                kl_pseudo = torch.tensor(0.0, device=data.x.device)
        else:
            kl_pseudo = torch.tensor(0.0, device=data.x.device)

        # Optional contrastive
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()

        # Eval
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_loss = ce(logp_val[data.val_mask], data.y[data.val_mask]).item()
            tr_acc = accuracy(logp[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_val[data.val_mask].argmax(1), data.y[data.val_mask])

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_loss)

        # ES
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | KL_p:{kl_pseudo:.3f} | Contr:{contr:.3f}"
            )

        if cfg["use_early_stopping_teacher"] and epochs_no_improve >= cfg["teacher_patience"]:
            print(f"\n⏹ Teacher early stopping at epoch {ep}, best epoch {best_epoch} "
                  f"with ValAcc={best_val_acc*100:.2f}%")
            break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    print(f"\n✅ Teacher finished; best ValAcc={best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    model.eval()
    with torch.no_grad():
        _, final_logp = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)

    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp


def train_student(student: StudentGCN, teacher_logp: torch.Tensor, data, cfg):
    E = cfg["epochs_student"]
    if cfg.get("sweep_short_mode", False):
        E = min(E, 800)

    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    train_acc_curve, val_acc_curve = [], []

    best_val_acc = -1.0
    best_epoch = -1
    best_state = None
    epochs_no_improve = 0

    print(f"\n🎓 Training STUDENT GCN for up to {E} epochs "
          f"(KD, early stopping={cfg['use_early_stopping_student']})...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(logp_s[data.train_mask].argmax(1), data.y[data.train_mask])
            val_acc = accuracy(logp_sv[data.val_mask].argmax(1), data.y[data.val_mask])
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = {k: v.cpu().clone() for k, v in student.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

        if cfg["use_early_stopping_student"] and epochs_no_improve >= cfg["student_patience"]:
            print(f"\n⏹ Student early stopping at epoch {ep}, best epoch {best_epoch} "
                  f"with ValAcc={best_val_acc*100:.2f}%")
            break

    if best_state is not None:
        student.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    print(f"\n✅ Student finished; best ValAcc={best_val_acc*100:.2f}% at epoch {best_epoch}\n")
    return student, train_acc_curve, val_acc_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred, logp


# Ensemble helper
@torch.no_grad()
def ensemble_performance(teacher_logp: torch.Tensor,
                         data,
                         student_logp: torch.Tensor):
    probs_teacher = teacher_logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs_teacher, data.edge_index)
    probs_student = student_logp.exp()

    probs_ens = (probs_teacher + lp_probs + probs_student) / 3.0
    pred_ens = probs_ens.argmax(1)

    ens_acc = accuracy(pred_ens[data.test_mask], data.y[data.test_mask])
    return ens_acc, pred_ens


# ============================================================
# 5) Diagnostics (plots etc.) – used only for main run
# ============================================================
def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)", fname=None):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(data.y.max().item()+1)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(data.y.max().item()+1)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    if fname is None:
        fname = title.lower().replace(" ", "_") + ".png"
    save_fig(fname)
    return cm


@torch.no_grad()
def tsne_static(model: BetterGAT, data, in_dim, cfg):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=cfg["teacher_hidden_dim"],
        dim_out=int(data.y.max().item())+1,
        heads=cfg["teacher_heads"],
        dropout=0.6,
        feature_mask_p=cfg["feature_mask_p"],
    ).to(device)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(n_components=2, init="pca", learning_rate="auto").fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(n_components=2, init="pca", learning_rate="auto").fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plt.tight_layout()
    save_fig("tsne_untrained_vs_trained.png")


@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs, color="#0A047A")

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    save_fig("teacher_accuracy_by_degree.png")


@torch.no_grad()
def per_class_accuracy(data, raw_pred, lp_pred, stu_pred, ens_pred, num_classes, title="Per-Class Accuracy"):
    spacer("Per-Class Accuracy (Teacher Raw vs LabelProp vs Student vs Ensemble)")
    y = data.y.cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()

    raw = raw_pred.cpu().numpy()
    lp = lp_pred.cpu().numpy()
    stu = stu_pred.cpu().numpy()
    ens = ens_pred.cpu().numpy()

    acc_raw, acc_lp, acc_stu, acc_ens = [], [], [], []

    for c in range(num_classes):
        mask = (y == c) & test_mask
        if mask.sum() == 0:
            acc_raw.append(0.0)
            acc_lp.append(0.0)
            acc_stu.append(0.0)
            acc_ens.append(0.0)
        else:
            acc_raw.append((raw[mask] == c).mean())
            acc_lp.append((lp[mask] == c).mean())
            acc_stu.append((stu[mask] == c).mean())
            acc_ens.append((ens[mask] == c).mean())

    x = np.arange(num_classes)
    width = 0.2

    plt.figure(figsize=(10, 4))
    plt.bar(x - 1.5*width, acc_raw, width, label="Teacher Raw")
    plt.bar(x - 0.5*width, acc_lp,  width, label="Teacher LP")
    plt.bar(x + 0.5*width, acc_stu, width, label="Student")
    plt.bar(x + 1.5*width, acc_ens, width, label="Ensemble")

    plt.xticks(x, list(range(num_classes)))
    plt.ylim(0, 1.05)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    save_fig("per_class_accuracy.png")

    return {
        "teacher_raw": acc_raw,
        "teacher_lp": acc_lp,
        "student": acc_stu,
        "ensemble": acc_ens,
    }


@torch.no_grad()
def mc_dropout_uncertainty(model: BetterGAT, data, num_samples: int = 30):
    spacer("MC-Dropout Uncertainty (Teacher)")
    model.train()

    probs_list = []
    for _ in range(num_samples):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))

    probs_mc = torch.cat(probs_list, dim=0)
    mean_probs = probs_mc.mean(dim=0)

    entropy = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)

    test_mask = data.test_mask
    _, logp_det = model(data.x, data.edge_index, training=False)
    pred_det = logp_det.argmax(1)

    correct_mask = (pred_det == data.y) & test_mask
    wrong_mask = (pred_det != data.y) & test_mask

    ent_correct = entropy[correct_mask].cpu().numpy()
    ent_wrong = entropy[wrong_mask].cpu().numpy()

    print(f"Avg entropy (correct test): {ent_correct.mean():.4f}")
    print(f"Avg entropy (wrong   test): {ent_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(ent_correct, bins=20, alpha=0.7, label="Correct", density=True)
    plt.hist(ent_wrong,   bins=20, alpha=0.7, label="Wrong",   density=True)
    plt.xlabel("Predictive Entropy")
    plt.ylabel("Density")
    plt.title("MC-Dropout Uncertainty on Test Nodes")
    plt.legend()
    plt.tight_layout()
    save_fig("mc_dropout_uncertainty.png")

    model.eval()
    return entropy, pred_det, float(ent_correct.mean()), float(ent_wrong.mean())


@torch.no_grad()
def calibration_plot(model: BetterGAT, data, n_bins: int = 10):
    spacer("Calibration (Teacher Raw Probs)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    probs = logp.exp()

    test_mask = data.test_mask
    y_true = data.y[test_mask]
    probs_test = probs[test_mask]

    conf, preds = probs_test.max(dim=1)
    conf = conf.cpu().numpy()
    preds = preds.cpu().numpy()
    y_true = y_true.cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    accs, avg_confs, counts = [], [], []
    total = len(conf)
    ece = 0.0

    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            accs.append(0.0)
            avg_confs.append(0.0)
            counts.append(0)
            continue
        counts.append(int(mask.sum()))
        avg_conf = conf[mask].mean()
        accuracy_b = (preds[mask] == y_true[mask]).mean()
        accs.append(accuracy_b)
        avg_confs.append(avg_conf)
        ece += (mask.sum() / total) * abs(accuracy_b - avg_conf)

    print(f"Expected Calibration Error (ECE): {ece:.4f}")

    centers = 0.5 * (bins[:-1] + bins[1:])
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    plt.bar(centers, accs, width=1.0 / n_bins, alpha=0.7, edgecolor="k", label="Accuracy")
    plt.plot(centers, avg_confs, "o-", label="Avg Confidence")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Teacher)")
    plt.legend()
    plt.tight_layout()
    save_fig("calibration_reliability_diagram.png")

    return float(ece)


@torch.no_grad()
def ego_graph_misclassified(model: BetterGAT, data, center_k: int = 2):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)

    test_mask = data.test_mask
    wrong_nodes = torch.where((pred != data.y) & test_mask)[0]
    if wrong_nodes.numel() == 0:
        print("No misclassified test nodes – nice!")
        return

    center = int(wrong_nodes[0].item())
    print(f"Visualising ego-graph for misclassified test node {center}")

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        center, num_hops=center_k, edge_index=data.edge_index, relabel_nodes=True
    )

    G_sub = nx.Graph()
    G_sub.add_edges_from(edge_index_sub.cpu().t().numpy())
    pos = nx.spring_layout(G_sub, seed=0)

    true_labels = data.y[subset].cpu().numpy()
    pred_labels = pred[subset].cpu().numpy()
    correct_flags = (true_labels == pred_labels)

    colors = ["#1f77b4" if c else "#d62728" for c in correct_flags]

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        G_sub,
        pos=pos,
        node_color=colors,
        node_size=200,
        with_labels=False,
        edge_color="gray",
    )
    plt.title("Ego-Graph Around Misclassified Node (Blue=Correct, Red=Wrong)")
    plt.axis("off")
    save_fig("ego_graph_misclassified.png")


@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, n_components: int = 2):
    if n_components not in (2, 3):
        raise ValueError("n_components must be 2 or 3")

    spacer(f"Interactive t-SNE ({n_components}D) – Teacher Embeddings")

    model.eval()
    emb, _ = model(data.x, data.edge_index, training=False)
    emb_np = emb.cpu().numpy()
    labels_np = data.y.cpu().numpy()

    tsne = TSNE(
        n_components=n_components, init="pca", learning_rate="auto"
    ).fit_transform(emb_np)

    if n_components == 2:
        fig = px.scatter(
            x=tsne[:, 0],
            y=tsne[:, 1],
            color=labels_np.astype(str),
            title="Interactive t-SNE (2D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Label"},
        )
    else:
        fig = px.scatter_3d(
            x=tsne[:, 0],
            y=tsne[:, 1],
            z=tsne[:, 2],
            color=labels_np.astype(str),
            title="Interactive t-SNE (3D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3", "color": "Label"},
        )

    fig.show()


# ============================================================
# 6) Markdown Report
# ============================================================
def generate_markdown_report(
    config,
    dataset_name,
    t_raw_acc,
    t_lp_acc,
    s_acc,
    ens_acc,
    ece,
    avg_ent_correct,
    avg_ent_wrong,
    t_best_epoch,
    t_best_val,
    s_best_epoch,
    s_best_val,
    per_class_stats,
    sweep_results=None,
    output_path="UltraGAT_report.md",
):
    lines = []
    lines.append(f"# Ultra-GAT++ Teacher–Student KD Report\n")
    lines.append(f"**Experiment tag:** `{config['experiment_tag']}`\n")
    lines.append(f"**Dataset:** `{dataset_name}`\n")
    lines.append("---\n")

    lines.append("## 1. Overall Results (Main Config)\n")
    lines.append("| Model | Test Accuracy |\n| --- | --- |\n")
    lines.append(f"| Teacher (Raw logits) | {t_raw_acc*100:.2f}% |\n")
    lines.append(f"| Teacher + Label Propagation | {t_lp_acc*100:.2f}% |\n")
    lines.append(f"| Student GCN (KD) | {s_acc*100:.2f}% |\n")
    lines.append(f"| Ensemble (Teacher + LP + Student) | {ens_acc*100:.2f}% |\n\n")

    lines.append(f"- Teacher best validation accuracy: **{t_best_val*100:.2f}%** at epoch **{t_best_epoch}**\n")
    lines.append(f"- Student best validation accuracy: **{s_best_val*100:.2f}%** at epoch **{s_best_epoch}**\n\n")

    lines.append("## 2. Calibration & Uncertainty (Teacher)\n")
    lines.append(f"- Expected Calibration Error (ECE): **{ece:.4f}**\n")
    lines.append(f"- Average MC-Dropout entropy (correct test nodes): **{avg_ent_correct:.4f}**\n")
    lines.append(f"- Average MC-Dropout entropy (wrong test nodes): **{avg_ent_wrong:.4f}**\n\n")

    lines.append("Lower entropy for correct vs. wrong predictions indicates that the teacher is more confident when it is right, which is desirable for active learning, human-in-the-loop, or risk-aware deployment.\n\n")

    lines.append("## 3. Per-Class Accuracies (Test, Main Config)\n")
    lines.append("| Class | Teacher Raw | Teacher LP | Student | Ensemble |\n| --- | --- | --- | --- | --- |\n")
    num_classes = len(per_class_stats["teacher_raw"])
    for c in range(num_classes):
        lines.append(
            f"| {c} | "
            f"{per_class_stats['teacher_raw'][c]*100:.1f}% | "
            f"{per_class_stats['teacher_lp'][c]*100:.1f}% | "
            f"{per_class_stats['student'][c]*100:.1f}% | "
            f"{per_class_stats['ensemble'][c]*100:.1f}% |\n"
        )
    lines.append("\n")

    if sweep_results is not None and len(sweep_results) > 0:
        lines.append("## 4. Hyperparameter Sweep Summary\n")
        lines.append("Sweep was run with reduced epochs (`sweep_short_mode=True`) to compare relative trends.\n\n")

        lines.append("| Sweep ID | Teacher hdim | Heads | DropEdge p | KD α | KD T | Teacher Test | Student Test | Ensemble Test |\n")
        lines.append("| --- | --- | --- | --- | --- | --- | --- | --- | --- |\n")
        for r in sweep_results:
            lines.append(
                f"| {r['id']} | {r['teacher_hidden_dim']} | {r['teacher_heads']} | "
                f"{r['dropedge_base_p']:.2f} | {r['distill_alpha']:.2f} | {r['distill_temperature']:.1f} | "
                f"{r['teacher_test_acc']*100:.2f}% | {r['student_test_acc']*100:.2f}% | {r['ensemble_test_acc']*100:.2f}% |\n"
            )
        lines.append("\n")

        best = max(sweep_results, key=lambda x: x["ensemble_test_acc"])
        lines.append(
            f"**Best sweep configuration by ensemble Test accuracy:** "
            f"Sweep `{best['id']}` with Ensemble Test = **{best['ensemble_test_acc']*100:.2f}%**.\n\n"
        )

    lines.append("## 5. Configuration (Main Config JSON)\n")
    lines.append("```json\n")
    lines.append(json.dumps(config, indent=2))
    lines.append("\n```\n")

    lines.append("## 6. Generated Plots & Artifacts\n")
    lines.append("- Static plots are saved in `plots/`:\n")
    lines.append("  - Graph + structure: `graph_layout.png`, `degree_distribution.png`\n")
    lines.append("  - Learning curves: `teacher_accuracy_curves.png`, `teacher_loss_curves.png`, `student_accuracy_curves.png`\n")
    lines.append("  - Confusions: `teacher_confusion_raw.png`, `teacher_confusion_lp.png`, `student_confusion_kd.png`, `ensemble_confusion.png`\n")
    lines.append("  - `per_class_accuracy.png`, `teacher_accuracy_by_degree.png`\n")
    lines.append("  - Uncertainty & calibration: `mc_dropout_uncertainty.png`, `calibration_reliability_diagram.png`\n")
    lines.append("  - Local structure: `ego_graph_misclassified.png`, `tsne_untrained_vs_trained.png`\n")
    lines.append("- Interactive Plotly t-SNE (2D/3D) is rendered inline in the notebook.\n\n")

    lines.append("> Ultra-GAT++ implements a stacked GATv2 teacher with DropEdge, feature masking, structural augmentation, label-propagation consistency, LP-based pseudo-label regularisation, and a compact GCN student distilled via knowledge distillation. A probability-space ensemble of Teacher, LP-smoothed Teacher, and Student boosts robustness and accuracy.\n")

    with open(output_path, "w") as f:
        f.writelines(lines)

    print(f"\n[+] Markdown report written to: {output_path}")


# ============================================================
# 7) Hyperparameter Sweep Helper
# ============================================================
def build_config_variants(base_config):
    """
    Define a small grid of configs to sweep over.
    Each entry overrides some keys from base_config.
    """
    variants = []

    # Sweep 1: slightly shallower heads, weaker DropEdge, softer KD
    cfg1 = deepcopy(base_config)
    cfg1.update({
        "experiment_tag": base_config["experiment_tag"] + "_sweep1",
        "teacher_hidden_dim": 16,
        "teacher_heads": (4, 4, 2),
        "dropedge_base_p": 0.15,
        "distill_alpha": 0.6,
        "distill_temperature": 2.0,
        "sweep_short_mode": True,
    })
    variants.append(("sweep1", cfg1))

    # Sweep 2: stronger regularisation, slightly higher KD alpha
    cfg2 = deepcopy(base_config)
    cfg2.update({
        "experiment_tag": base_config["experiment_tag"] + "_sweep2",
        "teacher_hidden_dim": 16,
        "teacher_heads": (8, 4, 2),
        "dropedge_base_p": 0.30,
        "distill_alpha": 0.7,
        "distill_temperature": 2.0,
        "sweep_short_mode": True,
    })
    variants.append(("sweep2", cfg2))

    # Sweep 3: deeper teacher hidden, lower KD alpha, higher T
    cfg3 = deepcopy(base_config)
    cfg3.update({
        "experiment_tag": base_config["experiment_tag"] + "_sweep3",
        "teacher_hidden_dim": 32,
        "teacher_heads": (4, 4, 2),
        "dropedge_base_p": 0.20,
        "distill_alpha": 0.5,
        "distill_temperature": 3.0,
        "sweep_short_mode": True,
    })
    variants.append(("sweep3", cfg3))

    return variants


def run_single_experiment(config, dataset=None, data=None, in_dim=None):
    """
    Core pipeline (no diagnostics, just metrics).
    Used by sweep AND by main experiment if desired.
    """
    if dataset is None or data is None or in_dim is None:
        dataset, data, in_dim = prepare_dataset(config)

    num_classes = int(data.y.max().item()) + 1

    teacher = BetterGAT(
        dim_in=in_dim,
        dim_h=config["teacher_hidden_dim"],
        dim_out=num_classes,
        heads=config["teacher_heads"],
        dropout=0.6,
        feature_mask_p=config["feature_mask_p"],
    ).to(device)

    teacher, teacher_logp, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss, t_best_epoch, t_best_val = train_teacher(
        teacher, data, config
    )
    t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, t_logp_eval = test_teacher(teacher, data)

    student = StudentGCN(in_dim, 16, num_classes).to(device)
    student, s_tr_acc, s_val_acc, s_best_epoch, s_best_val = train_student(
        student, teacher_logp, data, config
    )
    s_acc, s_pred, s_logp_eval = test_student(student, data)

    ens_acc, ens_pred = ensemble_performance(t_logp_eval, data, s_logp_eval)

    metrics = {
        "teacher_test_acc": t_raw_acc,
        "teacher_lp_test_acc": t_lp_acc,
        "student_test_acc": s_acc,
        "ensemble_test_acc": ens_acc,
        "t_best_epoch": t_best_epoch,
        "t_best_val": t_best_val,
        "s_best_epoch": s_best_epoch,
        "s_best_val": s_best_val,
    }

    return {
        "dataset": dataset,
        "data": data,
        "in_dim": in_dim,
        "teacher": teacher,
        "student": student,
        "t_logp_eval": t_logp_eval,
        "s_logp_eval": s_logp_eval,
        "t_tr_acc": t_tr_acc,
        "t_val_acc": t_val_acc,
        "t_tr_loss": t_tr_loss,
        "t_val_loss": t_val_loss,
        "t_raw_pred": t_raw_pred,
        "t_lp_pred": t_lp_pred,
        "s_pred": s_pred,
        "ens_pred": ens_pred,
        "metrics": metrics,
    }


def run_sweep(base_config):
    sweep_variants = build_config_variants(base_config)
    sweep_results = []

    print("\n================ HYPERPARAM SWEEP START ================\n")

    for sweep_id, cfg in sweep_variants:
        print(f"\n========== Running {sweep_id} ==========\n")
        # For speed and isolation, reload dataset fresh per config
        out = run_single_experiment(cfg)
        m = out["metrics"]
        sweep_results.append({
            "id": sweep_id,
            "teacher_hidden_dim": cfg["teacher_hidden_dim"],
            "teacher_heads": cfg["teacher_heads"],
            "dropedge_base_p": cfg["dropedge_base_p"],
            "distill_alpha": cfg["distill_alpha"],
            "distill_temperature": cfg["distill_temperature"],
            "teacher_test_acc": m["teacher_test_acc"],
            "student_test_acc": m["student_test_acc"],
            "ensemble_test_acc": m["ensemble_test_acc"],
        })

        print(
            f"[{sweep_id}] Teacher Test: {m['teacher_test_acc']*100:.2f}% | "
            f"Student Test: {m['student_test_acc']*100:.2f}% | "
            f"Ensemble Test: {m['ensemble_test_acc']*100:.2f}%"
        )

    print("\n================ HYPERPARAM SWEEP END =================\n")
    return sweep_results


# ============================================================
# 8) MAIN EXECUTION: Full experiment + sweep + report
# ============================================================
# 8.1 Main experiment (full diagnostics)
main_config = deepcopy(BASE_CONFIG)
main_config["sweep_short_mode"] = False
dataset, data, in_dim = prepare_dataset(main_config)

num_classes = int(data.y.max().item()) + 1

teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=main_config["teacher_hidden_dim"],
    dim_out=num_classes,
    heads=main_config["teacher_heads"],
    dropout=0.6,
    feature_mask_p=main_config["feature_mask_p"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss, t_best_epoch, t_best_val = train_teacher(
    teacher, data, main_config
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, t_logp_eval = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

student = StudentGCN(in_dim, 16, num_classes).to(device)
student, s_tr_acc, s_val_acc, s_best_epoch, s_best_val = train_student(
    student, teacher_logp, data, main_config
)
s_acc, s_pred, s_logp_eval = test_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {s_acc*100:.2f}%")

ens_acc, ens_pred = ensemble_performance(t_logp_eval, data, s_logp_eval)
print(f"🤝 Ensemble Test Accuracy (Teacher + LP + Student): {ens_acc*100:.2f}%")

# Diagnostics (only once for main)
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(t_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
save_fig("teacher_accuracy_curves.png")

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss", color="#1f77b4")
plt.plot(t_val_loss, label="Val Loss", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
save_fig("teacher_loss_curves.png")

spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc", color="#1f77b4")
plt.plot(s_val_acc, label="Val Acc", color="#ff7f0e")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student GCN (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
save_fig("student_accuracy_curves.png")

tsne_static(teacher, data, in_dim, main_config)
accuracy_by_degree(teacher, data)

cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)", "teacher_confusion_raw.png")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)", "teacher_confusion_lp.png")
cm_student = plot_confusion(data, s_pred, "Student Confusion (KD, Test)", "student_confusion_kd.png")
cm_ensemble = plot_confusion(data, ens_pred, "Ensemble Confusion (Test)", "ensemble_confusion.png")

per_class_stats = per_class_accuracy(
    data, t_raw_pred, t_lp_pred, s_pred, ens_pred, num_classes
)
entropy_mc, pred_det, avg_ent_correct, avg_ent_wrong = mc_dropout_uncertainty(teacher, data)
ece_val = calibration_plot(teacher, data)
ego_graph_misclassified(teacher, data)
interactive_tsne(teacher, data, n_components=2)
interactive_tsne(teacher, data, n_components=3)

# 8.2 Hyperparameter sweep
sweep_results = run_sweep(BASE_CONFIG)

# 8.3 Markdown report
generate_markdown_report(
    config=main_config,
    dataset_name=main_config["dataset_name"],
    t_raw_acc=t_raw_acc,
    t_lp_acc=t_lp_acc,
    s_acc=s_acc,
    ens_acc=ens_acc,
    ece=ece_val,
    avg_ent_correct=avg_ent_correct,
    avg_ent_wrong=avg_ent_wrong,
    t_best_epoch=t_best_epoch,
    t_best_val=t_best_val,
    s_best_epoch=s_best_epoch,
    s_best_val=s_best_val,
    per_class_stats=per_class_stats,
    sweep_results=sweep_results,
    output_path="UltraGAT_report.md",
)

print("\n✅ Ultra-GAT++ lab complete: main experiment, sweep, ensemble, diagnostics, and report ready.")


# ============================================================
# 9) (Optional) Package Layout Hint – HOW TO TURN INTO A LIB
# ============================================================
"""
Suggested repo structure (for GitHub):

ultragat/
  ├── ultragat/
  │     ├── __init__.py
  │     ├── config.py          # holds BASE_CONFIG & build_config_variants
  │     ├── data.py            # prepare_dataset()
  │     ├── models.py          # BetterGAT, StudentGCN, blocks
  │     ├── train.py           # train_teacher, train_student, distillation_loss
  │     ├── eval.py            # test_teacher, test_student, ensemble_performance
  │     ├── diagnostics.py     # plots, t-SNE, calibration, uncertainty
  │     ├── report.py          # generate_markdown_report
  │     └── sweep.py           # run_sweep(), run_single_experiment()
  ├── experiments/
  │     ├── citeseer_ultragat_kd.py   # calls into ultragat.* modules
  │     └── ...
  ├── requirements.txt
  ├── README.md
  └── setup.py / pyproject.toml

In your notebook you already have all functions.
You can copy-paste each logical block into the module above and then
import them in a clean `experiments/citeseer_ultragat_kd.py` script.
"""

In [ ]:
# ============================================================
# Ultra-GAT++ with Teacher–Student KD on CiteSeer
# ------------------------------------------------------------
# - Dataset: CiteSeer (Planetoid)
# - Teacher: BetterGAT (GATv2-based, deeper, residuals, DropEdge)
# - Student: GCN distilled from Teacher
# - Tricks:
#     * DropEdge (annealed)
#     * Feature masking
#     * Log-degree structural features
#     * Label smoothing on CE
#     * LP auxiliary consistency loss
#     * LP pseudo-label KL loss (semi-supervised regularisation)
#     * Knowledge Distillation (KD) with temperature
# - Diagnostics:
#     * Learning curves
#     * Confusion matrices
#     * Per-class accuracy (Teacher Raw / LabelProp / Student)
#     * t-SNE (untrained vs trained Teacher)
#     * Node-degree-wise accuracy
#     * MC-Dropout uncertainty histograms
#     * Calibration curve + ECE
#     * Ego-graph of misclassified node
#     * Interactive t-SNE in 2D & 3D (Plotly)
# - Auto-uses GPU (e.g. Tesla T4 on Colab/Kaggle) if available
# - Saves all static plots into ./plots/
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import os
import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
    k_hop_subgraph,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ============================================================
# 0) Config + Utilities
# ============================================================
BASE_CONFIG = {
    "experiment_tag": "UltraGAT_CiteSeer_v5",

    # Data
    "dataset_name": "CiteSeer",

    # Teacher architecture
    "teacher_hidden_dim": 16,
    "teacher_heads": (4, 4, 2),   # (h1, h2, h3)

    # Teacher training control
    "epochs_teacher": 3000,
    "use_early_stopping_teacher": True,
    "teacher_patience": 300,

    # Student training control
    "epochs_student": 1600,
    "use_early_stopping_student": True,
    "student_patience": 300,

    # Regularisation / augmentation
    "dropedge_base_p": 0.25,       # higher than before, annealed to 0
    "feature_mask_p": 0.10,
    "use_struct_features": True,   # add log-degree feature
    "label_smoothing": 0.10,       # smoothing for CE

    # Label propagation auxiliary consistency loss
    "use_lp_aux": True,
    "lp_aux_weight": 0.10,

    # LP-based pseudo-label KL regulariser (semi-supervised)
    "use_lp_pseudo": True,
    "lp_pseudo_conf_thr": 0.90,
    "lp_pseudo_weight": 0.10,

    # Contrastive (kept here as OFF, easy to extend later)
    "use_contrastive": False,
    "contrastive_weight": 0.0,
    "temperature_contrastive": 0.5,

    # KD hyperparams
    "distill_alpha": 0.60,       # weight for soft KD loss
    "distill_temperature": 2.0,  # KD temperature

    # Misc
    "sweep_short_mode": False,   # if True, lower epochs for quick tests
    "seed": 42,
}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


set_seed(BASE_CONFIG["seed"])
ensure_dir("plots")


# Auto-select device
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Device: CPU")


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.05)


def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    if y.numel() == 0:
        return 0.0
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


def smooth_one_hot(targets: torch.Tensor,
                   n_classes: int,
                   smoothing: float = 0.0) -> torch.Tensor:
    """
    Convert targets to smoothed one-hot (for label smoothing CE).
    """
    with torch.no_grad():
        assert 0.0 <= smoothing < 1.0
        confidence = 1.0 - smoothing
        label_shape = (targets.size(0), n_classes)
        smooth = torch.full(label_shape, smoothing / (n_classes - 1),
                            device=targets.device)
        smooth.scatter_(1, targets.unsqueeze(1), confidence)
        return smooth


# ============================================================
# 1) Dataset + Basic Stats + Structural Plots
# ============================================================
main_config = BASE_CONFIG.copy()

if main_config["sweep_short_mode"]:
    main_config["epochs_teacher"] = 400
    main_config["epochs_student"] = 400
    main_config["teacher_patience"] = 100
    main_config["student_patience"] = 100

dataset = Planetoid(root="./data", name=main_config["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Make graph undirected + add self-loops
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# Quick graph layout plot (saved)
spacer("Graph Layout (spring)")
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y.cpu().numpy(),
    width=0.5,
    edge_color="grey",
)
plt.title(f"{main_config['dataset_name']} Graph (Node-colored by Label)")
plt.tight_layout()
plt.savefig("plots/graph_layout.png", dpi=300)
plt.show()
print("[+] Saved figure to plots/graph_layout.png")

# Degree distribution plot (saved)
spacer("Node Degree Distribution")
degrees_arr = degree(data.edge_index[0]).cpu().numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
ax.bar(deg_counts.keys(), deg_counts.values())
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.savefig("plots/degree_distribution.png", dpi=300)
plt.show()
print("[+] Saved figure to plots/degree_distribution.png")

# ============================================================
# 2) Feature Scaling + Structural Augmentation
# ============================================================
x_np = data.x.cpu().numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).cpu().numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

if main_config["use_struct_features"]:
    x_aug = np.concatenate([x_scaled, log_deg], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)

print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

data = data.to(device)

# ============================================================
# 3) GATv2 Block + BetterGAT (Teacher)
# ============================================================
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(4, 4, 2),
        dropout: float = 0.6,
        feature_mask_p: float = 0.05,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # Residual linear head to support low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        # Optimiser: Adam (AdamW is also an option)
        self.opt = torch.optim.Adam(self.parameters(), lr=0.005, weight_decay=5e-4)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)

        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ============================================================
# 4) Optional Supervised Contrastive Stub (OFF by default)
# ============================================================
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9

    labels_eq = (y.unsqueeze(0) == y.unsqueeze(1)).float()
    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ============================================================
# 5) Teacher Training with Early Stopping + LP aux + LP pseudo
# ============================================================
def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    num_classes = dataset.num_classes
    ce = nn.KLDivLoss(reduction="batchmean")  # we'll use smoothed targets
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]
    smoothing = cfg["label_smoothing"]

    use_early = cfg["use_early_stopping_teacher"]
    patience = cfg["teacher_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    print(f"\n🧠 Training TEACHER for up to {E} epochs (early stopping={use_early})...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)
        emb, logp = model(data.x, edge_index_aug, training=True)

        # Label smoothing CE (implemented as KL with smoothed one-hot)
        y_train = data.y[data.train_mask]
        target_train = smooth_one_hot(
            y_train, dataset.num_classes, smoothing=smoothing  # <-- fixed call here
        )
        logp_train = logp[data.train_mask]
        ce_loss = ce(logp_train, target_train)
        loss = ce_loss

        # LP auxiliary consistency loss
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)

        # Optional LP pseudo-label KL (for confident unlabeled nodes)
        if cfg["use_lp_pseudo"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)  # [N, C]
                lp_conf, lp_pseudo = lp_probs.max(dim=1)

            unlabeled_mask = (~data.train_mask) & (~data.val_mask) & (~data.test_mask)
            high_conf_mask = unlabeled_mask & (lp_conf > cfg["lp_pseudo_conf_thr"])

            if high_conf_mask.any():
                logp_pseudo = logp[high_conf_mask]
                target_pseudo = lp_probs[high_conf_mask].detach()
                kl_pseudo = F.kl_div(
                    logp_pseudo, target_pseudo, reduction="batchmean"
                )
                loss = loss + cfg["lp_pseudo_weight"] * kl_pseudo
            else:
                kl_pseudo = torch.tensor(0.0, device=data.x.device)
        else:
            kl_pseudo = torch.tensor(0.0, device=data.x.device)

        # Optional contrastive (OFF by default)
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()

        # Eval
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_ce = F.nll_loss(
                logp_val[data.val_mask], data.y[data.val_mask]
            ).item()
            tr_acc = accuracy(
                logp[data.train_mask].argmax(1), data.y[data.train_mask]
            )
            val_acc = accuracy(
                logp_val[data.val_mask].argmax(1), data.y[data.val_mask]
            )

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_ce)

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_ce:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | KL_p:{kl_pseudo:.3f} | Contr:{contr:.3f}"
            )

        if use_early and wait >= patience:
            print(f"\n⏹ Early stopping triggered at epoch {ep}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"\n✅ Teacher training complete. Best Val Accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    model.eval()
    with torch.no_grad():
        _, final_logp = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)

    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp


# ============================================================
# 6) Student GCN + KD
# ============================================================
class StudentGCN(nn.Module):
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.gcn1(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.gcn2(h, edge_index)
        return F.log_softmax(h, dim=1)


def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_student(
    student: StudentGCN,
    teacher_logp: torch.Tensor,
    data,
    cfg,
):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    use_early = cfg["use_early_stopping_student"]
    patience = cfg["student_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    train_acc_curve, val_acc_curve = [], []

    print(f"\n🎓 Training STUDENT GCN for up to {E} epochs (KD, early stopping={use_early})...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(
                logp_s[data.train_mask].argmax(1), data.y[data.train_mask]
            )
            val_acc = accuracy(
                logp_sv[data.val_mask].argmax(1), data.y[data.val_mask]
            )
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = student.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

        if use_early and wait >= patience:
            print(f"\n⏹ Student early stopping triggered at epoch {ep}.")
            break

    if best_state is not None:
        student.load_state_dict(best_state)

    print(f"\n✅ Student training complete. Best Val Accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")
    return student, train_acc_curve, val_acc_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred


# ============================================================
# 7) Run Teacher + Student Training
# ============================================================
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=main_config["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=main_config["teacher_heads"],
    dropout=0.6,
    feature_mask_p=main_config["feature_mask_p"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss, t_best_epoch, t_best_val = train_teacher(
    teacher, data, main_config
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, _ = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

student = StudentGCN(in_dim, 32, dataset.num_classes).to(device)
student, s_tr_acc, s_val_acc, s_best_epoch, s_best_val = train_student(
    student, teacher_logp, data, main_config
)
s_acc, s_pred = test_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {s_acc*100:.2f}%")


# ============================================================
# 8) Plots & Diagnostics
# ============================================================
def plot_and_save(fig_name: str):
    plt.tight_layout()
    ensure_dir("plots")
    path = os.path.join("plots", fig_name)
    plt.savefig(path, dpi=300)
    print(f"[+] Saved figure to {path}")
    plt.show()


def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)", fname=None):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    if fname is None:
        fname = title.lower().replace(" ", "_").replace("(", "").replace(")", "") + ".png"
    plot_and_save(fname)
    return cm


# 8.1 Teacher learning curves
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc")
plt.plot(t_val_acc, label="Val Acc")
plt.axvline(t_best_epoch, linestyle="--", color="gray", label="Best Val Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("teacher_accuracy_curves.png")

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss")
plt.plot(t_val_loss, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("teacher_loss_curves.png")

# 8.2 Student learning curves
spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc")
plt.plot(s_val_acc, label="Val Acc")
plt.axvline(s_best_epoch, linestyle="--", color="gray", label="Best Val Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student GCN (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("student_accuracy_curves.png")


# 8.3 t-SNE (untrained vs trained teacher)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=main_config["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=main_config["teacher_heads"],
        dropout=0.6,
        feature_mask_p=main_config["feature_mask_p"],
    ).to(device)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plot_and_save("tsne_untrained_vs_trained_teacher.png")


tsne_static(teacher, data)


# 8.4 Accuracy by degree (teacher raw)
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs)

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plot_and_save("teacher_accuracy_by_degree.png")


accuracy_by_degree(teacher, data)

# 8.5 Confusion matrices
cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)",
                                fname="teacher_confusion_raw.png")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)",
                               fname="teacher_confusion_labelprop.png")
cm_student = plot_confusion(data, s_pred, "Student Confusion (KD, Test)",
                            fname="student_confusion_kd.png")


# 8.6 Per-Class Accuracy (Teacher Raw vs LP vs Student)
@torch.no_grad()
def per_class_accuracy(data, raw_pred, lp_pred, stu_pred, title="Per-Class Accuracy"):
    spacer("Per-Class Accuracy (Teacher Raw vs LabelProp vs Student)")
    y = data.y.cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()
    classes = np.arange(dataset.num_classes)

    raw = raw_pred.cpu().numpy()
    lp = lp_pred.cpu().numpy()
    stu = stu_pred.cpu().numpy()

    acc_raw, acc_lp, acc_stu = [], [], []

    for c in classes:
        mask = (y == c) & test_mask
        if mask.sum() == 0:
            acc_raw.append(0.0)
            acc_lp.append(0.0)
            acc_stu.append(0.0)
        else:
            acc_raw.append((raw[mask] == c).mean())
            acc_lp.append((lp[mask] == c).mean())
            acc_stu.append((stu[mask] == c).mean())

    x = np.arange(len(classes))
    width = 0.25

    plt.figure(figsize=(9, 4))
    plt.bar(x - width, acc_raw, width, label="Teacher Raw")
    plt.bar(x, acc_lp, width, label="Teacher LP")
    plt.bar(x + width, acc_stu, width, label="Student")

    plt.xticks(x, classes)
    plt.ylim(0, 1.05)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plot_and_save("per_class_accuracy.png")


per_class_accuracy(data, t_raw_pred, t_lp_pred, s_pred)


# 8.7 MC-Dropout Uncertainty (Teacher)
@torch.no_grad()
def mc_dropout_uncertainty(model: BetterGAT, data, num_samples: int = 30):
    spacer("MC-Dropout Uncertainty (Teacher)")
    model.train()  # enable dropout

    probs_list = []
    for _ in range(num_samples):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))

    probs_mc = torch.cat(probs_list, dim=0)  # [S, N, C]
    mean_probs = probs_mc.mean(dim=0)        # [N, C]

    entropy = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)

    test_mask = data.test_mask
    _, logp_det = model(data.x, data.edge_index, training=False)
    pred_det = logp_det.argmax(1)

    correct_mask = (pred_det == data.y) & test_mask
    wrong_mask = (pred_det != data.y) & test_mask

    ent_correct = entropy[correct_mask].cpu().numpy()
    ent_wrong = entropy[wrong_mask].cpu().numpy()

    print(f"Avg entropy (correct test): {ent_correct.mean():.4f}")
    print(f"Avg entropy (wrong   test): {ent_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(ent_correct, bins=20, alpha=0.7, label="Correct", density=True)
    plt.hist(ent_wrong, bins=20, alpha=0.7, label="Wrong", density=True)
    plt.xlabel("Predictive Entropy")
    plt.ylabel("Density")
    plt.title("MC-Dropout Uncertainty on Test Nodes")
    plt.legend()
    plot_and_save("mc_dropout_uncertainty.png")

    model.eval()
    return entropy, pred_det


entropy_mc, pred_det = mc_dropout_uncertainty(teacher, data)


# 8.8 Calibration (Teacher Raw Probs + ECE)
@torch.no_grad()
def calibration_plot(model: BetterGAT, data, n_bins: int = 10):
    spacer("Calibration (Teacher Raw Probs)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    probs = logp.exp()

    test_mask = data.test_mask
    y_true = data.y[test_mask]
    probs_test = probs[test_mask]

    conf, preds = probs_test.max(dim=1)
    conf = conf.cpu().numpy()
    preds = preds.cpu().numpy()
    y_true = y_true.cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    accs, avg_confs, counts = [], [], []
    total = len(conf)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            accs.append(0.0)
            avg_confs.append(0.0)
            counts.append(0)
            continue
        counts.append(int(mask.sum()))
        avg_conf = conf[mask].mean()
        accuracy_b = (preds[mask] == y_true[mask]).mean()
        accs.append(accuracy_b)
        avg_confs.append(avg_conf)
        ece += (mask.sum() / total) * abs(accuracy_b - avg_conf)

    print(f"Expected Calibration Error (ECE): {ece:.4f}")

    centers = 0.5 * (bins[:-1] + bins[1:])
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    plt.bar(centers, accs, width=1.0 / n_bins, alpha=0.7, edgecolor="k", label="Accuracy")
    plt.plot(centers, avg_confs, "o-", label="Avg Confidence")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Teacher)")
    plt.legend()
    plot_and_save("teacher_calibration_reliability.png")

    return ece


ece_val = calibration_plot(teacher, data)


# 8.9 Ego-Graph Around a Misclassified Test Node (Teacher Raw)
@torch.no_grad()
def ego_graph_misclassified(model: BetterGAT, data, center_k: int = 2):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)

    test_mask = data.test_mask
    wrong_nodes = torch.where((pred != data.y) & test_mask)[0]
    if wrong_nodes.numel() == 0:
        print("No misclassified test nodes – nice!")
        return

    center = int(wrong_nodes[0].item())
    print(f"Visualising ego-graph for misclassified test node {center}")

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        center, num_hops=center_k, edge_index=data.edge_index, relabel_nodes=True
    )

    G_sub = nx.Graph()
    G_sub.add_edges_from(edge_index_sub.cpu().t().numpy())
    pos = nx.spring_layout(G_sub, seed=0)

    true_labels = data.y[subset].cpu().numpy()
    pred_labels = pred[subset].cpu().numpy()
    correct_flags = (true_labels == pred_labels)

    colors = ["#1f77b4" if c else "#d62728" for c in correct_flags]

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        G_sub,
        pos=pos,
        node_color=colors,
        node_size=200,
        with_labels=False,
        edge_color="gray",
    )
    plt.title("Ego-Graph Around Misclassified Node (Blue=Correct, Red=Wrong)")
    plt.axis("off")
    plot_and_save("ego_graph_misclassified.png")


ego_graph_misclassified(teacher, data)


# 8.10 Interactive t-SNE (2D + 3D) with Plotly
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, n_components: int = 2):
    if n_components not in (2, 3):
        raise ValueError("n_components must be 2 or 3")

    spacer(f"Interactive t-SNE ({n_components}D) – Teacher Embeddings")

    model.eval()
    emb, _ = model(data.x, data.edge_index, training=False)
    emb_np = emb.cpu().numpy()
    labels_np = data.y.cpu().numpy()

    tsne = TSNE(
        n_components=n_components, init="pca", learning_rate="auto"
    ).fit_transform(emb_np)

    if n_components == 2:
        fig = px.scatter(
            x=tsne[:, 0],
            y=tsne[:, 1],
            color=labels_np.astype(str),
            title="Interactive t-SNE (2D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Label"},
        )
    else:
        fig = px.scatter_3d(
            x=tsne[:, 0],
            y=tsne[:, 1],
            z=tsne[:, 2],
            color=labels_np.astype(str),
            title="Interactive t-SNE (3D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3", "color": "Label"},
        )

    fig.show()


interactive_tsne(teacher, data, n_components=2)
interactive_tsne(teacher, data, n_components=3)

print("\n✅ Ultra-GAT++ Teacher + GCN Student KD lab finished.\n"
      "   - Teacher & Student trained with LP, KD, label smoothing, DropEdge\n"
      "   - Full diagnostics & plots saved in ./plots\n")

In [ ]:
# ============================================================
# Ultra-GAT++ v2 with Teacher–Student KD on CiteSeer
# ------------------------------------------------------------
# - Teacher: BetterGAT (GATv2-based, DropEdge, LP aux, LP pseudo, label smoothing)
# - Student: GCN distilled from MC-Dropout Teacher ensemble
# - Advancements:
#     * Cosine LR scheduler (optional) for teacher & student
#     * MC-Dropout teacher ensemble for KD targets
#     * Label smoothing CE
#     * LP consistency + LP pseudo-label regularisation
#     * DropEdge + feature masking + log-degree structural feature
# - Diagnostics:
#     * Learning curves, confusion matrices, per-class accuracy
#     * t-SNE (untrained vs trained)
#     * Accuracy vs node degree
#     * MC-Dropout uncertainty + calibration (ECE)
#     * Ego-graph visualisation for misclassified node
#     * Interactive Plotly t-SNE (2D & 3D)
# - Auto-uses GPU (T4) on Colab/Kaggle if available
# - Saves static plots in ./plots/
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import os
import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
    k_hop_subgraph,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ============================================================
# 0) Global Config + Utilities
# ============================================================
CONFIG = {
    "experiment_tag": "UltraGAT_CiteSeer_v2",

    # Dataset
    "dataset_name": "CiteSeer",

    # Teacher GAT architecture
    "teacher_hidden_dim": 16,
    "teacher_heads": (4, 4, 2),   # (h1, h2, h3)

    # Teacher training
    "epochs_teacher": 3000,
    "use_early_stopping_teacher": True,
    "teacher_patience": 300,
    "use_lr_scheduler_teacher": True,
    "teacher_scheduler_type": "cosine",   # ["cosine"]

    # Student training
    "epochs_student": 1600,
    "use_early_stopping_student": True,
    "student_patience": 300,
    "use_lr_scheduler_student": True,
    "student_scheduler_type": "cosine",   # ["cosine"]

    # Regularisation / augmentation
    "dropedge_base_p": 0.25,       # annealed to 0
    "feature_mask_p": 0.10,
    "use_struct_features": True,   # log-degree features
    "label_smoothing": 0.10,

    # Label propagation consistency aux
    "use_lp_aux": True,
    "lp_aux_weight": 0.10,

    # LP pseudo-label KL (semi-supervised)
    "use_lp_pseudo": True,
    "lp_pseudo_conf_thr": 0.90,
    "lp_pseudo_weight": 0.10,

    # Optional supervised contrastive (OFF by default)
    "use_contrastive": False,
    "contrastive_weight": 0.0,
    "temperature_contrastive": 0.5,

    # Knowledge Distillation
    "distill_alpha": 0.60,
    "distill_temperature": 2.0,

    # MC-Dropout teacher ensemble for KD
    "teacher_mc_samples_for_kd": 8,    # >1 = ensemble; 1 = deterministic

    # Misc
    "fast_mode": False,               # True = fewer epochs/patience (for debugging)
    "seed": 42,
}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.05)


def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    if y.numel() == 0:
        return 0.0
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


def smooth_one_hot(targets: torch.Tensor,
                   n_classes: int,
                   smoothing: float = 0.0) -> torch.Tensor:
    """
    Convert targets to smoothed one-hot (for label smoothing CE).
    """
    with torch.no_grad():
        assert 0.0 <= smoothing < 1.0
        confidence = 1.0 - smoothing
        label_shape = (targets.size(0), n_classes)
        smooth = torch.full(label_shape, smoothing / (n_classes - 1),
                            device=targets.device)
        smooth.scatter_(1, targets.unsqueeze(1), confidence)
        return smooth


set_seed(CONFIG["seed"])
ensure_dir("plots")

# Auto-select device
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Device: CPU")


# Fast mode for debugging
if CONFIG["fast_mode"]:
    CONFIG["epochs_teacher"] = 600
    CONFIG["epochs_student"] = 600
    CONFIG["teacher_patience"] = 150
    CONFIG["student_patience"] = 150


# ============================================================
# 1) Dataset + Stats + Basic Plots
# ============================================================
dataset = Planetoid(root="./data", name=CONFIG["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Make graph undirected + add self-loops
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# Graph layout (spring)
spacer("Graph Layout (spring)")
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y.cpu().numpy(),
    width=0.5,
    edge_color="grey",
)
plt.title(f"{CONFIG['dataset_name']} Graph (Node-colored by Label)")
plt.tight_layout()
plt.savefig("plots/graph_layout.png", dpi=300)
plt.show()
print("[+] Saved figure to plots/graph_layout.png")

# Degree distribution
spacer("Node Degree Distribution")
degrees_arr = degree(data.edge_index[0]).cpu().numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
ax.bar(deg_counts.keys(), deg_counts.values())
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.savefig("plots/degree_distribution.png", dpi=300)
plt.show()
print("[+] Saved figure to plots/degree_distribution.png")

# ============================================================
# 2) Feature Scaling + Structural Augmentation
# ============================================================
x_np = data.x.cpu().numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).cpu().numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

if CONFIG["use_struct_features"]:
    x_aug = np.concatenate([x_scaled, log_deg], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)

print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

data = data.to(device)


# ============================================================
# 3) GATv2 Block + BetterGAT (Teacher)
# ============================================================
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(4, 4, 2),
        dropout: float = 0.6,
        feature_mask_p: float = 0.05,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # Residual linear head to support low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        # Optimiser
        self.opt = torch.optim.Adam(self.parameters(), lr=0.005, weight_decay=5e-4)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)

        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ============================================================
# 4) Optional Supervised Contrastive Stub (OFF by default)
# ============================================================
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9

    labels_eq = (y.unsqueeze(0) == y.unsqueeze(1)).float()
    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ============================================================
# 5) Teacher Training + LP aux + LP pseudo + Cosine LR
# ============================================================
def build_scheduler(optimizer, cfg_epochs, scheduler_type: str):
    if scheduler_type == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg_epochs
        )
    else:
        return None


def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    num_classes = dataset.num_classes
    # label-smoothing CE implemented as KL to smoothed one-hot
    ce = nn.KLDivLoss(reduction="batchmean")
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]
    smoothing = cfg["label_smoothing"]

    use_early = cfg["use_early_stopping_teacher"]
    patience = cfg["teacher_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    # Optional cosine LR scheduler
    scheduler = None
    if cfg["use_lr_scheduler_teacher"]:
        scheduler = build_scheduler(model.opt, E, cfg["teacher_scheduler_type"])

    print(f"\n🧠 Training TEACHER for up to {E} epochs (early stopping={use_early})...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)
        emb, logp = model(data.x, edge_index_aug, training=True)

        # Label smoothing CE
        y_train = data.y[data.train_mask]
        target_train = smooth_one_hot(
            y_train, num_classes, smoothing=smoothing
        )
        logp_train = logp[data.train_mask]
        ce_loss = ce(logp_train, target_train)
        loss = ce_loss

        # LP auxiliary consistency
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)

        # LP pseudo-label KL for confident unlabeled nodes
        if cfg["use_lp_pseudo"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
                lp_conf, _ = lp_probs.max(dim=1)

            unlabeled_mask = (~data.train_mask) & (~data.val_mask) & (~data.test_mask)
            high_conf_mask = unlabeled_mask & (lp_conf > cfg["lp_pseudo_conf_thr"])

            if high_conf_mask.any():
                logp_pseudo = logp[high_conf_mask]
                target_pseudo = lp_probs[high_conf_mask].detach()
                kl_pseudo = F.kl_div(
                    logp_pseudo, target_pseudo, reduction="batchmean"
                )
                loss = loss + cfg["lp_pseudo_weight"] * kl_pseudo
            else:
                kl_pseudo = torch.tensor(0.0, device=data.x.device)
        else:
            kl_pseudo = torch.tensor(0.0, device=data.x.device)

        # Optional contrastive
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()
        if scheduler is not None:
            scheduler.step()

        # Eval
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_ce = F.nll_loss(
                logp_val[data.val_mask], data.y[data.val_mask]
            ).item()
            tr_acc = accuracy(
                logp[data.train_mask].argmax(1), data.y[data.train_mask]
            )
            val_acc = accuracy(
                logp_val[data.val_mask].argmax(1), data.y[data.val_mask]
            )

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_ce)

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_ce:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | KL_p:{kl_pseudo:.3f} | Contr:{contr:.3f}"
            )

        if use_early and wait >= patience:
            print(f"\n⏹ Early stopping triggered at epoch {ep}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"\n✅ Teacher training complete. Best Val Accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    model.eval()
    with torch.no_grad():
        _, final_logp = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)

    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp


# ============================================================
# 6) MC-Dropout Teacher Ensemble for KD
# ============================================================
@torch.no_grad()
def get_teacher_kd_logits(model: BetterGAT, data, cfg):
    """
    Returns log-probabilities to use for KD.
    If teacher_mc_samples_for_kd > 1: MC-Dropout ensemble.
    Else: deterministic teacher logp.
    """
    S = cfg["teacher_mc_samples_for_kd"]
    if S <= 1:
        model.eval()
        _, logp = model(data.x, data.edge_index, training=False)
        return logp

    spacer(f"MC-Dropout teacher ensemble for KD (S={S})")
    model.train()
    probs_list = []
    for _ in range(S):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))
    probs_mean = torch.cat(probs_list, dim=0).mean(dim=0)
    model.eval()
    return (probs_mean + 1e-12).log()


# ============================================================
# 7) Student GCN + KD + Cosine LR
# ============================================================
class StudentGCN(nn.Module):
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.gcn1(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.gcn2(h, edge_index)
        return F.log_softmax(h, dim=1)


def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_student(
    student: StudentGCN,
    teacher_logp_kd: torch.Tensor,
    data,
    cfg,
):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    use_early = cfg["use_early_stopping_student"]
    patience = cfg["student_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    train_acc_curve, val_acc_curve = [], []

    # Optional cosine LR scheduler
    scheduler = None
    if cfg["use_lr_scheduler_student"]:
        scheduler = build_scheduler(student.opt, E, cfg["student_scheduler_type"])

    print(f"\n🎓 Training STUDENT GCN for up to {E} epochs (KD, early stopping={use_early})...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp_kd, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()
        if scheduler is not None:
            scheduler.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(
                logp_s[data.train_mask].argmax(1), data.y[data.train_mask]
            )
            val_acc = accuracy(
                logp_sv[data.val_mask].argmax(1), data.y[data.val_mask]
            )
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = student.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

        if use_early and wait >= patience:
            print(f"\n⏹ Student early stopping triggered at epoch {ep}.")
            break

    if best_state is not None:
        student.load_state_dict(best_state)

    print(f"\n✅ Student training complete. Best Val Accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")
    return student, train_acc_curve, val_acc_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred


# ============================================================
# 8) Run Teacher + Student (with MC-KD)
# ============================================================
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=CONFIG["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=CONFIG["teacher_heads"],
    dropout=0.6,
    feature_mask_p=CONFIG["feature_mask_p"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp_det, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss, t_best_epoch, t_best_val = train_teacher(
    teacher, data, CONFIG
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, _ = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

# MC-Dropout teacher ensemble for KD targets
teacher_logp_kd = get_teacher_kd_logits(teacher, data, CONFIG)

student = StudentGCN(in_dim, 32, dataset.num_classes).to(device)
student, s_tr_acc, s_val_acc, s_best_epoch, s_best_val = train_student(
    student, teacher_logp_kd, data, CONFIG
)
s_acc, s_pred = test_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {s_acc*100:.2f}%")

print("\nSummary:")
print(f"  Teacher (Raw) Test Acc:       {t_raw_acc*100:.2f}%")
print(f"  Teacher (LP-refined) Test Acc:{t_lp_acc*100:.2f}%")
print(f"  Student (KD, MC Teacher) Acc: {s_acc*100:.2f}%")


# ============================================================
# 9) Plots & Diagnostics
# ============================================================
def plot_and_save(fig_name: str):
    plt.tight_layout()
    ensure_dir("plots")
    path = os.path.join("plots", fig_name)
    plt.savefig(path, dpi=300)
    print(f"[+] Saved figure to {path}")
    plt.show()


def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)", fname=None):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    if fname is None:
        fname = title.lower().replace(" ", "_").replace("(", "").replace(")", "") + ".png"
    plot_and_save(fname)
    return cm


# 9.1 Teacher learning curves
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc")
plt.plot(t_val_acc, label="Val Acc")
plt.axvline(t_best_epoch, linestyle="--", color="gray", label="Best Val Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("teacher_accuracy_curves.png")

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss")
plt.plot(t_val_loss, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("teacher_loss_curves.png")

# 9.2 Student learning curves
spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc")
plt.plot(s_val_acc, label="Val Acc")
plt.axvline(s_best_epoch, linestyle="--", color="gray", label="Best Val Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student GCN (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("student_accuracy_curves.png")


# 9.3 t-SNE (untrained vs trained teacher)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
    ).to(device)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plot_and_save("tsne_untrained_vs_trained_teacher.png")


tsne_static(teacher, data)


# 9.4 Accuracy by degree (teacher raw)
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs)

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plot_and_save("teacher_accuracy_by_degree.png")


accuracy_by_degree(teacher, data)

# 9.5 Confusion matrices
cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)",
                                fname="teacher_confusion_raw.png")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)",
                               fname="teacher_confusion_labelprop.png")
cm_student = plot_confusion(data, s_pred, "Student Confusion (KD, Test)",
                            fname="student_confusion_kd.png")


# 9.6 Per-Class Accuracy (Teacher Raw vs LP vs Student)
@torch.no_grad()
def per_class_accuracy(data, raw_pred, lp_pred, stu_pred, title="Per-Class Accuracy"):
    spacer("Per-Class Accuracy (Teacher Raw vs LabelProp vs Student)")
    y = data.y.cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()
    classes = np.arange(dataset.num_classes)

    raw = raw_pred.cpu().numpy()
    lp = lp_pred.cpu().numpy()
    stu = stu_pred.cpu().numpy()

    acc_raw, acc_lp, acc_stu = [], [], []

    for c in classes:
        mask = (y == c) & test_mask
        if mask.sum() == 0:
            acc_raw.append(0.0)
            acc_lp.append(0.0)
            acc_stu.append(0.0)
        else:
            acc_raw.append((raw[mask] == c).mean())
            acc_lp.append((lp[mask] == c).mean())
            acc_stu.append((stu[mask] == c).mean())

    x = np.arange(len(classes))
    width = 0.25

    plt.figure(figsize=(9, 4))
    plt.bar(x - width, acc_raw, width, label="Teacher Raw")
    plt.bar(x, acc_lp, width, label="Teacher LP")
    plt.bar(x + width, acc_stu, width, label="Student")

    plt.xticks(x, classes)
    plt.ylim(0, 1.05)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plot_and_save("per_class_accuracy.png")


per_class_accuracy(data, t_raw_pred, t_lp_pred, s_pred)


# 9.7 MC-Dropout Uncertainty (Teacher)
@torch.no_grad()
def mc_dropout_uncertainty(model: BetterGAT, data, num_samples: int = 30):
    spacer("MC-Dropout Uncertainty (Teacher)")
    model.train()  # enable dropout

    probs_list = []
    for _ in range(num_samples):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))

    probs_mc = torch.cat(probs_list, dim=0)  # [S, N, C]
    mean_probs = probs_mc.mean(dim=0)        # [N, C]

    entropy = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)

    test_mask = data.test_mask
    _, logp_det = model(data.x, data.edge_index, training=False)
    pred_det = logp_det.argmax(1)

    correct_mask = (pred_det == data.y) & test_mask
    wrong_mask = (pred_det != data.y) & test_mask

    ent_correct = entropy[correct_mask].cpu().numpy()
    ent_wrong = entropy[wrong_mask].cpu().numpy()

    print(f"Avg entropy (correct test): {ent_correct.mean():.4f}")
    print(f"Avg entropy (wrong   test): {ent_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(ent_correct, bins=20, alpha=0.7, label="Correct", density=True)
    plt.hist(ent_wrong, bins=20, alpha=0.7, label="Wrong", density=True)
    plt.xlabel("Predictive Entropy")
    plt.ylabel("Density")
    plt.title("MC-Dropout Uncertainty on Test Nodes")
    plt.legend()
    plot_and_save("mc_dropout_uncertainty.png")

    model.eval()
    return entropy, pred_det


entropy_mc, pred_det = mc_dropout_uncertainty(teacher, data)


# 9.8 Calibration (Teacher Raw Probs + ECE)
@torch.no_grad()
def calibration_plot(model: BetterGAT, data, n_bins: int = 10):
    spacer("Calibration (Teacher Raw Probs)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    probs = logp.exp()

    test_mask = data.test_mask
    y_true = data.y[test_mask]
    probs_test = probs[test_mask]

    conf, preds = probs_test.max(dim=1)
    conf = conf.cpu().numpy()
    preds = preds.cpu().numpy()
    y_true = y_true.cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    accs, avg_confs, counts = [], [], []
    total = len(conf)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            accs.append(0.0)
            avg_confs.append(0.0)
            counts.append(0)
            continue
        counts.append(int(mask.sum()))
        avg_conf = conf[mask].mean()
        accuracy_b = (preds[mask] == y_true[mask]).mean()
        accs.append(accuracy_b)
        avg_confs.append(avg_conf)
        ece += (mask.sum() / total) * abs(accuracy_b - avg_conf)

    print(f"Expected Calibration Error (ECE): {ece:.4f}")

    centers = 0.5 * (bins[:-1] + bins[1:])
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    plt.bar(centers, accs, width=1.0 / n_bins, alpha=0.7, edgecolor="k", label="Accuracy")
    plt.plot(centers, avg_confs, "o-", label="Avg Confidence")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Teacher)")
    plt.legend()
    plot_and_save("teacher_calibration_reliability.png")

    return ece


ece_val = calibration_plot(teacher, data)


# 9.9 Ego-Graph Around a Misclassified Test Node (Teacher Raw)
@torch.no_grad()
def ego_graph_misclassified(model: BetterGAT, data, center_k: int = 2):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)

    test_mask = data.test_mask
    wrong_nodes = torch.where((pred != data.y) & test_mask)[0]
    if wrong_nodes.numel() == 0:
        print("No misclassified test nodes – nice!")
        return

    center = int(wrong_nodes[0].item())
    print(f"Visualising ego-graph for misclassified test node {center}")

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        center, num_hops=center_k, edge_index=data.edge_index, relabel_nodes=True
    )

    G_sub = nx.Graph()
    G_sub.add_edges_from(edge_index_sub.cpu().t().numpy())
    pos = nx.spring_layout(G_sub, seed=0)

    true_labels = data.y[subset].cpu().numpy()
    pred_labels = pred[subset].cpu().numpy()
    correct_flags = (true_labels == pred_labels)

    colors = ["#1f77b4" if c else "#d62728" for c in correct_flags]

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        G_sub,
        pos=pos,
        node_color=colors,
        node_size=200,
        with_labels=False,
        edge_color="gray",
    )
    plt.title("Ego-Graph Around Misclassified Node (Blue=Correct, Red=Wrong)")
    plt.axis("off")
    plot_and_save("ego_graph_misclassified.png")


ego_graph_misclassified(teacher, data)


# 9.10 Interactive t-SNE (2D + 3D) with Plotly
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, n_components: int = 2):
    if n_components not in (2, 3):
        raise ValueError("n_components must be 2 or 3")

    spacer(f"Interactive t-SNE ({n_components}D) – Teacher Embeddings")

    model.eval()
    emb, _ = model(data.x, data.edge_index, training=False)
    emb_np = emb.cpu().numpy()
    labels_np = data.y.cpu().numpy()

    tsne = TSNE(
        n_components=n_components, init="pca", learning_rate="auto"
    ).fit_transform(emb_np)

    if n_components == 2:
        fig = px.scatter(
            x=tsne[:, 0],
            y=tsne[:, 1],
            color=labels_np.astype(str),
            title="Interactive t-SNE (2D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Label"},
        )
    else:
        fig = px.scatter_3d(
            x=tsne[:, 0],
            y=tsne[:, 1],
            z=tsne[:, 2],
            color=labels_np.astype(str),
            title="Interactive t-SNE (3D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3", "color": "Label"},
        )

    fig.show()


interactive_tsne(teacher, data, n_components=2)
interactive_tsne(teacher, data, n_components=3)

print("\n✅ Ultra-GAT++ v2 Teacher + GCN Student KD lab finished.\n"
      "   - Teacher & Student trained with DropEdge, LP aux, LP pseudo, label smoothing\n"
      "   - MC-Dropout teacher ensemble distilled into student\n"
      "   - Cosine LR schedulers enabled\n"
      "   - Full diagnostics & plots saved in ./plots\n")

In [ ]:
# ============================================================
# Ultra-GAT++ v2 with Teacher–Student KD on CiteSeer
# ------------------------------------------------------------
# - Teacher: BetterGAT (GATv2-based, DropEdge, LP aux, LP pseudo, label smoothing)
# - Student: GCN distilled from MC-Dropout Teacher ensemble
# - Advancements:
#     * Cosine LR scheduler (optional) for teacher & student
#     * MC-Dropout teacher ensemble for KD targets
#     * Label smoothing CE
#     * LP consistency + LP pseudo-label regularisation
#     * DropEdge + feature masking + log-degree structural feature
# - Diagnostics:
#     * Learning curves, confusion matrices, per-class accuracy
#     * t-SNE (untrained vs trained)
#     * Accuracy vs node degree
#     * MC-Dropout uncertainty + calibration (ECE)
#     * Ego-graph visualisation for misclassified node
#     * Interactive Plotly t-SNE (2D & 3D)
#     * AI-generated experiment overview + interactive pipeline flowchart
# - Auto-uses GPU (T4) on Colab/Kaggle if available
# - Saves static plots in ./plots/
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import os
import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go  # for interactive flowchart

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
    k_hop_subgraph,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ============================================================
# 0) Global Config + Utilities
# ============================================================
CONFIG = {
    "experiment_tag": "UltraGAT_CiteSeer_v2",

    # Dataset
    "dataset_name": "CiteSeer",

    # Teacher GAT architecture
    "teacher_hidden_dim": 16,
    "teacher_heads": (4, 4, 2),   # (h1, h2, h3)

    # Teacher training
    "epochs_teacher": 3000,
    "use_early_stopping_teacher": True,
    "teacher_patience": 300,
    "use_lr_scheduler_teacher": True,
    "teacher_scheduler_type": "cosine",   # ["cosine"]

    # Student training
    "epochs_student": 1600,
    "use_early_stopping_student": True,
    "student_patience": 300,
    "use_lr_scheduler_student": True,
    "student_scheduler_type": "cosine",   # ["cosine"]

    # Regularisation / augmentation
    "dropedge_base_p": 0.25,       # annealed to 0
    "feature_mask_p": 0.10,
    "use_struct_features": True,   # log-degree features
    "label_smoothing": 0.10,

    # Label propagation consistency aux
    "use_lp_aux": True,
    "lp_aux_weight": 0.10,

    # LP pseudo-label KL (semi-supervised)
    "use_lp_pseudo": True,
    "lp_pseudo_conf_thr": 0.90,
    "lp_pseudo_weight": 0.10,

    # Optional supervised contrastive (OFF by default)
    "use_contrastive": False,
    "contrastive_weight": 0.0,
    "temperature_contrastive": 0.5,

    # Knowledge Distillation
    "distill_alpha": 0.60,
    "distill_temperature": 2.0,

    # MC-Dropout teacher ensemble for KD
    "teacher_mc_samples_for_kd": 8,    # >1 = ensemble; 1 = deterministic

    # Misc
    "fast_mode": False,               # True = fewer epochs/patience (for debugging)
    "seed": 42,
}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.05)


def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    if y.numel() == 0:
        return 0.0
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


def smooth_one_hot(targets: torch.Tensor,
                   n_classes: int,
                   smoothing: float = 0.0) -> torch.Tensor:
    """
    Convert targets to smoothed one-hot (for label smoothing CE).
    """
    with torch.no_grad():
        assert 0.0 <= smoothing < 1.0
        confidence = 1.0 - smoothing
        label_shape = (targets.size(0), n_classes)
        smooth = torch.full(label_shape, smoothing / (n_classes - 1),
                            device=targets.device)
        smooth.scatter_(1, targets.unsqueeze(1), confidence)
        return smooth


set_seed(CONFIG["seed"])
ensure_dir("plots")

# Auto-select device
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Device: CPU")


# Fast mode for debugging
if CONFIG["fast_mode"]:
    CONFIG["epochs_teacher"] = 600
    CONFIG["epochs_student"] = 600
    CONFIG["teacher_patience"] = 150
    CONFIG["student_patience"] = 150


# ============================================================
# 1) Dataset + Stats + Basic Plots
# ============================================================
dataset = Planetoid(root="./data", name=CONFIG["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Make graph undirected + add self-loops
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# Graph layout (spring)
spacer("Graph Layout (spring)")
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y.cpu().numpy(),
    width=0.5,
    edge_color="grey",
)
plt.title(f"{CONFIG['dataset_name']} Graph (Node-colored by Label)")
plt.tight_layout()
plt.savefig("plots/graph_layout.png", dpi=300)
plt.show()
print("[+] Saved figure to plots/graph_layout.png")

# Degree distribution
spacer("Node Degree Distribution")
degrees_arr = degree(data.edge_index[0]).cpu().numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
ax.bar(deg_counts.keys(), deg_counts.values())
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.savefig("plots/degree_distribution.png", dpi=300)
plt.show()
print("[+] Saved figure to plots/degree_distribution.png")

# ============================================================
# 2) Feature Scaling + Structural Augmentation
# ============================================================
x_np = data.x.cpu().numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).cpu().numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

if CONFIG["use_struct_features"]:
    x_aug = np.concatenate([x_scaled, log_deg], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)

print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

data = data.to(device)


# ============================================================
# 3) GATv2 Block + BetterGAT (Teacher)
# ============================================================
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(4, 4, 2),
        dropout: float = 0.6,
        feature_mask_p: float = 0.05,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # Residual linear head to support low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        # Optimiser
        self.opt = torch.optim.Adam(self.parameters(), lr=0.005, weight_decay=5e-4)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)

        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ============================================================
# 4) Optional Supervised Contrastive Stub (OFF by default)
# ============================================================
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9

    labels_eq = (y.unsqueeze(0) == y.unsqueeze(1)).float()
    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ============================================================
# 5) Teacher Training + LP aux + LP pseudo + Cosine LR
# ============================================================
def build_scheduler(optimizer, cfg_epochs, scheduler_type: str):
    if scheduler_type == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg_epochs
        )
    else:
        return None


def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    num_classes = dataset.num_classes
    # label-smoothing CE implemented as KL to smoothed one-hot
    ce = nn.KLDivLoss(reduction="batchmean")
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]
    smoothing = cfg["label_smoothing"]

    use_early = cfg["use_early_stopping_teacher"]
    patience = cfg["teacher_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    # Optional cosine LR scheduler
    scheduler = None
    if cfg["use_lr_scheduler_teacher"]:
        scheduler = build_scheduler(model.opt, E, cfg["teacher_scheduler_type"])

    print(f"\n🧠 Training TEACHER for up to {E} epochs (early stopping={use_early})...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)
        emb, logp = model(data.x, edge_index_aug, training=True)

        # Label smoothing CE
        y_train = data.y[data.train_mask]
        target_train = smooth_one_hot(
            y_train, num_classes, smoothing=smoothing
        )
        logp_train = logp[data.train_mask]
        ce_loss = ce(logp_train, target_train)
        loss = ce_loss

        # LP auxiliary consistency
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)

        # LP pseudo-label KL for confident unlabeled nodes
        if cfg["use_lp_pseudo"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
                lp_conf, _ = lp_probs.max(dim=1)

            unlabeled_mask = (~data.train_mask) & (~data.val_mask) & (~data.test_mask)
            high_conf_mask = unlabeled_mask & (lp_conf > cfg["lp_pseudo_conf_thr"])

            if high_conf_mask.any():
                logp_pseudo = logp[high_conf_mask]
                target_pseudo = lp_probs[high_conf_mask].detach()
                kl_pseudo = F.kl_div(
                    logp_pseudo, target_pseudo, reduction="batchmean"
                )
                loss = loss + cfg["lp_pseudo_weight"] * kl_pseudo
            else:
                kl_pseudo = torch.tensor(0.0, device=data.x.device)
        else:
            kl_pseudo = torch.tensor(0.0, device=data.x.device)

        # Optional contrastive
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()
        if scheduler is not None:
            scheduler.step()

        # Eval
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_ce = F.nll_loss(
                logp_val[data.val_mask], data.y[data.val_mask]
            ).item()
            tr_acc = accuracy(
                logp[data.train_mask].argmax(1), data.y[data.train_mask]
            )
            val_acc = accuracy(
                logp_val[data.val_mask].argmax(1), data.y[data.val_mask]
            )

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_ce)

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_ce:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | KL_p:{kl_pseudo:.3f} | Contr:{contr:.3f}"
            )

        if use_early and wait >= patience:
            print(f"\n⏹ Early stopping triggered at epoch {ep}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"\n✅ Teacher training complete. Best Val Accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    model.eval()
    with torch.no_grad():
        _, final_logp = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)

    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp


# ============================================================
# 6) MC-Dropout Teacher Ensemble for KD
# ============================================================
@torch.no_grad()
def get_teacher_kd_logits(model: BetterGAT, data, cfg):
    """
    Returns log-probabilities to use for KD.
    If teacher_mc_samples_for_kd > 1: MC-Dropout ensemble.
    Else: deterministic teacher logp.
    """
    S = cfg["teacher_mc_samples_for_kd"]
    if S <= 1:
        model.eval()
        _, logp = model(data.x, data.edge_index, training=False)
        return logp

    spacer(f"MC-Dropout teacher ensemble for KD (S={S})")
    model.train()
    probs_list = []
    for _ in range(S):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))
    probs_mean = torch.cat(probs_list, dim=0).mean(dim=0)
    model.eval()
    return (probs_mean + 1e-12).log()


# ============================================================
# 7) Student GCN + KD + Cosine LR
# ============================================================
class StudentGCN(nn.Module):
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.gcn1(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.gcn2(h, edge_index)
        return F.log_softmax(h, dim=1)


def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_student(
    student: StudentGCN,
    teacher_logp_kd: torch.Tensor,
    data,
    cfg,
):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    use_early = cfg["use_early_stopping_student"]
    patience = cfg["student_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    train_acc_curve, val_acc_curve = [], []

    # Optional cosine LR scheduler
    scheduler = None
    if cfg["use_lr_scheduler_student"]:
        scheduler = build_scheduler(student.opt, E, cfg["student_scheduler_type"])

    print(f"\n🎓 Training STUDENT GCN for up to {E} epochs (KD, early stopping={use_early})...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp_kd, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()
        if scheduler is not None:
            scheduler.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(
                logp_s[data.train_mask].argmax(1), data.y[data.train_mask]
            )
            val_acc = accuracy(
                logp_sv[data.val_mask].argmax(1), data.y[data.val_mask]
            )
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = student.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

        if use_early and wait >= patience:
            print(f"\n⏹ Student early stopping triggered at epoch {ep}.")
            break

    if best_state is not None:
        student.load_state_dict(best_state)

    print(f"\n✅ Student training complete. Best Val Accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")
    return student, train_acc_curve, val_acc_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred


# ============================================================
# 8) Run Teacher + Student (with MC-KD)
# ============================================================
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=CONFIG["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=CONFIG["teacher_heads"],
    dropout=0.6,
    feature_mask_p=CONFIG["feature_mask_p"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp_det, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss, t_best_epoch, t_best_val = train_teacher(
    teacher, data, CONFIG
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, _ = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

# MC-Dropout teacher ensemble for KD targets
teacher_logp_kd = get_teacher_kd_logits(teacher, data, CONFIG)

student = StudentGCN(in_dim, 32, dataset.num_classes).to(device)
student, s_tr_acc, s_val_acc, s_best_epoch, s_best_val = train_student(
    student, teacher_logp_kd, data, CONFIG
)
s_acc, s_pred = test_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {s_acc*100:.2f}%")

print("\nSummary:")
print(f"  Teacher (Raw) Test Acc:       {t_raw_acc*100:.2f}%")
print(f"  Teacher (LP-refined) Test Acc:{t_lp_acc*100:.2f}%")
print(f"  Student (KD, MC Teacher) Acc: {s_acc*100:.2f}%")


# ============================================================
# 9) Plots & Diagnostics
# ============================================================
def plot_and_save(fig_name: str):
    plt.tight_layout()
    ensure_dir("plots")
    path = os.path.join("plots", fig_name)
    plt.savefig(path, dpi=300)
    print(f"[+] Saved figure to {path}")
    plt.show()


def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)", fname=None):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    if fname is None:
        fname = title.lower().replace(" ", "_").replace("(", "").replace(")", "") + ".png"
    plot_and_save(fname)
    return cm


# 9.1 Teacher learning curves
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc")
plt.plot(t_val_acc, label="Val Acc")
plt.axvline(t_best_epoch, linestyle="--", color="gray", label="Best Val Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("teacher_accuracy_curves.png")

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss")
plt.plot(t_val_loss, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("teacher_loss_curves.png")

# 9.2 Student learning curves
spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc")
plt.plot(s_val_acc, label="Val Acc")
plt.axvline(s_best_epoch, linestyle="--", color="gray", label="Best Val Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student GCN (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("student_accuracy_curves.png")


# 9.3 t-SNE (untrained vs trained teacher)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
    ).to(device)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plot_and_save("tsne_untrained_vs_trained_teacher.png")


tsne_static(teacher, data)


# 9.4 Accuracy by degree (teacher raw)
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs)

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plot_and_save("teacher_accuracy_by_degree.png")


accuracy_by_degree(teacher, data)

# 9.5 Confusion matrices
cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)",
                                fname="teacher_confusion_raw.png")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)",
                               fname="teacher_confusion_labelprop.png")
cm_student = plot_confusion(data, s_pred, "Student Confusion (KD, Test)",
                            fname="student_confusion_kd.png")


# 9.6 Per-Class Accuracy (Teacher Raw vs LP vs Student)
@torch.no_grad()
def per_class_accuracy(data, raw_pred, lp_pred, stu_pred, title="Per-Class Accuracy"):
    spacer("Per-Class Accuracy (Teacher Raw vs LabelProp vs Student)")
    y = data.y.cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()
    classes = np.arange(dataset.num_classes)

    raw = raw_pred.cpu().numpy()
    lp = lp_pred.cpu().numpy()
    stu = stu_pred.cpu().numpy()

    acc_raw, acc_lp, acc_stu = [], [], []

    for c in classes:
        mask = (y == c) & test_mask
        if mask.sum() == 0:
            acc_raw.append(0.0)
            acc_lp.append(0.0)
            acc_stu.append(0.0)
        else:
            acc_raw.append((raw[mask] == c).mean())
            acc_lp.append((lp[mask] == c).mean())
            acc_stu.append((stu[mask] == c).mean())

    x = np.arange(len(classes))
    width = 0.25

    plt.figure(figsize=(9, 4))
    plt.bar(x - width, acc_raw, width, label="Teacher Raw")
    plt.bar(x, acc_lp, width, label="Teacher LP")
    plt.bar(x + width, acc_stu, width, label="Student")

    plt.xticks(x, classes)
    plt.ylim(0, 1.05)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plot_and_save("per_class_accuracy.png")


per_class_accuracy(data, t_raw_pred, t_lp_pred, s_pred)


# 9.7 MC-Dropout Uncertainty (Teacher)
@torch.no_grad()
def mc_dropout_uncertainty(model: BetterGAT, data, num_samples: int = 30):
    spacer("MC-Dropout Uncertainty (Teacher)")
    model.train()  # enable dropout

    probs_list = []
    for _ in range(num_samples):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))

    probs_mc = torch.cat(probs_list, dim=0)  # [S, N, C]
    mean_probs = probs_mc.mean(dim=0)        # [N, C]

    entropy = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)

    test_mask = data.test_mask
    _, logp_det = model(data.x, data.edge_index, training=False)
    pred_det = logp_det.argmax(1)

    correct_mask = (pred_det == data.y) & test_mask
    wrong_mask = (pred_det != data.y) & test_mask

    ent_correct = entropy[correct_mask].cpu().numpy()
    ent_wrong = entropy[wrong_mask].cpu().numpy()

    print(f"Avg entropy (correct test): {ent_correct.mean():.4f}")
    print(f"Avg entropy (wrong   test): {ent_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(ent_correct, bins=20, alpha=0.7, label="Correct", density=True)
    plt.hist(ent_wrong, bins=20, alpha=0.7, label="Wrong", density=True)
    plt.xlabel("Predictive Entropy")
    plt.ylabel("Density")
    plt.title("MC-Dropout Uncertainty on Test Nodes")
    plt.legend()
    plot_and_save("mc_dropout_uncertainty.png")

    model.eval()
    return entropy, pred_det


entropy_mc, pred_det = mc_dropout_uncertainty(teacher, data)


# 9.8 Calibration (Teacher Raw Probs + ECE)
@torch.no_grad()
def calibration_plot(model: BetterGAT, data, n_bins: int = 10):
    spacer("Calibration (Teacher Raw Probs)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    probs = logp.exp()

    test_mask = data.test_mask
    y_true = data.y[test_mask]
    probs_test = probs[test_mask]

    conf, preds = probs_test.max(dim=1)
    conf = conf.cpu().numpy()
    preds = preds.cpu().numpy()
    y_true = y_true.cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    accs, avg_confs, counts = [], [], []
    total = len(conf)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            accs.append(0.0)
            avg_confs.append(0.0)
            counts.append(0)
            continue
        counts.append(int(mask.sum()))
        avg_conf = conf[mask].mean()
        accuracy_b = (preds[mask] == y_true[mask]).mean()
        accs.append(accuracy_b)
        avg_confs.append(avg_conf)
        ece += (mask.sum() / total) * abs(accuracy_b - avg_conf)

    print(f"Expected Calibration Error (ECE): {ece:.4f}")

    centers = 0.5 * (bins[:-1] + bins[1:])
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    plt.bar(centers, accs, width=1.0 / n_bins, alpha=0.7, edgecolor="k", label="Accuracy")
    plt.plot(centers, avg_confs, "o-", label="Avg Confidence")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Teacher)")
    plt.legend()
    plot_and_save("teacher_calibration_reliability.png")

    return ece


ece_val = calibration_plot(teacher, data)


# 9.9 Ego-Graph Around a Misclassified Test Node (Teacher Raw)
@torch.no_grad()
def ego_graph_misclassified(model: BetterGAT, data, center_k: int = 2):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)

    test_mask = data.test_mask
    wrong_nodes = torch.where((pred != data.y) & test_mask)[0]
    if wrong_nodes.numel() == 0:
        print("No misclassified test nodes – nice!")
        return

    center = int(wrong_nodes[0].item())
    print(f"Visualising ego-graph for misclassified test node {center}")

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        center, num_hops=center_k, edge_index=data.edge_index, relabel_nodes=True
    )

    G_sub = nx.Graph()
    G_sub.add_edges_from(edge_index_sub.cpu().t().numpy())
    pos = nx.spring_layout(G_sub, seed=0)

    true_labels = data.y[subset].cpu().numpy()
    pred_labels = pred[subset].cpu().numpy()
    correct_flags = (true_labels == pred_labels)

    colors = ["#1f77b4" if c else "#d62728" for c in correct_flags]

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        G_sub,
        pos=pos,
        node_color=colors,
        node_size=200,
        with_labels=False,
        edge_color="gray",
    )
    plt.title("Ego-Graph Around Misclassified Node (Blue=Correct, Red=Wrong)")
    plt.axis("off")
    plot_and_save("ego_graph_misclassified.png")


ego_graph_misclassified(teacher, data)


# 9.10 Interactive t-SNE (2D + 3D) with Plotly
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, n_components: int = 2):
    if n_components not in (2, 3):
        raise ValueError("n_components must be 2 or 3")

    spacer(f"Interactive t-SNE ({n_components}D) – Teacher Embeddings")

    model.eval()
    emb, _ = model(data.x, data.edge_index, training=False)
    emb_np = emb.cpu().numpy()
    labels_np = data.y.cpu().numpy()

    tsne = TSNE(
        n_components=n_components, init="pca", learning_rate="auto"
    ).fit_transform(emb_np)

    if n_components == 2:
        fig = px.scatter(
            x=tsne[:, 0],
            y=tsne[:, 1],
            color=labels_np.astype(str),
            title="Interactive t-SNE (2D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Label"},
        )
    else:
        fig = px.scatter_3d(
            x=tsne[:, 0],
            y=tsne[:, 1],
            z=tsne[:, 2],
            color=labels_np.astype(str),
            title="Interactive t-SNE (3D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3", "color": "Label"},
        )

    fig.show()


interactive_tsne(teacher, data, n_components=2)
interactive_tsne(teacher, data, n_components=3)


# ============================================================
# 10) AI-Generated Experiment Overview (Text Summary)
# ============================================================
spacer("AI-Generated Experiment Overview")

overview = f"""
============================================================
Ultra-GAT++ v2 – Experiment Overview
============================================================

• Experiment Tag      : {CONFIG['experiment_tag']}
• Dataset             : {CONFIG['dataset_name']}
• Nodes / Edges       : {data.num_nodes} / {data.edge_index.size(1)}
• Input Features      : {in_dim} (incl. log-degree={CONFIG['use_struct_features']})
• Num Classes         : {dataset.num_classes}

Teacher (BetterGAT):
    - Hidden Dim      : {CONFIG['teacher_hidden_dim']}
    - Heads           : {CONFIG['teacher_heads']}
    - Epochs (max)    : {CONFIG['epochs_teacher']}
    - Early Stopping  : {CONFIG['use_early_stopping_teacher']}
    - Best Val Epoch  : {t_best_epoch}
    - Best Val Acc    : {t_best_val * 100:.2f}%
    - DropEdge Base p : {CONFIG['dropedge_base_p']}
    - Feature Mask p  : {CONFIG['feature_mask_p']}
    - Label Smoothing : {CONFIG['label_smoothing']}
    - LP Aux Loss     : {CONFIG['use_lp_aux']} (w={CONFIG['lp_aux_weight']})
    - LP Pseudo KL    : {CONFIG['use_lp_pseudo']} (thr={CONFIG['lp_pseudo_conf_thr']}, w={CONFIG['lp_pseudo_weight']})
    - Cosine LR       : {CONFIG['use_lr_scheduler_teacher']}

Student (GCN + KD):
    - Hidden Dim      : 32
    - Epochs (max)    : {CONFIG['epochs_student']}
    - Early Stopping  : {CONFIG['use_early_stopping_student']}
    - Best Val Epoch  : {s_best_epoch}
    - Best Val Acc    : {s_best_val * 100:.2f}%
    - KD α / T        : {CONFIG['distill_alpha']} / {CONFIG['distill_temperature']}
    - Cosine LR       : {CONFIG['use_lr_scheduler_student']}

MC-Dropout Teacher Distillation:
    - Samples for KD  : {CONFIG['teacher_mc_samples_for_kd']}

Final Test Performance:
    - Teacher Raw     : {t_raw_acc * 100:.2f}%
    - Teacher + LP    : {t_lp_acc * 100:.2f}%
    - Student KD      : {s_acc * 100:.2f}%

Uncertainty & Calibration:
    - MC-Dropout Entropy (see histogram)
    - Teacher ECE      : {ece_val:.4f}

All core plots (curves, confusion, degree-accuracy, calibration, ego-graph)
have been saved under ./plots for offline analysis or paper-quality figures.
============================================================
"""

print(overview)


# ============================================================
# 11) Interactive Pipeline Flowchart (Plotly)
# ============================================================
spacer("Interactive Ultra-GAT++ Pipeline Flowchart")

def show_pipeline_flowchart():
    """
    Interactive flowchart of the end-to-end pipeline using Plotly Sankey.
    """

    # Nodes in the pipeline
    labels = [
        "CiteSeer Dataset",
        "Feature Scaling\n+ Log-Degree",
        "BetterGAT Teacher",
        "DropEdge + LP Aux\n+ LP Pseudo",
        "MC-Dropout\nTeacher Ensemble",
        "GCN Student\n(KD α/T)",
        "Evaluation &\nDiagnostics",
    ]

    # Map label index
    # 0: Dataset
    # 1: Features
    # 2: Teacher core
    # 3: Teacher regularisers
    # 4: MC-Dropout KD ensemble
    # 5: Student
    # 6: Evaluation

    source = [
        0,  # Dataset -> Feature Scaling
        1,  # Features -> Teacher
        2,  # Teacher -> Teacher Regularisers
        3,  # Teacher Regularisers -> MC-Dropout Ensemble
        4,  # MC-Dropout Ensemble -> Student
        2,  # Teacher (raw + LP) -> Evaluation
        5,  # Student -> Evaluation
    ]

    target = [
        1,
        2,
        3,
        4,
        5,
        6,
        6,
    ]

    values = [1, 1, 1, 1, 1, 1, 1]

    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    pad=25,
                    thickness=20,
                    line=dict(width=0.5),
                    label=labels,
                ),
                link=dict(
                    source=source,
                    target=target,
                    value=values,
                ),
            )
        ]
    )

    fig.update_layout(
        title_text="Ultra-GAT++ v2 Pipeline Flowchart",
        font_size=12,
        height=500,
    )
    fig.show()


show_pipeline_flowchart()

print("\n✅ Ultra-GAT++ v2 Teacher + GCN Student KD lab finished.\n"
      "   • Teacher & Student trained with DropEdge, LP aux, LP pseudo, label smoothing\n"
      "   • MC-Dropout teacher ensemble distilled into student\n"
      "   • Cosine LR schedulers enabled\n"
      "   • Full diagnostics & plots saved in ./plots\n"
      "   • AI-generated experiment overview + interactive pipeline flowchart displayed\n")

In [ ]:
# ============================================================
# Ultra-GAT++ v3 with Teacher–Student KD on CiteSeer
# ------------------------------------------------------------
# - Teacher: BetterGAT (GATv2-based, DropEdge, LP aux, LP pseudo, label smoothing)
# - Student: GCN distilled from MC-Dropout Teacher ensemble
# - New Advancements in v3:
#     * Optional Teacher Refit stage using train+val+LP pseudo-labels
#     * Teacher vs Student confidence scatter (interactive Plotly)
#     * Interactive misclassification inspector table
# - Existing Advancements:
#     * Cosine LR scheduler (teacher & student)
#     * MC-Dropout teacher ensemble for KD targets
#     * Label smoothing CE
#     * LP consistency + LP pseudo-label regularisation
#     * DropEdge + feature masking + log-degree structural feature
# - Diagnostics:
#     * Learning curves, confusion matrices, per-class accuracy
#     * t-SNE (untrained vs trained)
#     * Accuracy vs node degree
#     * MC-Dropout uncertainty + calibration (ECE)
#     * Ego-graph visualisation for misclassified node
#     * Interactive Plotly t-SNE (2D & 3D)
#     * AI-generated experiment overview + interactive pipeline flowchart
# - Auto-uses GPU (T4) on Colab/Kaggle if available
# - Saves static plots in ./plots/
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import os
import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go  # for interactive flowchart + tables

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
    k_hop_subgraph,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ============================================================
# 0) Global Config + Utilities
# ============================================================
CONFIG = {
    "experiment_tag": "UltraGAT_CiteSeer_v3",

    # Dataset
    "dataset_name": "CiteSeer",

    # Teacher GAT architecture
    "teacher_hidden_dim": 16,
    "teacher_heads": (4, 4, 2),   # (h1, h2, h3)

    # Teacher training
    "epochs_teacher": 3000,
    "use_early_stopping_teacher": True,
    "teacher_patience": 300,
    "use_lr_scheduler_teacher": True,
    "teacher_scheduler_type": "cosine",   # ["cosine"]

    # Student training
    "epochs_student": 1600,
    "use_early_stopping_student": True,
    "student_patience": 300,
    "use_lr_scheduler_student": True,
    "student_scheduler_type": "cosine",   # ["cosine"]

    # Regularisation / augmentation
    "dropedge_base_p": 0.25,       # annealed to 0
    "feature_mask_p": 0.10,
    "use_struct_features": True,   # log-degree features
    "label_smoothing": 0.10,

    # Label propagation consistency aux
    "use_lp_aux": True,
    "lp_aux_weight": 0.10,

    # LP pseudo-label KL (semi-supervised)
    "use_lp_pseudo": True,
    "lp_pseudo_conf_thr": 0.90,
    "lp_pseudo_weight": 0.10,

    # Optional supervised contrastive (OFF by default)
    "use_contrastive": False,
    "contrastive_weight": 0.0,
    "temperature_contrastive": 0.5,

    # Knowledge Distillation
    "distill_alpha": 0.60,
    "distill_temperature": 2.0,

    # MC-Dropout teacher ensemble for KD
    "teacher_mc_samples_for_kd": 8,    # >1 = ensemble; 1 = deterministic

    # NEW: Teacher refit stage
    "use_teacher_refit": True,
    "teacher_refit_epochs": 800,
    "teacher_refit_patience": 200,
    "teacher_refit_lp_thr": 0.92,

    # Misc
    "fast_mode": False,               # True = fewer epochs/patience (for debugging)
    "seed": 42,
}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.05)


def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    if y.numel() == 0:
        return 0.0
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


def smooth_one_hot(targets: torch.Tensor,
                   n_classes: int,
                   smoothing: float = 0.0) -> torch.Tensor:
    """
    Convert targets to smoothed one-hot (for label smoothing CE).
    """
    with torch.no_grad():
        assert 0.0 <= smoothing < 1.0
        confidence = 1.0 - smoothing
        label_shape = (targets.size(0), n_classes)
        smooth = torch.full(label_shape, smoothing / (n_classes - 1),
                            device=targets.device)
        smooth.scatter_(1, targets.unsqueeze(1), confidence)
        return smooth


set_seed(CONFIG["seed"])
ensure_dir("plots")

# Auto-select device
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Device: CPU")


# Fast mode to debug
if CONFIG["fast_mode"]:
    CONFIG["epochs_teacher"] = 600
    CONFIG["epochs_student"] = 600
    CONFIG["teacher_patience"] = 150
    CONFIG["student_patience"] = 150
    CONFIG["teacher_refit_epochs"] = 400
    CONFIG["teacher_refit_patience"] = 100


# ============================================================
# 1) Dataset + Stats + Basic Plots
# ============================================================
dataset = Planetoid(root="./data", name=CONFIG["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Make graph undirected + add self-loops
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# Graph layout (spring)
spacer("Graph Layout (spring)")
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y.cpu().numpy(),
    width=0.5,
    edge_color="grey",
)
plt.title(f"{CONFIG['dataset_name']} Graph (Node-colored by Label)")
plt.tight_layout()
plt.savefig("plots/graph_layout.png", dpi=300)
plt.show()
print("[+] Saved figure to plots/graph_layout.png")

# Degree distribution
spacer("Node Degree Distribution")
degrees_arr = degree(data.edge_index[0]).cpu().numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
ax.bar(deg_counts.keys(), deg_counts.values())
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.savefig("plots/degree_distribution.png", dpi=300)
plt.show()
print("[+] Saved figure to plots/degree_distribution.png")

# ============================================================
# 2) Feature Scaling + Structural Augmentation
# ============================================================
x_np = data.x.cpu().numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).cpu().numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

if CONFIG["use_struct_features"]:
    x_aug = np.concatenate([x_scaled, log_deg], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)

print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

data = data.to(device)


# ============================================================
# 3) GATv2 Block + BetterGAT (Teacher)
# ============================================================
class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(4, 4, 2),
        dropout: float = 0.6,
        feature_mask_p: float = 0.05,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        # Residual linear head to support low-degree / isolated nodes
        self.res_fc = nn.Linear(dim_in, dim_out)

        # Optimiser
        self.opt = torch.optim.Adam(self.parameters(), lr=0.005, weight_decay=5e-4)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)

        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        logits = self.out(h, edge_index)
        logits = logits + 0.1 * self.res_fc(x)

        return h, F.log_softmax(logits, dim=1)


# ============================================================
# 4) Optional Supervised Contrastive Stub (OFF by default)
# ============================================================
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9

    labels_eq = (y.unsqueeze(0) == y.unsqueeze(1)).float()
    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ============================================================
# 5) Teacher Training + LP aux + LP pseudo + Cosine LR
# ============================================================
def build_scheduler(optimizer, cfg_epochs, scheduler_type: str):
    if scheduler_type == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg_epochs
        )
    else:
        return None


def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    num_classes = dataset.num_classes
    ce = nn.KLDivLoss(reduction="batchmean")
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]
    smoothing = cfg["label_smoothing"]

    use_early = cfg["use_early_stopping_teacher"]
    patience = cfg["teacher_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    scheduler = None
    if cfg["use_lr_scheduler_teacher"]:
        scheduler = build_scheduler(model.opt, E, cfg["teacher_scheduler_type"])

    print(f"\n🧠 Training TEACHER for up to {E} epochs (early stopping={use_early})...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)
        emb, logp = model(data.x, edge_index_aug, training=True)

        # Label smoothing CE
        y_train = data.y[data.train_mask]
        target_train = smooth_one_hot(
            y_train, num_classes, smoothing=smoothing
        )
        logp_train = logp[data.train_mask]
        ce_loss = ce(logp_train, target_train)
        loss = ce_loss

        # LP auxiliary consistency
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)

        # LP pseudo-label KL for confident unlabeled nodes
        if cfg["use_lp_pseudo"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
                lp_conf, _ = lp_probs.max(dim=1)

            unlabeled_mask = (~data.train_mask) & (~data.val_mask) & (~data.test_mask)
            high_conf_mask = unlabeled_mask & (lp_conf > cfg["lp_pseudo_conf_thr"])

            if high_conf_mask.any():
                logp_pseudo = logp[high_conf_mask]
                target_pseudo = lp_probs[high_conf_mask].detach()
                kl_pseudo = F.kl_div(
                    logp_pseudo, target_pseudo, reduction="batchmean"
                )
                loss = loss + cfg["lp_pseudo_weight"] * kl_pseudo
            else:
                kl_pseudo = torch.tensor(0.0, device=data.x.device)
        else:
            kl_pseudo = torch.tensor(0.0, device=data.x.device)

        # Optional contrastive
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()
        if scheduler is not None:
            scheduler.step()

        # Eval
        model.eval()
        with torch.no_grad():
            emb_val, logp_val = model(data.x, data.edge_index, training=False)
            val_ce = F.nll_loss(
                logp_val[data.val_mask], data.y[data.val_mask]
            ).item()
            tr_acc = accuracy(
                logp[data.train_mask].argmax(1), data.y[data.train_mask]
            )
            val_acc = accuracy(
                logp_val[data.val_mask].argmax(1), data.y[data.val_mask]
            )

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_ce)

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_ce:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | KL_p:{kl_pseudo:.3f} | Contr:{contr:.3f}"
            )

        if use_early and wait >= patience:
            print(f"\n⏹ Early stopping triggered at epoch {ep}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"\n✅ Teacher training complete. Best Val Accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    model.eval()
    with torch.no_grad():
        _, final_logp = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)

    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp


# ============================================================
# 6) MC-Dropout Teacher Ensemble for KD
# ============================================================
@torch.no_grad()
def get_teacher_kd_logits(model: BetterGAT, data, cfg):
    """
    Returns log-probabilities to use for KD.
    If teacher_mc_samples_for_kd > 1: MC-Dropout ensemble.
    Else: deterministic teacher logp.
    """
    S = cfg["teacher_mc_samples_for_kd"]
    if S <= 1:
        model.eval()
        _, logp = model(data.x, data.edge_index, training=False)
        return logp

    spacer(f"MC-Dropout teacher ensemble for KD (S={S})")
    model.train()
    probs_list = []
    for _ in range(S):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))
    probs_mean = torch.cat(probs_list, dim=0).mean(dim=0)
    model.eval()
    return (probs_mean + 1e-12).log()


# ============================================================
# 7) Student GCN + KD + Cosine LR
# ============================================================
class StudentGCN(nn.Module):
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.gcn1(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.gcn2(h, edge_index)
        return F.log_softmax(h, dim=1)


def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_student(
    student: StudentGCN,
    teacher_logp_kd: torch.Tensor,
    data,
    cfg,
):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    use_early = cfg["use_early_stopping_student"]
    patience = cfg["student_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    train_acc_curve, val_acc_curve = [], []

    scheduler = None
    if cfg["use_lr_scheduler_student"]:
        scheduler = build_scheduler(student.opt, E, cfg["student_scheduler_type"])

    print(f"\n🎓 Training STUDENT GCN for up to {E} epochs (KD, early stopping={use_early})...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp_kd, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()
        if scheduler is not None:
            scheduler.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(
                logp_s[data.train_mask].argmax(1), data.y[data.train_mask]
            )
            val_acc = accuracy(
                logp_sv[data.val_mask].argmax(1), data.y[data.val_mask]
            )
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = student.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

        if use_early and wait >= patience:
            print(f"\n⏹ Student early stopping triggered at epoch {ep}.")
            break

    if best_state is not None:
        student.load_state_dict(best_state)

    print(f"\n✅ Student training complete. Best Val Accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")
    return student, train_acc_curve, val_acc_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_student(student: StudentGCN, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred


# ============================================================
# 8) Run Teacher + Student (with MC-KD)
# ============================================================
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=CONFIG["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=CONFIG["teacher_heads"],
    dropout=0.6,
    feature_mask_p=CONFIG["feature_mask_p"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp_det, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss, t_best_epoch, t_best_val = train_teacher(
    teacher, data, CONFIG
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, _ = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

# MC-Dropout teacher ensemble for KD targets
teacher_logp_kd = get_teacher_kd_logits(teacher, data, CONFIG)

student = StudentGCN(in_dim, 32, dataset.num_classes).to(device)
student, s_tr_acc, s_val_acc, s_best_epoch, s_best_val = train_student(
    student, teacher_logp_kd, data, CONFIG
)
s_acc, s_pred = test_student(student, data)
print(f"🎓 Student Test Accuracy (Distilled): {s_acc*100:.2f}%")

print("\nSummary:")
print(f"  Teacher (Raw) Test Acc:       {t_raw_acc*100:.2f}%")
print(f"  Teacher (LP-refined) Test Acc:{t_lp_acc*100:.2f}%")
print(f"  Student (KD, MC Teacher) Acc: {s_acc*100:.2f}%")


# ============================================================
# 9) Teacher Refit Stage (Train+Val+LP Pseudo Labels)
# ============================================================
@torch.no_grad()
def build_lp_pseudo_labels_for_refit(base_teacher: BetterGAT, data, thr: float):
    """
    Use the trained teacher + LP to create pseudo-labels for refit.
    """
    base_teacher.eval()
    _, logp = base_teacher(data.x, data.edge_index, training=False)
    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_conf, lp_pred = lp_probs.max(dim=1)

    mask_true = data.train_mask | data.val_mask
    unlabeled_mask = (~data.train_mask) & (~data.val_mask) & (~data.test_mask)
    pseudo_mask = unlabeled_mask & (lp_conf > thr)

    return lp_probs, lp_pred, lp_conf, mask_true, pseudo_mask


def train_teacher_refit(
    base_teacher: BetterGAT,
    data,
    cfg,
):
    """
    Refit teacher using:
      - True labels from train+val
      - LP pseudo-labels from unlabeled nodes above a confidence threshold
    """
    spacer("Teacher Refit Stage (Train+Val+LP Pseudo Labels)")
    thr = cfg["teacher_refit_lp_thr"]

    lp_probs, lp_pred, lp_conf, mask_true, pseudo_mask = build_lp_pseudo_labels_for_refit(
        base_teacher, data, thr
    )

    print(f"Refit LP threshold: {thr}")
    print(f"Pseudo-labeled nodes (unlabeled & conf>thr): {pseudo_mask.sum().item()}")

    # New teacher instance, initialised from original teacher weights
    refit_teacher = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
    ).to(device)
    refit_teacher.load_state_dict(base_teacher.state_dict())
    refit_teacher.opt = torch.optim.Adam(refit_teacher.parameters(), lr=0.003, weight_decay=5e-4)

    E = cfg["teacher_refit_epochs"]
    patience = cfg["teacher_refit_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    scheduler = build_scheduler(refit_teacher.opt, E, "cosine")

    ce = nn.NLLLoss()

    true_idx = torch.where(mask_true)[0]
    pseudo_idx = torch.where(pseudo_mask)[0]
    combined_idx = torch.cat([true_idx, pseudo_idx], dim=0)

    print(f"Refit true-labeled nodes:  {true_idx.numel()}")
    print(f"Refit pseudo-labeled nodes:{pseudo_idx.numel()}")

    if combined_idx.numel() == 0:
        print("No nodes available for refit, skipping.")
        return base_teacher, None, None

    # Targets: true labels for true_idx, LP pseudo for pseudo_idx
    targets_true = data.y[true_idx]
    targets_pseudo = lp_pred[pseudo_idx]
    targets_combined = torch.cat([targets_true, targets_pseudo], dim=0)

    print(f"\n🧠 Refit Teacher for up to {E} epochs (early stopping=True)...\n")

    for ep in range(1, E + 1):
        refit_teacher.train()
        refit_teacher.opt.zero_grad()

        _, logp = refit_teacher(data.x, data.edge_index, training=True)
        logp_combined = logp[combined_idx]
        loss = ce(logp_combined, targets_combined)

        loss.backward()
        nn.utils.clip_grad_norm_(refit_teacher.parameters(), 1.0)
        refit_teacher.opt.step()
        if scheduler is not None:
            scheduler.step()

        # Evaluate on val
        refit_teacher.eval()
        with torch.no_grad():
            _, logp_val = refit_teacher(data.x, data.edge_index, training=False)
            val_loss = ce(logp_val[data.val_mask], data.y[data.val_mask]).item()
            val_acc = accuracy(logp_val[data.val_mask].argmax(1), data.y[data.val_mask])

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = refit_teacher.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"[Refit] Epoch {ep:04d} | Loss:{loss:.3f} | "
                f"ValLoss:{val_loss:.3f} | ValAcc:{val_acc*100:5.2f}%"
            )

        if wait >= patience:
            print(f"\n⏹ Refit early stopping triggered at epoch {ep}.")
            break

    if best_state is not None:
        refit_teacher.load_state_dict(best_state)

    refit_teacher.eval()
    with torch.no_grad():
        _, logp_test = refit_teacher(data.x, data.edge_index, training=False)

    # Test metrics for refit teacher
    raw_pred_refit = logp_test.argmax(1)
    raw_acc_refit = accuracy(raw_pred_refit[data.test_mask], data.y[data.test_mask])

    probs_refit = logp_test.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs_refit = lp_layer(probs_refit, data.edge_index)
    lp_pred_refit = lp_probs_refit.argmax(1)
    lp_acc_refit = accuracy(lp_pred_refit[data.test_mask], data.y[data.test_mask])

    print(f"\n✅ Refit Teacher done. Best Val Acc: {best_val_acc*100:.2f}% at epoch {best_epoch}")
    print(f"   Refit Teacher Test (Raw) : {raw_acc_refit*100:.2f}%")
    print(f"   Refit Teacher Test (LP)  : {lp_acc_refit*100:.2f}%\n")

    return refit_teacher, raw_acc_refit, lp_acc_refit


t_refit_raw_acc = None
t_refit_lp_acc = None

if CONFIG["use_teacher_refit"]:
    teacher_refit, t_refit_raw_acc, t_refit_lp_acc = train_teacher_refit(
        teacher, data, CONFIG
    )
else:
    teacher_refit = teacher  # fall back to original


# ============================================================
# 10) Plots & Diagnostics
# ============================================================
def plot_and_save(fig_name: str):
    plt.tight_layout()
    ensure_dir("plots")
    path = os.path.join("plots", fig_name)
    plt.savefig(path, dpi=300)
    print(f"[+] Saved figure to {path}")
    plt.show()


def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)", fname=None):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    if fname is None:
        fname = title.lower().replace(" ", "_").replace("(", "").replace(")", "") + ".png"
    plot_and_save(fname)
    return cm


# 10.1 Teacher learning curves
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc")
plt.plot(t_val_acc, label="Val Acc")
plt.axvline(t_best_epoch, linestyle="--", color="gray", label="Best Val Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("teacher_accuracy_curves.png")

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss")
plt.plot(t_val_loss, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("teacher_loss_curves.png")

# 10.2 Student learning curves
spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc")
plt.plot(s_val_acc, label="Val Acc")
plt.axvline(s_best_epoch, linestyle="--", color="gray", label="Best Val Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student GCN (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("student_accuracy_curves.png")


# 10.3 t-SNE (untrained vs trained teacher)
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
    ).to(device)

    emb0, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plot_and_save("tsne_untrained_vs_trained_teacher.png")


tsne_static(teacher, data)


# 10.4 Accuracy by degree (teacher raw)
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs)

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plot_and_save("teacher_accuracy_by_degree.png")


accuracy_by_degree(teacher, data)

# 10.5 Confusion matrices
cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)",
                                fname="teacher_confusion_raw.png")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)",
                               fname="teacher_confusion_labelprop.png")
cm_student = plot_confusion(data, s_pred, "Student Confusion (KD, Test)",
                            fname="student_confusion_kd.png")


# 10.6 Per-Class Accuracy (Teacher Raw vs LP vs Student)
@torch.no_grad()
def per_class_accuracy(data, raw_pred, lp_pred, stu_pred, title="Per-Class Accuracy"):
    spacer("Per-Class Accuracy (Teacher Raw vs LabelProp vs Student)")
    y = data.y.cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()
    classes = np.arange(dataset.num_classes)

    raw = raw_pred.cpu().numpy()
    lp = lp_pred.cpu().numpy()
    stu = stu_pred.cpu().numpy()

    acc_raw, acc_lp, acc_stu = [], [], []

    for c in classes:
        mask = (y == c) & test_mask
        if mask.sum() == 0:
            acc_raw.append(0.0)
            acc_lp.append(0.0)
            acc_stu.append(0.0)
        else:
            acc_raw.append((raw[mask] == c).mean())
            acc_lp.append((lp[mask] == c).mean())
            acc_stu.append((stu[mask] == c).mean())

    x = np.arange(len(classes))
    width = 0.25

    plt.figure(figsize=(9, 4))
    plt.bar(x - width, acc_raw, width, label="Teacher Raw")
    plt.bar(x, acc_lp, width, label="Teacher LP")
    plt.bar(x + width, acc_stu, width, label="Student")

    plt.xticks(x, classes)
    plt.ylim(0, 1.05)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plot_and_save("per_class_accuracy.png")


per_class_accuracy(data, t_raw_pred, t_lp_pred, s_pred)


# 10.7 MC-Dropout Uncertainty (Teacher)
@torch.no_grad()
def mc_dropout_uncertainty(model: BetterGAT, data, num_samples: int = 30):
    spacer("MC-Dropout Uncertainty (Teacher)")
    model.train()  # enable dropout

    probs_list = []
    for _ in range(num_samples):
        _, logp_mc = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))

    probs_mc = torch.cat(probs_list, dim=0)  # [S, N, C]
    mean_probs = probs_mc.mean(dim=0)        # [N, C]

    entropy = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)

    test_mask = data.test_mask
    _, logp_det = model(data.x, data.edge_index, training=False)
    pred_det = logp_det.argmax(1)

    correct_mask = (pred_det == data.y) & test_mask
    wrong_mask = (pred_det != data.y) & test_mask

    ent_correct = entropy[correct_mask].cpu().numpy()
    ent_wrong = entropy[wrong_mask].cpu().numpy()

    print(f"Avg entropy (correct test): {ent_correct.mean():.4f}")
    print(f"Avg entropy (wrong   test): {ent_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(ent_correct, bins=20, alpha=0.7, label="Correct", density=True)
    plt.hist(ent_wrong, bins=20, alpha=0.7, label="Wrong", density=True)
    plt.xlabel("Predictive Entropy")
    plt.ylabel("Density")
    plt.title("MC-Dropout Uncertainty on Test Nodes")
    plt.legend()
    plot_and_save("mc_dropout_uncertainty.png")

    model.eval()
    return entropy, pred_det


entropy_mc, pred_det = mc_dropout_uncertainty(teacher, data)


# 10.8 Calibration (Teacher Raw Probs + ECE)
@torch.no_grad()
def calibration_plot(model: BetterGAT, data, n_bins: int = 10):
    spacer("Calibration (Teacher Raw Probs)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    probs = logp.exp()

    test_mask = data.test_mask
    y_true = data.y[test_mask]
    probs_test = probs[test_mask]

    conf, preds = probs_test.max(dim=1)
    conf = conf.cpu().numpy()
    preds = preds.cpu().numpy()
    y_true = y_true.cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    accs, avg_confs, counts = [], [], []
    total = len(conf)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            accs.append(0.0)
            avg_confs.append(0.0)
            counts.append(0)
            continue
        counts.append(int(mask.sum()))
        avg_conf = conf[mask].mean()
        accuracy_b = (preds[mask] == y_true[mask]).mean()
        accs.append(accuracy_b)
        avg_confs.append(avg_conf)
        ece += (mask.sum() / total) * abs(accuracy_b - avg_conf)

    print(f"Expected Calibration Error (ECE): {ece:.4f}")

    centers = 0.5 * (bins[:-1] + bins[1:])
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    plt.bar(centers, accs, width=1.0 / n_bins, alpha=0.7, edgecolor="k", label="Accuracy")
    plt.plot(centers, avg_confs, "o-", label="Avg Confidence")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Teacher)")
    plt.legend()
    plot_and_save("teacher_calibration_reliability.png")

    return ece


ece_val = calibration_plot(teacher, data)


# 10.9 Ego-Graph Around a Misclassified Test Node (Teacher Raw)
@torch.no_grad()
def ego_graph_misclassified(model: BetterGAT, data, center_k: int = 2):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")

    model.eval()
    _, logp = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)

    test_mask = data.test_mask
    wrong_nodes = torch.where((pred != data.y) & test_mask)[0]
    if wrong_nodes.numel() == 0:
        print("No misclassified test nodes – nice!")
        return

    center = int(wrong_nodes[0].item())
    print(f"Visualising ego-graph for misclassified test node {center}")

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        center, num_hops=center_k, edge_index=data.edge_index, relabel_nodes=True
    )

    G_sub = nx.Graph()
    G_sub.add_edges_from(edge_index_sub.cpu().t().numpy())
    pos = nx.spring_layout(G_sub, seed=0)

    true_labels = data.y[subset].cpu().numpy()
    pred_labels = pred[subset].cpu().numpy()
    correct_flags = (true_labels == pred_labels)

    colors = ["#1f77b4" if c else "#d62728" for c in correct_flags]

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        G_sub,
        pos=pos,
        node_color=colors,
        node_size=200,
        with_labels=False,
        edge_color="gray",
    )
    plt.title("Ego-Graph Around Misclassified Node (Blue=Correct, Red=Wrong)")
    plt.axis("off")
    plot_and_save("ego_graph_misclassified.png")


ego_graph_misclassified(teacher, data)


# 10.10 NEW: Teacher vs Student Confidence Scatter (Interactive)
@torch.no_grad()
def compare_confidence_scatter(teacher: BetterGAT, student: StudentGCN, data):
    spacer("Teacher vs Student Confidence Scatter (Test Nodes)")

    teacher.eval()
    student.eval()

    _, t_logp = teacher(data.x, data.edge_index, training=False)
    s_logp = student(data.x, data.edge_index, training=False)

    t_probs = t_logp.exp()
    s_probs = s_logp.exp()

    t_conf, t_pred = t_probs.max(dim=1)
    s_conf, s_pred = s_probs.max(dim=1)

    test_mask = data.test_mask
    correct = (s_pred == data.y)

    x = t_conf[test_mask].cpu().numpy()
    y = s_conf[test_mask].cpu().numpy()
    correctness = np.where(correct[test_mask].cpu().numpy(), "Correct", "Wrong")

    fig = px.scatter(
        x=x,
        y=y,
        color=correctness,
        labels={"x": "Teacher Confidence", "y": "Student Confidence", "color": "Student Correct?"},
        title="Teacher vs Student Confidence on Test Nodes",
        hover_data={"Teacher_Conf": x, "Student_Conf": y},
    )
    fig.show()


compare_confidence_scatter(teacher, student, data)


# 10.11 NEW: Misclassification Inspector Table (Interactive)
@torch.no_grad()
def misclassification_table(teacher: BetterGAT,
                            student: StudentGCN,
                            data,
                            entropy: torch.Tensor,
                            top_k: int = 20):
    spacer("Misclassification Inspector (Top High-Entropy Test Nodes)")

    teacher.eval()
    student.eval()

    _, t_logp = teacher(data.x, data.edge_index, training=False)
    s_logp = student(data.x, data.edge_index, training=False)

    t_pred = t_logp.argmax(1)
    s_pred = s_logp.argmax(1)

    test_idx = torch.where(data.test_mask)[0]
    ent_test = entropy[test_idx]
    y_test = data.y[test_idx]

    correct_teacher = (t_pred[test_idx] == y_test)
    correct_student = (s_pred[test_idx] == y_test)

    mis_mask = ~correct_student  # focus on student misclassifications
    if mis_mask.sum().item() == 0:
        print("No student misclassifications on test set – impressive!")
        return

    idx_mis = test_idx[mis_mask]
    ent_mis = ent_test[mis_mask]

    # Sort misclassified by entropy descending
    order = torch.argsort(ent_mis, descending=True)
    idx_top = idx_mis[order][:top_k]
    ent_top = ent_mis[order][:top_k]

    rows = []
    for node_id, ent_val in zip(idx_top.cpu().numpy(), ent_top.cpu().numpy()):
        node_id = int(node_id)
        rows.append(
            [
                node_id,
                int(data.y[node_id].item()),
                int(t_pred[node_id].item()),
                int(s_pred[node_id].item()),
                float(ent_val),
            ]
        )

    header = ["Node ID", "True Label", "Teacher Pred", "Student Pred", "Entropy"]

    fig = go.Figure(
        data=[
            go.Table(
                header=dict(values=header, fill_color="lightgrey", align="center"),
                cells=dict(values=list(zip(*rows)), align="center"),
            )
        ]
    )
    fig.update_layout(
        title=f"Top {len(rows)} High-Entropy Misclassified Test Nodes (Student Errors)",
        height=400,
    )
    fig.show()


misclassification_table(teacher, student, data, entropy_mc, top_k=20)


# ============================================================
# 11) Interactive t-SNE (2D + 3D) with Plotly
# ============================================================
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, n_components: int = 2):
    if n_components not in (2, 3):
        raise ValueError("n_components must be 2 or 3")

    spacer(f"Interactive t-SNE ({n_components}D) – Teacher Embeddings")

    model.eval()
    emb, _ = model(data.x, data.edge_index, training=False)
    emb_np = emb.cpu().numpy()
    labels_np = data.y.cpu().numpy()

    tsne = TSNE(
        n_components=n_components, init="pca", learning_rate="auto"
    ).fit_transform(emb_np)

    if n_components == 2:
        fig = px.scatter(
            x=tsne[:, 0],
            y=tsne[:, 1],
            color=labels_np.astype(str),
            title="Interactive t-SNE (2D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Label"},
        )
    else:
        fig = px.scatter_3d(
            x=tsne[:, 0],
            y=tsne[:, 1],
            z=tsne[:, 2],
            color=labels_np.astype(str),
            title="Interactive t-SNE (3D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3", "color": "Label"},
        )

    fig.show()


interactive_tsne(teacher, data, n_components=2)
interactive_tsne(teacher, data, n_components=3)


# ============================================================
# 12) AI-Generated Experiment Overview (Text Summary)
# ============================================================
spacer("AI-Generated Experiment Overview")

refit_raw_str = "N/A"
refit_lp_str = "N/A"
if t_refit_raw_acc is not None:
    refit_raw_str = f"{t_refit_raw_acc * 100:.2f}%"
if t_refit_lp_acc is not None:
    refit_lp_str = f"{t_refit_lp_acc * 100:.2f}%"

overview = f"""
============================================================
Ultra-GAT++ v3 – Experiment Overview
============================================================

• Experiment Tag      : {CONFIG['experiment_tag']}
• Dataset             : {CONFIG['dataset_name']}
• Nodes / Edges       : {data.num_nodes} / {data.edge_index.size(1)}
• Input Features      : {in_dim} (incl. log-degree={CONFIG['use_struct_features']})
• Num Classes         : {dataset.num_classes}

Teacher (BetterGAT):
    - Hidden Dim      : {CONFIG['teacher_hidden_dim']}
    - Heads           : {CONFIG['teacher_heads']}
    - Epochs (max)    : {CONFIG['epochs_teacher']}
    - Early Stopping  : {CONFIG['use_early_stopping_teacher']}
    - Best Val Epoch  : {t_best_epoch}
    - Best Val Acc    : {t_best_val * 100:.2f}%
    - DropEdge Base p : {CONFIG['dropedge_base_p']}
    - Feature Mask p  : {CONFIG['feature_mask_p']}
    - Label Smoothing : {CONFIG['label_smoothing']}
    - LP Aux Loss     : {CONFIG['use_lp_aux']} (w={CONFIG['lp_aux_weight']})
    - LP Pseudo KL    : {CONFIG['use_lp_pseudo']} (thr={CONFIG['lp_pseudo_conf_thr']}, w={CONFIG['lp_pseudo_weight']})
    - Cosine LR       : {CONFIG['use_lr_scheduler_teacher']}

Student (GCN + KD):
    - Hidden Dim      : 32
    - Epochs (max)    : {CONFIG['epochs_student']}
    - Early Stopping  : {CONFIG['use_early_stopping_student']}
    - Best Val Epoch  : {s_best_epoch}
    - Best Val Acc    : {s_best_val * 100:.2f}%
    - KD α / T        : {CONFIG['distill_alpha']} / {CONFIG['distill_temperature']}
    - Cosine LR       : {CONFIG['use_lr_scheduler_student']}

MC-Dropout Teacher Distillation:
    - Samples for KD  : {CONFIG['teacher_mc_samples_for_kd']}

Teacher Refit Stage:
    - Enabled         : {CONFIG['use_teacher_refit']}
    - Refit Epochs    : {CONFIG['teacher_refit_epochs']}
    - Refit Patience  : {CONFIG['teacher_refit_patience']}
    - Refit LP Thr    : {CONFIG['teacher_refit_lp_thr']}
    - Refit Test Raw  : {refit_raw_str}
    - Refit Test LP   : {refit_lp_str}

Final Test Performance:
    - Teacher Raw     : {t_raw_acc * 100:.2f}%
    - Teacher + LP    : {t_lp_acc * 100:.2f}%
    - Student KD      : {s_acc * 100:.2f}%

Uncertainty & Calibration:
    - MC-Dropout Entropy (see histogram + misclassification inspector)
    - Teacher ECE      : {ece_val:.4f}

All core plots (curves, confusion, degree-accuracy, calibration, ego-graph, etc.)
have been saved under ./plots for offline analysis or paper-quality figures.
Interactive diagnostics (t-SNE, flowchart, confidence scatter, misclassification table)
are rendered inline in this notebook.
============================================================
"""

print(overview)


# ============================================================
# 13) Interactive Ultra-GAT++ Pipeline Flowchart (Plotly)
# ============================================================
spacer("Interactive Ultra-GAT++ Pipeline Flowchart")

def show_pipeline_flowchart():
    """
    Interactive flowchart of the end-to-end pipeline using Plotly Sankey.
    """

    labels = [
        "CiteSeer Dataset",
        "Feature Scaling\n+ Log-Degree",
        "BetterGAT Teacher",
        "DropEdge + LP Aux\n+ LP Pseudo",
        "MC-Dropout\nTeacher Ensemble",
        "GCN Student\n(KD α/T)",
        "Teacher Refit\n(True+LP Pseudo)",
        "Evaluation &\nDiagnostics",
    ]

    # node indices:
    # 0: Dataset
    # 1: Features
    # 2: Teacher core
    # 3: Teacher regularisers
    # 4: MC-Dropout ensemble
    # 5: Student
    # 6: Refit Teacher
    # 7: Evaluation

    source = [
        0,  # Dataset -> Features
        1,  # Features -> Teacher
        2,  # Teacher -> Regularisers
        3,  # Regularisers -> MC-Dropout Ensemble
        4,  # Ensemble -> Student
        2,  # Teacher -> Evaluation
        5,  # Student -> Evaluation
        2,  # Teacher -> Refit Teacher
        6,  # Refit Teacher -> Evaluation
    ]

    target = [
        1,
        2,
        3,
        4,
        5,
        7,
        7,
        6,
        7,
    ]

    values = [1] * len(source)

    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    pad=25,
                    thickness=20,
                    line=dict(width=0.5),
                    label=labels,
                ),
                link=dict(
                    source=source,
                    target=target,
                    value=values,
                ),
            )
        ]
    )

    fig.update_layout(
        title_text="Ultra-GAT++ v3 Pipeline Flowchart",
        font_size=12,
        height=500,
    )
    fig.show()


show_pipeline_flowchart()

print("\n✅ Ultra-GAT++ v3 Teacher + GCN Student KD lab finished.\n"
      "   • Teacher & Student trained with DropEdge, LP aux, LP pseudo, label smoothing\n"
      "   • MC-Dropout teacher ensemble distilled into student\n"
      "   • Optional Teacher Refit stage using LP pseudo labels\n"
      "   • Cosine LR schedulers enabled\n"
      "   • Confidence scatter + misclassification inspector interactive tools\n"
      "   • Full diagnostics & plots saved in ./plots\n"
      "   • AI-generated experiment overview + interactive pipeline flowchart displayed\n")

In [ ]:
# ============================================================
# Ultra-GAT++ v4 with Dual-Teacher KD + Self-Training on Planetoid
# ------------------------------------------------------------
# Major Features:
#   • Dataset switcher: Cora / CiteSeer / PubMed
#   • Teacher: BetterGAT (GATv2 + DropEdge + PairNorm(optional)
#                        + LP aux + LP pseudo + label smoothing)
#   • Student: Residual 3-layer GCN + KD from
#       - Raw teacher logits
#       - LP-refined teacher probabilities
#     (dual-teacher KD)
#   • MC-Dropout teacher ensemble for KD targets
#   • Teacher Refit: Train+Val+LP pseudo labels
#   • Student Self-Training: pseudo-label fine-tune after KD
#   • Cosine LR schedulers (teacher & student)
#   • Extensive diagnostics:
#       - Learning curves, confusion matrices, per-class accuracy
#       - t-SNE (untrained vs trained)
#       - Accuracy vs degree
#       - MC-Dropout uncertainty, calibration (ECE)
#       - Ego-graph around misclassified node
#       - Attention entropy diagnostics
#       - Interactive: t-SNE (2D/3D), confidence scatter,
#                     misclassification table, Sankey flowchart
#   • AI-generated numeric experiment overview
#   • Optional multi-seed hook (set in CONFIG)
#
#   -> Ready to present as a full GAT Teacher–Student KD research lab.
# ============================================================

!pip install -q torch-geometric scikit-learn matplotlib networkx plotly

import os
import time
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    degree,
    add_self_loops,
    to_undirected,
    remove_isolated_nodes,
    to_networkx,
    k_hop_subgraph,
)
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.nn.models.label_prop import LabelPropagation


# ============================================================
# 0) Global Config + Utilities
# ============================================================
CONFIG = {
    "experiment_tag": "UltraGAT_CiteSeer_v4",

    # Dataset: choose from ["Cora", "CiteSeer", "PubMed"]
    "dataset_name": "CiteSeer",

    # Teacher GAT architecture
    "teacher_hidden_dim": 16,
    "teacher_heads": (4, 4, 2),   # (h1, h2, h3)

    # PairNorm for oversmoothing control in GAT (NEW)
    "use_pairnorm": True,

    # Teacher training
    "epochs_teacher": 3000,
    "use_early_stopping_teacher": True,
    "teacher_patience": 300,
    "use_lr_scheduler_teacher": True,
    "teacher_scheduler_type": "cosine",

    # Student training
    "epochs_student": 1600,
    "use_early_stopping_student": True,
    "student_patience": 300,
    "use_lr_scheduler_student": True,
    "student_scheduler_type": "cosine",

    # Regularisation / augmentation
    "dropedge_base_p": 0.25,
    "feature_mask_p": 0.10,
    "use_struct_features": True,   # log-degree feature
    "feature_noise_std": 0.01,     # NEW: Gaussian noise in features
    "label_smoothing": 0.10,

    # Label propagation consistency aux
    "use_lp_aux": True,
    "lp_aux_weight": 0.10,

    # LP pseudo-label KL (semi-supervised)
    "use_lp_pseudo": True,
    "lp_pseudo_conf_thr": 0.90,
    "lp_pseudo_weight": 0.10,

    # Optional supervised contrastive (OFF by default)
    "use_contrastive": False,
    "contrastive_weight": 0.0,
    "temperature_contrastive": 0.5,

    # Knowledge Distillation
    "distill_alpha": 0.60,          # weight for KD vs CE on hard labels
    "distill_temperature": 2.0,

    # Dual-teacher KD mixing (NEW)
    "dual_kd_mix": 0.5,             # 0: raw only, 1: LP-only, 0.5: mix

    # MC-Dropout teacher ensemble for KD
    "teacher_mc_samples_for_kd": 8,

    # Teacher refit stage
    "use_teacher_refit": True,
    "teacher_refit_epochs": 800,
    "teacher_refit_patience": 200,
    "teacher_refit_lp_thr": 0.92,

    # Student self-training stage (NEW)
    "use_student_self_training": True,
    "student_self_train_epochs": 400,
    "student_self_train_conf_thr": 0.90,
    "student_self_train_lr": 0.005,

    # Multi-seed hook (NEW, OFF by default to avoid long runtime)
    "multi_seed_eval": False,
    "multi_seed_values": [42, 52, 62],

    # Misc
    "fast_mode": False,
    "seed": 42,
}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


def spacer(title: str):
    print("\n" + "─" * 90)
    print("🎨", title)
    print("─" * 90)
    time.sleep(0.05)


def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    if y.numel() == 0:
        return 0.0
    return (pred.eq(y).sum() / y.numel()).item()


def dropedge(edge_index: torch.Tensor,
             epoch: int,
             total_epochs: int,
             base_p: float = 0.2) -> torch.Tensor:
    """
    Linearly decreases edge-drop probability from base_p -> 0 over training.
    Self-loops are always kept.
    """
    p = float(base_p) * max(0.0, 1.0 - epoch / float(total_epochs))
    if p <= 0:
        return edge_index

    E = edge_index.size(1)
    keep = torch.rand(E, device=edge_index.device) > p

    self_mask = edge_index[0] == edge_index[1]
    keep = torch.where(self_mask, torch.ones_like(keep, dtype=torch.bool), keep)

    return edge_index[:, keep]


def random_feature_mask(x: torch.Tensor, p: float, training: bool) -> torch.Tensor:
    if not training or p <= 0:
        return x
    mask = torch.empty_like(x).bernoulli_(1 - p)
    return x * mask


def add_feature_noise(x: torch.Tensor, std: float, training: bool) -> torch.Tensor:
    if not training or std <= 0:
        return x
    noise = torch.randn_like(x) * std
    return x + noise


def smooth_one_hot(targets: torch.Tensor,
                   n_classes: int,
                   smoothing: float = 0.0) -> torch.Tensor:
    """
    Convert targets to smoothed one-hot (for label smoothing CE).
    """
    with torch.no_grad():
        assert 0.0 <= smoothing < 1.0
        confidence = 1.0 - smoothing
        label_shape = (targets.size(0), n_classes)
        smooth = torch.full(label_shape, smoothing / (n_classes - 1),
                            device=targets.device)
        smooth.scatter_(1, targets.unsqueeze(1), confidence)
        return smooth


set_seed(CONFIG["seed"])
ensure_dir("plots")

# Auto-select device
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Device: CPU")

# Fast mode
if CONFIG["fast_mode"]:
    CONFIG["epochs_teacher"] = 600
    CONFIG["epochs_student"] = 600
    CONFIG["teacher_patience"] = 150
    CONFIG["student_patience"] = 150
    CONFIG["teacher_refit_epochs"] = 400
    CONFIG["teacher_refit_patience"] = 100
    CONFIG["student_self_train_epochs"] = 200


# ============================================================
# 1) Dataset + Stats + Basic Plots
# ============================================================
dataset = Planetoid(root="./data", name=CONFIG["dataset_name"])
data = dataset[0]

print(f"Dataset: {dataset}")
print("-------------------")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

print("\nGraph:")
print("------")
print(f"Edges are directed: {data.is_directed()}")
print(f"Graph has isolated nodes: {data.has_isolated_nodes()}")
print(f"Graph has self-loops: {data.has_self_loops()}")

isolated = (remove_isolated_nodes(data.edge_index)[2] == False).sum(dim=0).item()
print(f"Number of isolated nodes = {isolated}")

# Make graph undirected + add self-loops
data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
data.edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

# Graph layout (spring)
spacer("Graph Layout (spring)")
G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
nx.draw_networkx(
    G,
    pos=nx.spring_layout(G, seed=0),
    with_labels=False,
    node_size=20,
    node_color=data.y.cpu().numpy(),
    width=0.5,
    edge_color="grey",
)
plt.title(f"{CONFIG['dataset_name']} Graph (Node-colored by Label)")
plt.tight_layout()
plt.savefig("plots/graph_layout.png", dpi=300)
plt.show()
print("[+] Saved figure to plots/graph_layout.png")

# Degree distribution
spacer("Node Degree Distribution")
degrees_arr = degree(data.edge_index[0]).cpu().numpy()
deg_counts = Counter(degrees_arr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
ax.bar(deg_counts.keys(), deg_counts.values())
plt.title("Node Degree Distribution")
plt.tight_layout()
plt.savefig("plots/degree_distribution.png", dpi=300)
plt.show()
print("[+] Saved figure to plots/degree_distribution.png")

# ============================================================
# 2) Feature Scaling + Structural Augmentation
# ============================================================
x_np = data.x.cpu().numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

deg_np = degree(data.edge_index[0], num_nodes=data.num_nodes).cpu().numpy()
log_deg = np.log1p(deg_np).reshape(-1, 1)

if CONFIG["use_struct_features"]:
    x_aug = np.concatenate([x_scaled, log_deg], axis=1)
else:
    x_aug = x_scaled

data.x = torch.tensor(x_aug, dtype=torch.float32)
in_dim = data.x.size(1)

print(
    f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}, "
    f"Features (aug): {in_dim}, Classes: {dataset.num_classes}"
)

data = data.to(device)


# ============================================================
# 3) PairNorm + GATv2 Block + BetterGAT (Teacher)
# ============================================================
class PairNorm(nn.Module):
    """
    Simple PairNorm variant to mitigate oversmoothing.
    """
    def __init__(self, scale: float = 1.0, eps: float = 1e-6):
        super().__init__()
        self.scale = scale
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        col_mean = x.mean(dim=0, keepdim=True)
        x = x - col_mean
        row_norm = torch.sqrt(x.pow(2).sum(dim=1, keepdim=True) + self.eps)
        x = self.scale * x / row_norm
        return x


class GATv2Block(nn.Module):
    def __init__(
        self,
        in_d: int,
        out_d: int,
        heads: int = 8,
        dropout: float = 0.6,
        residual: bool = True,
        concat: bool = True,
        use_pairnorm: bool = False,
    ):
        super().__init__()
        self.conv = GATv2Conv(
            in_d, out_d, heads=heads, dropout=dropout, concat=concat
        )
        self.norm = nn.LayerNorm(out_d * heads if concat else out_d)
        self.pairnorm = PairNorm() if use_pairnorm else None
        self.drop = nn.Dropout(dropout)
        self.out_dim = out_d * heads if concat else out_d
        self.use_res = residual and (in_d == self.out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv(x, edge_index)
        h = self.norm(h)
        if self.pairnorm is not None:
            h = self.pairnorm(h)
        h = F.elu(h)
        h = self.drop(h)
        if self.use_res:
            h = h + x
        return h


class BetterGAT(nn.Module):
    def __init__(
        self,
        dim_in: int,
        dim_h: int,
        dim_out: int,
        heads=(4, 4, 2),
        dropout: float = 0.6,
        feature_mask_p: float = 0.05,
        use_pairnorm: bool = False,
    ):
        super().__init__()

        self.feature_mask_p = feature_mask_p
        self.feature_noise_std = CONFIG["feature_noise_std"]

        self.g1 = GATv2Block(dim_in, dim_h, heads=heads[0],
                             dropout=dropout, residual=False,
                             use_pairnorm=use_pairnorm)
        self.g2 = GATv2Block(dim_h * heads[0], dim_h, heads=heads[1],
                             dropout=dropout, residual=False,
                             use_pairnorm=use_pairnorm)
        self.g3 = GATv2Block(dim_h * heads[1], dim_h, heads=heads[2],
                             dropout=dropout, residual=False,
                             use_pairnorm=use_pairnorm)

        self.out = GATv2Conv(
            dim_h * heads[2], dim_out, heads=1, dropout=dropout, concat=False
        )

        self.res_fc = nn.Linear(dim_in, dim_out)
        self.opt = torch.optim.Adam(self.parameters(), lr=0.005, weight_decay=5e-4)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        training: bool = False,
        return_attention: bool = False,
    ):
        x = random_feature_mask(x, p=self.feature_mask_p, training=training)
        x = add_feature_noise(x, std=self.feature_noise_std, training=training)

        h = F.dropout(x, p=0.6, training=training)
        h = self.g1(h, edge_index)
        h = self.g2(h, edge_index)
        h = self.g3(h, edge_index)
        h = F.dropout(h, p=0.6, training=training)

        if return_attention:
            logits, (edge_idx_att, alpha) = self.out(
                h, edge_index, return_attention_weights=True
            )
        else:
            logits = self.out(h, edge_index)
            edge_idx_att, alpha = None, None

        logits = logits + 0.1 * self.res_fc(x)
        logp = F.log_softmax(logits, dim=1)
        return h, logp, edge_idx_att, alpha


# ============================================================
# 4) Optional Supervised Contrastive Loss (OFF by default)
# ============================================================
def supervised_contrastive_loss(
    emb: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() < 2:
        return torch.tensor(0.0, device=emb.device)

    z = F.normalize(emb[idx], dim=1)
    y = labels[idx]
    sim = torch.mm(z, z.t()) / temperature
    sim = sim - torch.eye(sim.size(0), device=sim.device) * 1e9

    labels_eq = (y.unsqueeze(0) == y.unsqueeze(1)).float()
    log_prob = F.log_softmax(sim, dim=1)
    loss = -(labels_eq * log_prob).sum() / (labels_eq.sum() + 1e-9)
    return loss


# ============================================================
# 5) Teacher Training (LP aux + LP pseudo + Cosine LR)
# ============================================================
def build_scheduler(optimizer, cfg_epochs, scheduler_type: str):
    if scheduler_type == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg_epochs
        )
    return None


def train_teacher(
    model: BetterGAT,
    data,
    cfg,
):
    num_classes = dataset.num_classes
    ce = nn.KLDivLoss(reduction="batchmean")
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    E = cfg["epochs_teacher"]
    base_drop = cfg["dropedge_base_p"]
    smoothing = cfg["label_smoothing"]

    use_early = cfg["use_early_stopping_teacher"]
    patience = cfg["teacher_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    train_acc_curve, val_acc_curve = [], []
    train_loss_curve, val_loss_curve = [], []

    scheduler = None
    if cfg["use_lr_scheduler_teacher"]:
        scheduler = build_scheduler(model.opt, E, cfg["teacher_scheduler_type"])

    print(f"\n🧠 Training TEACHER for up to {E} epochs (early stopping={use_early})...\n")

    for ep in range(1, E + 1):
        model.train()
        model.opt.zero_grad()

        edge_index_aug = dropedge(data.edge_index, ep, E, base_p=base_drop)
        emb, logp, _, _ = model(data.x, edge_index_aug, training=True)

        # label smoothing CE
        y_train = data.y[data.train_mask]
        target_train = smooth_one_hot(
            y_train, num_classes, smoothing=smoothing
        )
        logp_train = logp[data.train_mask]
        ce_loss = ce(logp_train, target_train)
        loss = ce_loss

        # LP auxiliary consistency
        if cfg["use_lp_aux"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
            cos_sim = F.cosine_similarity(probs, lp_probs, dim=1)
            lp_consistency = 1.0 - cos_sim.mean()
            loss = loss + cfg["lp_aux_weight"] * lp_consistency
        else:
            lp_consistency = torch.tensor(0.0, device=data.x.device)

        # LP pseudo-label KL for confident unlabeled nodes
        if cfg["use_lp_pseudo"]:
            with torch.no_grad():
                probs = logp.exp()
                lp_probs = lp_layer(probs, data.edge_index)
                lp_conf, _ = lp_probs.max(dim=1)

            unlabeled_mask = (~data.train_mask) & (~data.val_mask) & (~data.test_mask)
            high_conf_mask = unlabeled_mask & (lp_conf > cfg["lp_pseudo_conf_thr"])

            if high_conf_mask.any():
                logp_pseudo = logp[high_conf_mask]
                target_pseudo = lp_probs[high_conf_mask].detach()
                kl_pseudo = F.kl_div(
                    logp_pseudo, target_pseudo, reduction="batchmean"
                )
                loss = loss + cfg["lp_pseudo_weight"] * kl_pseudo
            else:
                kl_pseudo = torch.tensor(0.0, device=data.x.device)
        else:
            kl_pseudo = torch.tensor(0.0, device=data.x.device)

        # Optional contrastive
        if cfg["use_contrastive"]:
            contr = supervised_contrastive_loss(
                emb, data.y, data.train_mask, temperature=cfg["temperature_contrastive"]
            )
            loss = loss + cfg["contrastive_weight"] * contr
        else:
            contr = torch.tensor(0.0, device=data.x.device)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.opt.step()
        if scheduler is not None:
            scheduler.step()

        # Eval
        model.eval()
        with torch.no_grad():
            _, logp_val, _, _ = model(data.x, data.edge_index, training=False)
            val_ce = F.nll_loss(
                logp_val[data.val_mask], data.y[data.val_mask]
            ).item()
            tr_acc = accuracy(
                logp[data.train_mask].argmax(1), data.y[data.train_mask]
            )
            val_acc = accuracy(
                logp_val[data.val_mask].argmax(1), data.y[data.val_mask]
            )

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)
        train_loss_curve.append(float(loss.item()))
        val_loss_curve.append(val_ce)

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | "
                f"TrainLoss:{loss:.3f} | ValLoss:{val_ce:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}% | "
                f"LP_c:{lp_consistency:.3f} | KL_p:{kl_pseudo:.3f} | Contr:{contr:.3f}"
            )

        if use_early and wait >= patience:
            print(f"\n⏹ Teacher early stopping at epoch {ep}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"\n✅ Teacher training complete. Best Val Accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")

    model.eval()
    with torch.no_grad():
        _, final_logp, _, _ = model(data.x, data.edge_index, training=False)

    return model, final_logp, train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_teacher(model: BetterGAT, data):
    model.eval()
    _, logp, _, _ = model(data.x, data.edge_index, training=False)

    raw_pred = logp.argmax(1)
    raw_acc = accuracy(raw_pred[data.test_mask], data.y[data.test_mask])

    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_pred = lp_probs.argmax(1)
    lp_acc = accuracy(lp_pred[data.test_mask], data.y[data.test_mask])

    return raw_acc, raw_pred, lp_acc, lp_pred, logp, lp_probs


# ============================================================
# 6) MC-Dropout Teacher Ensemble + Dual KD Targets
# ============================================================
@torch.no_grad()
def get_teacher_kd_logits_dual(model: BetterGAT, data, cfg):
    """
    Dual-teacher KD target:
      - Raw teacher logits
      - LP-refined teacher probabilities
    Combined via cfg["dual_kd_mix"] and optionally MC-dropout.
    """
    S = cfg["teacher_mc_samples_for_kd"]
    dual_mix = cfg["dual_kd_mix"]
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)

    if S <= 1:
        model.eval()
        _, logp, _, _ = model(data.x, data.edge_index, training=False)
        probs = logp.exp()
        lp_probs = lp_layer(probs, data.edge_index)
        mixed_probs = (1 - dual_mix) * probs + dual_mix * lp_probs
        return (mixed_probs + 1e-12).log()

    spacer(f"MC-Dropout teacher ensemble for dual KD (S={S})")
    model.train()
    probs_list = []
    for _ in range(S):
        _, logp_mc, _, _ = model(data.x, data.edge_index, training=True)
        probs_mc = logp_mc.exp()
        probs_list.append(probs_mc.unsqueeze(0))
    probs_mean = torch.cat(probs_list, dim=0).mean(dim=0)

    # LP on ensemble-mean
    lp_probs = lp_layer(probs_mean, data.edge_index)
    mixed_probs = (1 - dual_mix) * probs_mean + dual_mix * lp_probs
    model.eval()
    return (mixed_probs + 1e-12).log()


# ============================================================
# 7) Residual 3-layer Student GCN + KD + Self-Training
# ============================================================
class ResidualGCN3(nn.Module):
    """
    Deeper 3-layer GCN with residual connections for Student (NEW).
    """
    def __init__(self, dim_in: int, dim_h: int, dim_out: int, dropout: float = 0.5):
        super().__init__()
        self.g1 = GCNConv(dim_in, dim_h)
        self.g2 = GCNConv(dim_h, dim_h)
        self.g3 = GCNConv(dim_h, dim_out)
        self.dropout = dropout
        self.res_fc = nn.Linear(dim_in, dim_h)
        self.opt = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                training: bool = False) -> torch.Tensor:
        h0 = x
        h = F.dropout(x, p=self.dropout, training=training)
        h = self.g1(h, edge_index)
        h = F.relu(h)

        # residual from input to 2nd layer hidden
        h = h + self.res_fc(h0)

        h = F.dropout(h, p=self.dropout, training=training)
        h = self.g2(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=training)
        h = self.g3(h, edge_index)
        return F.log_softmax(h, dim=1)


def distillation_loss(
    student_logp: torch.Tensor,
    teacher_logp: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    alpha: float,
    T: float,
) -> torch.Tensor:
    idx = mask.nonzero(as_tuple=False).view(-1)
    if idx.numel() == 0:
        return torch.tensor(0.0, device=student_logp.device)

    s_logp_T = F.log_softmax(student_logp[idx] / T, dim=1)
    t_prob_T = F.softmax(teacher_logp[idx] / T, dim=1)
    kd = F.kl_div(s_logp_T, t_prob_T, reduction="batchmean") * (T * T)

    ce = F.nll_loss(student_logp[idx], y[idx])

    return alpha * kd + (1.0 - alpha) * ce


def train_student(
    student: ResidualGCN3,
    teacher_logp_kd: torch.Tensor,
    data,
    cfg,
):
    E = cfg["epochs_student"]
    alpha = cfg["distill_alpha"]
    T = cfg["distill_temperature"]
    ce = nn.NLLLoss()

    use_early = cfg["use_early_stopping_student"]
    patience = cfg["student_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    train_acc_curve, val_acc_curve = [], []

    scheduler = None
    if cfg["use_lr_scheduler_student"]:
        scheduler = build_scheduler(student.opt, E, cfg["student_scheduler_type"])

    print(f"\n🎓 Training STUDENT Residual GCN3 for up to {E} epochs (KD, early stopping={use_early})...\n")

    for ep in range(1, E + 1):
        student.train()
        student.opt.zero_grad()

        logp_s = student(data.x, data.edge_index, training=True)
        kd_loss = distillation_loss(
            logp_s, teacher_logp_kd, data.y, data.train_mask,
            alpha=alpha, T=T
        )
        kd_loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        student.opt.step()
        if scheduler is not None:
            scheduler.step()

        student.eval()
        with torch.no_grad():
            logp_sv = student(data.x, data.edge_index, training=False)
            tr_acc = accuracy(
                logp_s[data.train_mask].argmax(1), data.y[data.train_mask]
            )
            val_acc = accuracy(
                logp_sv[data.val_mask].argmax(1), data.y[data.val_mask]
            )
            val_loss = ce(logp_sv[data.val_mask], data.y[data.val_mask]).item()

        train_acc_curve.append(tr_acc)
        val_acc_curve.append(val_acc)

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = student.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"Epoch {ep:04d} | ValLoss:{val_loss:.3f} | "
                f"TrainAcc:{tr_acc*100:5.2f}% | ValAcc:{val_acc*100:5.2f}%"
            )

        if use_early and wait >= patience:
            print(f"\n⏹ Student early stopping at epoch {ep}.")
            break

    if best_state is not None:
        student.load_state_dict(best_state)

    print(f"\n✅ Student KD training complete. Best Val Accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}\n")
    return student, train_acc_curve, val_acc_curve, best_epoch, best_val_acc


@torch.no_grad()
def test_student(student: ResidualGCN3, data):
    student.eval()
    logp = student(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
    return acc, pred


def student_self_training(student: ResidualGCN3, data, cfg):
    """
    NEW: Self-training stage for student with pseudo-labels on unlabeled nodes.
    """
    if not cfg["use_student_self_training"]:
        print("\n[Self-Training] Disabled in CONFIG.")
        return student, None

    spacer("Student Self-Training (pseudo-label fine-tune)")

    student.eval()
    with torch.no_grad():
        logp = student(data.x, data.edge_index, training=False)
        probs = logp.exp()
        conf, pred = probs.max(dim=1)

    unlabeled_mask = (~data.train_mask) & (~data.val_mask) & (~data.test_mask)
    pseudo_mask = unlabeled_mask & (conf > cfg["student_self_train_conf_thr"])

    n_pseudo = pseudo_mask.sum().item()
    print(f"Pseudo-labeled nodes for self-training: {n_pseudo}")
    if n_pseudo == 0:
        print("No pseudo-labeled nodes above threshold – skipping self-training.")
        return student, None

    # New optimiser for small fine-tune step
    opt = torch.optim.Adam(student.parameters(), lr=cfg["student_self_train_lr"], weight_decay=5e-4)
    ce = nn.NLLLoss()

    E = cfg["student_self_train_epochs"]
    for ep in range(1, E + 1):
        student.train()
        opt.zero_grad()
        logp_s = student(data.x, data.edge_index, training=True)
        loss = ce(logp_s[pseudo_mask], pred[pseudo_mask])
        loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step()

        if ep % 100 == 0 or ep == 1 or ep == E:
            student.eval()
            with torch.no_grad():
                logp_eval = student(data.x, data.edge_index, training=False)
                test_acc = accuracy(
                    logp_eval[data.test_mask].argmax(1), data.y[data.test_mask]
                )
            print(f"[SelfTrain] Epoch {ep:04d} | Loss:{loss:.4f} | TestAcc:{test_acc*100:.2f}%")

    return student, n_pseudo


# ============================================================
# 8) Teacher + Dual-KD Student + Self-Training Pipeline
# ============================================================
teacher = BetterGAT(
    dim_in=in_dim,
    dim_h=CONFIG["teacher_hidden_dim"],
    dim_out=dataset.num_classes,
    heads=CONFIG["teacher_heads"],
    dropout=0.6,
    feature_mask_p=CONFIG["feature_mask_p"],
    use_pairnorm=CONFIG["use_pairnorm"],
).to(device)

print("\nTeacher model:\n", teacher)

teacher, teacher_logp_det, t_tr_acc, t_val_acc, t_tr_loss, t_val_loss, t_best_epoch, t_best_val = train_teacher(
    teacher, data, CONFIG
)

t_raw_acc, t_raw_pred, t_lp_acc, t_lp_pred, t_logp_raw, t_probs_lp = test_teacher(teacher, data)
print(f"📈 Teacher Test Accuracy (Raw):       {t_raw_acc*100:.2f}%")
print(f"📈 Teacher Test Accuracy (LabelProp): {t_lp_acc*100:.2f}%")

# Dual-teacher KD logits (raw + LP, with MC-dropout)
teacher_logp_kd_dual = get_teacher_kd_logits_dual(teacher, data, CONFIG)

student = ResidualGCN3(in_dim, 32, dataset.num_classes).to(device)
student, s_tr_acc, s_val_acc, s_best_epoch, s_best_val = train_student(
    student, teacher_logp_kd_dual, data, CONFIG
)
s_acc_before_st, s_pred_before = test_student(student, data)
print(f"🎓 Student Test Accuracy (KD only): {s_acc_before_st*100:.2f}%")

# Optional student self-training
student, n_pseudo_st = student_self_training(student, data, CONFIG)
s_acc_after_st, s_pred_after = test_student(student, data)
print(f"🎓 Student Test Accuracy (after Self-Training): {s_acc_after_st*100:.2f}%")

print("\nSummary (Teacher + Student):")
print(f"  Teacher (Raw) Test Acc:       {t_raw_acc*100:.2f}%")
print(f"  Teacher (LP-refined) Test Acc:{t_lp_acc*100:.2f}%")
print(f"  Student KD (before ST) Acc:   {s_acc_before_st*100:.2f}%")
print(f"  Student KD+SelfTrain Acc:     {s_acc_after_st*100:.2f}%")

# ============================================================
# 9) Teacher Refit Stage
# ============================================================
@torch.no_grad()
def build_lp_pseudo_labels_for_refit(base_teacher: BetterGAT, data, thr: float):
    base_teacher.eval()
    _, logp, _, _ = base_teacher(data.x, data.edge_index, training=False)
    probs = logp.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs = lp_layer(probs, data.edge_index)
    lp_conf, lp_pred = lp_probs.max(dim=1)

    mask_true = data.train_mask | data.val_mask
    unlabeled_mask = (~data.train_mask) & (~data.val_mask) & (~data.test_mask)
    pseudo_mask = unlabeled_mask & (lp_conf > thr)

    return lp_probs, lp_pred, lp_conf, mask_true, pseudo_mask


def train_teacher_refit(
    base_teacher: BetterGAT,
    data,
    cfg,
):
    spacer("Teacher Refit Stage (Train+Val+LP Pseudo Labels)")
    thr = cfg["teacher_refit_lp_thr"]

    lp_probs, lp_pred, lp_conf, mask_true, pseudo_mask = build_lp_pseudo_labels_for_refit(
        base_teacher, data, thr
    )

    print(f"Refit LP threshold: {thr}")
    print(f"Pseudo-labeled nodes (unlabeled & conf>thr): {pseudo_mask.sum().item()}")

    refit_teacher = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
        use_pairnorm=CONFIG["use_pairnorm"],
    ).to(device)
    refit_teacher.load_state_dict(base_teacher.state_dict())
    refit_teacher.opt = torch.optim.Adam(refit_teacher.parameters(), lr=0.003, weight_decay=5e-4)

    E = cfg["teacher_refit_epochs"]
    patience = cfg["teacher_refit_patience"]
    wait = 0
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    scheduler = build_scheduler(refit_teacher.opt, E, "cosine")

    ce = nn.NLLLoss()

    true_idx = torch.where(mask_true)[0]
    pseudo_idx = torch.where(pseudo_mask)[0]
    combined_idx = torch.cat([true_idx, pseudo_idx], dim=0)

    print(f"Refit true-labeled nodes:  {true_idx.numel()}")
    print(f"Refit pseudo-labeled nodes:{pseudo_idx.numel()}")

    if combined_idx.numel() == 0:
        print("No nodes available for refit, skipping.")
        return base_teacher, None, None

    targets_true = data.y[true_idx]
    targets_pseudo = lp_pred[pseudo_idx]
    targets_combined = torch.cat([targets_true, targets_pseudo], dim=0)

    print(f"\n🧠 Refit Teacher for up to {E} epochs (early stopping=True)...\n")

    for ep in range(1, E + 1):
        refit_teacher.train()
        refit_teacher.opt.zero_grad()

        _, logp, _, _ = refit_teacher(data.x, data.edge_index, training=True)
        logp_combined = logp[combined_idx]
        loss = ce(logp_combined, targets_combined)

        loss.backward()
        nn.utils.clip_grad_norm_(refit_teacher.parameters(), 1.0)
        refit_teacher.opt.step()
        if scheduler is not None:
            scheduler.step()

        refit_teacher.eval()
        with torch.no_grad():
            _, logp_val, _, _ = refit_teacher(data.x, data.edge_index, training=False)
            val_loss = ce(logp_val[data.val_mask], data.y[data.val_mask]).item()
            val_acc = accuracy(logp_val[data.val_mask].argmax(1), data.y[data.val_mask])

        improved = val_acc > best_val_acc + 1e-5
        if improved:
            best_val_acc = val_acc
            best_epoch = ep
            best_state = refit_teacher.state_dict()
            wait = 0
        else:
            wait += 1

        if ep % 50 == 0 or ep == 1 or ep == E:
            print(
                f"[Refit] Epoch {ep:04d} | Loss:{loss:.3f} | "
                f"ValLoss:{val_loss:.3f} | ValAcc:{val_acc*100:5.2f}%"
            )

        if wait >= patience:
            print(f"\n⏹ Refit early stopping at epoch {ep}.")
            break

    if best_state is not None:
        refit_teacher.load_state_dict(best_state)

    refit_teacher.eval()
    with torch.no_grad():
        _, logp_test, _, _ = refit_teacher(data.x, data.edge_index, training=False)

    raw_pred_refit = logp_test.argmax(1)
    raw_acc_refit = accuracy(raw_pred_refit[data.test_mask], data.y[data.test_mask])

    probs_refit = logp_test.exp()
    lp_layer = LabelPropagation(num_layers=10, alpha=0.9)
    lp_probs_refit = lp_layer(probs_refit, data.edge_index)
    lp_pred_refit = lp_probs_refit.argmax(1)
    lp_acc_refit = accuracy(lp_pred_refit[data.test_mask], data.y[data.test_mask])

    print(f"\n✅ Refit Teacher done. Best Val Acc: {best_val_acc*100:.2f}% at epoch {best_epoch}")
    print(f"   Refit Teacher Test (Raw) : {raw_acc_refit*100:.2f}%")
    print(f"   Refit Teacher Test (LP)  : {lp_acc_refit*100:.2f}%\n")

    return refit_teacher, raw_acc_refit, lp_acc_refit


t_refit_raw_acc = None
t_refit_lp_acc = None

if CONFIG["use_teacher_refit"]:
    teacher_refit, t_refit_raw_acc, t_refit_lp_acc = train_teacher_refit(
        teacher, data, CONFIG
    )
else:
    teacher_refit = teacher

# ============================================================
# 10) Plots, Diagnostics, Attention Entropy, Interactive Tools
# (same spirit as v3 + NEW attention entropy & param counts)
# ============================================================
def plot_and_save(fig_name: str):
    plt.tight_layout()
    ensure_dir("plots")
    path = os.path.join("plots", fig_name)
    plt.savefig(path, dpi=300)
    print(f"[+] Saved figure to {path}")
    plt.show()


def plot_confusion(data, pred, title="Confusion Matrix (Test Mask)", fname=None):
    spacer(title)
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(dataset.num_classes)))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(dataset.num_classes)
    plt.xticks(ticks, ticks)
    plt.yticks(ticks, ticks)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

    if fname is None:
        fname = title.lower().replace(" ", "_").replace("(", "").replace(")", "") + ".png"
    plot_and_save(fname)
    return cm


# Learning curves
spacer("Teacher Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_acc, label="Train Acc")
plt.plot(t_val_acc, label="Val Acc")
plt.axvline(t_best_epoch, linestyle="--", color="gray", label="Best Val Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Teacher (BetterGAT) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("teacher_accuracy_curves.png")

spacer("Teacher Loss Curves")
plt.figure(figsize=(9, 4))
plt.plot(t_tr_loss, label="Train Loss")
plt.plot(t_val_loss, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Teacher (BetterGAT) Loss")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("teacher_loss_curves.png")

spacer("Student Learning Curves (Accuracy)")
plt.figure(figsize=(9, 4))
plt.plot(s_tr_acc, label="Train Acc")
plt.plot(s_val_acc, label="Val Acc")
plt.axvline(s_best_epoch, linestyle="--", color="gray", label="Best Val Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Student Residual GCN3 (KD) Accuracy")
plt.legend()
plt.grid(alpha=0.4)
plot_and_save("student_accuracy_curves.png")


# t-SNE static
@torch.no_grad()
def tsne_static(model: BetterGAT, data):
    spacer("t-SNE Embeddings (Untrained vs Trained Teacher)")

    untrained = BetterGAT(
        dim_in=in_dim,
        dim_h=CONFIG["teacher_hidden_dim"],
        dim_out=dataset.num_classes,
        heads=CONFIG["teacher_heads"],
        dropout=0.6,
        feature_mask_p=CONFIG["feature_mask_p"],
        use_pairnorm=CONFIG["use_pairnorm"],
    ).to(device)

    emb0, _, _, _ = untrained(data.x, data.edge_index, training=False)
    emb1, _, _, _ = model(data.x, data.edge_index, training=False)

    tsne0 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb0.cpu().numpy())
    tsne1 = TSNE(
        n_components=2, init="pca", learning_rate="auto"
    ).fit_transform(emb1.cpu().numpy())

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(tsne0[:, 0], tsne0[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Untrained Teacher Embeddings")

    plt.subplot(1, 2, 2)
    plt.scatter(tsne1[:, 0], tsne1[:, 1], c=data.y.cpu().numpy(), s=8)
    plt.axis("off")
    plt.title("Trained Teacher Embeddings")

    plot_and_save("tsne_untrained_vs_trained_teacher.png")


tsne_static(teacher, data)


# Accuracy vs degree
@torch.no_grad()
def accuracy_by_degree(model: BetterGAT, data):
    spacer("Accuracy by Node Degree (Teacher Raw)")
    _, logp, _, _ = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)
    degs = degree(data.edge_index[0], num_nodes=data.num_nodes)

    bins = [0, 1, 2, 3, 4, 5]
    accs, counts = [], []

    for d in bins:
        mask = (degs == d)
        counts.append(int(mask.sum().item()))
        accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    mask = (degs > 5)
    counts.append(int(mask.sum().item()))
    accs.append(accuracy(pred[mask], data.y[mask]) if mask.any() else 0.0)

    plt.figure(figsize=(8, 4))
    labels = [str(b) for b in bins] + [">5"]
    plt.bar(labels, accs)

    for i, a in enumerate(accs):
        plt.text(i, a + 0.01, f"{a*100:.1f}%\nN={counts[i]}", ha="center", fontsize=9)

    plt.title("Teacher Accuracy by Node Degree")
    plt.xlabel("Degree")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plot_and_save("teacher_accuracy_by_degree.png")


accuracy_by_degree(teacher, data)

# Confusion matrices
cm_teacher_raw = plot_confusion(data, t_raw_pred, "Teacher Confusion (Raw, Test)",
                                fname="teacher_confusion_raw.png")
cm_teacher_lp = plot_confusion(data, t_lp_pred, "Teacher Confusion (LabelProp, Test)",
                               fname="teacher_confusion_labelprop.png")
cm_student = plot_confusion(data, s_pred_after, "Student Confusion (KD+ST, Test)",
                            fname="student_confusion_kd_st.png")


# Per-class accuracy
@torch.no_grad()
def per_class_accuracy(data, raw_pred, lp_pred, stu_pred, title="Per-Class Accuracy"):
    spacer("Per-Class Accuracy (Teacher Raw vs LabelProp vs Student)")
    y = data.y.cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()
    classes = np.arange(dataset.num_classes)

    raw = raw_pred.cpu().numpy()
    lp = lp_pred.cpu().numpy()
    stu = stu_pred.cpu().numpy()

    acc_raw, acc_lp, acc_stu = [], [], []

    for c in classes:
        mask = (y == c) & test_mask
        if mask.sum() == 0:
            acc_raw.append(0.0)
            acc_lp.append(0.0)
            acc_stu.append(0.0)
        else:
            acc_raw.append((raw[mask] == c).mean())
            acc_lp.append((lp[mask] == c).mean())
            acc_stu.append((stu[mask] == c).mean())

    x = np.arange(len(classes))
    width = 0.25

    plt.figure(figsize=(9, 4))
    plt.bar(x - width, acc_raw, width, label="Teacher Raw")
    plt.bar(x, acc_lp, width, label="Teacher LP")
    plt.bar(x + width, acc_stu, width, label="Student (KD+ST)")

    plt.xticks(x, classes)
    plt.ylim(0, 1.05)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plot_and_save("per_class_accuracy.png")


per_class_accuracy(data, t_raw_pred, t_lp_pred, s_pred_after)


# MC-Dropout uncertainty
@torch.no_grad()
def mc_dropout_uncertainty(model: BetterGAT, data, num_samples: int = 30):
    spacer("MC-Dropout Uncertainty (Teacher)")
    model.train()

    probs_list = []
    for _ in range(num_samples):
        _, logp_mc, _, _ = model(data.x, data.edge_index, training=True)
        probs_list.append(logp_mc.exp().unsqueeze(0))

    probs_mc = torch.cat(probs_list, dim=0)
    mean_probs = probs_mc.mean(dim=0)

    entropy = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)

    test_mask = data.test_mask
    model.eval()
    _, logp_det, _, _ = model(data.x, data.edge_index, training=False)
    pred_det = logp_det.argmax(1)

    correct_mask = (pred_det == data.y) & test_mask
    wrong_mask = (pred_det != data.y) & test_mask

    ent_correct = entropy[correct_mask].cpu().numpy()
    ent_wrong = entropy[wrong_mask].cpu().numpy()

    print(f"Avg entropy (correct test): {ent_correct.mean():.4f}")
    print(f"Avg entropy (wrong   test): {ent_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(ent_correct, bins=20, alpha=0.7, label="Correct", density=True)
    plt.hist(ent_wrong, bins=20, alpha=0.7, label="Wrong", density=True)
    plt.xlabel("Predictive Entropy")
    plt.ylabel("Density")
    plt.title("MC-Dropout Uncertainty on Test Nodes")
    plt.legend()
    plot_and_save("mc_dropout_uncertainty.png")

    return entropy, pred_det


entropy_mc, pred_det = mc_dropout_uncertainty(teacher, data)


# Calibration
@torch.no_grad()
def calibration_plot(model: BetterGAT, data, n_bins: int = 10):
    spacer("Calibration (Teacher Raw Probs)")

    model.eval()
    _, logp, _, _ = model(data.x, data.edge_index, training=False)
    probs = logp.exp()

    test_mask = data.test_mask
    y_true = data.y[test_mask]
    probs_test = probs[test_mask]

    conf, preds = probs_test.max(dim=1)
    conf = conf.cpu().numpy()
    preds = preds.cpu().numpy()
    y_true = y_true.cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    accs, avg_confs, counts = [], [], []
    total = len(conf)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            accs.append(0.0)
            avg_confs.append(0.0)
            counts.append(0)
            continue
        counts.append(int(mask.sum()))
        avg_conf = conf[mask].mean()
        accuracy_b = (preds[mask] == y_true[mask]).mean()
        accs.append(accuracy_b)
        avg_confs.append(avg_conf)
        ece += (mask.sum() / total) * abs(accuracy_b - avg_conf)

    print(f"Expected Calibration Error (ECE): {ece:.4f}")

    centers = 0.5 * (bins[:-1] + bins[1:])
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    plt.bar(centers, accs, width=1.0 / n_bins, alpha=0.7, edgecolor="k", label="Accuracy")
    plt.plot(centers, avg_confs, "o-", label="Avg Confidence")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Teacher)")
    plt.legend()
    plot_and_save("teacher_calibration_reliability.png")

    return ece


ece_val = calibration_plot(teacher, data)


# Ego-graph misclassified
@torch.no_grad()
def ego_graph_misclassified(model: BetterGAT, data, center_k: int = 2):
    spacer("Ego-Graph Around a Misclassified Test Node (Teacher Raw)")
    model.eval()
    _, logp, _, _ = model(data.x, data.edge_index, training=False)
    pred = logp.argmax(1)

    test_mask = data.test_mask
    wrong_nodes = torch.where((pred != data.y) & test_mask)[0]
    if wrong_nodes.numel() == 0:
        print("No misclassified test nodes – nice!")
        return

    center = int(wrong_nodes[0].item())
    print(f"Visualising ego-graph for misclassified test node {center}")

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        center, num_hops=center_k, edge_index=data.edge_index, relabel_nodes=True
    )

    G_sub = nx.Graph()
    G_sub.add_edges_from(edge_index_sub.cpu().t().numpy())
    pos = nx.spring_layout(G_sub, seed=0)

    true_labels = data.y[subset].cpu().numpy()
    pred_labels = pred[subset].cpu().numpy()
    correct_flags = (true_labels == pred_labels)

    colors = ["#1f77b4" if c else "#d62728" for c in correct_flags]

    plt.figure(figsize=(6, 6))
    nx.draw_networkx(
        G_sub,
        pos=pos,
        node_color=colors,
        node_size=200,
        with_labels=False,
        edge_color="gray",
    )
    plt.title("Ego-Graph Around Misclassified Node (Blue=Correct, Red=Wrong)")
    plt.axis("off")
    plot_and_save("ego_graph_misclassified.png")


ego_graph_misclassified(teacher, data)


# NEW: attention entropy diagnostics
@torch.no_grad()
def attention_entropy_diagnostics(model: BetterGAT, data):
    spacer("Attention Entropy Diagnostics (Teacher)")

    model.eval()
    # run once with attention
    _, logp, edge_idx_att, alpha = model(data.x, data.edge_index, training=False, return_attention=True)
    if alpha is None:
        print("Attention weights not available for diagnostics.")
        return

    # alpha: [E, heads] or [E] – for GATv2Conv, shape is [E, heads]
    if alpha.dim() == 1:
        alpha_heads = alpha.unsqueeze(1)
    else:
        alpha_heads = alpha

    # entropy over neighbors per node for each head
    src = edge_idx_att[0]
    dst = edge_idx_att[1]
    num_heads = alpha_heads.size(1)

    entropies = [[] for _ in range(num_heads)]

    for head in range(num_heads):
        a = alpha_heads[:, head]
        # group by source node
        for u in torch.unique(src):
            mask = (src == u)
            w = a[mask]
            p = w / (w.sum() + 1e-12)
            ent = -(p * (p + 1e-12).log()).sum().item()
            entropies[head].append(ent)

    avg_ents = [np.mean(e) if len(e) > 0 else 0.0 for e in entropies]
    print("Average attention entropy per head:", ["{:.3f}".format(v) for v in avg_ents])

    plt.figure(figsize=(8, 4))
    plt.bar(np.arange(num_heads), avg_ents)
    plt.xlabel("Head")
    plt.ylabel("Avg Entropy")
    plt.title("Teacher Attention Entropy per Head")
    plot_and_save("teacher_attention_entropy_per_head.png")


attention_entropy_diagnostics(teacher, data)


# NEW: parameter counts
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


spacer("Parameter Counts")
print(f"Teacher params: {count_parameters(teacher):,}")
print(f"Student params: {count_parameters(student):,}")


# NEW: Teacher vs Student Confidence Scatter (interactive)
@torch.no_grad()
def compare_confidence_scatter(teacher: BetterGAT, student: ResidualGCN3, data):
    spacer("Teacher vs Student Confidence Scatter (Test Nodes)")

    teacher.eval()
    student.eval()

    _, t_logp, _, _ = teacher(data.x, data.edge_index, training=False)
    s_logp = student(data.x, data.edge_index, training=False)

    t_probs = t_logp.exp()
    s_probs = s_logp.exp()

    t_conf, t_pred = t_probs.max(dim=1)
    s_conf, s_pred = s_probs.max(dim=1)

    test_mask = data.test_mask
    correct = (s_pred == data.y)

    x = t_conf[test_mask].cpu().numpy()
    y = s_conf[test_mask].cpu().numpy()
    correctness = np.where(correct[test_mask].cpu().numpy(), "Correct", "Wrong")

    fig = px.scatter(
        x=x,
        y=y,
        color=correctness,
        labels={"x": "Teacher Confidence", "y": "Student Confidence", "color": "Student Correct?"},
        title="Teacher vs Student Confidence on Test Nodes",
        hover_data={"Teacher_Conf": x, "Student_Conf": y},
    )
    fig.show()


compare_confidence_scatter(teacher, student, data)


# NEW: Misclassification Inspector Table (interactive)
@torch.no_grad()
def misclassification_table(teacher: BetterGAT,
                            student: ResidualGCN3,
                            data,
                            entropy: torch.Tensor,
                            top_k: int = 20):
    spacer("Misclassification Inspector (Top High-Entropy Test Nodes)")

    teacher.eval()
    student.eval()

    _, t_logp, _, _ = teacher(data.x, data.edge_index, training=False)
    s_logp = student(data.x, data.edge_index, training=False)

    t_pred = t_logp.argmax(1)
    s_pred = s_logp.argmax(1)

    test_idx = torch.where(data.test_mask)[0]
    ent_test = entropy[test_idx]
    y_test = data.y[test_idx]

    correct_student = (s_pred[test_idx] == y_test)
    mis_mask = ~correct_student
    if mis_mask.sum().item() == 0:
        print("No student misclassifications on test set – impressive!")
        return

    idx_mis = test_idx[mis_mask]
    ent_mis = ent_test[mis_mask]

    order = torch.argsort(ent_mis, descending=True)
    idx_top = idx_mis[order][:top_k]
    ent_top = ent_mis[order][:top_k]

    rows = []
    for node_id, ent_val in zip(idx_top.cpu().numpy(), ent_top.cpu().numpy()):
        node_id = int(node_id)
        rows.append(
            [
                node_id,
                int(data.y[node_id].item()),
                int(t_pred[node_id].item()),
                int(s_pred[node_id].item()),
                float(ent_val),
            ]
        )

    header = ["Node ID", "True Label", "Teacher Pred", "Student Pred", "Entropy"]

    fig = go.Figure(
        data=[
            go.Table(
                header=dict(values=header, fill_color="lightgrey", align="center"),
                cells=dict(values=list(zip(*rows)), align="center"),
            )
        ]
    )
    fig.update_layout(
        title=f"Top {len(rows)} High-Entropy Misclassified Test Nodes (Student Errors)",
        height=400,
    )
    fig.show()


misclassification_table(teacher, student, data, entropy_mc, top_k=20)


# Interactive t-SNE
@torch.no_grad()
def interactive_tsne(model: BetterGAT, data, n_components: int = 2):
    if n_components not in (2, 3):
        raise ValueError("n_components must be 2 or 3")

    spacer(f"Interactive t-SNE ({n_components}D) – Teacher Embeddings")

    model.eval()
    emb, _, _, _ = model(data.x, data.edge_index, training=False)
    emb_np = emb.cpu().numpy()
    labels_np = data.y.cpu().numpy()

    tsne = TSNE(
        n_components=n_components, init="pca", learning_rate="auto"
    ).fit_transform(emb_np)

    if n_components == 2:
        fig = px.scatter(
            x=tsne[:, 0],
            y=tsne[:, 1],
            color=labels_np.astype(str),
            title="Interactive t-SNE (2D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Label"},
        )
    else:
        fig = px.scatter_3d(
            x=tsne[:, 0],
            y=tsne[:, 1],
            z=tsne[:, 2],
            color=labels_np.astype(str),
            title="Interactive t-SNE (3D) – Teacher Embeddings",
            labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3", "color": "Label"},
        )

    fig.show()


interactive_tsne(teacher, data, n_components=2)
interactive_tsne(teacher, data, n_components=3)


# ============================================================
# 11) AI-Generated Experiment Overview
# ============================================================
spacer("AI-Generated Experiment Overview")

refit_raw_str = "N/A"
refit_lp_str = "N/A"
if t_refit_raw_acc is not None:
    refit_raw_str = f"{t_refit_raw_acc * 100:.2f}%"
if t_refit_lp_acc is not None:
    refit_lp_str = f"{t_refit_lp_acc * 100:.2f}%"

overview = f"""
============================================================
Ultra-GAT++ v4 – Experiment Overview
============================================================

• Experiment Tag      : {CONFIG['experiment_tag']}
• Dataset             : {CONFIG['dataset_name']}
• Nodes / Edges       : {data.num_nodes} / {data.edge_index.size(1)}
• Input Features      : {in_dim} (log-degree used={CONFIG['use_struct_features']})
• Num Classes         : {dataset.num_classes}

Teacher (BetterGAT):
    - Hidden Dim      : {CONFIG['teacher_hidden_dim']}
    - Heads           : {CONFIG['teacher_heads']}
    - PairNorm        : {CONFIG['use_pairnorm']}
    - Epochs (max)    : {CONFIG['epochs_teacher']}
    - Early Stopping  : {CONFIG['use_early_stopping_teacher']}
    - Best Val Epoch  : {t_best_epoch}
    - Best Val Acc    : {t_best_val * 100:.2f}%
    - DropEdge Base p : {CONFIG['dropedge_base_p']}
    - Feature Mask p  : {CONFIG['feature_mask_p']}
    - Feature Noise   : std={CONFIG['feature_noise_std']}
    - Label Smoothing : {CONFIG['label_smoothing']}
    - LP Aux Loss     : {CONFIG['use_lp_aux']} (w={CONFIG['lp_aux_weight']})
    - LP Pseudo KL    : {CONFIG['use_lp_pseudo']} (thr={CONFIG['lp_pseudo_conf_thr']}, w={CONFIG['lp_pseudo_weight']})
    - Cosine LR       : {CONFIG['use_lr_scheduler_teacher']}

Student (Residual GCN3 + Dual KD):
    - Hidden Dim      : 32
    - Epochs (max)    : {CONFIG['epochs_student']}
    - Early Stopping  : {CONFIG['use_early_stopping_student']}
    - Best Val Epoch  : {s_best_epoch}
    - Best Val Acc    : {s_best_val * 100:.2f}%
    - KD α / T        : {CONFIG['distill_alpha']} / {CONFIG['distill_temperature']}
    - Dual KD mix     : raw:{1-CONFIG['dual_kd_mix']:.2f} vs LP:{CONFIG['dual_kd_mix']:.2f}
    - Cosine LR       : {CONFIG['use_lr_scheduler_student']}

Student Self-Training:
    - Enabled         : {CONFIG['use_student_self_training']}
    - Pseudo Thr      : {CONFIG['student_self_train_conf_thr']}
    - ST Epochs       : {CONFIG['student_self_train_epochs']}
    - Pseudo Nodes    : {n_pseudo_st if n_pseudo_st is not None else 0}

MC-Dropout KD:
    - Teacher Samples : {CONFIG['teacher_mc_samples_for_kd']}

Teacher Refit Stage:
    - Enabled         : {CONFIG['use_teacher_refit']}
    - Refit Epochs    : {CONFIG['teacher_refit_epochs']}
    - Refit Patience  : {CONFIG['teacher_refit_patience']}
    - Refit LP Thr    : {CONFIG['teacher_refit_lp_thr']}
    - Refit Test Raw  : {refit_raw_str}
    - Refit Test LP   : {refit_lp_str}

Final Test Performance:
    - Teacher Raw     : {t_raw_acc * 100:.2f}%
    - Teacher + LP    : {t_lp_acc * 100:.2f}%
    - Student KD only : {s_acc_before_st * 100:.2f}%
    - Student KD + ST : {s_acc_after_st * 100:.2f}%

Uncertainty & Calibration:
    - MC-Dropout      : see histogram + misclassification inspector
    - Teacher ECE     : {ece_val:.4f}

Capacity:
    - Teacher Params  : {count_parameters(teacher):,}
    - Student Params  : {count_parameters(student):,}

All key plots have been saved under ./plots
(accuracy curves, loss curves, confusion matrices, per-class accuracy,
 degree-accuracy, calibration, uncertainty, attention entropy, ego-graph),
and interactive Plotly views (t-SNE, confidence scatter, flowchart, etc.)
render inline in this notebook.
============================================================
"""

print(overview)


# ============================================================
# 12) Interactive Pipeline Flowchart (Sankey)
# ============================================================
spacer("Interactive Ultra-GAT++ v4 Pipeline Flowchart")

def show_pipeline_flowchart():
    labels = [
        "Planetoid\nDataset",
        "Feature Scaling\n+ Log-Degree",
        "BetterGAT\nTeacher",
        "DropEdge + LP Aux\n+ LP Pseudo",
        "MC-Dropout\nDual KD Mix",
        "Residual GCN3\nStudent KD",
        "Student\nSelf-Training",
        "Teacher Refit\n(True+LP Pseudo)",
        "Evaluation &\nDiagnostics",
    ]

    source = [
        0,  # Dataset -> Features
        1,  # Features -> Teacher
        2,  # Teacher -> Regularisers
        3,  # Regularisers -> Dual KD
        4,  # Dual KD -> Student
        5,  # Student -> Self-Training
        2,  # Teacher -> Refit
        6,  # Self-Training -> Eval
        7,  # Refit -> Eval
        2,  # Teacher -> Eval
    ]

    target = [
        1,
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        8,
        8,
    ]

    values = [1] * len(source)

    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    pad=25,
                    thickness=20,
                    line=dict(width=0.5),
                    label=labels,
                ),
                link=dict(
                    source=source,
                    target=target,
                    value=values,
                ),
            )
        ]
    )

    fig.update_layout(
        title_text="Ultra-GAT++ v4 Pipeline Flowchart",
        font_size=12,
        height=550,
    )
    fig.show()


show_pipeline_flowchart()

print("\n✅ Ultra-GAT++ v4 Teacher + Dual-KD Residual GCN3 Student lab finished.\n"
      "   • Ready as a full research-style project: teacher, student, refit,\n"
      "     self-training, uncertainty, calibration, interpretability, and\n"
      "     rich plots + interactive dashboards all in one notebook.\n")

In [ ]:
# ============================================================
# 13) Harsh Robustness & Failure Geometry
#    - Extreme sparsity
#    - Edge corruption / rewiring
#    - Teacher vs Student robustness curves
#    - Failure geometry report (who flips, how)
# ------------------------------------------------------------
# Assumes you already ran Ultra-GAT++ v4 code and you have:
#   - CONFIG, data, teacher, student, dataset, device, accuracy()
# ============================================================

from copy import deepcopy

spacer("Harsh Robustness Experiments – Failure Geometry")


# 13.1 Edge perturbation operators
# -------------------------------

def perturb_edges_global(edge_index: torch.Tensor,
                         num_nodes: int,
                         drop_frac: float = 0.0,
                         flip_frac: float = 0.0,
                         keep_self_loops: bool = True) -> torch.Tensor:
    """
    Global structural noise:
      - drop_frac: fraction of existing edges dropped at random
      - flip_frac: fraction of original edges replaced by random edges
    """
    device = edge_index.device
    E = edge_index.size(1)

    # 1) Drop edges
    if drop_frac > 0:
        keep_mask = torch.rand(E, device=device) > drop_frac
        ei_kept = edge_index[:, keep_mask]
    else:
        ei_kept = edge_index

    # 2) Edge rewiring / random edges
    if flip_frac > 0:
        num_flip = int(E * flip_frac)
        if num_flip > 0:
            rand_src = torch.randint(0, num_nodes, (num_flip,), device=device)
            rand_dst = torch.randint(0, num_nodes, (num_flip,), device=device)
            ei_flip = torch.stack([rand_src, rand_dst], dim=0)
            edge_all = torch.cat([ei_kept, ei_flip], dim=1)
        else:
            edge_all = ei_kept
    else:
        edge_all = ei_kept

    # 3) Make undirected and (optionally) enforce self-loops
    edge_all = to_undirected(edge_all, num_nodes=num_nodes)

    if keep_self_loops:
        edge_all, _ = add_self_loops(edge_all, num_nodes=num_nodes)

    return edge_all


def clone_data_with_new_edges(data, new_edge_index: torch.Tensor):
    """
    Create a shallow clone of `data` but with a new edge_index.
    Masks, features, labels, etc. are kept identical.
    """
    new_data = deepcopy(data)
    new_data.edge_index = new_edge_index
    return new_data


# 13.2 Robustness evaluation under sparsity / corruption
# ------------------------------------------------------
@torch.no_grad()
def eval_on_perturbed_graph(teacher: BetterGAT,
                            student: ResidualGCN3,
                            data_base,
                            drop_frac: float,
                            flip_frac: float):
    """
    Evaluate teacher + student on a perturbed version of the graph
    without retraining.
    """
    edge_pert = perturb_edges_global(
        data_base.edge_index,
        num_nodes=data_base.num_nodes,
        drop_frac=drop_frac,
        flip_frac=flip_frac,
        keep_self_loops=True,
    )

    data_pert = clone_data_with_new_edges(data_base, edge_pert).to(device)

    teacher.eval()
    student.eval()

    # Teacher
    _, t_logp, _, _ = teacher(data_pert.x, data_pert.edge_index, training=False)
    t_pred = t_logp.argmax(1)
    t_acc = accuracy(t_pred[data_pert.test_mask], data_pert.y[data_pert.test_mask])

    # Student
    s_logp = student(data_pert.x, data_pert.edge_index, training=False)
    s_pred = s_logp.argmax(1)
    s_acc = accuracy(s_pred[data_pert.test_mask], data_pert.y[data_pert.test_mask])

    return data_pert, t_pred, s_pred, float(t_acc), float(s_acc)


def robustness_sweep(teacher: BetterGAT,
                     student: ResidualGCN3,
                     data_base,
                     drop_levels=None,
                     flip_levels=None):
    """
    Sweep across increasing sparsity and edge corruption,
    tracking teacher & student test accuracy.
    """
    if drop_levels is None:
        # from mild to brutal sparsity
        drop_levels = [0.0, 0.3, 0.6, 0.8]
    if flip_levels is None:
        # from clean to heavily rewired
        flip_levels = [0.0, 0.1, 0.3]

    results = []

    spacer("Robustness Sweep – sparsity x corruption")

    for df in drop_levels:
        for ff in flip_levels:
            data_pert, t_pred, s_pred, t_acc, s_acc = eval_on_perturbed_graph(
                teacher, student, data_base,
                drop_frac=df,
                flip_frac=ff,
            )
            results.append({
                "drop_frac": df,
                "flip_frac": ff,
                "teacher_acc": t_acc,
                "student_acc": s_acc,
            })
            print(
                f"[Robust] drop={df:.2f}, flip={ff:.2f} | "
                f"Teacher:{t_acc*100:5.2f}%  Student:{s_acc*100:5.2f}%"
            )

    # Simple visualisation: heatmaps for teacher & student robustness
    # Convert to grids
    dfs = sorted(set(r["drop_frac"] for r in results))
    ffs = sorted(set(r["flip_frac"] for r in results))

    T_mat = np.zeros((len(dfs), len(ffs)))
    S_mat = np.zeros((len(dfs), len(ffs)))

    for r in results:
        i = dfs.index(r["drop_frac"])
        j = ffs.index(r["flip_frac"])
        T_mat[i, j] = r["teacher_acc"]
        S_mat[i, j] = r["student_acc"]

    # Teacher heatmap
    spacer("Teacher Robustness Heatmap (Accuracy)")
    plt.figure(figsize=(7, 5))
    plt.imshow(T_mat, cmap="viridis", vmin=0, vmax=1)
    plt.title("Teacher Accuracy vs Sparsity / Corruption")
    plt.xlabel("Edge flip fraction")
    plt.ylabel("Edge drop fraction")
    plt.xticks(range(len(ffs)), [f"{v:.1f}" for v in ffs])
    plt.yticks(range(len(dfs)), [f"{v:.1f}" for v in dfs])
    for i in range(len(dfs)):
        for j in range(len(ffs)):
            plt.text(j, i, f"{T_mat[i,j]*100:.1f}%",
                     ha="center", va="center", color="white", fontsize=8)
    plot_and_save("robustness_teacher_heatmap.png")

    # Student heatmap
    spacer("Student Robustness Heatmap (Accuracy)")
    plt.figure(figsize=(7, 5))
    plt.imshow(S_mat, cmap="viridis", vmin=0, vmax=1)
    plt.title("Student Accuracy vs Sparsity / Corruption")
    plt.xlabel("Edge flip fraction")
    plt.ylabel("Edge drop fraction")
    plt.xticks(range(len(ffs)), [f"{v:.1f}" for v in ffs])
    plt.yticks(range(len(dfs)), [f"{v:.1f}" for v in dfs])
    for i in range(len(dfs)):
        for j in range(len(ffs)):
            plt.text(j, i, f"{S_mat[i,j]*100:.1f}%",
                     ha="center", va="center", color="white", fontsize=8)
    plot_and_save("robustness_student_heatmap.png")

    return results


# 13.3 Failure geometry: who flips when graph is harsh?
# -----------------------------------------------------
@torch.no_grad()
def failure_geometry_report(teacher: BetterGAT,
                            student: ResidualGCN3,
                            data_base,
                            drop_frac: float,
                            flip_frac: float):
    """
    Compare predictions on clean vs harsh graph:
      - Correct -> Wrong
      - Wrong   -> Correct
      - etc.
    For both teacher and student.
    """
    spacer(
        f"Failure Geometry Report "
        f"(drop={drop_frac:.2f}, flip={flip_frac:.2f})"
    )

    # 1) Baseline (clean) predictions
    teacher.eval()
    student.eval()

    _, t_logp_clean, _, _ = teacher(data_base.x, data_base.edge_index, training=False)
    s_logp_clean = student(data_base.x, data_base.edge_index, training=False)

    t_pred_clean = t_logp_clean.argmax(1)
    s_pred_clean = s_logp_clean.argmax(1)

    # 2) Perturbed predictions
    edge_pert = perturb_edges_global(
        data_base.edge_index,
        num_nodes=data_base.num_nodes,
        drop_frac=drop_frac,
        flip_frac=flip_frac,
        keep_self_loops=True,
    )
    data_pert = clone_data_with_new_edges(data_base, edge_pert).to(device)

    _, t_logp_pert, _, _ = teacher(data_pert.x, data_pert.edge_index, training=False)
    s_logp_pert = student(data_pert.x, data_pert.edge_index, training=False)

    t_pred_pert = t_logp_pert.argmax(1)
    s_pred_pert = s_logp_pert.argmax(1)

    y = data_base.y
    test_mask = data_base.test_mask

    def classify_flips(pred_clean, pred_pert, who: str):
        clean_correct = (pred_clean == y)
        pert_correct = (pred_pert == y)

        cc = (clean_correct & pert_correct & test_mask).sum().item()
        cw = (clean_correct & ~pert_correct & test_mask).sum().item()
        wc = (~clean_correct & pert_correct & test_mask).sum().item()
        ww = (~clean_correct & ~pert_correct & test_mask).sum().item()

        total = int(test_mask.sum().item())
        print(f"\n[{who}] Test nodes: {total}")
        print(f"  Correct -> Correct : {cc:4d}")
        print(f"  Correct -> Wrong   : {cw:4d}")
        print(f"  Wrong   -> Correct : {wc:4d}")
        print(f"  Wrong   -> Wrong   : {ww:4d}")

        return cc, cw, wc, ww

    print("\n--- Teacher failure geometry ---")
    t_cc, t_cw, t_wc, t_ww = classify_flips(t_pred_clean, t_pred_pert, "Teacher")

    print("\n--- Student failure geometry ---")
    s_cc, s_cw, s_wc, s_ww = classify_flips(s_pred_clean, s_pred_pert, "Student")

    # Small bar plot comparing “Correct -> Wrong” vulnerability
    labels = ["Teacher", "Student"]
    cw_vals = [t_cw, s_cw]
    wc_vals = [t_wc, s_wc]

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(7, 4))
    plt.bar(x - width / 2, cw_vals, width, label="Correct → Wrong")
    plt.bar(x + width / 2, wc_vals, width, label="Wrong → Correct")
    plt.xticks(x, labels)
    plt.ylabel("Number of test nodes")
    plt.title(f"Failure Geometry (drop={drop_frac:.2f}, flip={flip_frac:.2f})")
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plot_and_save("failure_geometry_teacher_vs_student.png")


# 13.4 Run the robustness suite (configurable)
# --------------------------------------------

# You can tune these to go more/less brutal.
ROBUST_DROP_LEVELS = [0.0, 0.3, 0.6, 0.85]
ROBUST_FLIP_LEVELS = [0.0, 0.1, 0.3]

robust_results = robustness_sweep(
    teacher,
    student,
    data,
    drop_levels=ROBUST_DROP_LEVELS,
    flip_levels=ROBUST_FLIP_LEVELS,
)

# Pick a particularly harsh configuration and inspect failure geometry
HARSH_DROP = 0.60
HARSH_FLIP = 0.30
failure_geometry_report(
    teacher,
    student,
    data,
    drop_frac=HARSH_DROP,
    flip_frac=HARSH_FLIP,
)

print("\n✅ Harsh Robustness & Failure Geometry analysis completed.")
print("   • See robustness heatmaps + failure_geometry_teacher_vs_student.png\n"
      "   • These show how quickly each model breaks under sparsity/corruption,\n"
      "     and which nodes flip from correct to wrong (and vice versa).")

In [ ]:
# ============================================================
# 14) Advanced Decision Geometry & Feature-Space Harshness
#     - Margin diagnostics (confidence gaps)
#     - Margin vs correctness
#     - Feature ablation robustness (Teacher vs Student)
#     - Optional: structure + feature harshness grid
# ------------------------------------------------------------
# Assumes:
#   - teacher : BetterGAT (trained)
#   - student : ResidualGCN3 (trained)
#   - data, dataset, device, accuracy, spacer, plot_and_save
# ============================================================

from copy import deepcopy

spacer("Advanced Decision Geometry & Feature-Space Harshness")


# 14.1 Helper to safely unpack teacher outputs
# --------------------------------------------
@torch.no_grad()
def forward_teacher_logp(model, x, edge_index, training: bool = False):
    """
    Robustly handle different forward signatures:
      - (emb, logp)
      - (emb, logp, ..., ...)
      - logp only
    """
    out = model(x, edge_index, training=training)
    if isinstance(out, tuple):
        # Expect at least (emb, logp)
        emb = out[0]
        logp = out[1]
    else:
        emb = None
        logp = out
    return emb, logp


# 14.2 Margin diagnostics: how "close" decisions are
# --------------------------------------------------
@torch.no_grad()
def margin_diagnostics(model, data):
    spacer("Margin Diagnostics – Teacher (Clean Graph)")

    model.eval()
    emb, logp = forward_teacher_logp(model, data.x, data.edge_index, training=False)
    probs = logp.exp()

    # Top-2 margin per node: p1 - p2
    top2 = torch.topk(probs, k=2, dim=1)
    margin = (top2.values[:, 0] - top2.values[:, 1]).cpu().numpy()

    y_true = data.y.cpu().numpy()
    preds = probs.argmax(1).cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()

    correct = (preds == y_true)
    margin_correct = margin[(correct & test_mask)]
    margin_wrong   = margin[(~correct & test_mask)]

    print(f"Mean margin (all test):   {margin[test_mask].mean():.4f}")
    print(f"Mean margin (correct):    {margin_correct.mean():.4f}")
    print(f"Mean margin (incorrect):  {margin_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(margin_correct, bins=30, alpha=0.7, label="Correct", density=True)
    plt.hist(margin_wrong,   bins=30, alpha=0.7, label="Wrong",   density=True)
    plt.xlabel("Top-2 Margin (p1 - p2)")
    plt.ylabel("Density")
    plt.title("Teacher Top-2 Margin Distribution (Test Nodes)")
    plt.legend()
    plot_and_save("margin_distribution_teacher.png")

    return margin, preds, y_true


margin_vals, margin_preds, margin_y = margin_diagnostics(teacher, data)


# 14.3 Margin vs degree and vs uncertainty (if entropy already computed)
# ----------------------------------------------------------------------
@torch.no_grad()
def margin_vs_degree(model, data, margin_vals):
    spacer("Margin vs Node Degree (Teacher)")

    degs = degree(data.edge_index[0], num_nodes=data.num_nodes).cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()

    degs_test = degs[test_mask]
    margins_test = margin_vals[test_mask]

    plt.figure(figsize=(8, 4))
    plt.scatter(degs_test + 1e-3, margins_test, s=10, alpha=0.6)
    plt.xscale("log")
    plt.xlabel("Degree (log scale)")
    plt.ylabel("Top-2 Margin")
    plt.title("Teacher Margin vs Node Degree (Test Nodes)")
    plt.grid(alpha=0.3)
    plot_and_save("margin_vs_degree_teacher.png")


margin_vs_degree(teacher, data, margin_vals)


# 14.4 Feature ablation robustness (no retraining)
# ------------------------------------------------
def feature_ablate(x: torch.Tensor, ablate_frac: float) -> torch.Tensor:
    """
    Randomly zero out a given fraction of features (per node)
    *consistently* across evaluation, i.e. structured feature mask.
    """
    device = x.device
    num_feats = x.size(1)
    num_abl = int(num_feats * ablate_frac)

    if num_abl <= 0:
        return x

    feat_indices = torch.randperm(num_feats, device=device)[:num_abl]
    x_ab = x.clone()
    x_ab[:, feat_indices] = 0.0
    return x_ab


@torch.no_grad()
def eval_feature_ablation(teacher, student, data, ablate_fracs=None):
    spacer("Feature-Space Ablation Robustness – Teacher vs Student")

    if ablate_fracs is None:
        ablate_fracs = [0.0, 0.25, 0.5, 0.75, 0.9]

    results = []

    for f in ablate_fracs:
        x_ab = feature_ablate(data.x, f)

        # Teacher
        teacher.eval()
        _, t_logp = forward_teacher_logp(teacher, x_ab, data.edge_index, training=False)
        t_pred = t_logp.argmax(1)
        t_acc = accuracy(t_pred[data.test_mask], data.y[data.test_mask])

        # Student
        student.eval()
        s_logp = student(x_ab, data.edge_index, training=False)
        s_pred = s_logp.argmax(1)
        s_acc = accuracy(s_pred[data.test_mask], data.y[data.test_mask])

        results.append((f, float(t_acc), float(s_acc)))
        print(
            f"[FeatAbl] ablate={f*100:5.1f}% | "
            f"Teacher:{t_acc*100:5.2f}%  Student:{s_acc*100:5.2f}%"
        )

    # Plot robustness curves
    fracs = [r[0] for r in results]
    t_accs = [r[1] for r in results]
    s_accs = [r[2] for r in results]

    plt.figure(figsize=(8, 4))
    plt.plot(fracs, t_accs, marker="o", label="Teacher")
    plt.plot(fracs, s_accs, marker="o", label="Student")
    plt.xlabel("Fraction of Features Ablated")
    plt.ylabel("Test Accuracy")
    plt.title("Feature Ablation Robustness – Teacher vs Student")
    plt.ylim(0, 1.05)
    plt.grid(alpha=0.4)
    plt.legend()
    plot_and_save("feature_ablation_robustness.png")

    return results


feat_ablation_results = eval_feature_ablation(teacher, student, data)


# 14.5 Joint structural + feature harshness grid (optional but fun)
# -----------------------------------------------------------------
@torch.no_grad()
def joint_harshness_grid(teacher,
                         student,
                         data,
                         drop_levels=None,
                         feat_levels=None,
                         flip_frac: float = 0.0):
    """
    Combine structural sparsity + feature ablation in one grid.
    This is *very* harsh – you probably don't want to go crazy here.
    """
    spacer("Joint Structural + Feature Harshness Grid")

    if drop_levels is None:
        drop_levels = [0.0, 0.3, 0.6]
    if feat_levels is None:
        feat_levels = [0.0, 0.5, 0.9]

    T_mat = np.zeros((len(drop_levels), len(feat_levels)))
    S_mat = np.zeros((len(drop_levels), len(feat_levels)))

    for i, df in enumerate(drop_levels):
        for j, ff in enumerate(feat_levels):

            # 1) Structural corruption
            edge_pert = perturb_edges_global(
                data.edge_index,
                num_nodes=data.num_nodes,
                drop_frac=df,
                flip_frac=flip_frac,
                keep_self_loops=True,
            )
            data_pert = clone_data_with_new_edges(data, edge_pert).to(device)

            # 2) Feature ablation
            x_ab = feature_ablate(data_pert.x, ff)

            # Teacher
            teacher.eval()
            _, t_logp = forward_teacher_logp(
                teacher, x_ab, data_pert.edge_index, training=False
            )
            t_pred = t_logp.argmax(1)
            t_acc = accuracy(
                t_pred[data_pert.test_mask], data_pert.y[data_pert.test_mask]
            )

            # Student
            student.eval()
            s_logp = student(x_ab, data_pert.edge_index, training=False)
            s_pred = s_logp.argmax(1)
            s_acc = accuracy(
                s_pred[data_pert.test_mask], data_pert.y[data_pert.test_mask]
            )

            T_mat[i, j] = t_acc
            S_mat[i, j] = s_acc

            print(
                f"[Joint] drop={df:.2f}, featAbl={ff:.2f} | "
                f"Teacher:{t_acc*100:5.2f}%  Student:{s_acc*100:5.2f}%"
            )

    # Plot Teacher grid
    spacer("Joint Harshness – Teacher")
    plt.figure(figsize=(7, 5))
    plt.imshow(T_mat, cmap="magma", vmin=0, vmax=1)
    plt.title(f"Teacher Accuracy – drop vs feature ablation (flip={flip_frac:.2f})")
    plt.xlabel("Feature ablation fraction")
    plt.ylabel("Edge drop fraction")
    plt.xticks(range(len(feat_levels)), [f"{v:.1f}" for v in feat_levels])
    plt.yticks(range(len(drop_levels)), [f"{v:.1f}" for v in drop_levels])
    for i in range(len(drop_levels)):
        for j in range(len(feat_levels)):
            plt.text(j, i, f"{T_mat[i,j]*100:.1f}%",
                     ha="center", va="center", color="white", fontsize=8)
    plot_and_save("joint_harshness_teacher.png")

    # Plot Student grid
    spacer("Joint Harshness – Student")
    plt.figure(figsize=(7, 5))
    plt.imshow(S_mat, cmap="magma", vmin=0, vmax=1)
    plt.title(f"Student Accuracy – drop vs feature ablation (flip={flip_frac:.2f})")
    plt.xlabel("Feature ablation fraction")
    plt.ylabel("Edge drop fraction")
    plt.xticks(range(len(feat_levels)), [f"{v:.1f}" for v in feat_levels])
    plt.yticks(range(len(drop_levels)), [f"{v:.1f}" for v in drop_levels])
    for i in range(len(drop_levels)):
        for j in range(len(feat_levels)):
            plt.text(j, i, f"{S_mat[i,j]*100:.1f}%",
                     ha="center", va="center", color="white", fontsize=8)
    plot_and_save("joint_harshness_student.png")

    return T_mat, S_mat


# This is computationally heavier; you can comment it out if needed
JOINT_DROP_LEVELS = [0.0, 0.3, 0.6]
JOINT_FEAT_LEVELS = [0.0, 0.5, 0.9]
JOINT_FLIP_FRAC   = 0.10   # small structural corruption on top

T_joint, S_joint = joint_harshness_grid(
    teacher,
    student,
    data,
    drop_levels=JOINT_DROP_LEVELS,
    feat_levels=JOINT_FEAT_LEVELS,
    flip_frac=JOINT_FLIP_FRAC,
)

print("\n✅ Advanced decision geometry + feature-space harshness completed.")
print("   • margin_distribution_teacher.png")
print("   • margin_vs_degree_teacher.png")
print("   • feature_ablation_robustness.png")
print("   • joint_harshness_teacher.png / joint_harshness_student.png")

In [ ]:
# ============================================================
# 14) Advanced Decision Geometry & Feature-Space Harshness
#     - Margin diagnostics (confidence gaps)
#     - Margin vs correctness
#     - Feature ablation robustness (Teacher vs Student)
#     - Optional: structure + feature harshness grid
# ------------------------------------------------------------
# Assumes:
#   - teacher : BetterGAT (trained)
#   - student : ResidualGCN3 (trained)
#   - data, dataset, device, accuracy, spacer, plot_and_save
# ============================================================

from copy import deepcopy

spacer("Advanced Decision Geometry & Feature-Space Harshness")


# 14.1 Helper to safely unpack teacher outputs
# --------------------------------------------
@torch.no_grad()
def forward_teacher_logp(model, x, edge_index, training: bool = False):
    """
    Robustly handle different forward signatures:
      - (emb, logp)
      - (emb, logp, ..., ...)
      - logp only
    """
    out = model(x, edge_index, training=training)
    if isinstance(out, tuple):
        # Expect at least (emb, logp)
        emb = out[0]
        logp = out[1]
    else:
        emb = None
        logp = out
    return emb, logp


# 14.2 Margin diagnostics: how "close" decisions are
# --------------------------------------------------
@torch.no_grad()
def margin_diagnostics(model, data):
    spacer("Margin Diagnostics – Teacher (Clean Graph)")

    model.eval()
    emb, logp = forward_teacher_logp(model, data.x, data.edge_index, training=False)
    probs = logp.exp()

    # Top-2 margin per node: p1 - p2
    top2 = torch.topk(probs, k=2, dim=1)
    margin = (top2.values[:, 0] - top2.values[:, 1]).cpu().numpy()

    y_true = data.y.cpu().numpy()
    preds = probs.argmax(1).cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()

    correct = (preds == y_true)
    margin_correct = margin[(correct & test_mask)]
    margin_wrong   = margin[(~correct & test_mask)]

    print(f"Mean margin (all test):   {margin[test_mask].mean():.4f}")
    print(f"Mean margin (correct):    {margin_correct.mean():.4f}")
    print(f"Mean margin (incorrect):  {margin_wrong.mean():.4f}")

    plt.figure(figsize=(9, 4))
    plt.hist(margin_correct, bins=30, alpha=0.7, label="Correct", density=True)
    plt.hist(margin_wrong,   bins=30, alpha=0.7, label="Wrong",   density=True)
    plt.xlabel("Top-2 Margin (p1 - p2)")
    plt.ylabel("Density")
    plt.title("Teacher Top-2 Margin Distribution (Test Nodes)")
    plt.legend()
    plot_and_save("margin_distribution_teacher.png")

    return margin, preds, y_true


margin_vals, margin_preds, margin_y = margin_diagnostics(teacher, data)


# 14.3 Margin vs degree and vs uncertainty (if entropy already computed)
# ----------------------------------------------------------------------
@torch.no_grad()
def margin_vs_degree(model, data, margin_vals):
    spacer("Margin vs Node Degree (Teacher)")

    degs = degree(data.edge_index[0], num_nodes=data.num_nodes).cpu().numpy()
    test_mask = data.test_mask.cpu().numpy()

    degs_test = degs[test_mask]
    margins_test = margin_vals[test_mask]

    plt.figure(figsize=(8, 4))
    plt.scatter(degs_test + 1e-3, margins_test, s=10, alpha=0.6)
    plt.xscale("log")
    plt.xlabel("Degree (log scale)")
    plt.ylabel("Top-2 Margin")
    plt.title("Teacher Margin vs Node Degree (Test Nodes)")
    plt.grid(alpha=0.3)
    plot_and_save("margin_vs_degree_teacher.png")


margin_vs_degree(teacher, data, margin_vals)


# 14.4 Feature ablation robustness (no retraining)
# ------------------------------------------------
def feature_ablate(x: torch.Tensor, ablate_frac: float) -> torch.Tensor:
    """
    Randomly zero out a given fraction of features (per node)
    *consistently* across evaluation, i.e. structured feature mask.
    """
    device = x.device
    num_feats = x.size(1)
    num_abl = int(num_feats * ablate_frac)

    if num_abl <= 0:
        return x

    feat_indices = torch.randperm(num_feats, device=device)[:num_abl]
    x_ab = x.clone()
    x_ab[:, feat_indices] = 0.0
    return x_ab


@torch.no_grad()
def eval_feature_ablation(teacher, student, data, ablate_fracs=None):
    spacer("Feature-Space Ablation Robustness – Teacher vs Student")

    if ablate_fracs is None:
        ablate_fracs = [0.0, 0.25, 0.5, 0.75, 0.9]

    results = []

    for f in ablate_fracs:
        x_ab = feature_ablate(data.x, f)

        # Teacher
        teacher.eval()
        _, t_logp = forward_teacher_logp(teacher, x_ab, data.edge_index, training=False)
        t_pred = t_logp.argmax(1)
        t_acc = accuracy(t_pred[data.test_mask], data.y[data.test_mask])

        # Student
        student.eval()
        s_logp = student(x_ab, data.edge_index, training=False)
        s_pred = s_logp.argmax(1)
        s_acc = accuracy(s_pred[data.test_mask], data.y[data.test_mask])

        results.append((f, float(t_acc), float(s_acc)))
        print(
            f"[FeatAbl] ablate={f*100:5.1f}% | "
            f"Teacher:{t_acc*100:5.2f}%  Student:{s_acc*100:5.2f}%"
        )

    # Plot robustness curves
    fracs = [r[0] for r in results]
    t_accs = [r[1] for r in results]
    s_accs = [r[2] for r in results]

    plt.figure(figsize=(8, 4))
    plt.plot(fracs, t_accs, marker="o", label="Teacher")
    plt.plot(fracs, s_accs, marker="o", label="Student")
    plt.xlabel("Fraction of Features Ablated")
    plt.ylabel("Test Accuracy")
    plt.title("Feature Ablation Robustness – Teacher vs Student")
    plt.ylim(0, 1.05)
    plt.grid(alpha=0.4)
    plt.legend()
    plot_and_save("feature_ablation_robustness.png")

    return results


feat_ablation_results = eval_feature_ablation(teacher, student, data)


# 14.5 Joint structural + feature harshness grid (optional but fun)
# -----------------------------------------------------------------
@torch.no_grad()
def joint_harshness_grid(teacher,
                         student,
                         data,
                         drop_levels=None,
                         feat_levels=None,
                         flip_frac: float = 0.0):
    """
    Combine structural sparsity + feature ablation in one grid.
    This is *very* harsh – you probably don't want to go crazy here.
    """
    spacer("Joint Structural + Feature Harshness Grid")

    if drop_levels is None:
        drop_levels = [0.0, 0.3, 0.6]
    if feat_levels is None:
        feat_levels = [0.0, 0.5, 0.9]

    T_mat = np.zeros((len(drop_levels), len(feat_levels)))
    S_mat = np.zeros((len(drop_levels), len(feat_levels)))

    for i, df in enumerate(drop_levels):
        for j, ff in enumerate(feat_levels):

            # 1) Structural corruption
            edge_pert = perturb_edges_global(
                data.edge_index,
                num_nodes=data.num_nodes,
                drop_frac=df,
                flip_frac=flip_frac,
                keep_self_loops=True,
            )
            data_pert = clone_data_with_new_edges(data, edge_pert).to(device)

            # 2) Feature ablation
            x_ab = feature_ablate(data_pert.x, ff)

            # Teacher
            teacher.eval()
            _, t_logp = forward_teacher_logp(
                teacher, x_ab, data_pert.edge_index, training=False
            )
            t_pred = t_logp.argmax(1)
            t_acc = accuracy(
                t_pred[data_pert.test_mask], data_pert.y[data_pert.test_mask]
            )

            # Student
            student.eval()
            s_logp = student(x_ab, data_pert.edge_index, training=False)
            s_pred = s_logp.argmax(1)
            s_acc = accuracy(
                s_pred[data_pert.test_mask], data_pert.y[data_pert.test_mask]
            )

            T_mat[i, j] = t_acc
            S_mat[i, j] = s_acc

            print(
                f"[Joint] drop={df:.2f}, featAbl={ff:.2f} | "
                f"Teacher:{t_acc*100:5.2f}%  Student:{s_acc*100:5.2f}%"
            )

    # Plot Teacher grid
    spacer("Joint Harshness – Teacher")
    plt.figure(figsize=(7, 5))
    plt.imshow(T_mat, cmap="magma", vmin=0, vmax=1)
    plt.title(f"Teacher Accuracy – drop vs feature ablation (flip={flip_frac:.2f})")
    plt.xlabel("Feature ablation fraction")
    plt.ylabel("Edge drop fraction")
    plt.xticks(range(len(feat_levels)), [f"{v:.1f}" for v in feat_levels])
    plt.yticks(range(len(drop_levels)), [f"{v:.1f}" for v in drop_levels])
    for i in range(len(drop_levels)):
        for j in range(len(feat_levels)):
            plt.text(j, i, f"{T_mat[i,j]*100:.1f}%",
                     ha="center", va="center", color="white", fontsize=8)
    plot_and_save("joint_harshness_teacher.png")

    # Plot Student grid
    spacer("Joint Harshness – Student")
    plt.figure(figsize=(7, 5))
    plt.imshow(S_mat, cmap="magma", vmin=0, vmax=1)
    plt.title(f"Student Accuracy – drop vs feature ablation (flip={flip_frac:.2f})")
    plt.xlabel("Feature ablation fraction")
    plt.ylabel("Edge drop fraction")
    plt.xticks(range(len(feat_levels)), [f"{v:.1f}" for v in feat_levels])
    plt.yticks(range(len(drop_levels)), [f"{v:.1f}" for v in drop_levels])
    for i in range(len(drop_levels)):
        for j in range(len(feat_levels)):
            plt.text(j, i, f"{S_mat[i,j]*100:.1f}%",
                     ha="center", va="center", color="white", fontsize=8)
    plot_and_save("joint_harshness_student.png")

    return T_mat, S_mat


# This is computationally heavier; you can comment it out if needed
JOINT_DROP_LEVELS = [0.0, 0.3, 0.6]
JOINT_FEAT_LEVELS = [0.0, 0.5, 0.9]
JOINT_FLIP_FRAC   = 0.10   # small structural corruption on top

T_joint, S_joint = joint_harshness_grid(
    teacher,
    student,
    data,
    drop_levels=JOINT_DROP_LEVELS,
    feat_levels=JOINT_FEAT_LEVELS,
    flip_frac=JOINT_FLIP_FRAC,
)

print("\n✅ Advanced decision geometry + feature-space harshness completed.")
print("   • margin_distribution_teacher.png")
print("   • margin_vs_degree_teacher.png")
print("   • feature_ablation_robustness.png")
print("   • joint_harshness_teacher.png / joint_harshness_student.png")